# 04 — Event Stream Construction

## Notebook purpose

This notebook reconstructs the V0.1 BUY/SELL market-event streams from the causally aligned trade-book table produced by Notebook 03.

Its primary task is:

> Convert every authoritative aligned trade into auditable individual and grouped event representations without violating causal ordering, partition boundaries, timestamp semantics, or conservation.

The resulting event tables, membership ledgers, observation windows, and event arrays become the authorized inputs for Notebook 05 market-state feature construction and the later point-process notebooks.

This notebook is a clean rewrite.

All earlier Notebook 04 event tables, arrays, contracts, manifests, hashes, summaries, and handoffs are invalid because they were constructed using a malformed inherited partition assignment. They are provenance only and must not be loaded as inputs.

---

## Current V0.1 stage

**Project:** The Clown Project  
**Pipeline:** V0.1 causal market-data reconstruction pipeline  
**Notebook:** `04_EVENT_STREAM_CONSTRUCTION.ipynb`  
**Previous notebook:** `03_CAUSAL_TRADE_BOOK_ALIGNMENT.ipynb`  
**Next notebook:** `05_MARKET_STATE_FEATURES.ipynb`

**Source run prefix:** `BTCUSDT_spot_20260710T063746Z_c8b5bf12`  
**V0.1 run ID:** `v0_1_20260714T090616Z_e82325081a81`  
**Primary ordering authority:** `collector_sequence`  
**Primary alignment authority:** `LOCAL_STRICT`  
**Canonical timezone:** UTC  
**Operating mode:** `ENGINEERING_REPRODUCTION_MODE`  
**Initial status:** `NOT EVALUATED`

---

## Core authority principle

A valid file hash proves that the exact expected file was loaded.

It does not prove that every field inside that file is semantically correct.

Notebook 04 must therefore independently reconstruct every field that controls:

- event grouping;
- partition membership;
- protected-partition access;
- observation windows;
- likelihood eligibility;
- downstream model selection.

Notebook 03 remains authoritative for the causal trade-to-book alignment itself, but its inherited partition labels are diagnostic only.

---

## Authority hierarchy

### 1. Notebook 00 frozen run and split contract

Notebook 00 is authoritative for:

- source identity;
- V0.1 run identity;
- collector-sequence partition boundaries;
- partition order;
- half-open interval convention `[start, end)`;
- protected-partition rules;
- frozen capture-end authority.

### 2. Notebook 03 LOCAL_STRICT alignment

Notebook 03 is authoritative for:

- trade identity;
- trade collector sequence;
- trade exchange and local timestamps;
- aggressor side;
- trade price, quantity, and notional;
- matched prior-book identity;
- matched book collector sequence;
- matched book local receipt time;
- causal book-state fields;
- alignment warning fields.

Every retained match must satisfy:

1. `book_collector_sequence < trade_collector_sequence`
2. `book_local_receipt_time_ns <= trade_local_receipt_time_ns`

### 3. Notebook 03 partition labels

Notebook 03 partition labels are diagnostic only.

They must be preserved as:

- `trade_partition_notebook_03`
- `matched_book_partition_notebook_03`

They must not control event construction, event grouping, partition counts, observation windows, candidate selection, or event arrays.

### 4. V0.0 outputs

V0.0 event and point-process outputs are reconciliation and provenance inputs only.

They may not make a V0.1 gate pass or fail and may not replace any Notebook 04 reconstruction.

---

## Frozen authoritative trade partitions

Notebook 04 must derive `trade_partition_authoritative` directly from `trade_collector_sequence` using the verified Notebook 00 split contract.

| Partition | Collector-sequence interval | Expected trades |
|---|---:|---:|
| DEVELOPMENT | `[1, 51840)` | 33,820 |
| CALIBRATION | `[51840, 72576)` | 13,532 |
| VALIDATION | `[72576, 88127)` | 10,198 |
| ENGINEERING_HOLDOUT | `[88127, 103678)` | 10,133 |

Every trade must map to exactly one interval.

The global authoritative trade count must remain:

`67,683`

Any inherited-versus-authoritative discrepancy must be retained in a partition-mismatch audit. No discrepancy may be silently corrected or discarded.

---

## Trade partition and matched-book partition

A trade’s statistical partition is determined only by its own collector sequence.

The matched prior book state may belong to the preceding partition. This is permitted causal history carry when the LOCAL_STRICT conditions remain satisfied.

Notebook 04 must keep separate:

- `event_partition`
- `matched_book_partition`
- `cross_partition_history_flag`

A prior-partition book state must never cause a trade to be reassigned to the earlier partition.

---

## Required event representations

Notebook 04 must construct and preserve all of the following.

### Individual-trade reference events

One event per authoritative aligned trade.

The event time is the trade’s observed local receipt time.

This representation must conserve all 67,683 trades exactly.

### Same-exchange-millisecond diagnostic buckets

Grouping key:

`(trade_partition_authoritative, trade_exchange_trade_time_ms)`

These buckets diagnose exchange-millisecond fragmentation and mixed-side activity.

They are not automatically eligible as a bivariate BUY/SELL point-process stream.

### Same-millisecond same-side bursts

Grouping key:

`(trade_partition_authoritative, trade_exchange_trade_time_ms, trade_aggressor_side)`

No burst may cross an authoritative partition boundary.

For each burst:

- event time is the local receipt time of the earliest constituent in collector-sequence order;
- causal state is the LOCAL_STRICT matched book state of that earliest constituent;
- print count, quantity, and notional are aggregated exactly;
- later constituent states must not replace or average the initiation state;
- aggregate marks unavailable at initiation must carry an explicit availability time.

### Membership ledgers

Every raw trade must map exactly once to:

- one individual event;
- one exchange-millisecond diagnostic bucket;
- one same-millisecond same-side burst.

No trade may appear zero times or multiple times in a required membership ledger.

---

## Timestamp policy

Local receipt timestamps contain exact ties.

Notebook 04 must not:

- jitter timestamps;
- create artificial nanosecond spacing;
- drop tied events;
- interpret collector sequence as physical elapsed time;
- silently pass duplicate event times into a simple-point-process estimator.

The selected event representation must either:

1. satisfy strict timestamp requirements; or
2. require a frozen simultaneous-event batch interface.

When simultaneous batches are required, later estimators must:

- evaluate all members using history strictly before the shared timestamp;
- prevent zero-time excitation among members of the same batch;
- apply the batch’s combined excitation only after all members have been scored.

---

## Observation-window policy

Observation windows must be reconstructed only after authoritative partition assignment.

For every partition:

- the hard start and exclusive end come from Notebook 00;
- nonterminal exclusive ends equal the start of the next authoritative partition;
- the terminal holdout end comes from the frozen capture-end authority;
- no arbitrary epsilon or timestamp jitter may be introduced;
- every event and simultaneous-event batch must lie strictly before the exclusive end.

Every partition-relative event array must satisfy:

`0 <= relative_event_time_ns < observation_window_duration_ns`

The first selected event in each partition must have relative time zero.

The left-censoring interval between the frozen partition start and the first event must be recorded explicitly rather than silently removed.

---

## Selection protection

Primary event-definition selection may inspect only:

- DEVELOPMENT;
- CALIBRATION.

Selection must not use:

- VALIDATION;
- ENGINEERING_HOLDOUT;
- future labels;
- predictive performance;
- Poisson or Hawkes likelihood;
- residual diagnostics;
- model calibration results;
- V0.0 event counts as acceptance criteria.

The event definition must be frozen and written to disk before detailed VALIDATION or ENGINEERING_HOLDOUT event construction is opened.

Validation and holdout must use the already frozen event-definition contract.

---

## Required outputs

Authoritative Notebook 04 outputs include:

- canonical individual-trade event table;
- same-exchange-millisecond diagnostic bucket table;
- same-millisecond same-side burst table;
- selected primary event table;
- exact-time simultaneous-event batch table;
- trade-to-event membership ledgers;
- authoritative partition-reconstruction audit;
- inherited partition-mismatch audit;
- cross-partition causal-history audit;
- timestamp-tie and fragmentation audits;
- global, side-level, and partition-level conservation audits;
- candidate-readiness and protected-access audits;
- event-definition contract;
- mark-availability contract;
- simultaneous-event batch contract;
- observation-window contract;
- partition-specific event arrays;
- V0.0 reconciliation outputs;
- Notebook 04 output manifest;
- controlled Notebook 04-to-Notebook 05 handoff.

Every authoritative output must record:

- source run prefix;
- V0.1 run ID;
- source input paths;
- source checksums;
- producing notebook;
- schema version;
- row count;
- acceptance status;
- output checksum.

Every authoritative artifact must be reloaded from disk and verified before the notebook may pass.

---

## Non-goals

This notebook must not:

- compute market-state predictors;
- construct future-return labels;
- estimate Poisson models;
- estimate Hawkes models;
- compare model likelihoods;
- run residual diagnostics;
- claim that Hawkes is superior to a simpler baseline;
- build quoting logic;
- infer queue position;
- simulate fills;
- calculate inventory, cash, fees, or P&L;
- evaluate a market-making strategy;
- make claim-bearing out-of-sample conclusions.

Those tasks belong to later notebooks.

---

## Blocking failure conditions

This notebook must fail if any of the following occur:

- a required upstream artifact is missing;
- an upstream hash or run identity fails verification;
- a trade maps to zero or multiple authoritative partitions;
- authoritative partition counts differ from the frozen Notebook 00 contract;
- a trade sequence lies outside its assigned half-open interval;
- grouping uses the inherited Notebook 03 partition label;
- an event crosses an authoritative partition boundary;
- a trade appears zero or multiple times in a membership ledger;
- trade count, quantity, or notional conservation fails globally, by side, or by partition;
- a burst uses a noncausal or post-initiation book state;
- an event or likelihood batch reaches or exceeds its exclusive window end;
- VALIDATION or ENGINEERING_HOLDOUT affects event-definition selection;
- timestamps are jittered;
- tied events are silently removed;
- V0.0 output is used as acceptance authority;
- an authoritative output fails read-back, schema, row-count, or checksum verification;
- the output manifest or Notebook 05 handoff cannot be verified.

---

## Required warning outputs

Warnings must explicitly report:

- Notebook 03 partition-label mismatches;
- matched-book partition-label mismatches;
- cross-partition causal-history observations;
- exact local-time ties;
- mixed-side exchange-millisecond buckets;
- multi-book-state bursts;
- inherited alignment-warning counts;
- V0.0 reconciliation differences;
- event candidates requiring the simultaneous-event batch interface.

Warnings must not silently remove observations.

---

## Expected terminal authority

A successful run is expected to end with:

- **Notebook status:** `CONDITIONAL PASS`
- **Operating authority:** `ENGINEERING_REPRODUCTION_MODE`
- **Event-stream authority:** `V0.1_CAUSAL_EVENT_CONSTRUCTION`
- **Notebook 05 authorization:** `TRUE`
- **Hawkes estimation authority:** `NONE`
- **Market-making authority:** `NONE`

The conditional status remains necessary because the current source consists of one continuous one-hour collection and does not provide an independent claim-bearing holdout.

A visually clean event table is not the objective.

A causally valid, partition-safe, conservation-complete, timestamp-explicit, and independently auditable event-stream handoff is the objective.

In [1]:
# ============================================================
# 04_EVENT_STREAM_CONSTRUCTION
# Cell 02 — Imports, frozen authority, partition specification,
#           paths, and core audit helpers
# ============================================================

from __future__ import annotations

import hashlib
import json
import platform
import re
import sys
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from types import MappingProxyType
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Display and pandas behavior
# ------------------------------------------------------------

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 180)

pd.options.mode.copy_on_write = True


# ------------------------------------------------------------
# Fixed notebook identity
# ------------------------------------------------------------

NOTEBOOK_NAME = "04_EVENT_STREAM_CONSTRUCTION"
NOTEBOOK_FILENAME = f"{NOTEBOOK_NAME}.ipynb"

PREVIOUS_NOTEBOOK_NAME = "03_CAUSAL_TRADE_BOOK_ALIGNMENT"
NEXT_NOTEBOOK_NAME = "05_MARKET_STATE_FEATURES"

PROJECT_NAME = "The Clown Project"
PIPELINE_VERSION = "V0.1"
OPERATING_MODE = "ENGINEERING_REPRODUCTION_MODE"

SOURCE_RUN_PREFIX = "BTCUSDT_spot_20260710T063746Z_c8b5bf12"
V01_RUN_ID = "v0_1_20260714T090616Z_e82325081a81"
COMBINED_PREFIX = f"{SOURCE_RUN_PREFIX}__{V01_RUN_ID}"

SYMBOL = "BTCUSDT"
CANONICAL_TIMEZONE = "UTC"
PRIMARY_ORDERING_AUTHORITY = "collector_sequence"
PRIMARY_ALIGNMENT_POLICY = "LOCAL_STRICT"
INTERVAL_CONVENTION = "[start, end)"


# ------------------------------------------------------------
# Notebook 04 semantic authority
# ------------------------------------------------------------

AUTHORITATIVE_TRADE_PARTITION_FIELD = (
    "trade_partition_authoritative"
)

INHERITED_TRADE_PARTITION_FIELD = (
    "trade_partition_notebook_03"
)

AUTHORITATIVE_BOOK_PARTITION_FIELD = (
    "matched_book_partition_authoritative"
)

INHERITED_BOOK_PARTITION_FIELD = (
    "matched_book_partition_notebook_03"
)

EVENT_PARTITION_FIELD = "event_partition"
MATCHED_BOOK_PARTITION_FIELD = "matched_book_partition"
CROSS_PARTITION_HISTORY_FIELD = "cross_partition_history_flag"

EVENT_TIME_AUTHORITY = "trade_local_receipt_time_ns"
BURST_TIME_AUTHORITY = "earliest_constituent_local_receipt_time_ns"
BURST_STATE_AUTHORITY = "earliest_constituent_local_strict_book_state"

EVENT_DEFINITION_SELECTION_PARTITIONS = (
    "DEVELOPMENT",
    "CALIBRATION",
)

PROTECTED_PARTITIONS = (
    "VALIDATION",
    "ENGINEERING_HOLDOUT",
)

PARTITION_ORDER = (
    "DEVELOPMENT",
    "CALIBRATION",
    "VALIDATION",
    "ENGINEERING_HOLDOUT",
)

VALID_AGGRESSOR_SIDES = (
    "BUY",
    "SELL",
)

SIDE_CODE = MappingProxyType(
    {
        "BUY": 0,
        "SELL": 1,
    }
)

INDIVIDUAL_EVENT_REPRESENTATION = "INDIVIDUAL_TRADE_EVENTS"
BURST_EVENT_REPRESENTATION = "SAME_MS_SAME_SIDE_BURSTS"
MS_BUCKET_REPRESENTATION = "SAME_EXCHANGE_MS_DIAGNOSTIC_BUCKETS"

NO_TIMESTAMP_JITTER_ALLOWED = True
NO_SILENT_EVENT_REMOVAL_ALLOWED = True
V00_ACCEPTANCE_AUTHORITY_ALLOWED = False

EXPECTED_TERMINAL_STATUS = "CONDITIONAL PASS"
EXPECTED_EVENT_STREAM_AUTHORITY = "V0.1_CAUSAL_EVENT_CONSTRUCTION"


# ------------------------------------------------------------
# Fixed expected upstream counts
# ------------------------------------------------------------

EXPECTED_ALIGNED_TRADE_ROWS = 67_683
EXPECTED_MATCHED_TRADE_ROWS = 67_683
EXPECTED_UNMATCHED_TRADE_ROWS = 0

EXPECTED_GLOBAL_BUY_TRADES = 30_596
EXPECTED_GLOBAL_SELL_TRADES = 37_087

EXPECTED_FIRST_COLLECTOR_SEQUENCE = 1
EXPECTED_LAST_COLLECTOR_SEQUENCE = 103_677
EXPECTED_FINAL_SEQUENCE_END_EXCLUSIVE = 103_678


# ------------------------------------------------------------
# Frozen Notebook 00 partition specification
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class PartitionSpec:
    """
    Immutable Notebook 04 view of one frozen Notebook 00 partition.

    Collector-sequence and receipt-time intervals are half-open:
    [start, end).
    """

    order: int
    name: str

    collector_sequence_start: int
    collector_sequence_end_exclusive: int

    receipt_time_start_ns: int
    receipt_time_end_exclusive_ns: int

    expected_trade_count: int
    expected_buy_count: int
    expected_sell_count: int

    event_definition_selection_allowed: bool
    protected_from_event_definition_selection: bool

    @property
    def collector_sequence_last_included(self) -> int:
        return self.collector_sequence_end_exclusive - 1

    @property
    def sequence_interval_size(self) -> int:
        return (
            self.collector_sequence_end_exclusive
            - self.collector_sequence_start
        )

    @property
    def receipt_window_duration_ns(self) -> int:
        return (
            self.receipt_time_end_exclusive_ns
            - self.receipt_time_start_ns
        )

    @property
    def expected_side_total(self) -> int:
        return self.expected_buy_count + self.expected_sell_count

    def contains_trade_sequence(self, sequence: int) -> bool:
        return (
            self.collector_sequence_start
            <= int(sequence)
            < self.collector_sequence_end_exclusive
        )

    def contains_receipt_time(self, timestamp_ns: int) -> bool:
        return (
            self.receipt_time_start_ns
            <= int(timestamp_ns)
            < self.receipt_time_end_exclusive_ns
        )


PARTITION_SPECS = (
    PartitionSpec(
        order=1,
        name="DEVELOPMENT",
        collector_sequence_start=1,
        collector_sequence_end_exclusive=51_840,
        receipt_time_start_ns=1_783_665_467_531_985_400,
        receipt_time_end_exclusive_ns=1_783_667_269_391_572_100,
        expected_trade_count=33_820,
        expected_buy_count=15_194,
        expected_sell_count=18_626,
        event_definition_selection_allowed=True,
        protected_from_event_definition_selection=False,
    ),
    PartitionSpec(
        order=2,
        name="CALIBRATION",
        collector_sequence_start=51_840,
        collector_sequence_end_exclusive=72_576,
        receipt_time_start_ns=1_783_667_269_391_572_100,
        receipt_time_end_exclusive_ns=1_783_667_989_690_751_200,
        expected_trade_count=13_532,
        expected_buy_count=7_267,
        expected_sell_count=6_265,
        event_definition_selection_allowed=True,
        protected_from_event_definition_selection=False,
    ),
    PartitionSpec(
        order=3,
        name="VALIDATION",
        collector_sequence_start=72_576,
        collector_sequence_end_exclusive=88_127,
        receipt_time_start_ns=1_783_667_989_690_751_200,
        receipt_time_end_exclusive_ns=1_783_668_525_057_534_700,
        expected_trade_count=10_198,
        expected_buy_count=5_774,
        expected_sell_count=4_424,
        event_definition_selection_allowed=False,
        protected_from_event_definition_selection=True,
    ),
    PartitionSpec(
        order=4,
        name="ENGINEERING_HOLDOUT",
        collector_sequence_start=88_127,
        collector_sequence_end_exclusive=103_678,
        receipt_time_start_ns=1_783_668_525_057_534_700,
        receipt_time_end_exclusive_ns=1_783_669_066_749_750_801,
        expected_trade_count=10_133,
        expected_buy_count=2_361,
        expected_sell_count=7_772,
        event_definition_selection_allowed=False,
        protected_from_event_definition_selection=True,
    ),
)

PARTITION_SPEC_BY_NAME: Mapping[str, PartitionSpec] = (
    MappingProxyType(
        {
            spec.name: spec
            for spec in PARTITION_SPECS
        }
    )
)

CAPTURE_END_AUTHORITY_NS = (
    PARTITION_SPECS[-1].receipt_time_end_exclusive_ns
)


# ------------------------------------------------------------
# Validate the frozen in-notebook partition declaration
#
# This does not replace loading and verifying Notebook 00.
# It only prevents accidental transcription errors in this cell.
# ------------------------------------------------------------

if tuple(spec.order for spec in PARTITION_SPECS) != (1, 2, 3, 4):
    raise RuntimeError("Partition orders are not exactly 1, 2, 3, 4.")

if tuple(spec.name for spec in PARTITION_SPECS) != PARTITION_ORDER:
    raise RuntimeError("Partition names do not match the frozen order.")

for current_spec, next_spec in zip(
    PARTITION_SPECS[:-1],
    PARTITION_SPECS[1:],
):
    if (
        current_spec.collector_sequence_end_exclusive
        != next_spec.collector_sequence_start
    ):
        raise RuntimeError(
            "Collector-sequence partition intervals are not contiguous: "
            f"{current_spec.name} -> {next_spec.name}."
        )

    if (
        current_spec.receipt_time_end_exclusive_ns
        != next_spec.receipt_time_start_ns
    ):
        raise RuntimeError(
            "Receipt-time partition intervals are not contiguous: "
            f"{current_spec.name} -> {next_spec.name}."
        )

for spec in PARTITION_SPECS:
    if spec.sequence_interval_size <= 0:
        raise RuntimeError(
            f"Nonpositive sequence interval for {spec.name}."
        )

    if spec.receipt_window_duration_ns <= 0:
        raise RuntimeError(
            f"Nonpositive receipt-time window for {spec.name}."
        )

    if spec.expected_side_total != spec.expected_trade_count:
        raise RuntimeError(
            "Expected BUY and SELL counts do not sum to the expected "
            f"trade count for {spec.name}."
        )

if (
    PARTITION_SPECS[0].collector_sequence_start
    != EXPECTED_FIRST_COLLECTOR_SEQUENCE
):
    raise RuntimeError(
        "The first partition does not begin at collector sequence 1."
    )

if (
    PARTITION_SPECS[-1].collector_sequence_end_exclusive
    != EXPECTED_FINAL_SEQUENCE_END_EXCLUSIVE
):
    raise RuntimeError(
        "The final partition does not end at collector sequence 103678."
    )

if (
    sum(spec.expected_trade_count for spec in PARTITION_SPECS)
    != EXPECTED_ALIGNED_TRADE_ROWS
):
    raise RuntimeError(
        "Frozen partition trade counts do not sum to 67,683."
    )

if (
    sum(spec.expected_buy_count for spec in PARTITION_SPECS)
    != EXPECTED_GLOBAL_BUY_TRADES
):
    raise RuntimeError(
        "Frozen partition BUY counts do not sum to 30,596."
    )

if (
    sum(spec.expected_sell_count for spec in PARTITION_SPECS)
    != EXPECTED_GLOBAL_SELL_TRADES
):
    raise RuntimeError(
        "Frozen partition SELL counts do not sum to 37,087."
    )

if tuple(
    spec.name
    for spec in PARTITION_SPECS
    if spec.event_definition_selection_allowed
) != EVENT_DEFINITION_SELECTION_PARTITIONS:
    raise RuntimeError(
        "Event-definition selection permissions are inconsistent."
    )

if tuple(
    spec.name
    for spec in PARTITION_SPECS
    if spec.protected_from_event_definition_selection
) != PROTECTED_PARTITIONS:
    raise RuntimeError(
        "Protected partition declarations are inconsistent."
    )


# ------------------------------------------------------------
# Frozen upstream hashes
# ------------------------------------------------------------

EXPECTED_SOURCE_SET_SHA256 = (
    "132c83531eec615d279408b5c06f402973114ba3058dfadd2fe58e2e67184c4b"
)

EXPECTED_V01_RUN_CONFIG_SHA256 = (
    "14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a770bee995f688d59617"
)

EXPECTED_V01_RUN_IDENTITY_SHA256 = (
    "5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9df19204c42e37fe4198"
)

EXPECTED_CHRONOLOGICAL_SPLIT_CONTRACT_SHA256 = (
    "3b7e8d46f43d8a4dd7b7a1f3d9a61c2c92d3f3643042a8b01968ddc7e0fb4624"
)

EXPECTED_NOTEBOOK_00_OUTPUT_MANIFEST_FILE_SHA256 = (
    "e12966301d92e61656715f2a6d397786820de836d8156dd70092250619cbb00e"
)

EXPECTED_NOTEBOOK_03_OUTPUT_MANIFEST_FILE_SHA256 = (
    "4daa7fa17c2f953e3b931b607e1b5b066f4e29e7dea7397d86b3c9e3b77406ca"
)

EXPECTED_NOTEBOOK_03_TO_04_HANDOFF_PAYLOAD_SHA256 = (
    "6f4d48e4fbb56d9ac80b415b3c0fe6c4625526b555938b846ee2876626b6152e"
)

EXPECTED_NOTEBOOK_03_MATCHED_ALIGNMENT_FILE_SHA256 = (
    "a18f66ca6a97a66b8f42a4d61b2aa34d26bcee5ae5a8d2054e576c88beede7a3"
)


# ------------------------------------------------------------
# Project roots
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"D:\Clown Project")
V00_ROOT = PROJECT_ROOT / "V0.0"
V01_ROOT = PROJECT_ROOT / "V0.1"

V01_CONFIG_ROOT = V01_ROOT / "config"
V01_DATA_ROOT = V01_ROOT / "data"
V01_ARTIFACTS_ROOT = V01_ROOT / "artifacts"

V01_PROCESSED_ALIGNMENT_DIR = (
    V01_DATA_ROOT
    / "processed"
    / "trade_book_alignment"
)

V01_PROCESSED_EVENT_DIR = (
    V01_DATA_ROOT
    / "processed"
    / "events"
)

V01_MANIFEST_DIR = (
    V01_ARTIFACTS_ROOT
    / "manifests"
)

V01_HANDOFF_DIR = (
    V01_ARTIFACTS_ROOT
    / "handoff"
)

V01_AUDIT_TABLE_DIR = (
    V01_ARTIFACTS_ROOT
    / "audit_tables"
    / NOTEBOOK_NAME
)

V01_DIAGNOSTICS_DIR = (
    V01_ARTIFACTS_ROOT
    / "diagnostics"
    / NOTEBOOK_NAME
)

V01_RECONCILIATION_DIR = (
    V01_ARTIFACTS_ROOT
    / "reconciliation"
    / NOTEBOOK_NAME
)


# ------------------------------------------------------------
# Required Notebook 00 inputs
# ------------------------------------------------------------

NOTEBOOK_00_OUTPUT_MANIFEST_PATH = (
    V01_MANIFEST_DIR
    / (
        f"{COMBINED_PREFIX}"
        "__00_V01_RUN_CONTRACT"
        "__notebook_00_output_manifest.json"
    )
)


# ------------------------------------------------------------
# Required Notebook 03 inputs
# ------------------------------------------------------------

NOTEBOOK_03_OUTPUT_MANIFEST_PATH = (
    V01_MANIFEST_DIR
    / (
        f"{COMBINED_PREFIX}"
        f"__{PREVIOUS_NOTEBOOK_NAME}"
        "__notebook_03_output_manifest.json"
    )
)

NOTEBOOK_03_TO_04_HANDOFF_PATH = (
    V01_HANDOFF_DIR
    / (
        f"{COMBINED_PREFIX}"
        f"__{PREVIOUS_NOTEBOOK_NAME}"
        "__notebook_03_to_notebook_04_handoff.json"
    )
)

NOTEBOOK_03_ALL_ALIGNMENT_PATH = (
    V01_PROCESSED_ALIGNMENT_DIR
    / (
        f"{COMBINED_PREFIX}"
        f"__{PREVIOUS_NOTEBOOK_NAME}"
        "__local_strict_all_trades_alignment.csv"
    )
)

NOTEBOOK_03_MATCHED_ALIGNMENT_PATH = (
    V01_PROCESSED_ALIGNMENT_DIR
    / (
        f"{COMBINED_PREFIX}"
        f"__{PREVIOUS_NOTEBOOK_NAME}"
        "__local_strict_matched_trade_book.csv"
    )
)


# ------------------------------------------------------------
# Contract output location
#
# Individual output paths are registered in the next cell.
# ------------------------------------------------------------

EVENT_DEFINITION_CONTRACT_PATH = (
    V01_CONFIG_ROOT
    / "event_definition_contract.json"
)


# ------------------------------------------------------------
# Basic validation and serialization helpers
# ------------------------------------------------------------

SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$")
HASH_CHUNK_SIZE = 8 * 1024 * 1024


def utc_now_iso() -> str:
    """Return the current UTC time as an ISO-8601 string."""
    return datetime.now(timezone.utc).isoformat()


def require(condition: bool, message: str) -> None:
    """Raise immediately when a blocking invariant is false."""
    if not bool(condition):
        raise AssertionError(message)


def require_sha256(value: str, field_name: str) -> str:
    """Require a normalized lowercase SHA-256 literal."""
    require(
        isinstance(value, str),
        f"{field_name} must be a string.",
    )

    normalized = value.lower()

    require(
        SHA256_PATTERN.fullmatch(normalized) is not None,
        f"{field_name} is not a valid SHA-256 value.",
    )

    return normalized


def require_file(path: Path) -> Path:
    """Require that a path exists and is a regular file."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Required file does not exist: {path}"
        )

    if not path.is_file():
        raise FileNotFoundError(
            f"Required path is not a regular file: {path}"
        )

    return path


def require_directory(path: Path) -> Path:
    """Require that a path exists and is a directory."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Required directory does not exist: {path}"
        )

    if not path.is_dir():
        raise NotADirectoryError(
            f"Required path is not a directory: {path}"
        )

    return path


def require_columns(
    frame: pd.DataFrame,
    required_columns: Iterable[str],
    frame_name: str,
) -> None:
    """Require a DataFrame to contain all declared columns."""
    required = tuple(required_columns)

    duplicate_requirements = sorted(
        {
            column
            for column in required
            if required.count(column) > 1
        }
    )

    require(
        not duplicate_requirements,
        (
            f"{frame_name} required-column declaration contains "
            f"duplicates: {duplicate_requirements}"
        ),
    )

    missing = [
        column
        for column in required
        if column not in frame.columns
    ]

    require(
        not missing,
        f"{frame_name} is missing required columns: {missing}",
    )


def require_unique_columns(
    frame: pd.DataFrame,
    frame_name: str,
) -> None:
    """Require that a DataFrame has no duplicate column names."""
    duplicates = (
        frame.columns[
            frame.columns.duplicated(keep=False)
        ]
        .astype(str)
        .tolist()
    )

    require(
        not duplicates,
        f"{frame_name} has duplicate columns: {duplicates}",
    )


def sha256_file(
    path: Path,
    chunk_size: int = HASH_CHUNK_SIZE,
) -> str:
    """Compute SHA-256 without loading the full file into memory."""
    path = require_file(path)

    require(
        chunk_size > 0,
        "chunk_size must be positive.",
    )

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def _json_default(value: Any) -> Any:
    """Normalize selected scientific-Python values for JSON."""
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        scalar = float(value)

        if not np.isfinite(scalar):
            raise ValueError(
                "Non-finite floating-point values are not valid "
                "in canonical JSON."
            )

        return scalar

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, datetime):
        return value.isoformat()

    raise TypeError(
        f"Object of type {type(value).__name__} is not JSON serializable."
    )


def canonical_json_bytes(payload: Any) -> bytes:
    """Serialize a payload deterministically for hashing."""
    return json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
        default=_json_default,
    ).encode("utf-8")


def canonical_json_sha256(payload: Any) -> str:
    """Compute SHA-256 of deterministic canonical JSON bytes."""
    return hashlib.sha256(
        canonical_json_bytes(payload)
    ).hexdigest()


def read_json_object(path: Path) -> dict[str, Any]:
    """Read a JSON file and require an object at the top level."""
    path = require_file(path)

    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    if not isinstance(payload, dict):
        raise TypeError(
            f"JSON top level must be an object: {path}"
        )

    return payload


def summarize_file(path: Path) -> dict[str, Any]:
    """Return deterministic file metadata for an audit record."""
    path = require_file(path)
    resolved = path.resolve()

    return {
        "path": str(resolved),
        "filename": resolved.name,
        "size_bytes": int(resolved.stat().st_size),
        "sha256": sha256_file(resolved),
    }


def assign_partition_from_sequence(
    collector_sequence: int,
) -> str:
    """
    Map one collector sequence to exactly one frozen partition.

    Raises when the sequence is outside the Notebook 00 contract.
    """
    sequence = int(collector_sequence)

    matches = [
        spec.name
        for spec in PARTITION_SPECS
        if spec.contains_trade_sequence(sequence)
    ]

    require(
        len(matches) == 1,
        (
            "Collector sequence must map to exactly one partition: "
            f"sequence={sequence}, matches={matches}"
        ),
    )

    return matches[0]


# ------------------------------------------------------------
# Normalized gate records
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class GateResult:
    gate: str
    status: str
    severity: str
    detail: str


def make_gate(
    gate: str,
    passed: bool,
    detail: str,
    severity: str = "BLOCKING",
) -> GateResult:
    """Create one normalized audit-gate result."""
    normalized_severity = severity.upper()

    require(
        normalized_severity in {"BLOCKING", "WARNING"},
        (
            "Gate severity must be BLOCKING or WARNING: "
            f"{severity}"
        ),
    )

    return GateResult(
        gate=str(gate),
        status="PASS" if bool(passed) else "FAIL",
        severity=normalized_severity,
        detail=str(detail),
    )


def gate_results_to_frame(
    gates: Sequence[GateResult],
) -> pd.DataFrame:
    """Convert gate results to a stable DataFrame."""
    return pd.DataFrame.from_records(
        [
            {
                "gate": gate.gate,
                "status": gate.status,
                "severity": gate.severity,
                "detail": gate.detail,
            }
            for gate in gates
        ],
        columns=[
            "gate",
            "status",
            "severity",
            "detail",
        ],
    )


def fail_if_blocking_gate_failed(
    gate_frame: pd.DataFrame,
) -> None:
    """Stop execution if any blocking gate did not pass."""
    require_columns(
        gate_frame,
        [
            "gate",
            "status",
            "severity",
            "detail",
        ],
        "gate_frame",
    )

    failed_blocking = gate_frame.loc[
        gate_frame["severity"].eq("BLOCKING")
        & ~gate_frame["status"].eq("PASS")
    ]

    if not failed_blocking.empty:
        display(failed_blocking)
        raise AssertionError(
            "At least one blocking Notebook 04 gate failed."
        )


# ------------------------------------------------------------
# Validate fixed hash literals now
# ------------------------------------------------------------

_FIXED_HASHES = {
    "EXPECTED_SOURCE_SET_SHA256": EXPECTED_SOURCE_SET_SHA256,
    "EXPECTED_V01_RUN_CONFIG_SHA256": EXPECTED_V01_RUN_CONFIG_SHA256,
    "EXPECTED_V01_RUN_IDENTITY_SHA256": (
        EXPECTED_V01_RUN_IDENTITY_SHA256
    ),
    "EXPECTED_CHRONOLOGICAL_SPLIT_CONTRACT_SHA256": (
        EXPECTED_CHRONOLOGICAL_SPLIT_CONTRACT_SHA256
    ),
    "EXPECTED_NOTEBOOK_00_OUTPUT_MANIFEST_FILE_SHA256": (
        EXPECTED_NOTEBOOK_00_OUTPUT_MANIFEST_FILE_SHA256
    ),
    "EXPECTED_NOTEBOOK_03_OUTPUT_MANIFEST_FILE_SHA256": (
        EXPECTED_NOTEBOOK_03_OUTPUT_MANIFEST_FILE_SHA256
    ),
    "EXPECTED_NOTEBOOK_03_TO_04_HANDOFF_PAYLOAD_SHA256": (
        EXPECTED_NOTEBOOK_03_TO_04_HANDOFF_PAYLOAD_SHA256
    ),
    "EXPECTED_NOTEBOOK_03_MATCHED_ALIGNMENT_FILE_SHA256": (
        EXPECTED_NOTEBOOK_03_MATCHED_ALIGNMENT_FILE_SHA256
    ),
}

for hash_name, hash_value in _FIXED_HASHES.items():
    require_sha256(hash_value, hash_name)


# ------------------------------------------------------------
# Runtime record
# ------------------------------------------------------------

runtime_record = {
    "project_name": PROJECT_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "notebook_name": NOTEBOOK_NAME,
    "notebook_filename": NOTEBOOK_FILENAME,
    "previous_notebook": PREVIOUS_NOTEBOOK_NAME,
    "next_notebook": NEXT_NOTEBOOK_NAME,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_1_run_id": V01_RUN_ID,
    "combined_prefix": COMBINED_PREFIX,
    "symbol": SYMBOL,
    "canonical_timezone": CANONICAL_TIMEZONE,
    "primary_ordering_authority": PRIMARY_ORDERING_AUTHORITY,
    "primary_alignment_policy": PRIMARY_ALIGNMENT_POLICY,
    "authoritative_trade_partition_field": (
        AUTHORITATIVE_TRADE_PARTITION_FIELD
    ),
    "event_definition_selection_partitions": list(
        EVENT_DEFINITION_SELECTION_PARTITIONS
    ),
    "protected_partitions": list(PROTECTED_PARTITIONS),
    "interval_convention": INTERVAL_CONVENTION,
    "operating_mode": OPERATING_MODE,
    "no_timestamp_jitter_allowed": NO_TIMESTAMP_JITTER_ALLOWED,
    "started_at_utc": utc_now_iso(),
    "python_version": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "working_directory": str(Path.cwd()),
}

runtime_record

{'project_name': 'The Clown Project',
 'pipeline_version': 'V0.1',
 'notebook_name': '04_EVENT_STREAM_CONSTRUCTION',
 'notebook_filename': '04_EVENT_STREAM_CONSTRUCTION.ipynb',
 'previous_notebook': '03_CAUSAL_TRADE_BOOK_ALIGNMENT',
 'next_notebook': '05_MARKET_STATE_FEATURES',
 'source_run_prefix': 'BTCUSDT_spot_20260710T063746Z_c8b5bf12',
 'v0_1_run_id': 'v0_1_20260714T090616Z_e82325081a81',
 'combined_prefix': 'BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81',
 'symbol': 'BTCUSDT',
 'canonical_timezone': 'UTC',
 'primary_ordering_authority': 'collector_sequence',
 'primary_alignment_policy': 'LOCAL_STRICT',
 'authoritative_trade_partition_field': 'trade_partition_authoritative',
 'event_definition_selection_partitions': ['DEVELOPMENT', 'CALIBRATION'],
 'protected_partitions': ['VALIDATION', 'ENGINEERING_HOLDOUT'],
 'interval_convention': '[start, end)',
 'operating_mode': 'ENGINEERING_REPRODUCTION_MODE',
 'no_timestamp_jitter_allowed': True,
 'started_at_u

In [2]:
# ============================================================
# 04_EVENT_STREAM_CONSTRUCTION
# Cell 03 — Output-directory initialization and upstream
#           artifact discovery / cryptographic inspection
# ============================================================

from dataclasses import asdict


# ------------------------------------------------------------
# Create Notebook 04-owned output directories
#
# No directory under V0.0 is created or modified.
# ------------------------------------------------------------

NOTEBOOK_04_OUTPUT_DIRECTORIES = (
    V01_PROCESSED_EVENT_DIR,
    V01_MANIFEST_DIR,
    V01_HANDOFF_DIR,
    V01_AUDIT_TABLE_DIR,
    V01_DIAGNOSTICS_DIR,
    V01_RECONCILIATION_DIR,
)

for directory in NOTEBOOK_04_OUTPUT_DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

require_directory(V01_ROOT)
require_directory(V01_CONFIG_ROOT)
require_directory(V01_DATA_ROOT)
require_directory(V01_ARTIFACTS_ROOT)

for directory in NOTEBOOK_04_OUTPUT_DIRECTORIES:
    require_directory(directory)


# ------------------------------------------------------------
# Artifact-discovery helpers
#
# Discovery is fail-closed:
# - prefer an exact registered path;
# - otherwise search only declared roots;
# - require all semantic tokens;
# - reject ambiguous matches;
# - never select a prior Notebook 04 output as an input.
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class DiscoveredArtifact:
    label: str
    path: Path
    discovery_method: str
    size_bytes: int
    file_sha256: str
    canonical_json_sha256: str | None


def _normalized_path_text(path: Path) -> str:
    """Return a case-normalized searchable path string."""
    return str(Path(path)).replace("\\", "/").lower()


def _deduplicate_paths(paths: Iterable[Path]) -> list[Path]:
    """Deduplicate paths by resolved, case-normalized identity."""
    unique: dict[str, Path] = {}

    for path in paths:
        candidate = Path(path)

        if not candidate.is_file():
            continue

        key = str(candidate.resolve()).casefold()
        unique[key] = candidate.resolve()

    return sorted(
        unique.values(),
        key=lambda item: str(item).casefold(),
    )


def _search_files(
    roots: Iterable[Path],
    suffixes: Iterable[str],
) -> list[Path]:
    """Recursively enumerate files with allowed suffixes."""
    normalized_suffixes = {
        suffix.lower()
        if suffix.startswith(".")
        else f".{suffix.lower()}"
        for suffix in suffixes
    }

    results: list[Path] = []

    for root in roots:
        root = Path(root)

        if not root.exists():
            continue

        require_directory(root)

        for candidate in root.rglob("*"):
            if (
                candidate.is_file()
                and candidate.suffix.lower() in normalized_suffixes
            ):
                results.append(candidate.resolve())

    return _deduplicate_paths(results)


def resolve_artifact(
    *,
    label: str,
    exact_paths: Iterable[Path],
    search_roots: Iterable[Path],
    required_tokens: Iterable[str],
    suffixes: Iterable[str],
    forbidden_tokens: Iterable[str] = (
        "04_event_stream_construction",
    ),
) -> tuple[Path, str]:
    """
    Resolve one upstream artifact without silently choosing
    between ambiguous candidates.
    """
    exact_matches = _deduplicate_paths(exact_paths)

    if len(exact_matches) == 1:
        return exact_matches[0], "EXACT_REGISTERED_PATH"

    if len(exact_matches) > 1:
        raise RuntimeError(
            f"{label}: multiple exact registered paths exist:\n"
            + "\n".join(f"  - {path}" for path in exact_matches)
        )

    required = tuple(
        token.casefold()
        for token in required_tokens
    )

    forbidden = tuple(
        token.casefold()
        for token in forbidden_tokens
    )

    candidates = []

    for candidate in _search_files(
        search_roots,
        suffixes,
    ):
        searchable = _normalized_path_text(candidate)

        if not all(token in searchable for token in required):
            continue

        if any(token in searchable for token in forbidden):
            continue

        candidates.append(candidate)

    candidates = _deduplicate_paths(candidates)

    if not candidates:
        raise FileNotFoundError(
            f"{label}: no matching artifact was found.\n"
            f"Required tokens: {required}\n"
            f"Search roots:\n"
            + "\n".join(
                f"  - {Path(root)}"
                for root in search_roots
            )
        )

    if len(candidates) > 1:
        raise RuntimeError(
            f"{label}: artifact discovery is ambiguous.\n"
            f"Required tokens: {required}\n"
            f"Candidates:\n"
            + "\n".join(
                f"  - {path}"
                for path in candidates
            )
        )

    return candidates[0], "SEMANTIC_DISCOVERY"


def inspect_artifact(
    label: str,
    path: Path,
    discovery_method: str,
) -> DiscoveredArtifact:
    """Compute immutable identity information for one artifact."""
    path = require_file(path)
    canonical_hash: str | None = None

    if path.suffix.lower() == ".json":
        payload = read_json_object(path)
        canonical_hash = canonical_json_sha256(payload)

    return DiscoveredArtifact(
        label=label,
        path=path.resolve(),
        discovery_method=discovery_method,
        size_bytes=int(path.stat().st_size),
        file_sha256=sha256_file(path),
        canonical_json_sha256=canonical_hash,
    )


# ------------------------------------------------------------
# Candidate paths
#
# Exact paths are attempted first. Alternative extensions are
# declared explicitly rather than inferred after selection.
# ------------------------------------------------------------

NOTEBOOK_03_MATCHED_ALIGNMENT_EXACT_CANDIDATES = (
    NOTEBOOK_03_MATCHED_ALIGNMENT_PATH,
    NOTEBOOK_03_MATCHED_ALIGNMENT_PATH.with_suffix(".parquet"),
    NOTEBOOK_03_MATCHED_ALIGNMENT_PATH.with_suffix(".feather"),
)

SPLIT_CONTRACT_EXACT_CANDIDATES = (
    V01_CONFIG_ROOT / "split_contract.json",
    V01_CONFIG_ROOT / "chronological_split_contract.json",
    V01_CONFIG_ROOT / "v0_1_split_contract.json",
)


# ------------------------------------------------------------
# Resolve required upstream artifacts
# ------------------------------------------------------------

resolved_paths: dict[str, Path] = {}
discovery_methods: dict[str, str] = {}


(
    resolved_paths["notebook_00_output_manifest"],
    discovery_methods["notebook_00_output_manifest"],
) = resolve_artifact(
    label="Notebook 00 output manifest",
    exact_paths=(
        NOTEBOOK_00_OUTPUT_MANIFEST_PATH,
    ),
    search_roots=(
        V01_MANIFEST_DIR,
        V01_ARTIFACTS_ROOT,
    ),
    required_tokens=(
        "00_v01_run_contract",
        "manifest",
    ),
    suffixes=(".json",),
)


(
    resolved_paths["split_contract"],
    discovery_methods["split_contract"],
) = resolve_artifact(
    label="Notebook 00 split contract",
    exact_paths=SPLIT_CONTRACT_EXACT_CANDIDATES,
    search_roots=(
        V01_CONFIG_ROOT,
        V01_ARTIFACTS_ROOT,
    ),
    required_tokens=(
        "split",
        "contract",
    ),
    suffixes=(".json",),
)


(
    resolved_paths["notebook_03_output_manifest"],
    discovery_methods["notebook_03_output_manifest"],
) = resolve_artifact(
    label="Notebook 03 output manifest",
    exact_paths=(
        NOTEBOOK_03_OUTPUT_MANIFEST_PATH,
    ),
    search_roots=(
        V01_MANIFEST_DIR,
        V01_ARTIFACTS_ROOT,
    ),
    required_tokens=(
        "03_causal_trade_book_alignment",
        "manifest",
    ),
    suffixes=(".json",),
)


(
    resolved_paths["notebook_03_to_04_handoff"],
    discovery_methods["notebook_03_to_04_handoff"],
) = resolve_artifact(
    label="Notebook 03 to Notebook 04 handoff",
    exact_paths=(
        NOTEBOOK_03_TO_04_HANDOFF_PATH,
    ),
    search_roots=(
        V01_HANDOFF_DIR,
        V01_ARTIFACTS_ROOT,
    ),
    required_tokens=(
        "03_causal_trade_book_alignment",
        "handoff",
    ),
    suffixes=(".json",),
    forbidden_tokens=(),
)


(
    resolved_paths["notebook_03_matched_alignment"],
    discovery_methods["notebook_03_matched_alignment"],
) = resolve_artifact(
    label="Notebook 03 LOCAL_STRICT matched trade-book table",
    exact_paths=NOTEBOOK_03_MATCHED_ALIGNMENT_EXACT_CANDIDATES,
    search_roots=(
        V01_PROCESSED_ALIGNMENT_DIR,
        V01_DATA_ROOT,
    ),
    required_tokens=(
        "03_causal_trade_book_alignment",
        "local_strict",
        "matched",
    ),
    suffixes=(
        ".csv",
        ".parquet",
        ".feather",
    ),
)


# ------------------------------------------------------------
# Inspect resolved artifacts
# ------------------------------------------------------------

upstream_artifacts: dict[str, DiscoveredArtifact] = {}

for label, path in resolved_paths.items():
    upstream_artifacts[label] = inspect_artifact(
        label=label,
        path=path,
        discovery_method=discovery_methods[label],
    )


upstream_discovery_table = pd.DataFrame.from_records(
    [
        {
            **asdict(artifact),
            "path": str(artifact.path),
        }
        for artifact in upstream_artifacts.values()
    ],
    columns=[
        "label",
        "path",
        "discovery_method",
        "size_bytes",
        "file_sha256",
        "canonical_json_sha256",
    ],
).sort_values(
    "label",
    kind="stable",
).reset_index(drop=True)


# ------------------------------------------------------------
# Load JSON control artifacts for structural inspection
# ------------------------------------------------------------

notebook_00_manifest_payload = read_json_object(
    resolved_paths["notebook_00_output_manifest"]
)

split_contract_payload = read_json_object(
    resolved_paths["split_contract"]
)

notebook_03_manifest_payload = read_json_object(
    resolved_paths["notebook_03_output_manifest"]
)

notebook_03_to_04_handoff_payload = read_json_object(
    resolved_paths["notebook_03_to_04_handoff"]
)


# ------------------------------------------------------------
# Recursive JSON inspection helpers
# ------------------------------------------------------------

def iter_json_nodes(
    value: Any,
    location: str = "$",
):
    """Yield every JSON node with a stable structural location."""
    yield location, value

    if isinstance(value, dict):
        for key, child in value.items():
            child_location = f"{location}.{key}"
            yield from iter_json_nodes(
                child,
                child_location,
            )

    elif isinstance(value, list):
        for index, child in enumerate(value):
            child_location = f"{location}[{index}]"
            yield from iter_json_nodes(
                child,
                child_location,
            )


def find_json_key_values(
    payload: Any,
    key_names: Iterable[str],
) -> list[dict[str, Any]]:
    """Find values whose key matches any requested key name."""
    targets = {
        str(key).casefold()
        for key in key_names
    }

    matches: list[dict[str, Any]] = []

    for location, node in iter_json_nodes(payload):
        if not isinstance(node, dict):
            continue

        for key, value in node.items():
            if str(key).casefold() in targets:
                matches.append(
                    {
                        "location": f"{location}.{key}",
                        "key": str(key),
                        "value": value,
                    }
                )

    return matches


def find_embedded_sha256_values(
    payload: Any,
) -> list[dict[str, str]]:
    """Find all SHA-256-looking strings embedded in JSON."""
    matches: list[dict[str, str]] = []

    for location, node in iter_json_nodes(payload):
        if not isinstance(node, str):
            continue

        normalized = node.strip().lower()

        if SHA256_PATTERN.fullmatch(normalized):
            matches.append(
                {
                    "location": location,
                    "sha256": normalized,
                }
            )

    return matches


def scalar_identity_values(
    payload: Any,
    keys: Iterable[str],
) -> set[str]:
    """Collect normalized scalar identity values for selected keys."""
    values: set[str] = set()

    for match in find_json_key_values(payload, keys):
        value = match["value"]

        if isinstance(
            value,
            (str, int, float, bool),
        ):
            values.add(str(value))

    return values


# ------------------------------------------------------------
# Verify run identities wherever they are declared
# ------------------------------------------------------------

json_payloads = {
    "notebook_00_output_manifest": notebook_00_manifest_payload,
    "split_contract": split_contract_payload,
    "notebook_03_output_manifest": notebook_03_manifest_payload,
    "notebook_03_to_04_handoff": (
        notebook_03_to_04_handoff_payload
    ),
}

identity_rows: list[dict[str, Any]] = []

for artifact_label, payload in json_payloads.items():
    source_run_values = scalar_identity_values(
        payload,
        (
            "source_run_prefix",
            "source_prefix",
            "v0_0_source_run_prefix",
        ),
    )

    v01_run_values = scalar_identity_values(
        payload,
        (
            "v0_1_run_id",
            "v01_run_id",
            "run_id",
        ),
    )

    if source_run_values:
        require(
            source_run_values == {SOURCE_RUN_PREFIX},
            (
                f"{artifact_label} contains unexpected source-run "
                f"identities: {sorted(source_run_values)}"
            ),
        )

    if v01_run_values:
        require(
            v01_run_values == {V01_RUN_ID},
            (
                f"{artifact_label} contains unexpected V0.1 run "
                f"identities: {sorted(v01_run_values)}"
            ),
        )

    identity_rows.append(
        {
            "artifact": artifact_label,
            "source_run_values_found": sorted(source_run_values),
            "source_run_identity_pass": (
                source_run_values == {SOURCE_RUN_PREFIX}
                if source_run_values
                else None
            ),
            "v0_1_run_values_found": sorted(v01_run_values),
            "v0_1_run_identity_pass": (
                v01_run_values == {V01_RUN_ID}
                if v01_run_values
                else None
            ),
        }
    )

upstream_identity_table = pd.DataFrame.from_records(
    identity_rows
)


# ------------------------------------------------------------
# Cryptographic probes
#
# For JSON control artifacts, both the raw file hash and the
# canonical parsed-payload hash are retained. Embedded hashes
# are also inspected because upstream artifacts may distinguish:
#
# - file SHA-256;
# - payload SHA-256;
# - manifest SHA-256;
# - referenced artifact SHA-256.
#
# Notebook 04 does not silently treat these as interchangeable.
# ------------------------------------------------------------

expected_hash_probes = {
    "notebook_00_output_manifest": (
        EXPECTED_NOTEBOOK_00_OUTPUT_MANIFEST_FILE_SHA256
    ),
    "notebook_03_output_manifest": (
        EXPECTED_NOTEBOOK_03_OUTPUT_MANIFEST_FILE_SHA256
    ),
    "notebook_03_to_04_handoff": (
        EXPECTED_NOTEBOOK_03_TO_04_HANDOFF_PAYLOAD_SHA256
    ),
    "notebook_03_matched_alignment": (
        EXPECTED_NOTEBOOK_03_MATCHED_ALIGNMENT_FILE_SHA256
    ),
}

hash_probe_rows: list[dict[str, Any]] = []

for artifact_label, expected_hash in expected_hash_probes.items():
    artifact = upstream_artifacts[artifact_label]

    embedded_hashes: set[str] = set()

    if artifact_label in json_payloads:
        embedded_hashes = {
            item["sha256"]
            for item in find_embedded_sha256_values(
                json_payloads[artifact_label]
            )
        }

    hash_probe_rows.append(
        {
            "artifact": artifact_label,
            "expected_sha256": expected_hash,
            "raw_file_sha256": artifact.file_sha256,
            "canonical_json_sha256": (
                artifact.canonical_json_sha256
            ),
            "matches_raw_file_sha256": (
                artifact.file_sha256 == expected_hash
            ),
            "matches_canonical_json_sha256": (
                artifact.canonical_json_sha256 == expected_hash
                if artifact.canonical_json_sha256 is not None
                else None
            ),
            "found_as_embedded_sha256": (
                expected_hash in embedded_hashes
            ),
        }
    )

upstream_hash_probe_table = pd.DataFrame.from_records(
    hash_probe_rows
)


# ------------------------------------------------------------
# The matched alignment is a data file with an explicitly
# registered file hash. It must match exactly at this stage.
# ------------------------------------------------------------

require(
    upstream_artifacts[
        "notebook_03_matched_alignment"
    ].file_sha256
    == EXPECTED_NOTEBOOK_03_MATCHED_ALIGNMENT_FILE_SHA256,
    (
        "Notebook 03 matched alignment file SHA-256 does not "
        "match the registered authority value."
    ),
)


# ------------------------------------------------------------
# Publish resolved canonical paths for downstream cells
# ------------------------------------------------------------

NOTEBOOK_00_OUTPUT_MANIFEST_PATH_RESOLVED = (
    resolved_paths["notebook_00_output_manifest"]
)

SPLIT_CONTRACT_PATH_RESOLVED = (
    resolved_paths["split_contract"]
)

NOTEBOOK_03_OUTPUT_MANIFEST_PATH_RESOLVED = (
    resolved_paths["notebook_03_output_manifest"]
)

NOTEBOOK_03_TO_04_HANDOFF_PATH_RESOLVED = (
    resolved_paths["notebook_03_to_04_handoff"]
)

NOTEBOOK_03_MATCHED_ALIGNMENT_PATH_RESOLVED = (
    resolved_paths["notebook_03_matched_alignment"]
)


# ------------------------------------------------------------
# Cell output
# ------------------------------------------------------------

display(upstream_discovery_table)
display(upstream_identity_table)
display(upstream_hash_probe_table)

{
    "status": "PASS",
    "resolved_artifact_count": len(upstream_artifacts),
    "output_directories_verified": len(
        NOTEBOOK_04_OUTPUT_DIRECTORIES
    ),
    "matched_alignment_hash_verified": True,
    "next_action": (
        "Verify manifest and handoff semantics against the "
        "resolved split contract and matched-table artifact."
    ),
}

,label,path,discovery_method,size_bytes,file_sha256,canonical_json_sha256
0,notebook_00_output_manifest,D:\Clown Project\V0.1\artifacts\manifests\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__00_V01_RUN_CONTRACT__notebook_00_output_manifest.json,EXACT_REGISTERED_PATH,15290,e12966301d92e61656715f2a6d397786820de836d8156dd70092250619cbb00e,6e2162b5fe2b6afc2b14934414381769790ca84bc4bef981519c2958d1c95fb2
1,notebook_03_matched_alignment,D:\Clown Project\V0.1\data\processed\trade_book_alignment\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__local_stri...,EXACT_REGISTERED_PATH,131540683,a18f66ca6a97a66b8f42a4d61b2aa34d26bcee5ae5a8d2054e576c88beede7a3,None
2,notebook_03_output_manifest,D:\Clown Project\V0.1\artifacts\manifests\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__notebook_03_output_manifes...,EXACT_REGISTERED_PATH,19576,4daa7fa17c2f953e3b931b607e1b5b066f4e29e7dea7397d86b3c9e3b77406ca,387bad090da471e20fbc0c443a398236f6d0df12c43fbad3fa829afaaf77064b
3,notebook_03_to_04_handoff,D:\Clown Project\V0.1\artifacts\handoff\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__notebook_03_to_notebook_04_h...,EXACT_REGISTERED_PATH,28856,6f4d48e4fbb56d9ac80b415b3c0fe6c4625526b555938b846ee2876626b6152e,fbceb30d63f6b5ea9684a1af324e973f2e72a134595bb35734dfa56ed7f86d21
4,split_contract,D:\Clown Project\V0.1\config\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__00_V01_RUN_CONTRACT__split_contract.json,SEMANTIC_DISCOVERY,7330,1037581968095b5e262429fd86004d1abf3e5b691b330e7a4aa8e89f590d09fe,f2dc5639b1ffbb7f96fe2566f310ac6e1cec125cbd45d5cc48a7c7dbbbc22398


,artifact,source_run_values_found,source_run_identity_pass,v0_1_run_values_found,v0_1_run_identity_pass
0,notebook_00_output_manifest,[BTCUSDT_spot_20260710T063746Z_c8b5bf12],True,[v0_1_20260714T090616Z_e82325081a81],True
1,split_contract,[BTCUSDT_spot_20260710T063746Z_c8b5bf12],True,[v0_1_20260714T090616Z_e82325081a81],True
2,notebook_03_output_manifest,[BTCUSDT_spot_20260710T063746Z_c8b5bf12],True,[v0_1_20260714T090616Z_e82325081a81],True
3,notebook_03_to_04_handoff,[BTCUSDT_spot_20260710T063746Z_c8b5bf12],True,[v0_1_20260714T090616Z_e82325081a81],True


,artifact,expected_sha256,raw_file_sha256,canonical_json_sha256,matches_raw_file_sha256,matches_canonical_json_sha256,found_as_embedded_sha256
0,notebook_00_output_manifest,e12966301d92e61656715f2a6d397786820de836d8156dd70092250619cbb00e,e12966301d92e61656715f2a6d397786820de836d8156dd70092250619cbb00e,6e2162b5fe2b6afc2b14934414381769790ca84bc4bef981519c2958d1c95fb2,True,False,False
1,notebook_03_output_manifest,4daa7fa17c2f953e3b931b607e1b5b066f4e29e7dea7397d86b3c9e3b77406ca,4daa7fa17c2f953e3b931b607e1b5b066f4e29e7dea7397d86b3c9e3b77406ca,387bad090da471e20fbc0c443a398236f6d0df12c43fbad3fa829afaaf77064b,True,False,False
2,notebook_03_to_04_handoff,6f4d48e4fbb56d9ac80b415b3c0fe6c4625526b555938b846ee2876626b6152e,6f4d48e4fbb56d9ac80b415b3c0fe6c4625526b555938b846ee2876626b6152e,fbceb30d63f6b5ea9684a1af324e973f2e72a134595bb35734dfa56ed7f86d21,True,False,False
3,notebook_03_matched_alignment,a18f66ca6a97a66b8f42a4d61b2aa34d26bcee5ae5a8d2054e576c88beede7a3,a18f66ca6a97a66b8f42a4d61b2aa34d26bcee5ae5a8d2054e576c88beede7a3,None,True,None,False


{'status': 'PASS',
 'resolved_artifact_count': 5,
 'output_directories_verified': 6,
 'matched_alignment_hash_verified': True,
 'next_action': 'Verify manifest and handoff semantics against the resolved split contract and matched-table artifact.'}

In [3]:
# ============================================================
# 04_EVENT_STREAM_CONSTRUCTION
# Cell 04 — Upstream semantic-contract verification
#
# This cell verifies:
# - Notebook 00 manifest envelope and split-contract registration
# - split-contract payload integrity and frozen boundaries
# - Notebook 03 manifest and Notebook 04 authorization
# - matched-table registration in both manifest and handoff
# - LOCAL_STRICT event-stream authority
# - Notebook 04-specific partition-protection rules
#
# No event data are constructed in this cell.
# Notebook 03 partition summaries remain diagnostic only.
# ============================================================


# ------------------------------------------------------------
# Structural helper functions
# ------------------------------------------------------------

def require_mapping(
    value: Any,
    value_name: str,
) -> dict[str, Any]:
    """Require a JSON-like mapping."""
    require(
        isinstance(value, dict),
        f"{value_name} must be a mapping; "
        f"observed={type(value).__name__}.",
    )

    return value


def require_list(
    value: Any,
    value_name: str,
) -> list[Any]:
    """Require a JSON-like list."""
    require(
        isinstance(value, list),
        f"{value_name} must be a list; "
        f"observed={type(value).__name__}.",
    )

    return value


def normalized_path_key(path: Path | str) -> str:
    """
    Return a stable, case-insensitive Windows-path identity key.

    strict=False is intentional because the function is also used
    to compare paths recorded inside manifests.
    """
    return str(
        Path(path).resolve(strict=False)
    ).replace("\\", "/").casefold()


def paths_match(
    left: Path | str,
    right: Path | str,
) -> bool:
    """Compare two paths by normalized resolved identity."""
    return (
        normalized_path_key(left)
        == normalized_path_key(right)
    )


def unwrap_artifact_envelope(
    artifact: Any,
    *,
    artifact_name: str,
    expected_artifact_type: str | None = None,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """
    Validate and unwrap a Notebook 00-style artifact envelope.

    Required structure:
        {
            "artifact_metadata": {...},
            "payload": {...}
        }
    """
    envelope = require_mapping(
        artifact,
        artifact_name,
    )

    metadata = require_mapping(
        envelope.get("artifact_metadata"),
        f"{artifact_name}.artifact_metadata",
    )

    payload = require_mapping(
        envelope.get("payload"),
        f"{artifact_name}.payload",
    )

    required_metadata_fields = {
        "artifact_type",
        "artifact_schema_version",
        "pipeline_version",
        "source_run_prefix",
        "source_set_hash",
        "v0_1_run_id",
        "producing_notebook",
        "acceptance_status",
        "payload_sha256",
    }

    missing_metadata_fields = sorted(
        required_metadata_fields
        - set(metadata)
    )

    require(
        not missing_metadata_fields,
        (
            f"{artifact_name} metadata is missing fields: "
            f"{missing_metadata_fields}"
        ),
    )

    if expected_artifact_type is not None:
        require(
            metadata["artifact_type"]
            == expected_artifact_type,
            (
                f"{artifact_name} artifact type mismatch: "
                f"observed={metadata['artifact_type']!r}, "
                f"expected={expected_artifact_type!r}."
            ),
        )

    require(
        metadata["pipeline_version"]
        == PIPELINE_VERSION,
        (
            f"{artifact_name} pipeline version mismatch: "
            f"{metadata['pipeline_version']!r}"
        ),
    )

    require(
        metadata["source_run_prefix"]
        == SOURCE_RUN_PREFIX,
        (
            f"{artifact_name} source-run mismatch: "
            f"{metadata['source_run_prefix']!r}"
        ),
    )

    require(
        metadata["source_set_hash"]
        == EXPECTED_SOURCE_SET_SHA256,
        (
            f"{artifact_name} source-set hash mismatch: "
            f"{metadata['source_set_hash']!r}"
        ),
    )

    require(
        metadata["v0_1_run_id"]
        == V01_RUN_ID,
        (
            f"{artifact_name} V0.1 run mismatch: "
            f"{metadata['v0_1_run_id']!r}"
        ),
    )

    observed_payload_sha256 = canonical_json_sha256(
        payload
    )

    expected_payload_sha256 = require_sha256(
        str(metadata["payload_sha256"]),
        f"{artifact_name}.payload_sha256",
    )

    require(
        observed_payload_sha256
        == expected_payload_sha256,
        (
            f"{artifact_name} payload hash mismatch: "
            f"observed={observed_payload_sha256}, "
            f"expected={expected_payload_sha256}."
        ),
    )

    return metadata, payload


def records_matching_path(
    records: Sequence[Mapping[str, Any]],
    target_path: Path,
) -> list[dict[str, Any]]:
    """
    Return manifest records whose path field matches target_path.

    Recognized path fields:
    - resolved_path
    - path
    """
    target_key = normalized_path_key(target_path)
    matches: list[dict[str, Any]] = []

    for raw_record in records:
        record = dict(raw_record)

        candidate_path = (
            record.get("resolved_path")
            or record.get("path")
        )

        if candidate_path is None:
            continue

        if normalized_path_key(candidate_path) == target_key:
            matches.append(record)

    return matches


def require_single_record(
    records: Sequence[Mapping[str, Any]],
    *,
    record_name: str,
) -> dict[str, Any]:
    """Require exactly one record from an already filtered list."""
    records = [
        dict(record)
        for record in records
    ]

    require(
        len(records) == 1,
        (
            f"{record_name} must resolve to exactly one record; "
            f"observed={len(records)}."
        ),
    )

    return records[0]


def require_file_metadata_match(
    *,
    metadata: Mapping[str, Any],
    actual_path: Path,
    metadata_name: str,
) -> None:
    """
    Verify path, size, and file hash against one metadata record.
    """
    actual_path = require_file(actual_path)

    registered_path = (
        metadata.get("resolved_path")
        or metadata.get("path")
    )

    require(
        registered_path is not None,
        f"{metadata_name} does not contain a path field.",
    )

    require(
        paths_match(
            registered_path,
            actual_path,
        ),
        (
            f"{metadata_name} path mismatch: "
            f"registered={registered_path}, "
            f"actual={actual_path}"
        ),
    )

    registered_size = metadata.get("size_bytes")

    require(
        registered_size is not None,
        f"{metadata_name} lacks size_bytes.",
    )

    require(
        int(registered_size)
        == int(actual_path.stat().st_size),
        (
            f"{metadata_name} size mismatch: "
            f"registered={registered_size}, "
            f"actual={actual_path.stat().st_size}"
        ),
    )

    registered_hash = require_sha256(
        str(metadata.get("sha256")),
        f"{metadata_name}.sha256",
    )

    actual_hash = sha256_file(actual_path)

    require(
        registered_hash == actual_hash,
        (
            f"{metadata_name} SHA-256 mismatch: "
            f"registered={registered_hash}, "
            f"actual={actual_hash}"
        ),
    )


# ------------------------------------------------------------
# Verify Notebook 00 output-manifest envelope
# ------------------------------------------------------------

(
    notebook_00_manifest_metadata,
    notebook_00_manifest_body,
) = unwrap_artifact_envelope(
    notebook_00_manifest_payload,
    artifact_name="Notebook 00 output manifest",
    expected_artifact_type="NOTEBOOK_00_OUTPUT_MANIFEST",
)

require(
    notebook_00_manifest_metadata[
        "producing_notebook"
    ]
    == "00_V01_RUN_CONTRACT.ipynb",
    (
        "Notebook 00 manifest producing-notebook mismatch: "
        f"{notebook_00_manifest_metadata['producing_notebook']!r}"
    ),
)

require(
    notebook_00_manifest_body.get(
        "source_run_prefix"
    )
    == SOURCE_RUN_PREFIX,
    "Notebook 00 manifest body has the wrong source run.",
)

require(
    notebook_00_manifest_body.get(
        "v0_1_run_id"
    )
    == V01_RUN_ID,
    "Notebook 00 manifest body has the wrong V0.1 run.",
)

require(
    notebook_00_manifest_body.get(
        "source_set_hash"
    )
    == EXPECTED_SOURCE_SET_SHA256,
    "Notebook 00 manifest body has the wrong source-set hash.",
)

require(
    notebook_00_manifest_body.get(
        "v0_1_run_config_hash"
    )
    == EXPECTED_V01_RUN_CONFIG_SHA256,
    "Notebook 00 manifest body has the wrong run-config hash.",
)

require(
    notebook_00_manifest_body.get(
        "v0_1_run_identity_hash"
    )
    == EXPECTED_V01_RUN_IDENTITY_SHA256,
    "Notebook 00 manifest body has the wrong run-identity hash.",
)

require(
    notebook_00_manifest_body.get(
        "operating_mode"
    )
    == OPERATING_MODE,
    "Notebook 00 manifest body has the wrong operating mode.",
)

notebook_00_manifest_artifacts = require_list(
    notebook_00_manifest_body.get("artifacts"),
    "Notebook 00 manifest artifacts",
)

require(
    int(
        notebook_00_manifest_body.get(
            "artifact_count",
            -1,
        )
    )
    == len(notebook_00_manifest_artifacts),
    (
        "Notebook 00 manifest artifact_count does not match "
        "the artifact-record list."
    ),
)

notebook_00_manifest_artifact_table = (
    pd.DataFrame.from_records(
        notebook_00_manifest_artifacts
    )
)

require_unique_columns(
    notebook_00_manifest_artifact_table,
    "notebook_00_manifest_artifact_table",
)

require_columns(
    notebook_00_manifest_artifact_table,
    [
        "artifact_type",
        "format",
        "resolved_path",
        "size_bytes",
        "sha256",
    ],
    "notebook_00_manifest_artifact_table",
)


# ------------------------------------------------------------
# Verify that the resolved split contract is exactly the
# CHRONOLOGICAL_SPLIT_CONTRACT registered by Notebook 00
# ------------------------------------------------------------

split_manifest_records = (
    notebook_00_manifest_artifact_table.loc[
        notebook_00_manifest_artifact_table[
            "artifact_type"
        ].eq("CHRONOLOGICAL_SPLIT_CONTRACT")
    ]
    .to_dict(orient="records")
)

split_manifest_record = require_single_record(
    split_manifest_records,
    record_name=(
        "Notebook 00 CHRONOLOGICAL_SPLIT_CONTRACT record"
    ),
)

require_file_metadata_match(
    metadata=split_manifest_record,
    actual_path=SPLIT_CONTRACT_PATH_RESOLVED,
    metadata_name=(
        "Notebook 00 split-contract manifest record"
    ),
)


# ------------------------------------------------------------
# Unwrap and verify the split-contract artifact
# ------------------------------------------------------------

(
    split_contract_metadata,
    authoritative_split_contract,
) = unwrap_artifact_envelope(
    split_contract_payload,
    artifact_name="Notebook 00 split contract",
    expected_artifact_type=(
        "CHRONOLOGICAL_SPLIT_CONTRACT"
    ),
)

AUTHORITATIVE_SPLIT_CONTRACT_FILE_SHA256 = (
    sha256_file(
        SPLIT_CONTRACT_PATH_RESOLVED
    )
)

AUTHORITATIVE_SPLIT_CONTRACT_PAYLOAD_SHA256 = (
    require_sha256(
        str(
            split_contract_metadata[
                "payload_sha256"
            ]
        ),
        "authoritative split-contract payload SHA-256",
    )
)

require(
    AUTHORITATIVE_SPLIT_CONTRACT_PAYLOAD_SHA256
    == EXPECTED_CHRONOLOGICAL_SPLIT_CONTRACT_SHA256,
    (
        "The split-contract payload hash does not match the "
        "frozen Notebook 00 hash registered in Notebook 04: "
        f"observed="
        f"{AUTHORITATIVE_SPLIT_CONTRACT_PAYLOAD_SHA256}, "
        f"expected="
        f"{EXPECTED_CHRONOLOGICAL_SPLIT_CONTRACT_SHA256}."
    ),
)

require(
    authoritative_split_contract.get(
        "source_run_prefix"
    )
    == SOURCE_RUN_PREFIX,
    "Split contract source-run identity mismatch.",
)

require(
    authoritative_split_contract.get(
        "source_set_hash"
    )
    == EXPECTED_SOURCE_SET_SHA256,
    "Split contract source-set hash mismatch.",
)

require(
    authoritative_split_contract.get(
        "pipeline_version"
    )
    == PIPELINE_VERSION,
    "Split contract pipeline version mismatch.",
)

require(
    authoritative_split_contract.get(
        "operating_mode"
    )
    == OPERATING_MODE,
    "Split contract operating mode mismatch.",
)

require(
    authoritative_split_contract.get(
        "primary_boundary_authority"
    )
    == PRIMARY_ORDERING_AUTHORITY,
    (
        "Split contract must use collector_sequence as its "
        "primary boundary authority."
    ),
)

require(
    authoritative_split_contract.get(
        "interval_convention"
    )
    == INTERVAL_CONVENTION,
    "Split contract does not use the frozen half-open rule.",
)

require(
    authoritative_split_contract.get(
        "canonical_timezone"
    )
    == CANONICAL_TIMEZONE,
    "Split contract timezone mismatch.",
)

require(
    tuple(
        authoritative_split_contract.get(
            "partition_order",
            [],
        )
    )
    == PARTITION_ORDER,
    "Split contract partition order mismatch.",
)

require(
    authoritative_split_contract.get(
        "partitions_are_independent"
    )
    is False,
    (
        "Engineering-reproduction partitions must not be "
        "represented as independent samples."
    ),
)

require(
    authoritative_split_contract.get(
        "claim_bearing_holdout_available"
    )
    is False,
    (
        "This source run must not claim an independent "
        "claim-bearing holdout."
    ),
)

require(
    authoritative_split_contract.get(
        "final_partition_model_selection_access"
    )
    is False,
    (
        "The final partition must remain unavailable for "
        "model selection."
    ),
)

require(
    authoritative_split_contract.get(
        "producing_notebook"
    )
    == "00_V01_RUN_CONTRACT.ipynb",
    "Split contract producing-notebook mismatch.",
)


# ------------------------------------------------------------
# Verify the frozen boundary table
# ------------------------------------------------------------

split_boundary_table = pd.DataFrame.from_records(
    require_list(
        authoritative_split_contract.get(
            "boundary_table"
        ),
        "split contract boundary_table",
    )
)

require_unique_columns(
    split_boundary_table,
    "split_boundary_table",
)

require_columns(
    split_boundary_table,
    [
        "partition_order",
        "partition",
        "collector_sequence_start",
        "collector_sequence_end_exclusive",
        "collector_sequence_last_included",
        "receipt_time_start_ns",
        "receipt_time_end_ns_exclusive",
        "row_count",
        "trade_row_count",
        "depth_row_count",
        "independent_partition",
        "claim_authority",
        "interval_convention",
    ],
    "split_boundary_table",
)

split_boundary_table = (
    split_boundary_table
    .sort_values(
        "partition_order",
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    tuple(
        split_boundary_table[
            "partition"
        ].astype(str)
    )
    == PARTITION_ORDER,
    "Boundary-table partition order mismatch.",
)

require(
    split_boundary_table[
        "partition_order"
    ].astype(int).tolist()
    == [1, 2, 3, 4],
    "Boundary-table partition numbers are invalid.",
)

require(
    split_boundary_table[
        "interval_convention"
    ].eq(INTERVAL_CONVENTION).all(),
    "Boundary table contains a non-half-open interval.",
)

require(
    ~split_boundary_table[
        "independent_partition"
    ].astype(bool).any(),
    (
        "No engineering-reproduction partition may be marked "
        "independent."
    ),
)

require(
    ~split_boundary_table[
        "claim_authority"
    ].astype(bool).any(),
    (
        "No partition in this one-session engineering run may "
        "carry claim authority."
    ),
)

require(
    (
        split_boundary_table[
            "trade_row_count"
        ].astype(int)
        + split_boundary_table[
            "depth_row_count"
        ].astype(int)
    ).equals(
        split_boundary_table[
            "row_count"
        ].astype(int)
    ),
    (
        "Boundary-table trade and depth counts do not sum to "
        "the combined partition row counts."
    ),
)


boundary_comparison_rows: list[dict[str, Any]] = []

for spec in PARTITION_SPECS:
    matching_rows = split_boundary_table.loc[
        split_boundary_table[
            "partition"
        ].eq(spec.name)
    ]

    require(
        len(matching_rows) == 1,
        (
            f"Expected exactly one split-contract boundary row "
            f"for {spec.name}; observed={len(matching_rows)}."
        ),
    )

    row = matching_rows.iloc[0]

    checks = {
        "partition_order_matches": (
            int(row["partition_order"])
            == spec.order
        ),
        "sequence_start_matches": (
            int(
                row[
                    "collector_sequence_start"
                ]
            )
            == spec.collector_sequence_start
        ),
        "sequence_end_exclusive_matches": (
            int(
                row[
                    "collector_sequence_end_exclusive"
                ]
            )
            == spec.collector_sequence_end_exclusive
        ),
        "sequence_last_matches": (
            int(
                row[
                    "collector_sequence_last_included"
                ]
            )
            == spec.collector_sequence_last_included
        ),
        "receipt_start_matches": (
            int(
                row[
                    "receipt_time_start_ns"
                ]
            )
            == spec.receipt_time_start_ns
        ),
        "receipt_end_exclusive_matches": (
            int(
                row[
                    "receipt_time_end_ns_exclusive"
                ]
            )
            == spec.receipt_time_end_exclusive_ns
        ),
        "trade_count_matches": (
            int(row["trade_row_count"])
            == spec.expected_trade_count
        ),
        "independence_rule_matches": (
            bool(row["independent_partition"])
            is False
        ),
        "claim_authority_rule_matches": (
            bool(row["claim_authority"])
            is False
        ),
    }

    boundary_comparison_rows.append(
        {
            "partition": spec.name,
            "expected_sequence_start": (
                spec.collector_sequence_start
            ),
            "observed_sequence_start": int(
                row[
                    "collector_sequence_start"
                ]
            ),
            "expected_sequence_end_exclusive": (
                spec.collector_sequence_end_exclusive
            ),
            "observed_sequence_end_exclusive": int(
                row[
                    "collector_sequence_end_exclusive"
                ]
            ),
            "expected_receipt_start_ns": (
                spec.receipt_time_start_ns
            ),
            "observed_receipt_start_ns": int(
                row["receipt_time_start_ns"]
            ),
            "expected_receipt_end_exclusive_ns": (
                spec.receipt_time_end_exclusive_ns
            ),
            "observed_receipt_end_exclusive_ns": int(
                row[
                    "receipt_time_end_ns_exclusive"
                ]
            ),
            "expected_trade_count": (
                spec.expected_trade_count
            ),
            "observed_trade_count": int(
                row["trade_row_count"]
            ),
            **checks,
            "all_checks_pass": all(
                checks.values()
            ),
        }
    )

split_boundary_comparison = pd.DataFrame.from_records(
    boundary_comparison_rows
)

require(
    split_boundary_comparison[
        "all_checks_pass"
    ].all(),
    (
        "At least one Notebook 00 boundary differs from the "
        "frozen Notebook 04 declaration."
    ),
)

require(
    int(
        split_boundary_table[
            "trade_row_count"
        ].sum()
    )
    == EXPECTED_ALIGNED_TRADE_ROWS,
    (
        "Notebook 00 partition trade counts do not sum to "
        f"{EXPECTED_ALIGNED_TRADE_ROWS:,}."
    ),
)

for current_spec, next_spec in zip(
    PARTITION_SPECS[:-1],
    PARTITION_SPECS[1:],
):
    require(
        (
            current_spec.collector_sequence_end_exclusive
            == next_spec.collector_sequence_start
        ),
        (
            "Frozen collector-sequence intervals are not "
            f"contiguous: {current_spec.name} -> "
            f"{next_spec.name}."
        ),
    )

    require(
        (
            current_spec.receipt_time_end_exclusive_ns
            == next_spec.receipt_time_start_ns
        ),
        (
            "Frozen receipt-time intervals are not contiguous: "
            f"{current_spec.name} -> {next_spec.name}."
        ),
    )


# ------------------------------------------------------------
# Inspect Notebook 00 generic access matrix, then freeze a
# stricter Notebook 04 event-definition selection policy.
#
# Notebook 00's generic model-selection matrix is preserved.
# Notebook 04 does not use it to grant VALIDATION access for
# event-definition selection.
# ------------------------------------------------------------

split_access_matrix = pd.DataFrame.from_records(
    require_list(
        authoritative_split_contract.get(
            "access_matrix"
        ),
        "split contract access_matrix",
    )
)

require_unique_columns(
    split_access_matrix,
    "split_access_matrix",
)

require_columns(
    split_access_matrix,
    [
        "partition",
        "model_selection_access",
        "parameter_estimation_access",
        "final_evaluation_access",
        "claim_authority",
    ],
    "split_access_matrix",
)

require(
    set(
        split_access_matrix[
            "partition"
        ].astype(str)
    )
    == set(PARTITION_ORDER),
    "Split access matrix does not cover all partitions.",
)

final_access_row = split_access_matrix.loc[
    split_access_matrix[
        "partition"
    ].eq("ENGINEERING_HOLDOUT")
]

require(
    len(final_access_row) == 1,
    (
        "Split access matrix must contain exactly one "
        "ENGINEERING_HOLDOUT row."
    ),
)

require(
    not bool(
        final_access_row.iloc[0][
            "model_selection_access"
        ]
    ),
    (
        "ENGINEERING_HOLDOUT must be excluded from generic "
        "model-selection access."
    ),
)

notebook_04_partition_access = pd.DataFrame(
    [
        {
            "partition_order": spec.order,
            "partition": spec.name,
            "generic_notebook_00_model_selection_access": bool(
                split_access_matrix.loc[
                    split_access_matrix[
                        "partition"
                    ].eq(spec.name),
                    "model_selection_access",
                ].iloc[0]
            ),
            "notebook_04_event_definition_selection_access": (
                spec.event_definition_selection_allowed
            ),
            "notebook_04_protected_partition": (
                spec.protected_from_event_definition_selection
            ),
            "construction_allowed_before_selection_freeze": (
                spec.event_definition_selection_allowed
            ),
            "construction_allowed_after_selection_freeze": True,
        }
        for spec in PARTITION_SPECS
    ]
)

require(
    tuple(
        notebook_04_partition_access.loc[
            notebook_04_partition_access[
                "notebook_04_event_definition_selection_access"
            ],
            "partition",
        ]
    )
    == EVENT_DEFINITION_SELECTION_PARTITIONS,
    (
        "Notebook 04 selection access must be limited to "
        "DEVELOPMENT and CALIBRATION."
    ),
)

require(
    tuple(
        notebook_04_partition_access.loc[
            notebook_04_partition_access[
                "notebook_04_protected_partition"
            ],
            "partition",
        ]
    )
    == PROTECTED_PARTITIONS,
    (
        "Notebook 04 protected partitions must be VALIDATION "
        "and ENGINEERING_HOLDOUT."
    ),
)


# ------------------------------------------------------------
# Verify relevant split-history rules
# ------------------------------------------------------------

split_history_rules = pd.DataFrame.from_records(
    require_list(
        authoritative_split_contract.get(
            "history_rules"
        ),
        "split contract history_rules",
    )
)

require_unique_columns(
    split_history_rules,
    "split_history_rules",
)

require_columns(
    split_history_rules,
    [
        "stage",
        "history_carry_allowed",
        "history_usage",
        "scored_pre_boundary_events",
    ],
    "split_history_rules",
)

for required_stage in (
    "TRADE_BOOK_ALIGNMENT",
    "HAWKES_INTENSITY",
    "MODEL_SELECTION",
):
    matching_stage = split_history_rules.loc[
        split_history_rules[
            "stage"
        ].eq(required_stage)
    ]

    require(
        len(matching_stage) == 1,
        (
            f"Split-history rules must contain exactly one "
            f"{required_stage} row."
        ),
    )

trade_alignment_history_rule = (
    split_history_rules.loc[
        split_history_rules[
            "stage"
        ].eq("TRADE_BOOK_ALIGNMENT")
    ]
    .iloc[0]
)

require(
    bool(
        trade_alignment_history_rule[
            "history_carry_allowed"
        ]
    ),
    (
        "Trade-book alignment must permit causal history carry "
        "across partition boundaries."
    ),
)

require(
    not bool(
        trade_alignment_history_rule[
            "scored_pre_boundary_events"
        ]
    ),
    (
        "Pre-boundary history may initialize state but may not "
        "be scored inside the later partition."
    ),
)

hawkes_history_rule = (
    split_history_rules.loc[
        split_history_rules[
            "stage"
        ].eq("HAWKES_INTENSITY")
    ]
    .iloc[0]
)

require(
    bool(
        hawkes_history_rule[
            "history_carry_allowed"
        ]
    ),
    (
        "Hawkes intensity history must be allowed to carry "
        "causally across partition boundaries."
    ),
)

require(
    not bool(
        hawkes_history_rule[
            "scored_pre_boundary_events"
        ]
    ),
    (
        "Pre-boundary events may initialize excitation history "
        "but may not be scored in the later partition."
    ),
)


# ------------------------------------------------------------
# Verify Notebook 03 manifest semantics
# ------------------------------------------------------------

accepted_notebook_03_terminal_statuses = {
    "PASS",
    "PASS_WITH_ALIGNMENT_WARNINGS",
    "PASS_WITH_REFERENCE_WARNINGS",
}

require(
    notebook_03_manifest_payload.get(
        "project_name"
    )
    == PROJECT_NAME,
    "Notebook 03 manifest project-name mismatch.",
)

require(
    notebook_03_manifest_payload.get(
        "pipeline_version"
    )
    == PIPELINE_VERSION,
    "Notebook 03 manifest pipeline-version mismatch.",
)

require(
    notebook_03_manifest_payload.get(
        "notebook_name"
    )
    == PREVIOUS_NOTEBOOK_NAME,
    "Notebook 03 manifest notebook-name mismatch.",
)

require(
    notebook_03_manifest_payload.get(
        "source_run_prefix"
    )
    == SOURCE_RUN_PREFIX,
    "Notebook 03 manifest source-run mismatch.",
)

require(
    notebook_03_manifest_payload.get(
        "v0_1_run_id"
    )
    == V01_RUN_ID,
    "Notebook 03 manifest V0.1 run mismatch.",
)

require(
    notebook_03_manifest_payload.get(
        "combined_prefix"
    )
    == COMBINED_PREFIX,
    "Notebook 03 manifest combined-prefix mismatch.",
)

require(
    notebook_03_manifest_payload.get(
        "operating_mode"
    )
    == OPERATING_MODE,
    "Notebook 03 manifest operating-mode mismatch.",
)

require(
    notebook_03_manifest_payload.get(
        "primary_ordering_authority"
    )
    == PRIMARY_ORDERING_AUTHORITY,
    "Notebook 03 manifest ordering-authority mismatch.",
)

require(
    notebook_03_manifest_payload.get(
        "primary_alignment_policy"
    )
    == PRIMARY_ALIGNMENT_POLICY,
    "Notebook 03 manifest alignment-policy mismatch.",
)

require(
    notebook_03_manifest_payload.get(
        "canonical_timezone"
    )
    == CANONICAL_TIMEZONE,
    "Notebook 03 manifest timezone mismatch.",
)

require(
    notebook_03_manifest_payload.get(
        "terminal_status"
    )
    in accepted_notebook_03_terminal_statuses,
    (
        "Notebook 03 terminal status does not authorize "
        "controlled downstream work."
    ),
)

require(
    notebook_03_manifest_payload.get(
        "notebook_04_authorized"
    )
    is True,
    "Notebook 03 manifest does not authorize Notebook 04.",
)

notebook_03_gate_summary = require_mapping(
    notebook_03_manifest_payload.get(
        "gate_summary"
    ),
    "Notebook 03 gate_summary",
)

require(
    int(
        notebook_03_gate_summary.get(
            "blocking_gate_failure_count",
            -1,
        )
    )
    == 0,
    (
        "Notebook 03 manifest reports one or more blocking "
        "gate failures."
    ),
)


# ------------------------------------------------------------
# Verify Notebook 03 output-manifest registration of the
# matched LOCAL_STRICT trade-book table
# ------------------------------------------------------------

notebook_03_output_artifacts = require_list(
    notebook_03_manifest_payload.get(
        "output_artifacts"
    ),
    "Notebook 03 output_artifacts",
)

notebook_03_matched_manifest_records = (
    records_matching_path(
        notebook_03_output_artifacts,
        NOTEBOOK_03_MATCHED_ALIGNMENT_PATH_RESOLVED,
    )
)

notebook_03_matched_manifest_record = (
    require_single_record(
        notebook_03_matched_manifest_records,
        record_name=(
            "Notebook 03 matched-table manifest record"
        ),
    )
)

require_file_metadata_match(
    metadata=notebook_03_matched_manifest_record,
    actual_path=(
        NOTEBOOK_03_MATCHED_ALIGNMENT_PATH_RESOLVED
    ),
    metadata_name=(
        "Notebook 03 matched-table manifest record"
    ),
)

require(
    int(
        notebook_03_matched_manifest_record.get(
            "row_count",
            -1,
        )
    )
    == EXPECTED_MATCHED_TRADE_ROWS,
    (
        "Notebook 03 matched-table manifest row count "
        f"must equal {EXPECTED_MATCHED_TRADE_ROWS:,}."
    ),
)


# ------------------------------------------------------------
# Verify Notebook 03-to-04 handoff semantics
# ------------------------------------------------------------

require(
    notebook_03_to_04_handoff_payload.get(
        "project_name"
    )
    == PROJECT_NAME,
    "Notebook 03 handoff project-name mismatch.",
)

require(
    notebook_03_to_04_handoff_payload.get(
        "pipeline_version"
    )
    == PIPELINE_VERSION,
    "Notebook 03 handoff pipeline-version mismatch.",
)

require(
    notebook_03_to_04_handoff_payload.get(
        "source_run_prefix"
    )
    == SOURCE_RUN_PREFIX,
    "Notebook 03 handoff source-run mismatch.",
)

require(
    notebook_03_to_04_handoff_payload.get(
        "v0_1_run_id"
    )
    == V01_RUN_ID,
    "Notebook 03 handoff V0.1 run mismatch.",
)

require(
    notebook_03_to_04_handoff_payload.get(
        "combined_prefix"
    )
    == COMBINED_PREFIX,
    "Notebook 03 handoff combined-prefix mismatch.",
)

require(
    notebook_03_to_04_handoff_payload.get(
        "producing_notebook"
    )
    == PREVIOUS_NOTEBOOK_NAME,
    "Notebook 03 handoff producing-notebook mismatch.",
)

require(
    notebook_03_to_04_handoff_payload.get(
        "next_notebook"
    )
    == NOTEBOOK_NAME,
    "Notebook 03 handoff next-notebook mismatch.",
)

require(
    notebook_03_to_04_handoff_payload.get(
        "terminal_status"
    )
    == notebook_03_manifest_payload.get(
        "terminal_status"
    ),
    (
        "Notebook 03 manifest and handoff terminal statuses "
        "do not agree."
    ),
)

require(
    notebook_03_to_04_handoff_payload.get(
        "notebook_04_authorized"
    )
    is True,
    "Notebook 03 handoff does not authorize Notebook 04.",
)

require(
    notebook_03_to_04_handoff_payload.get(
        "primary_alignment_policy"
    )
    == PRIMARY_ALIGNMENT_POLICY,
    "Notebook 03 handoff alignment-policy mismatch.",
)

require(
    notebook_03_to_04_handoff_payload.get(
        "primary_ordering_authority"
    )
    == PRIMARY_ORDERING_AUTHORITY,
    "Notebook 03 handoff ordering-authority mismatch.",
)

require(
    notebook_03_to_04_handoff_payload.get(
        "canonical_timezone"
    )
    == CANONICAL_TIMEZONE,
    "Notebook 03 handoff timezone mismatch.",
)

require(
    paths_match(
        notebook_03_to_04_handoff_payload.get(
            "notebook_03_manifest_path"
        ),
        NOTEBOOK_03_OUTPUT_MANIFEST_PATH_RESOLVED,
    ),
    (
        "Notebook 03 handoff does not reference the loaded "
        "Notebook 03 manifest."
    ),
)


# ------------------------------------------------------------
# Verify authoritative matched-table metadata in the handoff
# ------------------------------------------------------------

authoritative_handoff_inputs = require_mapping(
    notebook_03_to_04_handoff_payload.get(
        "authoritative_inputs_for_notebook_04"
    ),
    "Notebook 03 authoritative_inputs_for_notebook_04",
)

matched_handoff_metadata = require_mapping(
    authoritative_handoff_inputs.get(
        "local_strict_matched_trade_book"
    ),
    (
        "Notebook 03 handoff "
        "local_strict_matched_trade_book"
    ),
)

require_file_metadata_match(
    metadata=matched_handoff_metadata,
    actual_path=(
        NOTEBOOK_03_MATCHED_ALIGNMENT_PATH_RESOLVED
    ),
    metadata_name=(
        "Notebook 03 handoff matched-table metadata"
    ),
)

require(
    int(
        matched_handoff_metadata.get(
            "row_count",
            -1,
        )
    )
    == EXPECTED_MATCHED_TRADE_ROWS,
    (
        "Notebook 03 handoff matched-table row count "
        f"must equal {EXPECTED_MATCHED_TRADE_ROWS:,}."
    ),
)


# ------------------------------------------------------------
# Verify Notebook 03 handoff counts
# ------------------------------------------------------------

notebook_03_handoff_counts = require_mapping(
    notebook_03_to_04_handoff_payload.get(
        "counts"
    ),
    "Notebook 03 handoff counts",
)

required_handoff_count_values = {
    "raw_trade_count": (
        EXPECTED_ALIGNED_TRADE_ROWS
    ),
    "local_strict_all_alignment_rows": (
        EXPECTED_ALIGNED_TRADE_ROWS
    ),
    "local_strict_matched_rows": (
        EXPECTED_MATCHED_TRADE_ROWS
    ),
    "local_strict_unmatched_rows": (
        EXPECTED_UNMATCHED_TRADE_ROWS
    ),
    "buy_trade_count": (
        EXPECTED_GLOBAL_BUY_TRADES
    ),
    "sell_trade_count": (
        EXPECTED_GLOBAL_SELL_TRADES
    ),
}

handoff_count_rows: list[dict[str, Any]] = []

for count_name, expected_value in (
    required_handoff_count_values.items()
):
    observed_value = notebook_03_handoff_counts.get(
        count_name
    )

    passed = bool(
        observed_value is not None
        and int(observed_value)
        == int(expected_value)
    )

    handoff_count_rows.append(
        {
            "count_name": count_name,
            "expected_value": int(
                expected_value
            ),
            "observed_value": (
                None
                if observed_value is None
                else int(observed_value)
            ),
            "passed": passed,
        }
    )

notebook_03_handoff_count_audit = (
    pd.DataFrame.from_records(
        handoff_count_rows
    )
)

require(
    notebook_03_handoff_count_audit[
        "passed"
    ].all(),
    (
        "At least one Notebook 03 handoff count differs from "
        "the frozen Notebook 04 expectation."
    ),
)


# ------------------------------------------------------------
# Verify event-stream authority fields
# ------------------------------------------------------------

notebook_03_event_stream_authority = (
    require_mapping(
        notebook_03_to_04_handoff_payload.get(
            "event_stream_authority"
        ),
        "Notebook 03 event_stream_authority",
    )
)

required_event_authority_fields = {
    "primary_event_time_field_for_notebook_04": (
        "trade_local_receipt_time_ns"
    ),
    "secondary_exchange_trade_time_field": (
        "trade_exchange_trade_time_ms"
    ),
    "primary_ordering_field": (
        "trade_collector_sequence"
    ),
    "trade_identity_field": "trade_id",
    "aggressor_side_field": (
        "trade_aggressor_side"
    ),
    "price_field": "trade_price",
    "quantity_field": "trade_quantity",
    "notional_field": "trade_notional",
    "matched_book_sequence_field": (
        "book_collector_sequence"
    ),
    "matched_book_local_time_field": (
        "book_local_receipt_time_ns"
    ),
    "alignment_policy": PRIMARY_ALIGNMENT_POLICY,
}

event_authority_rows: list[dict[str, Any]] = []

for field_name, expected_value in (
    required_event_authority_fields.items()
):
    observed_value = (
        notebook_03_event_stream_authority.get(
            field_name
        )
    )

    event_authority_rows.append(
        {
            "field": field_name,
            "expected_value": expected_value,
            "observed_value": observed_value,
            "passed": (
                observed_value
                == expected_value
            ),
        }
    )

notebook_03_event_authority_audit = (
    pd.DataFrame.from_records(
        event_authority_rows
    )
)

require(
    notebook_03_event_authority_audit[
        "passed"
    ].all(),
    (
        "Notebook 03 event-stream authority does not match "
        "Notebook 04 requirements."
    ),
)


# ------------------------------------------------------------
# Verify that EXCHANGE_STRICT remains sensitivity only
# ------------------------------------------------------------

sensitivity_only_inputs = require_mapping(
    notebook_03_to_04_handoff_payload.get(
        "sensitivity_only_inputs"
    ),
    "Notebook 03 sensitivity_only_inputs",
)

require(
    "exchange_strict_alignment_sensitivity"
    in sensitivity_only_inputs,
    (
        "Notebook 03 handoff must classify EXCHANGE_STRICT "
        "as sensitivity-only evidence."
    ),
)

require(
    notebook_03_to_04_handoff_payload.get(
        "primary_alignment_policy"
    )
    != "EXCHANGE_STRICT",
    (
        "EXCHANGE_STRICT must not become Notebook 04's "
        "primary alignment authority."
    ),
)


# ------------------------------------------------------------
# Preserve Notebook 03 partition summary as diagnostic only
#
# These counts are intentionally not used as blocking
# partition authority. Notebook 04 will reconstruct partitions
# from collector sequence in a later cell.
# ------------------------------------------------------------

notebook_03_partition_summary_diagnostic = (
    pd.DataFrame.from_records(
        require_list(
            notebook_03_to_04_handoff_payload.get(
                "partition_summary",
                [],
            ),
            "Notebook 03 partition_summary",
        )
    )
)

partition_summary_comparison_rows: list[
    dict[str, Any]
] = []

if (
    not notebook_03_partition_summary_diagnostic.empty
    and {
        "trade_partition",
        "raw_trade_count",
    }.issubset(
        notebook_03_partition_summary_diagnostic.columns
    )
):
    for spec in PARTITION_SPECS:
        inherited_rows = (
            notebook_03_partition_summary_diagnostic.loc[
                notebook_03_partition_summary_diagnostic[
                    "trade_partition"
                ].eq(spec.name)
            ]
        )

        inherited_count = (
            None
            if len(inherited_rows) != 1
            else int(
                inherited_rows.iloc[0][
                    "raw_trade_count"
                ]
            )
        )

        partition_summary_comparison_rows.append(
            {
                "partition": spec.name,
                "notebook_00_authoritative_trade_count": (
                    spec.expected_trade_count
                ),
                "notebook_03_inherited_trade_count": (
                    inherited_count
                ),
                "count_difference": (
                    None
                    if inherited_count is None
                    else (
                        inherited_count
                        - spec.expected_trade_count
                    )
                ),
                "matches_authoritative_count": (
                    inherited_count
                    == spec.expected_trade_count
                ),
                "authority": (
                    "DIAGNOSTIC_ONLY"
                ),
            }
        )

notebook_03_partition_summary_comparison = (
    pd.DataFrame.from_records(
        partition_summary_comparison_rows
    )
)

if (
    notebook_03_partition_summary_comparison.empty
):
    inherited_partition_mismatch_count = None
else:
    inherited_partition_mismatch_count = int(
        (
            ~notebook_03_partition_summary_comparison[
                "matches_authoritative_count"
            ]
        ).sum()
    )


# ------------------------------------------------------------
# Preserve inherited warnings without treating them as filters
# ------------------------------------------------------------

notebook_03_warnings_to_carry_forward = (
    require_list(
        notebook_03_to_04_handoff_payload.get(
            "warnings_to_carry_forward",
            [],
        ),
        "Notebook 03 warnings_to_carry_forward",
    )
)

notebook_03_warning_table = (
    pd.DataFrame.from_records(
        notebook_03_warnings_to_carry_forward
    )
)


# ------------------------------------------------------------
# Combined semantic gate ledger
# ------------------------------------------------------------

upstream_semantic_gates: list[GateResult] = [
    make_gate(
        gate="notebook_00_manifest_payload_verified",
        passed=True,
        severity="BLOCKING",
        detail=(
            "Notebook 00 output-manifest envelope and payload "
            "hash verified."
        ),
    ),
    make_gate(
        gate="split_contract_manifest_registration_verified",
        passed=True,
        severity="BLOCKING",
        detail=(
            f"path={SPLIT_CONTRACT_PATH_RESOLVED}; "
            f"file_sha256="
            f"{AUTHORITATIVE_SPLIT_CONTRACT_FILE_SHA256}"
        ),
    ),
    make_gate(
        gate="split_contract_payload_hash_verified",
        passed=(
            AUTHORITATIVE_SPLIT_CONTRACT_PAYLOAD_SHA256
            == EXPECTED_CHRONOLOGICAL_SPLIT_CONTRACT_SHA256
        ),
        severity="BLOCKING",
        detail=(
            "payload_sha256="
            f"{AUTHORITATIVE_SPLIT_CONTRACT_PAYLOAD_SHA256}"
        ),
    ),
    make_gate(
        gate="split_boundary_table_matches_frozen_declaration",
        passed=bool(
            split_boundary_comparison[
                "all_checks_pass"
            ].all()
        ),
        severity="BLOCKING",
        detail=(
            f"verified_partitions="
            f"{len(split_boundary_comparison)}"
        ),
    ),
    make_gate(
        gate="notebook_04_selection_access_frozen",
        passed=(
            tuple(
                notebook_04_partition_access.loc[
                    notebook_04_partition_access[
                        "notebook_04_event_definition_selection_access"
                    ],
                    "partition",
                ]
            )
            == EVENT_DEFINITION_SELECTION_PARTITIONS
        ),
        severity="BLOCKING",
        detail=(
            "selection_partitions="
            + ",".join(
                EVENT_DEFINITION_SELECTION_PARTITIONS
            )
        ),
    ),
    make_gate(
        gate="notebook_04_protected_partitions_frozen",
        passed=(
            tuple(
                notebook_04_partition_access.loc[
                    notebook_04_partition_access[
                        "notebook_04_protected_partition"
                    ],
                    "partition",
                ]
            )
            == PROTECTED_PARTITIONS
        ),
        severity="BLOCKING",
        detail=(
            "protected_partitions="
            + ",".join(PROTECTED_PARTITIONS)
        ),
    ),
    make_gate(
        gate="notebook_03_manifest_semantics_verified",
        passed=True,
        severity="BLOCKING",
        detail=(
            "Identity, operating mode, alignment authority, "
            "terminal status, and authorization verified."
        ),
    ),
    make_gate(
        gate="notebook_03_matched_table_manifest_verified",
        passed=True,
        severity="BLOCKING",
        detail=(
            f"rows="
            f"{notebook_03_matched_manifest_record['row_count']}; "
            f"sha256="
            f"{notebook_03_matched_manifest_record['sha256']}"
        ),
    ),
    make_gate(
        gate="notebook_03_handoff_semantics_verified",
        passed=True,
        severity="BLOCKING",
        detail=(
            "Notebook 03-to-04 handoff identity and "
            "authorization verified."
        ),
    ),
    make_gate(
        gate="notebook_03_handoff_counts_verified",
        passed=bool(
            notebook_03_handoff_count_audit[
                "passed"
            ].all()
        ),
        severity="BLOCKING",
        detail=(
            f"verified_counts="
            f"{len(notebook_03_handoff_count_audit)}"
        ),
    ),
    make_gate(
        gate="notebook_03_event_authority_verified",
        passed=bool(
            notebook_03_event_authority_audit[
                "passed"
            ].all()
        ),
        severity="BLOCKING",
        detail=(
            f"verified_fields="
            f"{len(notebook_03_event_authority_audit)}"
        ),
    ),
    make_gate(
        gate="exchange_strict_remains_sensitivity_only",
        passed=True,
        severity="BLOCKING",
        detail=(
            "LOCAL_STRICT remains primary authority."
        ),
    ),
    make_gate(
        gate="notebook_03_partition_summary_matches_authority",
        passed=(
            inherited_partition_mismatch_count == 0
            if inherited_partition_mismatch_count
            is not None
            else False
        ),
        severity="WARNING",
        detail=(
            "Notebook 03 partition summary is diagnostic only; "
            f"mismatched_partition_count="
            f"{inherited_partition_mismatch_count}"
        ),
    ),
    make_gate(
        gate="notebook_03_warning_findings_absent",
        passed=(
            len(
                notebook_03_warnings_to_carry_forward
            )
            == 0
        ),
        severity="WARNING",
        detail=(
            "Warnings are carried forward without filtering; "
            f"warning_count="
            f"{len(notebook_03_warnings_to_carry_forward)}"
        ),
    ),
]

upstream_semantic_gate_frame = (
    gate_results_to_frame(
        upstream_semantic_gates
    )
)

fail_if_blocking_gate_failed(
    upstream_semantic_gate_frame
)


# ------------------------------------------------------------
# Compact audit summary
# ------------------------------------------------------------

upstream_semantic_summary = pd.DataFrame(
    [
        {
            "source_run_prefix": SOURCE_RUN_PREFIX,
            "v0_1_run_id": V01_RUN_ID,
            "operating_mode": OPERATING_MODE,
            "split_contract_file_sha256": (
                AUTHORITATIVE_SPLIT_CONTRACT_FILE_SHA256
            ),
            "split_contract_payload_sha256": (
                AUTHORITATIVE_SPLIT_CONTRACT_PAYLOAD_SHA256
            ),
            "matched_table_path": str(
                NOTEBOOK_03_MATCHED_ALIGNMENT_PATH_RESOLVED
            ),
            "matched_table_rows": int(
                notebook_03_matched_manifest_record[
                    "row_count"
                ]
            ),
            "matched_table_sha256": (
                notebook_03_matched_manifest_record[
                    "sha256"
                ]
            ),
            "notebook_03_terminal_status": (
                notebook_03_manifest_payload[
                    "terminal_status"
                ]
            ),
            "notebook_03_warning_count": len(
                notebook_03_warnings_to_carry_forward
            ),
            "inherited_partition_mismatch_count": (
                inherited_partition_mismatch_count
            ),
            "notebook_04_authorized": True,
        }
    ]
)

display(split_boundary_comparison)
display(notebook_04_partition_access)
display(notebook_03_handoff_count_audit)
display(notebook_03_event_authority_audit)

if not (
    notebook_03_partition_summary_comparison.empty
):
    display(
        notebook_03_partition_summary_comparison
    )

if not notebook_03_warning_table.empty:
    display(notebook_03_warning_table)

display(upstream_semantic_gate_frame)
display(upstream_semantic_summary)

{
    "status": "PASS",
    "notebook_04_authorized": True,
    "authoritative_split_contract_verified": True,
    "authoritative_matched_table_verified": True,
    "partition_summary_authority": "DIAGNOSTIC_ONLY",
    "next_action": (
        "Load the matched LOCAL_STRICT table from disk, "
        "canonicalize required fields, and independently "
        "reconstruct authoritative trade and book partitions."
    ),
}

,partition,expected_sequence_start,observed_sequence_start,expected_sequence_end_exclusive,observed_sequence_end_exclusive,expected_receipt_start_ns,observed_receipt_start_ns,expected_receipt_end_exclusive_ns,observed_receipt_end_exclusive_ns,expected_trade_count,observed_trade_count,partition_order_matches,sequence_start_matches,sequence_end_exclusive_matches,sequence_last_matches,receipt_start_matches,receipt_end_exclusive_matches,trade_count_matches,independence_rule_matches,claim_authority_rule_matches,all_checks_pass
0,DEVELOPMENT,1,1,51840,51840,1783665467531985400,1783665467531985400,1783667269391572100,1783667269391572100,33820,33820,True,True,True,True,True,True,True,True,True,True
1,CALIBRATION,51840,51840,72576,72576,1783667269391572100,1783667269391572100,1783667989690751200,1783667989690751200,13532,13532,True,True,True,True,True,True,True,True,True,True
2,VALIDATION,72576,72576,88127,88127,1783667989690751200,1783667989690751200,1783668525057534700,1783668525057534700,10198,10198,True,True,True,True,True,True,True,True,True,True
3,ENGINEERING_HOLDOUT,88127,88127,103678,103678,1783668525057534700,1783668525057534700,1783669066749750801,1783669066749750801,10133,10133,True,True,True,True,True,True,True,True,True,True


,partition_order,partition,generic_notebook_00_model_selection_access,notebook_04_event_definition_selection_access,notebook_04_protected_partition,construction_allowed_before_selection_freeze,construction_allowed_after_selection_freeze
0,1,DEVELOPMENT,True,True,False,True,True
1,2,CALIBRATION,True,True,False,True,True
2,3,VALIDATION,True,False,True,False,True
3,4,ENGINEERING_HOLDOUT,False,False,True,False,True


,count_name,expected_value,observed_value,passed
0,raw_trade_count,67683,67683,True
1,local_strict_all_alignment_rows,67683,67683,True
2,local_strict_matched_rows,67683,67683,True
3,local_strict_unmatched_rows,0,0,True
4,buy_trade_count,30596,30596,True
5,sell_trade_count,37087,37087,True


,field,expected_value,observed_value,passed
0,primary_event_time_field_for_notebook_04,trade_local_receipt_time_ns,trade_local_receipt_time_ns,True
1,secondary_exchange_trade_time_field,trade_exchange_trade_time_ms,trade_exchange_trade_time_ms,True
2,primary_ordering_field,trade_collector_sequence,trade_collector_sequence,True
3,trade_identity_field,trade_id,trade_id,True
4,aggressor_side_field,trade_aggressor_side,trade_aggressor_side,True
5,price_field,trade_price,trade_price,True
6,quantity_field,trade_quantity,trade_quantity,True
7,notional_field,trade_notional,trade_notional,True
8,matched_book_sequence_field,book_collector_sequence,book_collector_sequence,True
9,matched_book_local_time_field,book_local_receipt_time_ns,book_local_receipt_time_ns,True


,partition,notebook_00_authoritative_trade_count,notebook_03_inherited_trade_count,count_difference,matches_authoritative_count,authority
0,DEVELOPMENT,33820,33820,0,True,DIAGNOSTIC_ONLY
1,CALIBRATION,13532,13533,1,False,DIAGNOSTIC_ONLY
2,VALIDATION,10198,10197,-1,False,DIAGNOSTIC_ONLY
3,ENGINEERING_HOLDOUT,10133,10133,0,True,DIAGNOSTIC_ONLY


,detail,gate,gate_frame_name,severity,status
0,inside_spread_trade_count=67,price_book_inside_spread_trades,price_book_gate_frame,WARNING,FAIL
1,aggressor_book_inconsistency_count=818,price_book_aggressor_inconsistencies,price_book_gate_frame,WARNING,FAIL
2,outside_visible_book_count=30926,price_book_outside_visible_book_trades,price_book_gate_frame,WARNING,FAIL
3,wide_spread_state_match_count=95,price_book_wide_spread_state_matches,price_book_gate_frame,WARNING,FAIL
4,different_book_from_local_strict_count=1510,exchange_strict_differs_from_local_strict,exchange_strict_gate_frame,WARNING,FAIL
5,NO_LOADABLE_TRADE_LIKE_REFERENCE_SELECTED,v00_reference_selected,v00_reference_reconciliation_gate_frame,WARNING,FAIL


,gate,status,severity,detail
0,notebook_00_manifest_payload_verified,PASS,BLOCKING,Notebook 00 output-manifest envelope and payload hash verified.
1,split_contract_manifest_registration_verified,PASS,BLOCKING,path=D:\Clown Project\V0.1\config\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__00_V01_RUN_CONTRACT__split_contract.json; file_sha256=103758196809...
2,split_contract_payload_hash_verified,PASS,BLOCKING,payload_sha256=3b7e8d46f43d8a4dd7b7a1f3d9a61c2c92d3f3643042a8b01968ddc7e0fb4624
3,split_boundary_table_matches_frozen_declaration,PASS,BLOCKING,verified_partitions=4
4,notebook_04_selection_access_frozen,PASS,BLOCKING,"selection_partitions=DEVELOPMENT,CALIBRATION"
5,notebook_04_protected_partitions_frozen,PASS,BLOCKING,"protected_partitions=VALIDATION,ENGINEERING_HOLDOUT"
6,notebook_03_manifest_semantics_verified,PASS,BLOCKING,"Identity, operating mode, alignment authority, terminal status, and authorization verified."
7,notebook_03_matched_table_manifest_verified,PASS,BLOCKING,rows=67683; sha256=a18f66ca6a97a66b8f42a4d61b2aa34d26bcee5ae5a8d2054e576c88beede7a3
8,notebook_03_handoff_semantics_verified,PASS,BLOCKING,Notebook 03-to-04 handoff identity and authorization verified.
9,notebook_03_handoff_counts_verified,PASS,BLOCKING,verified_counts=6


,source_run_prefix,v0_1_run_id,operating_mode,split_contract_file_sha256,split_contract_payload_sha256,matched_table_path,matched_table_rows,matched_table_sha256,notebook_03_terminal_status,notebook_03_warning_count,inherited_partition_mismatch_count,notebook_04_authorized
0,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,ENGINEERING_REPRODUCTION_MODE,1037581968095b5e262429fd86004d1abf3e5b691b330e7a4aa8e89f590d09fe,3b7e8d46f43d8a4dd7b7a1f3d9a61c2c92d3f3643042a8b01968ddc7e0fb4624,D:\Clown Project\V0.1\data\processed\trade_book_alignment\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__local_stri...,67683,a18f66ca6a97a66b8f42a4d61b2aa34d26bcee5ae5a8d2054e576c88beede7a3,PASS_WITH_ALIGNMENT_WARNINGS,6,2,True


{'status': 'PASS',
 'notebook_04_authorized': True,
 'authoritative_split_contract_verified': True,
 'authoritative_matched_table_verified': True,
 'partition_summary_authority': 'DIAGNOSTIC_ONLY',
 'next_action': 'Load the matched LOCAL_STRICT table from disk, canonicalize required fields, and independently reconstruct authoritative trade and book partitions.'}

In [7]:
# ============================================================
# 04_EVENT_STREAM_CONSTRUCTION
# Cell 05 — Load the LOCAL_STRICT matched table, preserve
#           inherited Notebook 03 partitions, and reconstruct
#           authoritative trade/book partitions from Notebook 00
# ============================================================


# ------------------------------------------------------------
# Input loading
# ------------------------------------------------------------

def read_matched_alignment(path: Path) -> pd.DataFrame:
    """Load the verified Notebook 03 matched table without filtering rows."""
    path = require_file(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        header = pd.read_csv(path, nrows=0).columns.tolist()

        string_columns = {
            column: "string"
            for column in (
                "alignment_policy",
                "trade_partition",
                "book_partition",
                "matched_book_partition",
                "trade_price",
                "trade_quantity",
                "trade_notional",
            )
            if column in header
        }

        return pd.read_csv(
            path,
            low_memory=False,
            dtype=string_columns,
        )

    if suffix == ".parquet":
        return pd.read_parquet(path)

    if suffix == ".feather":
        return pd.read_feather(path)

    raise ValueError(
        "Unsupported matched-alignment format: "
        f"{path.suffix!r}"
    )


require(
    sha256_file(NOTEBOOK_03_MATCHED_ALIGNMENT_PATH_RESOLVED)
    == EXPECTED_NOTEBOOK_03_MATCHED_ALIGNMENT_FILE_SHA256,
    "The matched-table hash changed after upstream verification.",
)

matched_trade_book_raw = read_matched_alignment(
    NOTEBOOK_03_MATCHED_ALIGNMENT_PATH_RESOLVED
)

require_unique_columns(
    matched_trade_book_raw,
    "matched_trade_book_raw",
)

require(
    len(matched_trade_book_raw) == EXPECTED_MATCHED_TRADE_ROWS,
    (
        "Matched-table row count mismatch: "
        f"observed={len(matched_trade_book_raw):,}, "
        f"expected={EXPECTED_MATCHED_TRADE_ROWS:,}."
    ),
)

require(
    matched_trade_book_raw.shape[1] == 148,
    (
        "Matched-table column count mismatch: "
        f"observed={matched_trade_book_raw.shape[1]}, "
        "expected=148."
    ),
)


# ------------------------------------------------------------
# Required schema
# ------------------------------------------------------------

MATCHED_TABLE_REQUIRED_COLUMNS = (
    "source_run_prefix",
    "v0_1_run_id",
    "alignment_policy",
    "trade_id",
    "trade_collector_sequence",
    "trade_local_receipt_time_ns",
    "trade_exchange_trade_time_ms",
    "trade_price",
    "trade_quantity",
    "trade_notional",
    "trade_aggressor_side",
    "trade_partition",
    "is_local_strict_match",
    "book_state_id",
    "book_collector_sequence",
    "book_local_receipt_time_ns",
    "book_partition",
    "matched_book_partition",
    "local_observation_lag_ns",
    "local_strict_sequence_condition",
    "local_strict_time_condition",
)

require_columns(
    matched_trade_book_raw,
    MATCHED_TABLE_REQUIRED_COLUMNS,
    "matched_trade_book_raw",
)


# ------------------------------------------------------------
# Parsing helpers
# ------------------------------------------------------------

def parse_required_int64(
    series: pd.Series,
    field_name: str,
) -> pd.Series:
    """Parse a required integral field without silently truncating values."""
    numeric = pd.to_numeric(series, errors="raise")

    require(
        numeric.notna().all(),
        f"{field_name} contains missing values.",
    )

    if pd.api.types.is_integer_dtype(numeric.dtype):
        return numeric.astype("int64")

    numeric_float = numeric.astype("float64")

    require(
        np.isfinite(numeric_float.to_numpy()).all(),
        f"{field_name} contains non-finite values.",
    )

    require(
        numeric_float.eq(np.floor(numeric_float)).all(),
        f"{field_name} contains non-integral values.",
    )

    int64_info = np.iinfo(np.int64)

    require(
        numeric_float.ge(int64_info.min).all()
        and numeric_float.le(int64_info.max).all(),
        f"{field_name} lies outside the int64 range.",
    )

    return numeric.astype("int64")


def parse_required_float64(
    series: pd.Series,
    field_name: str,
) -> pd.Series:
    """Parse a required finite floating-point field."""
    numeric = pd.to_numeric(
        series,
        errors="raise",
    ).astype("float64")

    require(
        numeric.notna().all(),
        f"{field_name} contains missing values.",
    )

    require(
        np.isfinite(numeric.to_numpy()).all(),
        f"{field_name} contains non-finite values.",
    )

    return numeric


def parse_required_bool(
    series: pd.Series,
    field_name: str,
) -> pd.Series:
    """Parse strict Boolean representations from CSV or binary tables."""
    if pd.api.types.is_bool_dtype(series.dtype):
        require(
            series.notna().all(),
            f"{field_name} contains missing values.",
        )
        return series.astype(bool)

    normalized = (
        series.astype("string")
        .str.strip()
        .str.casefold()
    )

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    invalid_mask = ~normalized.isin(mapping)

    invalid_values = sorted(
        normalized.loc[invalid_mask]
        .dropna()
        .unique()
        .tolist()
    )

    require(
        not invalid_mask.any(),
        (
            f"{field_name} contains invalid Boolean values: "
            f"{invalid_values}"
        ),
    )

    return normalized.map(mapping).astype(bool)


def normalize_partition_labels(
    series: pd.Series,
) -> pd.Series:
    """Normalize inherited partition labels for comparison only."""
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .replace(
            {
                "HOLDOUT": "ENGINEERING_HOLDOUT",
                "FINAL_HOLDOUT": "ENGINEERING_HOLDOUT",
            }
        )
    )


def map_sequences_to_partitions(
    sequence: pd.Series,
    field_name: str,
) -> pd.Series:
    """Map each collector sequence to exactly one Notebook 00 interval."""
    sequence_int = parse_required_int64(
        sequence,
        field_name,
    )

    mapped = pd.Series(
        pd.NA,
        index=sequence_int.index,
        dtype="string",
    )

    assignment_count = pd.Series(
        0,
        index=sequence_int.index,
        dtype="int8",
    )

    for spec in PARTITION_SPECS:
        mask = (
            sequence_int.ge(
                spec.collector_sequence_start
            )
            & sequence_int.lt(
                spec.collector_sequence_end_exclusive
            )
        )

        mapped.loc[mask] = spec.name
        assignment_count.loc[mask] += 1

    bad_assignment_mask = assignment_count.ne(1)

    bad_sequences = (
        sequence_int.loc[bad_assignment_mask]
        .drop_duplicates()
        .sort_values()
        .head(20)
        .tolist()
    )

    require(
        not bad_assignment_mask.any(),
        (
            f"{field_name} must map to exactly one frozen "
            f"partition. Example invalid sequences: {bad_sequences}"
        ),
    )

    partition_dtype = pd.CategoricalDtype(
        categories=list(PARTITION_ORDER),
        ordered=True,
    )

    return mapped.astype(partition_dtype)


def receipt_time_inside_assigned_partition(
    receipt_time_ns: pd.Series,
    assigned_partition: pd.Series,
    field_name: str,
) -> pd.Series:
    """Check receipt times against their assigned frozen intervals."""
    receipt = parse_required_int64(
        receipt_time_ns,
        field_name,
    )

    result = pd.Series(
        False,
        index=receipt.index,
        dtype=bool,
    )

    partition_as_string = assigned_partition.astype(
        "string"
    )

    for spec in PARTITION_SPECS:
        partition_mask = partition_as_string.eq(
            spec.name
        )

        result.loc[partition_mask] = (
            receipt.loc[partition_mask]
            .ge(spec.receipt_time_start_ns)
            & receipt.loc[partition_mask]
            .lt(spec.receipt_time_end_exclusive_ns)
        )

    return result


# ------------------------------------------------------------
# Preserve inherited Notebook 03 fields
# ------------------------------------------------------------

partition_audited_trade_book = (
    matched_trade_book_raw.copy(deep=True)
)

partition_audited_trade_book.insert(
    0,
    "notebook_04_source_row_number",
    np.arange(
        1,
        len(partition_audited_trade_book) + 1,
        dtype=np.int64,
    ),
)

partition_audited_trade_book = (
    partition_audited_trade_book.rename(
        columns={
            "trade_partition": (
                INHERITED_TRADE_PARTITION_FIELD
            ),
            "book_partition": (
                "book_partition_notebook_03"
            ),
            "matched_book_partition": (
                INHERITED_BOOK_PARTITION_FIELD
            ),
            "partition_crossing_match_flag": (
                "partition_crossing_match_flag_notebook_03"
            ),
        }
    )
)

require_columns(
    partition_audited_trade_book,
    (
        INHERITED_TRADE_PARTITION_FIELD,
        "book_partition_notebook_03",
        INHERITED_BOOK_PARTITION_FIELD,
    ),
    "partition_audited_trade_book",
)


# ------------------------------------------------------------
# Canonical types and identity checks
# ------------------------------------------------------------

for integer_field in (
    "trade_id",
    "trade_collector_sequence",
    "trade_local_receipt_time_ns",
    "trade_exchange_trade_time_ms",
    "book_state_id",
    "book_collector_sequence",
    "book_local_receipt_time_ns",
    "local_observation_lag_ns",
):
    partition_audited_trade_book[integer_field] = (
        parse_required_int64(
            partition_audited_trade_book[
                integer_field
            ],
            integer_field,
        )
    )

for numeric_field in (
    "trade_price",
    "trade_quantity",
    "trade_notional",
):
    source_text_field = (
        f"{numeric_field}_source_text"
    )

    partition_audited_trade_book[
        source_text_field
    ] = (
        partition_audited_trade_book[
            numeric_field
        ]
        .astype("string")
        .str.strip()
    )

    partition_audited_trade_book[
        numeric_field
    ] = parse_required_float64(
        partition_audited_trade_book[
            numeric_field
        ],
        numeric_field,
    )

for boolean_field in (
    "is_local_strict_match",
    "local_strict_sequence_condition",
    "local_strict_time_condition",
):
    partition_audited_trade_book[
        boolean_field
    ] = parse_required_bool(
        partition_audited_trade_book[
            boolean_field
        ],
        boolean_field,
    )

partition_audited_trade_book[
    "trade_aggressor_side"
] = (
    partition_audited_trade_book[
        "trade_aggressor_side"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
)

partition_audited_trade_book[
    "alignment_policy"
] = (
    partition_audited_trade_book[
        "alignment_policy"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
)

require(
    partition_audited_trade_book[
        "source_run_prefix"
    ]
    .astype("string")
    .eq(SOURCE_RUN_PREFIX)
    .all(),
    (
        "Matched table contains an unexpected "
        "source_run_prefix."
    ),
)

require(
    partition_audited_trade_book[
        "v0_1_run_id"
    ]
    .astype("string")
    .eq(V01_RUN_ID)
    .all(),
    (
        "Matched table contains an unexpected "
        "v0_1_run_id."
    ),
)

require(
    partition_audited_trade_book[
        "alignment_policy"
    ]
    .eq(PRIMARY_ALIGNMENT_POLICY)
    .all(),
    (
        "Matched table contains a non-LOCAL_STRICT "
        "alignment policy."
    ),
)

require(
    partition_audited_trade_book[
        "is_local_strict_match"
    ].all(),
    (
        "Matched-only input contains at least one "
        "unmatched row."
    ),
)

require(
    partition_audited_trade_book[
        "local_strict_sequence_condition"
    ].all(),
    (
        "Matched input contains a failed LOCAL_STRICT "
        "sequence condition."
    ),
)

require(
    partition_audited_trade_book[
        "local_strict_time_condition"
    ].all(),
    (
        "Matched input contains a failed LOCAL_STRICT "
        "time condition."
    ),
)

require(
    partition_audited_trade_book[
        "trade_aggressor_side"
    ]
    .isin(VALID_AGGRESSOR_SIDES)
    .all(),
    "Matched table contains an invalid aggressor side.",
)

require(
    partition_audited_trade_book[
        "trade_id"
    ].is_unique,
    "Matched table contains duplicate trade IDs.",
)

require(
    partition_audited_trade_book[
        "trade_collector_sequence"
    ].is_unique,
    (
        "Matched table contains duplicate trade "
        "collector sequences."
    ),
)

require(
    partition_audited_trade_book[
        "trade_quantity"
    ].gt(0).all(),
    "Matched table contains nonpositive trade quantities.",
)

require(
    partition_audited_trade_book[
        "trade_price"
    ].gt(0).all(),
    "Matched table contains nonpositive trade prices.",
)

require(
    partition_audited_trade_book[
        "trade_notional"
    ].gt(0).all(),
    "Matched table contains nonpositive trade notionals.",
)


# ------------------------------------------------------------
# Reconstruct authoritative trade and book partitions
# ------------------------------------------------------------

partition_audited_trade_book[
    AUTHORITATIVE_TRADE_PARTITION_FIELD
] = map_sequences_to_partitions(
    partition_audited_trade_book[
        "trade_collector_sequence"
    ],
    "trade_collector_sequence",
)

partition_audited_trade_book[
    AUTHORITATIVE_BOOK_PARTITION_FIELD
] = map_sequences_to_partitions(
    partition_audited_trade_book[
        "book_collector_sequence"
    ],
    "book_collector_sequence",
)

partition_audited_trade_book[
    EVENT_PARTITION_FIELD
] = partition_audited_trade_book[
    AUTHORITATIVE_TRADE_PARTITION_FIELD
]

partition_audited_trade_book[
    MATCHED_BOOK_PARTITION_FIELD
] = partition_audited_trade_book[
    AUTHORITATIVE_BOOK_PARTITION_FIELD
]

partition_audited_trade_book[
    CROSS_PARTITION_HISTORY_FIELD
] = (
    partition_audited_trade_book[
        EVENT_PARTITION_FIELD
    ]
    .astype("string")
    .ne(
        partition_audited_trade_book[
            MATCHED_BOOK_PARTITION_FIELD
        ].astype("string")
    )
)

partition_order_code = {
    partition: order
    for order, partition in enumerate(
        PARTITION_ORDER,
        start=1,
    )
}

partition_audited_trade_book[
    "event_partition_order"
] = (
    partition_audited_trade_book[
        EVENT_PARTITION_FIELD
    ]
    .astype("string")
    .map(partition_order_code)
    .astype("int8")
)

partition_audited_trade_book[
    "matched_book_partition_order"
] = (
    partition_audited_trade_book[
        MATCHED_BOOK_PARTITION_FIELD
    ]
    .astype("string")
    .map(partition_order_code)
    .astype("int8")
)

partition_audited_trade_book[
    "trade_receipt_inside_partition_flag"
] = receipt_time_inside_assigned_partition(
    partition_audited_trade_book[
        "trade_local_receipt_time_ns"
    ],
    partition_audited_trade_book[
        AUTHORITATIVE_TRADE_PARTITION_FIELD
    ],
    "trade_local_receipt_time_ns",
)

partition_audited_trade_book[
    "book_receipt_inside_partition_flag"
] = receipt_time_inside_assigned_partition(
    partition_audited_trade_book[
        "book_local_receipt_time_ns"
    ],
    partition_audited_trade_book[
        AUTHORITATIVE_BOOK_PARTITION_FIELD
    ],
    "book_local_receipt_time_ns",
)


# ------------------------------------------------------------
# Compare inherited and authoritative partitions
# ------------------------------------------------------------

partition_audited_trade_book[
    "trade_partition_notebook_03_normalized"
] = normalize_partition_labels(
    partition_audited_trade_book[
        INHERITED_TRADE_PARTITION_FIELD
    ]
)

partition_audited_trade_book[
    "matched_book_partition_notebook_03_normalized"
] = normalize_partition_labels(
    partition_audited_trade_book[
        INHERITED_BOOK_PARTITION_FIELD
    ]
)

partition_audited_trade_book[
    "trade_partition_mismatch_flag"
] = (
    partition_audited_trade_book[
        "trade_partition_notebook_03_normalized"
    ]
    .ne(
        partition_audited_trade_book[
            AUTHORITATIVE_TRADE_PARTITION_FIELD
        ].astype("string")
    )
)

partition_audited_trade_book[
    "matched_book_partition_mismatch_flag"
] = (
    partition_audited_trade_book[
        "matched_book_partition_notebook_03_normalized"
    ]
    .ne(
        partition_audited_trade_book[
            AUTHORITATIVE_BOOK_PARTITION_FIELD
        ].astype("string")
    )
)

trade_partition_mismatch_audit = (
    partition_audited_trade_book.loc[
        partition_audited_trade_book[
            "trade_partition_mismatch_flag"
        ],
        [
            "notebook_04_source_row_number",
            "trade_id",
            "trade_collector_sequence",
            "trade_local_receipt_time_ns",
            "trade_aggressor_side",
            INHERITED_TRADE_PARTITION_FIELD,
            AUTHORITATIVE_TRADE_PARTITION_FIELD,
            "book_state_id",
            "book_collector_sequence",
            INHERITED_BOOK_PARTITION_FIELD,
            AUTHORITATIVE_BOOK_PARTITION_FIELD,
        ],
    ]
    .sort_values(
        "trade_collector_sequence",
        kind="stable",
    )
    .reset_index(drop=True)
)

matched_book_partition_mismatch_audit = (
    partition_audited_trade_book.loc[
        partition_audited_trade_book[
            "matched_book_partition_mismatch_flag"
        ],
        [
            "notebook_04_source_row_number",
            "trade_id",
            "trade_collector_sequence",
            "book_state_id",
            "book_collector_sequence",
            "book_local_receipt_time_ns",
            INHERITED_BOOK_PARTITION_FIELD,
            AUTHORITATIVE_BOOK_PARTITION_FIELD,
            AUTHORITATIVE_TRADE_PARTITION_FIELD,
            CROSS_PARTITION_HISTORY_FIELD,
        ],
    ]
    .sort_values(
        [
            "book_collector_sequence",
            "trade_collector_sequence",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

cross_partition_history_audit = (
    partition_audited_trade_book.loc[
        partition_audited_trade_book[
            CROSS_PARTITION_HISTORY_FIELD
        ],
        [
            "notebook_04_source_row_number",
            "trade_id",
            "trade_collector_sequence",
            "trade_local_receipt_time_ns",
            AUTHORITATIVE_TRADE_PARTITION_FIELD,
            "book_state_id",
            "book_collector_sequence",
            "book_local_receipt_time_ns",
            AUTHORITATIVE_BOOK_PARTITION_FIELD,
            "local_observation_lag_ns",
        ],
    ]
    .sort_values(
        "trade_collector_sequence",
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Authoritative partition counts
# ------------------------------------------------------------

authoritative_partition_observed = (
    partition_audited_trade_book.groupby(
        AUTHORITATIVE_TRADE_PARTITION_FIELD,
        observed=False,
        sort=False,
    )
    .agg(
        observed_trade_count=(
            "trade_id",
            "size",
        ),
        observed_buy_count=(
            "trade_aggressor_side",
            lambda values: int(
                values.eq("BUY").sum()
            ),
        ),
        observed_sell_count=(
            "trade_aggressor_side",
            lambda values: int(
                values.eq("SELL").sum()
            ),
        ),
        observed_sequence_min=(
            "trade_collector_sequence",
            "min",
        ),
        observed_sequence_max=(
            "trade_collector_sequence",
            "max",
        ),
        observed_receipt_time_min_ns=(
            "trade_local_receipt_time_ns",
            "min",
        ),
        observed_receipt_time_max_ns=(
            "trade_local_receipt_time_ns",
            "max",
        ),
    )
    .reset_index()
    .rename(
        columns={
            AUTHORITATIVE_TRADE_PARTITION_FIELD: (
                "partition"
            )
        }
    )
)

authoritative_partition_observed[
    "partition"
] = authoritative_partition_observed[
    "partition"
].astype("string")

expected_partition_frame = pd.DataFrame.from_records(
    [
        {
            "partition_order": spec.order,
            "partition": spec.name,
            "expected_sequence_start": (
                spec.collector_sequence_start
            ),
            "expected_sequence_end_exclusive": (
                spec.collector_sequence_end_exclusive
            ),
            "expected_trade_count": (
                spec.expected_trade_count
            ),
            "expected_buy_count": (
                spec.expected_buy_count
            ),
            "expected_sell_count": (
                spec.expected_sell_count
            ),
            "expected_receipt_start_ns": (
                spec.receipt_time_start_ns
            ),
            "expected_receipt_end_exclusive_ns": (
                spec.receipt_time_end_exclusive_ns
            ),
        }
        for spec in PARTITION_SPECS
    ]
)

authoritative_partition_count_audit = (
    expected_partition_frame.merge(
        authoritative_partition_observed,
        on="partition",
        how="left",
        validate="one_to_one",
    )
)

require(
    authoritative_partition_count_audit[
        "observed_trade_count"
    ].notna().all(),
    (
        "At least one authoritative partition has no "
        "observed trade-count record."
    ),
)

authoritative_partition_count_audit[
    "trade_count_pass"
] = (
    authoritative_partition_count_audit[
        "observed_trade_count"
    ]
    .astype("int64")
    .eq(
        authoritative_partition_count_audit[
            "expected_trade_count"
        ]
    )
)

authoritative_partition_count_audit[
    "buy_count_pass"
] = (
    authoritative_partition_count_audit[
        "observed_buy_count"
    ]
    .astype("int64")
    .eq(
        authoritative_partition_count_audit[
            "expected_buy_count"
        ]
    )
)

authoritative_partition_count_audit[
    "sell_count_pass"
] = (
    authoritative_partition_count_audit[
        "observed_sell_count"
    ]
    .astype("int64")
    .eq(
        authoritative_partition_count_audit[
            "expected_sell_count"
        ]
    )
)

authoritative_partition_count_audit[
    "sequence_interval_pass"
] = (
    authoritative_partition_count_audit[
        "observed_sequence_min"
    ]
    .ge(
        authoritative_partition_count_audit[
            "expected_sequence_start"
        ]
    )
    & authoritative_partition_count_audit[
        "observed_sequence_max"
    ]
    .lt(
        authoritative_partition_count_audit[
            "expected_sequence_end_exclusive"
        ]
    )
)

authoritative_partition_count_audit[
    "receipt_interval_pass"
] = (
    authoritative_partition_count_audit[
        "observed_receipt_time_min_ns"
    ]
    .ge(
        authoritative_partition_count_audit[
            "expected_receipt_start_ns"
        ]
    )
    & authoritative_partition_count_audit[
        "observed_receipt_time_max_ns"
    ]
    .lt(
        authoritative_partition_count_audit[
            "expected_receipt_end_exclusive_ns"
        ]
    )
)

authoritative_partition_count_audit[
    "all_checks_pass"
] = (
    authoritative_partition_count_audit[
        [
            "trade_count_pass",
            "buy_count_pass",
            "sell_count_pass",
            "sequence_interval_pass",
            "receipt_interval_pass",
        ]
    ].all(axis=1)
)


# ------------------------------------------------------------
# Blocking invariants
# ------------------------------------------------------------

sequence_causality_pass = bool(
    partition_audited_trade_book[
        "book_collector_sequence"
    ]
    .lt(
        partition_audited_trade_book[
            "trade_collector_sequence"
        ]
    )
    .all()
)

local_time_causality_pass = bool(
    partition_audited_trade_book[
        "book_local_receipt_time_ns"
    ]
    .le(
        partition_audited_trade_book[
            "trade_local_receipt_time_ns"
        ]
    )
    .all()
)

nonnegative_lag_pass = bool(
    partition_audited_trade_book[
        "local_observation_lag_ns"
    ]
    .ge(0)
    .all()
)

matched_book_not_from_future_partition_pass = bool(
    partition_audited_trade_book[
        "matched_book_partition_order"
    ]
    .le(
        partition_audited_trade_book[
            "event_partition_order"
        ]
    )
    .all()
)

trade_receipt_interval_pass = bool(
    partition_audited_trade_book[
        "trade_receipt_inside_partition_flag"
    ].all()
)

book_receipt_interval_pass = bool(
    partition_audited_trade_book[
        "book_receipt_inside_partition_flag"
    ].all()
)

authoritative_counts_pass = bool(
    authoritative_partition_count_audit[
        "all_checks_pass"
    ].all()
)

duplicate_trade_id_count = int(
    partition_audited_trade_book[
        "trade_id"
    ]
    .duplicated()
    .sum()
)

duplicate_trade_sequence_count = int(
    partition_audited_trade_book[
        "trade_collector_sequence"
    ]
    .duplicated()
    .sum()
)

trade_receipt_outside_count = int(
    (
        ~partition_audited_trade_book[
            "trade_receipt_inside_partition_flag"
        ]
    ).sum()
)

book_receipt_outside_count = int(
    (
        ~partition_audited_trade_book[
            "book_receipt_inside_partition_flag"
        ]
    ).sum()
)

future_book_partition_count = int(
    (
        partition_audited_trade_book[
            "matched_book_partition_order"
        ]
        > partition_audited_trade_book[
            "event_partition_order"
        ]
    ).sum()
)


# ------------------------------------------------------------
# Gate ledger
# ------------------------------------------------------------

partition_reconstruction_gates = [
    make_gate(
        gate="matched_table_file_hash_reverified",
        passed=True,
        severity="BLOCKING",
        detail=(
            "sha256="
            f"{EXPECTED_NOTEBOOK_03_MATCHED_ALIGNMENT_FILE_SHA256}"
        ),
    ),
    make_gate(
        gate="matched_table_row_count",
        passed=(
            len(partition_audited_trade_book)
            == EXPECTED_MATCHED_TRADE_ROWS
        ),
        severity="BLOCKING",
        detail=(
            f"observed={len(partition_audited_trade_book):,}; "
            f"expected={EXPECTED_MATCHED_TRADE_ROWS:,}"
        ),
    ),
    make_gate(
        gate="trade_identity_unique",
        passed=(duplicate_trade_id_count == 0),
        severity="BLOCKING",
        detail=(
            "duplicate_trade_id_count="
            f"{duplicate_trade_id_count}"
        ),
    ),
    make_gate(
        gate="trade_collector_sequence_unique",
        passed=(duplicate_trade_sequence_count == 0),
        severity="BLOCKING",
        detail=(
            "duplicate_trade_sequence_count="
            f"{duplicate_trade_sequence_count}"
        ),
    ),
    make_gate(
        gate="local_strict_sequence_causality",
        passed=sequence_causality_pass,
        severity="BLOCKING",
        detail=(
            "Requires book_collector_sequence "
            "< trade_collector_sequence for every row."
        ),
    ),
    make_gate(
        gate="local_strict_receipt_time_causality",
        passed=local_time_causality_pass,
        severity="BLOCKING",
        detail=(
            "Requires book_local_receipt_time_ns "
            "<= trade_local_receipt_time_ns for every row."
        ),
    ),
    make_gate(
        gate="local_observation_lag_nonnegative",
        passed=nonnegative_lag_pass,
        severity="BLOCKING",
        detail=(
            "Requires local_observation_lag_ns >= 0."
        ),
    ),
    make_gate(
        gate="authoritative_partition_counts_and_intervals",
        passed=authoritative_counts_pass,
        severity="BLOCKING",
        detail=(
            "verified_partition_count="
            f"{len(authoritative_partition_count_audit)}"
        ),
    ),
    make_gate(
        gate=(
            "trade_receipts_inside_authoritative_"
            "partition_intervals"
        ),
        passed=trade_receipt_interval_pass,
        severity="BLOCKING",
        detail=(
            "outside_interval_count="
            f"{trade_receipt_outside_count}"
        ),
    ),
    make_gate(
        gate=(
            "book_receipts_inside_authoritative_"
            "partition_intervals"
        ),
        passed=book_receipt_interval_pass,
        severity="BLOCKING",
        detail=(
            "outside_interval_count="
            f"{book_receipt_outside_count}"
        ),
    ),
    make_gate(
        gate="matched_book_never_from_future_partition",
        passed=matched_book_not_from_future_partition_pass,
        severity="BLOCKING",
        detail=(
            "future_partition_count="
            f"{future_book_partition_count}"
        ),
    ),
    make_gate(
        gate=(
            "notebook_03_trade_partition_"
            "matches_authority"
        ),
        passed=(
            len(trade_partition_mismatch_audit)
            == 0
        ),
        severity="WARNING",
        detail=(
            "mismatch_count="
            f"{len(trade_partition_mismatch_audit)}"
        ),
    ),
    make_gate(
        gate=(
            "notebook_03_matched_book_partition_"
            "matches_authority"
        ),
        passed=(
            len(
                matched_book_partition_mismatch_audit
            )
            == 0
        ),
        severity="WARNING",
        detail=(
            "mismatch_count="
            f"{len(matched_book_partition_mismatch_audit)}"
        ),
    ),
    make_gate(
        gate="cross_partition_history_absent",
        passed=(
            len(cross_partition_history_audit)
            == 0
        ),
        severity="WARNING",
        detail=(
            "causal_history_carry_count="
            f"{len(cross_partition_history_audit)}"
        ),
    ),
]

partition_reconstruction_gate_frame = (
    gate_results_to_frame(
        partition_reconstruction_gates
    )
)

fail_if_blocking_gate_failed(
    partition_reconstruction_gate_frame
)


# ------------------------------------------------------------
# Freeze the construction frame without inherited partition
# columns. The full audited frame remains available separately.
# ------------------------------------------------------------

INHERITED_PARTITION_COLUMNS = (
    INHERITED_TRADE_PARTITION_FIELD,
    "book_partition_notebook_03",
    INHERITED_BOOK_PARTITION_FIELD,
    "trade_partition_notebook_03_normalized",
    "matched_book_partition_notebook_03_normalized",
    "partition_crossing_match_flag_notebook_03",
)

columns_to_drop_from_construction = [
    column
    for column in INHERITED_PARTITION_COLUMNS
    if column in partition_audited_trade_book.columns
]

canonical_trade_book = (
    partition_audited_trade_book
    .drop(
        columns=columns_to_drop_from_construction
    )
    .sort_values(
        [
            "trade_collector_sequence",
            "trade_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    not any(
        column in canonical_trade_book.columns
        for column in INHERITED_PARTITION_COLUMNS
    ),
    (
        "Inherited partition fields leaked into "
        "canonical_trade_book."
    ),
)

require(
    canonical_trade_book[
        EVENT_PARTITION_FIELD
    ]
    .astype("string")
    .eq(
        canonical_trade_book[
            AUTHORITATIVE_TRADE_PARTITION_FIELD
        ].astype("string")
    )
    .all(),
    (
        "event_partition differs from the authoritative "
        "trade partition."
    ),
)

require(
    canonical_trade_book[
        MATCHED_BOOK_PARTITION_FIELD
    ]
    .astype("string")
    .eq(
        canonical_trade_book[
            AUTHORITATIVE_BOOK_PARTITION_FIELD
        ].astype("string")
    )
    .all(),
    (
        "matched_book_partition differs from the "
        "authoritative book partition."
    ),
)

require(
    len(canonical_trade_book)
    == len(partition_audited_trade_book),
    (
        "Canonicalization changed the matched-table "
        "row count."
    ),
)


# ------------------------------------------------------------
# Compact outputs
# ------------------------------------------------------------

partition_reconstruction_summary = pd.DataFrame(
    [
        {
            "input_rows": int(
                len(matched_trade_book_raw)
            ),
            "canonical_rows": int(
                len(canonical_trade_book)
            ),
            "input_columns": int(
                matched_trade_book_raw.shape[1]
            ),
            "canonical_columns": int(
                canonical_trade_book.shape[1]
            ),
            "trade_partition_mismatch_rows": int(
                len(trade_partition_mismatch_audit)
            ),
            "matched_book_partition_mismatch_rows": int(
                len(
                    matched_book_partition_mismatch_audit
                )
            ),
            "cross_partition_history_rows": int(
                len(cross_partition_history_audit)
            ),
            "minimum_trade_sequence": int(
                canonical_trade_book[
                    "trade_collector_sequence"
                ].min()
            ),
            "maximum_trade_sequence": int(
                canonical_trade_book[
                    "trade_collector_sequence"
                ].max()
            ),
            "minimum_book_sequence": int(
                canonical_trade_book[
                    "book_collector_sequence"
                ].min()
            ),
            "maximum_book_sequence": int(
                canonical_trade_book[
                    "book_collector_sequence"
                ].max()
            ),
            "partition_authority": (
                "NOTEBOOK_00_COLLECTOR_SEQUENCE_INTERVALS"
            ),
            "alignment_authority": (
                PRIMARY_ALIGNMENT_POLICY
            ),
        }
    ]
)

display(authoritative_partition_count_audit)
display(trade_partition_mismatch_audit)
display(matched_book_partition_mismatch_audit)
display(cross_partition_history_audit)
display(partition_reconstruction_gate_frame)
display(partition_reconstruction_summary)

{
    "status": "PASS",
    "canonical_rows": int(
        len(canonical_trade_book)
    ),
    "authoritative_partition_counts_verified": (
        authoritative_counts_pass
    ),
    "trade_partition_mismatch_rows": int(
        len(trade_partition_mismatch_audit)
    ),
    "matched_book_partition_mismatch_rows": int(
        len(matched_book_partition_mismatch_audit)
    ),
    "cross_partition_history_rows": int(
        len(cross_partition_history_audit)
    ),
    "construction_partition_field": (
        EVENT_PARTITION_FIELD
    ),
    "next_action": (
        "Audit exact local-time ties, exchange-millisecond "
        "fragmentation, and candidate event representations "
        "using canonical_trade_book."
    ),
}

,partition_order,partition,expected_sequence_start,expected_sequence_end_exclusive,expected_trade_count,expected_buy_count,expected_sell_count,expected_receipt_start_ns,expected_receipt_end_exclusive_ns,observed_trade_count,observed_buy_count,observed_sell_count,observed_sequence_min,observed_sequence_max,observed_receipt_time_min_ns,observed_receipt_time_max_ns,trade_count_pass,buy_count_pass,sell_count_pass,sequence_interval_pass,receipt_interval_pass,all_checks_pass
0,1,DEVELOPMENT,1,51840,33820,15194,18626,1783665467531985400,1783667269391572100,33820,15194,18626,14,51838,1783665468766951600,1783667269232205000,True,True,True,True,True,True
1,2,CALIBRATION,51840,72576,13532,7267,6265,1783667269391572100,1783667989690751200,13532,7267,6265,51852,72571,1783667270546982000,1783667989349685300,True,True,True,True,True,True
2,3,VALIDATION,72576,88127,10198,5774,4424,1783667989690751200,1783668525057534700,10198,5774,4424,72576,88125,1783667989690751200,1783668524924691300,True,True,True,True,True,True
3,4,ENGINEERING_HOLDOUT,88127,103678,10133,2361,7772,1783668525057534700,1783669066749750801,10133,2361,7772,88129,103676,1783668525205920700,1783669066707304900,True,True,True,True,True,True


,notebook_04_source_row_number,trade_id,trade_collector_sequence,trade_local_receipt_time_ns,trade_aggressor_side,trade_partition_notebook_03,trade_partition_authoritative,book_state_id,book_collector_sequence,matched_book_partition_notebook_03,matched_book_partition_authoritative
0,47353,6494643393,72576,1783667989690751200,SELL,CALIBRATION,VALIDATION,25213,72575,CALIBRATION,CALIBRATION


,notebook_04_source_row_number,trade_id,trade_collector_sequence,book_state_id,book_collector_sequence,book_local_receipt_time_ns,matched_book_partition_notebook_03,matched_book_partition_authoritative,trade_partition_authoritative,cross_partition_history_flag


,notebook_04_source_row_number,trade_id,trade_collector_sequence,trade_local_receipt_time_ns,trade_partition_authoritative,book_state_id,book_collector_sequence,book_local_receipt_time_ns,matched_book_partition_authoritative,local_observation_lag_ns
0,47353,6494643393,72576,1783667989690751200,VALIDATION,25213,72575,1783667989674233200,CALIBRATION,16518000


,gate,status,severity,detail
0,matched_table_file_hash_reverified,PASS,BLOCKING,sha256=a18f66ca6a97a66b8f42a4d61b2aa34d26bcee5ae5a8d2054e576c88beede7a3
1,matched_table_row_count,PASS,BLOCKING,"observed=67,683; expected=67,683"
2,trade_identity_unique,PASS,BLOCKING,duplicate_trade_id_count=0
3,trade_collector_sequence_unique,PASS,BLOCKING,duplicate_trade_sequence_count=0
4,local_strict_sequence_causality,PASS,BLOCKING,Requires book_collector_sequence < trade_collector_sequence for every row.
5,local_strict_receipt_time_causality,PASS,BLOCKING,Requires book_local_receipt_time_ns <= trade_local_receipt_time_ns for every row.
6,local_observation_lag_nonnegative,PASS,BLOCKING,Requires local_observation_lag_ns >= 0.
7,authoritative_partition_counts_and_intervals,PASS,BLOCKING,verified_partition_count=4
8,trade_receipts_inside_authoritative_partition_intervals,PASS,BLOCKING,outside_interval_count=0
9,book_receipts_inside_authoritative_partition_intervals,PASS,BLOCKING,outside_interval_count=0


,input_rows,canonical_rows,input_columns,canonical_columns,trade_partition_mismatch_rows,matched_book_partition_mismatch_rows,cross_partition_history_rows,minimum_trade_sequence,maximum_trade_sequence,minimum_book_sequence,maximum_book_sequence,partition_authority,alignment_authority
0,67683,67683,148,159,1,0,1,14,103676,13,103659,NOTEBOOK_00_COLLECTOR_SEQUENCE_INTERVALS,LOCAL_STRICT


{'status': 'PASS',
 'canonical_rows': 67683,
 'authoritative_partition_counts_verified': True,
 'trade_partition_mismatch_rows': 1,
 'matched_book_partition_mismatch_rows': 0,
 'cross_partition_history_rows': 1,
 'construction_partition_field': 'event_partition',
 'next_action': 'Audit exact local-time ties, exchange-millisecond fragmentation, and candidate event representations using canonical_trade_book.'}

In [11]:
# ============================================================
# 04_EVENT_STREAM_CONSTRUCTION
# Cell 06 — Timestamp fragmentation audit and selection-eligible
#           candidate event representations
#
# This cell constructs candidate event representations only for
# DEVELOPMENT and CALIBRATION. VALIDATION and ENGINEERING_HOLDOUT
# are summarized only for audit counts and cannot affect event-
# definition selection.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    "canonical_trade_book" in globals(),
    "canonical_trade_book does not exist. Run Cell 05 first.",
)

require_columns(
    canonical_trade_book,
    [
        "notebook_04_source_row_number",
        "trade_id",
        "trade_collector_sequence",
        "trade_local_receipt_time_ns",
        "trade_exchange_trade_time_ms",
        "trade_aggressor_side",
        "trade_price",
        "trade_quantity",
        "trade_notional",
        "book_state_id",
        "book_collector_sequence",
        "book_local_receipt_time_ns",
        "local_observation_lag_ns",
        EVENT_PARTITION_FIELD,
        MATCHED_BOOK_PARTITION_FIELD,
        AUTHORITATIVE_TRADE_PARTITION_FIELD,
        AUTHORITATIVE_BOOK_PARTITION_FIELD,
        CROSS_PARTITION_HISTORY_FIELD,
    ],
    "canonical_trade_book",
)

for forbidden_column in INHERITED_PARTITION_COLUMNS:
    require(
        forbidden_column not in canonical_trade_book.columns,
        (
            "Inherited Notebook 03 partition field leaked into "
            f"canonical_trade_book: {forbidden_column}"
        ),
    )

require(
    len(canonical_trade_book) == EXPECTED_MATCHED_TRADE_ROWS,
    (
        "canonical_trade_book row count changed unexpectedly: "
        f"observed={len(canonical_trade_book):,}, "
        f"expected={EXPECTED_MATCHED_TRADE_ROWS:,}."
    ),
)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def ordered_partition_string(
    series: pd.Series,
) -> pd.Series:
    """Return normalized partition labels as ordinary strings."""
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
    )


def make_event_id_series(
    prefix: str,
    count: int,
) -> pd.Series:
    """Create deterministic one-based event IDs."""
    require(
        count >= 0,
        "Event-ID count must be nonnegative.",
    )

    return pd.Series(
        [
            f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__{prefix}__{index:08d}"
            for index in range(1, count + 1)
        ],
        dtype="string",
    )


def summarize_exact_time_groups(
    frame: pd.DataFrame,
    *,
    scope_name: str,
    time_field: str,
    event_count_field_name: str,
) -> pd.DataFrame:
    """
    Summarize exact timestamp groups.

    This is an audit only. It does not jitter, remove, or reorder
    tied observations.
    """
    require_columns(
        frame,
        [
            EVENT_PARTITION_FIELD,
            time_field,
            "trade_aggressor_side",
            "trade_collector_sequence",
            "trade_id",
        ],
        f"{scope_name}_exact_time_input",
    )

    if frame.empty:
        return pd.DataFrame(
            columns=[
                "scope",
                "event_partition",
                time_field,
                event_count_field_name,
                "buy_count",
                "sell_count",
                "unique_side_count",
                "mixed_side_flag",
                "first_trade_collector_sequence",
                "last_trade_collector_sequence",
                "first_trade_id",
                "last_trade_id",
            ]
        )

    working = frame.copy()

    working["_event_partition_key"] = ordered_partition_string(
        working[EVENT_PARTITION_FIELD]
    )

    working = working.sort_values(
        [
            "_event_partition_key",
            time_field,
            "trade_collector_sequence",
            "trade_id",
        ],
        kind="stable",
    )

    grouped = (
        working.groupby(
            [
                "_event_partition_key",
                time_field,
            ],
            observed=True,
            sort=True,
            dropna=False,
        )
        .agg(
            **{
                event_count_field_name: (
                    "trade_id",
                    "size",
                ),
                "buy_count": (
                    "trade_aggressor_side",
                    lambda values: int(
                        values.eq("BUY").sum()
                    ),
                ),
                "sell_count": (
                    "trade_aggressor_side",
                    lambda values: int(
                        values.eq("SELL").sum()
                    ),
                ),
                "unique_side_count": (
                    "trade_aggressor_side",
                    "nunique",
                ),
                "first_trade_collector_sequence": (
                    "trade_collector_sequence",
                    "first",
                ),
                "last_trade_collector_sequence": (
                    "trade_collector_sequence",
                    "last",
                ),
                "first_trade_id": (
                    "trade_id",
                    "first",
                ),
                "last_trade_id": (
                    "trade_id",
                    "last",
                ),
            }
        )
        .reset_index()
        .rename(
            columns={
                "_event_partition_key": "event_partition",
            }
        )
    )

    grouped.insert(
        0,
        "scope",
        scope_name,
    )

    grouped["mixed_side_flag"] = (
        grouped["unique_side_count"].gt(1)
    )

    return grouped


def summarize_exchange_ms_buckets(
    frame: pd.DataFrame,
    *,
    scope_name: str,
) -> pd.DataFrame:
    """Construct exchange-millisecond diagnostic buckets."""
    require_columns(
        frame,
        [
            EVENT_PARTITION_FIELD,
            "trade_exchange_trade_time_ms",
            "trade_local_receipt_time_ns",
            "trade_aggressor_side",
            "trade_collector_sequence",
            "trade_id",
            "trade_quantity",
            "trade_notional",
            "book_state_id",
        ],
        f"{scope_name}_exchange_ms_bucket_input",
    )

    if frame.empty:
        return pd.DataFrame()

    working = frame.copy()

    working["_event_partition_key"] = ordered_partition_string(
        working[EVENT_PARTITION_FIELD]
    )

    working = working.sort_values(
        [
            "_event_partition_key",
            "trade_exchange_trade_time_ms",
            "trade_collector_sequence",
            "trade_id",
        ],
        kind="stable",
    )

    buckets = (
        working.groupby(
            [
                "_event_partition_key",
                "trade_exchange_trade_time_ms",
            ],
            observed=True,
            sort=True,
            dropna=False,
        )
        .agg(
            bucket_print_count=(
                "trade_id",
                "size",
            ),
            buy_print_count=(
                "trade_aggressor_side",
                lambda values: int(
                    values.eq("BUY").sum()
                ),
            ),
            sell_print_count=(
                "trade_aggressor_side",
                lambda values: int(
                    values.eq("SELL").sum()
                ),
            ),
            unique_side_count=(
                "trade_aggressor_side",
                "nunique",
            ),
            total_quantity=(
                "trade_quantity",
                "sum",
            ),
            total_notional=(
                "trade_notional",
                "sum",
            ),
            first_trade_collector_sequence=(
                "trade_collector_sequence",
                "first",
            ),
            last_trade_collector_sequence=(
                "trade_collector_sequence",
                "last",
            ),
            first_trade_id=(
                "trade_id",
                "first",
            ),
            last_trade_id=(
                "trade_id",
                "last",
            ),
            first_local_receipt_time_ns=(
                "trade_local_receipt_time_ns",
                "first",
            ),
            last_local_receipt_time_ns=(
                "trade_local_receipt_time_ns",
                "last",
            ),
            distinct_local_receipt_time_count=(
                "trade_local_receipt_time_ns",
                "nunique",
            ),
            unique_book_state_count=(
                "book_state_id",
                "nunique",
            ),
        )
        .reset_index()
        .rename(
            columns={
                "_event_partition_key": "event_partition",
            }
        )
    )

    buckets.insert(
        0,
        "scope",
        scope_name,
    )

    buckets["mixed_side_flag"] = (
        buckets["unique_side_count"].gt(1)
    )

    buckets["multi_print_bucket_flag"] = (
        buckets["bucket_print_count"].gt(1)
    )

    buckets["multi_local_time_bucket_flag"] = (
        buckets[
            "distinct_local_receipt_time_count"
        ].gt(1)
    )

    buckets["multi_book_state_bucket_flag"] = (
        buckets["unique_book_state_count"].gt(1)
    )

    buckets["bucket_duration_ns"] = (
        buckets["last_local_receipt_time_ns"]
        - buckets["first_local_receipt_time_ns"]
    )

    buckets["vwap"] = (
        buckets["total_notional"]
        / buckets["total_quantity"]
    )

    buckets["diagnostic_bucket_id"] = make_event_id_series(
        "MS_BUCKET",
        len(buckets),
    )

    return buckets


def candidate_event_time_tie_summary(
    event_frame: pd.DataFrame,
    *,
    representation_name: str,
    event_time_field: str,
    event_id_field: str,
    side_field: str,
) -> dict[str, Any]:
    """Summarize whether candidate event times are simple or batched."""
    require_columns(
        event_frame,
        [
            EVENT_PARTITION_FIELD,
            event_time_field,
            event_id_field,
            side_field,
        ],
        f"{representation_name}_time_tie_input",
    )

    if event_frame.empty:
        return {
            "representation": representation_name,
            "event_count": 0,
            "exact_time_tie_group_count": 0,
            "events_in_exact_time_tie_groups": 0,
            "mixed_side_exact_time_group_count": 0,
            "requires_simultaneous_event_batch_interface": False,
        }

    working = event_frame.copy()

    working["_event_partition_key"] = ordered_partition_string(
        working[EVENT_PARTITION_FIELD]
    )

    grouped = (
        working.groupby(
            [
                "_event_partition_key",
                event_time_field,
            ],
            observed=True,
            sort=True,
            dropna=False,
        )
        .agg(
            candidate_event_count=(
                event_id_field,
                "size",
            ),
            unique_side_count=(
                side_field,
                "nunique",
            ),
        )
        .reset_index()
    )

    tied = grouped.loc[
        grouped["candidate_event_count"].gt(1)
    ]

    return {
        "representation": representation_name,
        "event_count": int(len(event_frame)),
        "exact_time_tie_group_count": int(len(tied)),
        "events_in_exact_time_tie_groups": int(
            tied["candidate_event_count"].sum()
        ),
        "mixed_side_exact_time_group_count": int(
            tied["unique_side_count"].gt(1).sum()
        ),
        "requires_simultaneous_event_batch_interface": bool(
            len(tied) > 0
        ),
    }


# ------------------------------------------------------------
# Build audit-only all-partition views and selection-only views
# ------------------------------------------------------------

audit_trade_book_all_partitions = (
    canonical_trade_book.copy()
)

audit_trade_book_all_partitions[
    EVENT_PARTITION_FIELD
] = ordered_partition_string(
    audit_trade_book_all_partitions[
        EVENT_PARTITION_FIELD
    ]
)

selection_candidate_trade_book = (
    canonical_trade_book.loc[
        ordered_partition_string(
            canonical_trade_book[
                EVENT_PARTITION_FIELD
            ]
        ).isin(EVENT_DEFINITION_SELECTION_PARTITIONS)
    ]
    .copy()
    .sort_values(
        [
            "event_partition_order",
            "trade_collector_sequence",
            "trade_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

selection_candidate_trade_book[
    EVENT_PARTITION_FIELD
] = ordered_partition_string(
    selection_candidate_trade_book[
        EVENT_PARTITION_FIELD
    ]
)

require(
    not selection_candidate_trade_book.empty,
    "Selection-eligible candidate trade book is empty.",
)

require(
    selection_candidate_trade_book[
        EVENT_PARTITION_FIELD
    ]
    .isin(EVENT_DEFINITION_SELECTION_PARTITIONS)
    .all(),
    (
        "Selection-eligible candidate trade book contains a "
        "protected partition."
    ),
)

selection_protected_row_count = int(
    selection_candidate_trade_book[
        EVENT_PARTITION_FIELD
    ]
    .isin(PROTECTED_PARTITIONS)
    .sum()
)

require(
    selection_protected_row_count == 0,
    (
        "Protected rows leaked into selection candidate "
        f"construction: {selection_protected_row_count}"
    ),
)


# ------------------------------------------------------------
# Audit exact local-time ties
# ------------------------------------------------------------

all_partition_exact_local_time_groups = (
    summarize_exact_time_groups(
        audit_trade_book_all_partitions,
        scope_name="ALL_PARTITIONS_AUDIT_ONLY",
        time_field="trade_local_receipt_time_ns",
        event_count_field_name="trade_count",
    )
)

selection_exact_local_time_groups = (
    summarize_exact_time_groups(
        selection_candidate_trade_book,
        scope_name="SELECTION_ELIGIBLE_ONLY",
        time_field="trade_local_receipt_time_ns",
        event_count_field_name="trade_count",
    )
)

all_partition_exact_local_time_ties = (
    all_partition_exact_local_time_groups.loc[
        all_partition_exact_local_time_groups[
            "trade_count"
        ].gt(1)
    ]
    .copy()
    .reset_index(drop=True)
)

selection_exact_local_time_ties = (
    selection_exact_local_time_groups.loc[
        selection_exact_local_time_groups[
            "trade_count"
        ].gt(1)
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Exchange-millisecond diagnostic buckets
# ------------------------------------------------------------

all_partition_exchange_ms_buckets = (
    summarize_exchange_ms_buckets(
        audit_trade_book_all_partitions,
        scope_name="ALL_PARTITIONS_AUDIT_ONLY",
    )
)

selection_exchange_ms_buckets = (
    summarize_exchange_ms_buckets(
        selection_candidate_trade_book,
        scope_name="SELECTION_ELIGIBLE_ONLY",
    )
)

require(
    int(
        selection_exchange_ms_buckets[
            "bucket_print_count"
        ].sum()
    )
    == len(selection_candidate_trade_book),
    (
        "Selection exchange-millisecond buckets do not "
        "conserve trade membership."
    ),
)


# ------------------------------------------------------------
# Individual-trade candidate events for selection-eligible rows
# ------------------------------------------------------------

individual_trade_candidate_events = (
    selection_candidate_trade_book[
        [
            "notebook_04_source_row_number",
            "trade_id",
            "trade_collector_sequence",
            "trade_local_receipt_time_ns",
            "trade_exchange_trade_time_ms",
            "trade_aggressor_side",
            "trade_price",
            "trade_quantity",
            "trade_notional",
            "book_state_id",
            "book_collector_sequence",
            "book_local_receipt_time_ns",
            "local_observation_lag_ns",
            EVENT_PARTITION_FIELD,
            MATCHED_BOOK_PARTITION_FIELD,
            CROSS_PARTITION_HISTORY_FIELD,
        ]
    ]
    .copy()
    .sort_values(
        [
            EVENT_PARTITION_FIELD,
            "trade_local_receipt_time_ns",
            "trade_collector_sequence",
            "trade_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

individual_trade_candidate_events.insert(
    0,
    "individual_event_id",
    make_event_id_series(
        "INDIVIDUAL",
        len(individual_trade_candidate_events),
    ),
)

individual_trade_candidate_events[
    "event_representation"
] = INDIVIDUAL_EVENT_REPRESENTATION

individual_trade_candidate_events[
    "event_time_ns"
] = individual_trade_candidate_events[
    "trade_local_receipt_time_ns"
]

individual_trade_candidate_events[
    "event_side"
] = individual_trade_candidate_events[
    "trade_aggressor_side"
]

individual_trade_candidate_events[
    "event_quantity"
] = individual_trade_candidate_events[
    "trade_quantity"
]

individual_trade_candidate_events[
    "event_notional"
] = individual_trade_candidate_events[
    "trade_notional"
]

individual_trade_candidate_events[
    "event_print_count"
] = 1

individual_trade_candidate_events[
    "aggregate_mark_available_time_ns"
] = individual_trade_candidate_events[
    "event_time_ns"
]


# ------------------------------------------------------------
# Same-ms same-side burst candidate events
#
# Direct groupby aggregation is used for initiation fields.
# No post-hoc merge is required to discover the first constituent,
# so every burst candidate must have its initiation constituent
# by construction.
# ------------------------------------------------------------

burst_sort_columns = [
    EVENT_PARTITION_FIELD,
    "trade_exchange_trade_time_ms",
    "trade_aggressor_side",
    "trade_collector_sequence",
    "trade_id",
]

burst_constituent_frame = (
    selection_candidate_trade_book
    .sort_values(
        burst_sort_columns,
        kind="stable",
    )
    .reset_index(drop=True)
    .copy()
)

burst_group_columns = [
    EVENT_PARTITION_FIELD,
    "trade_exchange_trade_time_ms",
    "trade_aggressor_side",
]

burst_constituent_frame[
    "same_ms_same_side_burst_number"
] = (
    burst_constituent_frame
    .groupby(
        burst_group_columns,
        observed=True,
        sort=True,
        dropna=False,
    )
    .ngroup()
    .astype("int64")
    + 1
)

require(
    burst_constituent_frame[
        "same_ms_same_side_burst_number"
    ].notna().all(),
    (
        "At least one trade did not receive a same-ms "
        "same-side burst number."
    ),
)

same_ms_same_side_burst_events = (
    burst_constituent_frame.groupby(
        [
            "same_ms_same_side_burst_number",
            EVENT_PARTITION_FIELD,
            "trade_exchange_trade_time_ms",
            "trade_aggressor_side",
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
    .agg(
        burst_print_count=(
            "trade_id",
            "size",
        ),
        initiation_trade_id=(
            "trade_id",
            "first",
        ),
        final_trade_id=(
            "trade_id",
            "last",
        ),
        initiation_trade_collector_sequence=(
            "trade_collector_sequence",
            "first",
        ),
        final_trade_collector_sequence=(
            "trade_collector_sequence",
            "last",
        ),
        event_time_ns=(
            "trade_local_receipt_time_ns",
            "first",
        ),
        final_constituent_local_receipt_time_ns=(
            "trade_local_receipt_time_ns",
            "last",
        ),
        total_quantity=(
            "trade_quantity",
            "sum",
        ),
        total_notional=(
            "trade_notional",
            "sum",
        ),
        min_trade_price=(
            "trade_price",
            "min",
        ),
        max_trade_price=(
            "trade_price",
            "max",
        ),
        first_trade_price=(
            "trade_price",
            "first",
        ),
        last_trade_price=(
            "trade_price",
            "last",
        ),
        initiation_book_state_id=(
            "book_state_id",
            "first",
        ),
        initiation_book_collector_sequence=(
            "book_collector_sequence",
            "first",
        ),
        initiation_book_local_receipt_time_ns=(
            "book_local_receipt_time_ns",
            "first",
        ),
        initiation_local_observation_lag_ns=(
            "local_observation_lag_ns",
            "first",
        ),
        unique_book_state_count=(
            "book_state_id",
            "nunique",
        ),
        unique_local_receipt_time_count=(
            "trade_local_receipt_time_ns",
            "nunique",
        ),
        cross_partition_history_constituent_count=(
            CROSS_PARTITION_HISTORY_FIELD,
            "sum",
        ),
    )
    .reset_index()
)

same_ms_same_side_burst_events.insert(
    0,
    "burst_event_id",
    make_event_id_series(
        "BURST",
        len(same_ms_same_side_burst_events),
    ),
)

same_ms_same_side_burst_events[
    "event_representation"
] = BURST_EVENT_REPRESENTATION

same_ms_same_side_burst_events[
    "event_side"
] = same_ms_same_side_burst_events[
    "trade_aggressor_side"
]

same_ms_same_side_burst_events[
    "event_quantity"
] = same_ms_same_side_burst_events[
    "total_quantity"
]

same_ms_same_side_burst_events[
    "event_notional"
] = same_ms_same_side_burst_events[
    "total_notional"
]

same_ms_same_side_burst_events[
    "event_print_count"
] = same_ms_same_side_burst_events[
    "burst_print_count"
]

same_ms_same_side_burst_events[
    "vwap"
] = (
    same_ms_same_side_burst_events[
        "total_notional"
    ]
    / same_ms_same_side_burst_events[
        "total_quantity"
    ]
)

same_ms_same_side_burst_events[
    "burst_duration_ns"
] = (
    same_ms_same_side_burst_events[
        "final_constituent_local_receipt_time_ns"
    ]
    - same_ms_same_side_burst_events[
        "event_time_ns"
    ]
)

same_ms_same_side_burst_events[
    "aggregate_mark_available_time_ns"
] = same_ms_same_side_burst_events[
    "final_constituent_local_receipt_time_ns"
]

same_ms_same_side_burst_events[
    "aggregate_mark_available_at_initiation_flag"
] = (
    same_ms_same_side_burst_events[
        "aggregate_mark_available_time_ns"
    ]
    .eq(
        same_ms_same_side_burst_events[
            "event_time_ns"
        ]
    )
    & same_ms_same_side_burst_events[
        "burst_print_count"
    ].eq(1)
)

same_ms_same_side_burst_events[
    "multi_print_burst_flag"
] = same_ms_same_side_burst_events[
    "burst_print_count"
].gt(1)

same_ms_same_side_burst_events[
    "multi_local_time_burst_flag"
] = same_ms_same_side_burst_events[
    "unique_local_receipt_time_count"
].gt(1)

same_ms_same_side_burst_events[
    "multi_book_state_burst_flag"
] = same_ms_same_side_burst_events[
    "unique_book_state_count"
].gt(1)

same_ms_same_side_burst_events[
    "has_cross_partition_history_flag"
] = same_ms_same_side_burst_events[
    "cross_partition_history_constituent_count"
].gt(0)

same_ms_same_side_burst_events[
    MATCHED_BOOK_PARTITION_FIELD
] = same_ms_same_side_burst_events[
    EVENT_PARTITION_FIELD
]

require(
    same_ms_same_side_burst_events[
        "initiation_trade_id"
    ].notna().all(),
    (
        "At least one burst candidate lacks an initiation "
        "constituent."
    ),
)

require(
    same_ms_same_side_burst_events[
        "initiation_trade_collector_sequence"
    ].notna().all(),
    (
        "At least one burst candidate lacks an initiation "
        "collector sequence."
    ),
)

require(
    same_ms_same_side_burst_events[
        "event_time_ns"
    ].notna().all(),
    "At least one burst candidate lacks an event time.",
)

require(
    same_ms_same_side_burst_events[
        "burst_duration_ns"
    ].ge(0).all(),
    "At least one burst has negative duration.",
)


# ------------------------------------------------------------
# Burst membership ledger
# ------------------------------------------------------------

burst_event_id_map = (
    same_ms_same_side_burst_events
    .set_index("same_ms_same_side_burst_number")[
        "burst_event_id"
    ]
)

trade_to_same_ms_same_side_burst_membership = (
    burst_constituent_frame[
        [
            "notebook_04_source_row_number",
            "trade_id",
            "trade_collector_sequence",
            "trade_local_receipt_time_ns",
            "trade_exchange_trade_time_ms",
            "trade_aggressor_side",
            EVENT_PARTITION_FIELD,
            "same_ms_same_side_burst_number",
        ]
    ]
    .copy()
    .sort_values(
        [
            EVENT_PARTITION_FIELD,
            "trade_collector_sequence",
            "trade_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

trade_to_same_ms_same_side_burst_membership[
    "burst_event_id"
] = (
    trade_to_same_ms_same_side_burst_membership[
        "same_ms_same_side_burst_number"
    ]
    .map(burst_event_id_map)
    .astype("string")
)

require(
    trade_to_same_ms_same_side_burst_membership[
        "burst_event_id"
    ].notna().all(),
    (
        "At least one selection-eligible trade lacks a "
        "burst_event_id."
    ),
)

require(
    len(trade_to_same_ms_same_side_burst_membership)
    == len(selection_candidate_trade_book),
    (
        "Burst membership row count does not match "
        "selection-eligible trade count."
    ),
)

require(
    trade_to_same_ms_same_side_burst_membership[
        "trade_id"
    ].is_unique,
    (
        "A selection-eligible trade appears more than once "
        "in burst membership."
    ),
)

require(
    int(
        same_ms_same_side_burst_events[
            "burst_print_count"
        ].sum()
    )
    == len(selection_candidate_trade_book),
    (
        "Same-ms same-side bursts do not conserve "
        "selection-eligible trade membership."
    ),
)


# ------------------------------------------------------------
# Individual-event membership ledger
# ------------------------------------------------------------

trade_to_individual_event_membership = (
    individual_trade_candidate_events[
        [
            "individual_event_id",
            "notebook_04_source_row_number",
            "trade_id",
            "trade_collector_sequence",
            "trade_local_receipt_time_ns",
            "trade_exchange_trade_time_ms",
            "trade_aggressor_side",
            EVENT_PARTITION_FIELD,
        ]
    ]
    .copy()
)

require(
    len(trade_to_individual_event_membership)
    == len(selection_candidate_trade_book),
    (
        "Individual-event membership row count does not "
        "match selection-eligible trade count."
    ),
)

require(
    trade_to_individual_event_membership[
        "trade_id"
    ].is_unique,
    (
        "A selection-eligible trade appears more than once "
        "in individual-event membership."
    ),
)


# ------------------------------------------------------------
# Candidate-readiness summaries
# ------------------------------------------------------------

individual_time_tie_summary = (
    candidate_event_time_tie_summary(
        individual_trade_candidate_events,
        representation_name=INDIVIDUAL_EVENT_REPRESENTATION,
        event_time_field="event_time_ns",
        event_id_field="individual_event_id",
        side_field="event_side",
    )
)

burst_time_tie_summary = (
    candidate_event_time_tie_summary(
        same_ms_same_side_burst_events,
        representation_name=BURST_EVENT_REPRESENTATION,
        event_time_field="event_time_ns",
        event_id_field="burst_event_id",
        side_field="event_side",
    )
)

selection_same_ms_same_side_multiplicity_count = int(
    same_ms_same_side_burst_events[
        "multi_print_burst_flag"
    ].sum()
)

selection_multi_book_state_burst_count = int(
    same_ms_same_side_burst_events[
        "multi_book_state_burst_flag"
    ].sum()
)

selection_multi_local_time_burst_count = int(
    same_ms_same_side_burst_events[
        "multi_local_time_burst_flag"
    ].sum()
)

candidate_event_readiness = pd.DataFrame.from_records(
    [
        {
            **individual_time_tie_summary,
            "candidate_scope": "DEVELOPMENT_CALIBRATION_ONLY",
            "input_trade_count": int(
                len(selection_candidate_trade_book)
            ),
            "membership_conserved": bool(
                len(trade_to_individual_event_membership)
                == len(selection_candidate_trade_book)
            ),
            "print_count_conserved": True,
            "quantity_conserved_float": bool(
                np.isclose(
                    individual_trade_candidate_events[
                        "event_quantity"
                    ].sum(),
                    selection_candidate_trade_book[
                        "trade_quantity"
                    ].sum(),
                    rtol=0.0,
                    atol=1e-12,
                )
            ),
            "notional_conserved_float": bool(
                np.isclose(
                    individual_trade_candidate_events[
                        "event_notional"
                    ].sum(),
                    selection_candidate_trade_book[
                        "trade_notional"
                    ].sum(),
                    rtol=0.0,
                    atol=1e-8,
                )
            ),
            "candidate_blocking_ready": True,
        },
        {
            **burst_time_tie_summary,
            "candidate_scope": "DEVELOPMENT_CALIBRATION_ONLY",
            "input_trade_count": int(
                len(selection_candidate_trade_book)
            ),
            "membership_conserved": bool(
                len(
                    trade_to_same_ms_same_side_burst_membership
                )
                == len(selection_candidate_trade_book)
            ),
            "print_count_conserved": bool(
                int(
                    same_ms_same_side_burst_events[
                        "burst_print_count"
                    ].sum()
                )
                == len(selection_candidate_trade_book)
            ),
            "quantity_conserved_float": bool(
                np.isclose(
                    same_ms_same_side_burst_events[
                        "event_quantity"
                    ].sum(),
                    selection_candidate_trade_book[
                        "trade_quantity"
                    ].sum(),
                    rtol=0.0,
                    atol=1e-12,
                )
            ),
            "notional_conserved_float": bool(
                np.isclose(
                    same_ms_same_side_burst_events[
                        "event_notional"
                    ].sum(),
                    selection_candidate_trade_book[
                        "trade_notional"
                    ].sum(),
                    rtol=0.0,
                    atol=1e-8,
                )
            ),
            "candidate_blocking_ready": bool(
                same_ms_same_side_burst_events[
                    "initiation_trade_id"
                ].notna().all()
            ),
        },
    ]
)


# ------------------------------------------------------------
# Fragmentation summary
# ------------------------------------------------------------

timestamp_fragmentation_summary = pd.DataFrame.from_records(
    [
        {
            "scope": "ALL_PARTITIONS_AUDIT_ONLY",
            "trade_rows": int(
                len(audit_trade_book_all_partitions)
            ),
            "exact_local_time_group_count": int(
                len(all_partition_exact_local_time_groups)
            ),
            "exact_local_time_tie_group_count": int(
                len(all_partition_exact_local_time_ties)
            ),
            "trades_in_exact_local_time_ties": int(
                all_partition_exact_local_time_ties[
                    "trade_count"
                ].sum()
                if not all_partition_exact_local_time_ties.empty
                else 0
            ),
            "mixed_side_exact_local_time_tie_count": int(
                all_partition_exact_local_time_ties[
                    "mixed_side_flag"
                ].sum()
                if not all_partition_exact_local_time_ties.empty
                else 0
            ),
            "exchange_ms_bucket_count": int(
                len(all_partition_exchange_ms_buckets)
            ),
            "multi_print_exchange_ms_bucket_count": int(
                all_partition_exchange_ms_buckets[
                    "multi_print_bucket_flag"
                ].sum()
                if not all_partition_exchange_ms_buckets.empty
                else 0
            ),
            "mixed_side_exchange_ms_bucket_count": int(
                all_partition_exchange_ms_buckets[
                    "mixed_side_flag"
                ].sum()
                if not all_partition_exchange_ms_buckets.empty
                else 0
            ),
            "multi_book_state_exchange_ms_bucket_count": int(
                all_partition_exchange_ms_buckets[
                    "multi_book_state_bucket_flag"
                ].sum()
                if not all_partition_exchange_ms_buckets.empty
                else 0
            ),
        },
        {
            "scope": "SELECTION_ELIGIBLE_ONLY",
            "trade_rows": int(
                len(selection_candidate_trade_book)
            ),
            "exact_local_time_group_count": int(
                len(selection_exact_local_time_groups)
            ),
            "exact_local_time_tie_group_count": int(
                len(selection_exact_local_time_ties)
            ),
            "trades_in_exact_local_time_ties": int(
                selection_exact_local_time_ties[
                    "trade_count"
                ].sum()
                if not selection_exact_local_time_ties.empty
                else 0
            ),
            "mixed_side_exact_local_time_tie_count": int(
                selection_exact_local_time_ties[
                    "mixed_side_flag"
                ].sum()
                if not selection_exact_local_time_ties.empty
                else 0
            ),
            "exchange_ms_bucket_count": int(
                len(selection_exchange_ms_buckets)
            ),
            "multi_print_exchange_ms_bucket_count": int(
                selection_exchange_ms_buckets[
                    "multi_print_bucket_flag"
                ].sum()
                if not selection_exchange_ms_buckets.empty
                else 0
            ),
            "mixed_side_exchange_ms_bucket_count": int(
                selection_exchange_ms_buckets[
                    "mixed_side_flag"
                ].sum()
                if not selection_exchange_ms_buckets.empty
                else 0
            ),
            "multi_book_state_exchange_ms_bucket_count": int(
                selection_exchange_ms_buckets[
                    "multi_book_state_bucket_flag"
                ].sum()
                if not selection_exchange_ms_buckets.empty
                else 0
            ),
        },
    ]
)


# ------------------------------------------------------------
# Gate ledger
# ------------------------------------------------------------

candidate_representation_gates = [
    make_gate(
        gate="selection_candidate_scope_excludes_protected_partitions",
        passed=(selection_protected_row_count == 0),
        severity="BLOCKING",
        detail=(
            "protected_row_count="
            f"{selection_protected_row_count}"
        ),
    ),
    make_gate(
        gate="individual_candidate_membership_conserved",
        passed=(
            len(trade_to_individual_event_membership)
            == len(selection_candidate_trade_book)
        ),
        severity="BLOCKING",
        detail=(
            "membership_rows="
            f"{len(trade_to_individual_event_membership)}; "
            "selection_trade_rows="
            f"{len(selection_candidate_trade_book)}"
        ),
    ),
    make_gate(
        gate="burst_candidate_membership_conserved",
        passed=(
            len(trade_to_same_ms_same_side_burst_membership)
            == len(selection_candidate_trade_book)
        ),
        severity="BLOCKING",
        detail=(
            "membership_rows="
            f"{len(trade_to_same_ms_same_side_burst_membership)}; "
            "selection_trade_rows="
            f"{len(selection_candidate_trade_book)}"
        ),
    ),
    make_gate(
        gate="burst_candidate_print_count_conserved",
        passed=(
            int(
                same_ms_same_side_burst_events[
                    "burst_print_count"
                ].sum()
            )
            == len(selection_candidate_trade_book)
        ),
        severity="BLOCKING",
        detail=(
            "burst_print_sum="
            f"{int(same_ms_same_side_burst_events['burst_print_count'].sum())}; "
            "selection_trade_rows="
            f"{len(selection_candidate_trade_book)}"
        ),
    ),
    make_gate(
        gate="burst_candidate_initiation_constituents_present",
        passed=(
            same_ms_same_side_burst_events[
                "initiation_trade_id"
            ].notna().all()
            and same_ms_same_side_burst_events[
                "event_time_ns"
            ].notna().all()
        ),
        severity="BLOCKING",
        detail=(
            "burst_event_count="
            f"{len(same_ms_same_side_burst_events)}"
        ),
    ),
    make_gate(
        gate="burst_candidate_durations_nonnegative",
        passed=(
            same_ms_same_side_burst_events[
                "burst_duration_ns"
            ].ge(0).all()
        ),
        severity="BLOCKING",
        detail=(
            "negative_duration_count="
            f"{int(same_ms_same_side_burst_events['burst_duration_ns'].lt(0).sum())}"
        ),
    ),
    make_gate(
        gate="exchange_ms_diagnostic_bucket_membership_conserved",
        passed=(
            int(
                selection_exchange_ms_buckets[
                    "bucket_print_count"
                ].sum()
            )
            == len(selection_candidate_trade_book)
        ),
        severity="BLOCKING",
        detail=(
            "bucket_print_sum="
            f"{int(selection_exchange_ms_buckets['bucket_print_count'].sum())}; "
            "selection_trade_rows="
            f"{len(selection_candidate_trade_book)}"
        ),
    ),
    make_gate(
        gate="selection_exact_local_time_ties_absent",
        passed=(len(selection_exact_local_time_ties) == 0),
        severity="WARNING",
        detail=(
            "tie_group_count="
            f"{len(selection_exact_local_time_ties)}; "
            "trades_in_ties="
            f"{int(selection_exact_local_time_ties['trade_count'].sum()) if not selection_exact_local_time_ties.empty else 0}"
        ),
    ),
    make_gate(
        gate="selection_mixed_side_exchange_ms_buckets_absent",
        passed=(
            int(
                selection_exchange_ms_buckets[
                    "mixed_side_flag"
                ].sum()
            )
            == 0
        ),
        severity="WARNING",
        detail=(
            "mixed_side_bucket_count="
            f"{int(selection_exchange_ms_buckets['mixed_side_flag'].sum())}"
        ),
    ),
    make_gate(
        gate="selection_same_ms_same_side_multiplicity_absent",
        passed=(
            selection_same_ms_same_side_multiplicity_count == 0
        ),
        severity="WARNING",
        detail=(
            "multi_print_burst_count="
            f"{selection_same_ms_same_side_multiplicity_count}"
        ),
    ),
    make_gate(
        gate="selection_multi_book_state_bursts_absent",
        passed=(
            selection_multi_book_state_burst_count == 0
        ),
        severity="WARNING",
        detail=(
            "multi_book_state_burst_count="
            f"{selection_multi_book_state_burst_count}"
        ),
    ),
    make_gate(
        gate="selection_multi_local_time_bursts_absent",
        passed=(
            selection_multi_local_time_burst_count == 0
        ),
        severity="WARNING",
        detail=(
            "multi_local_time_burst_count="
            f"{selection_multi_local_time_burst_count}"
        ),
    ),
]

candidate_representation_gate_frame = (
    gate_results_to_frame(
        candidate_representation_gates
    )
)

fail_if_blocking_gate_failed(
    candidate_representation_gate_frame
)


# ------------------------------------------------------------
# Cell output
# ------------------------------------------------------------

display(timestamp_fragmentation_summary)
display(candidate_event_readiness)
display(candidate_representation_gate_frame)

display(
    selection_exchange_ms_buckets.head(20)
)

display(
    same_ms_same_side_burst_events.head(20)
)

display(
    trade_to_same_ms_same_side_burst_membership.head(20)
)

{
    "status": "PASS",
    "selection_trade_rows": int(
        len(selection_candidate_trade_book)
    ),
    "individual_candidate_event_rows": int(
        len(individual_trade_candidate_events)
    ),
    "same_ms_same_side_burst_event_rows": int(
        len(same_ms_same_side_burst_events)
    ),
    "selection_exchange_ms_bucket_rows": int(
        len(selection_exchange_ms_buckets)
    ),
    "selection_exact_local_time_tie_groups": int(
        len(selection_exact_local_time_ties)
    ),
    "selection_same_ms_same_side_multi_print_bursts": int(
        selection_same_ms_same_side_multiplicity_count
    ),
    "selection_candidate_scope": (
        "DEVELOPMENT_AND_CALIBRATION_ONLY"
    ),
    "protected_partitions_used_for_selection": False,
    "next_action": (
        "Freeze the structural primary-event selection rule "
        "using only DEVELOPMENT and CALIBRATION candidate "
        "readiness, then write and reload the selection "
        "checkpoint before opening protected partitions."
    ),
}

,scope,trade_rows,exact_local_time_group_count,exact_local_time_tie_group_count,trades_in_exact_local_time_ties,mixed_side_exact_local_time_tie_count,exchange_ms_bucket_count,multi_print_exchange_ms_bucket_count,mixed_side_exchange_ms_bucket_count,multi_book_state_exchange_ms_bucket_count
0,ALL_PARTITIONS_AUDIT_ONLY,67683,16212,5063,56534,52,13860,3011,27,3
1,SELECTION_ELIGIBLE_ONLY,47352,11079,3430,39703,41,9475,1990,22,3


,representation,event_count,exact_time_tie_group_count,events_in_exact_time_tie_groups,mixed_side_exact_time_group_count,requires_simultaneous_event_batch_interface,candidate_scope,input_trade_count,membership_conserved,print_count_conserved,quantity_conserved_float,notional_conserved_float,candidate_blocking_ready
0,INDIVIDUAL_TRADE_EVENTS,47352,3430,39703,41,True,DEVELOPMENT_CALIBRATION_ONLY,47352,True,True,True,True,True
1,SAME_MS_SAME_SIDE_BURSTS,9497,179,417,31,True,DEVELOPMENT_CALIBRATION_ONLY,47352,True,True,True,True,True


,gate,status,severity,detail
0,selection_candidate_scope_excludes_protected_partitions,PASS,BLOCKING,protected_row_count=0
1,individual_candidate_membership_conserved,PASS,BLOCKING,membership_rows=47352; selection_trade_rows=47352
2,burst_candidate_membership_conserved,PASS,BLOCKING,membership_rows=47352; selection_trade_rows=47352
3,burst_candidate_print_count_conserved,PASS,BLOCKING,burst_print_sum=47352; selection_trade_rows=47352
4,burst_candidate_initiation_constituents_present,PASS,BLOCKING,burst_event_count=9497
5,burst_candidate_durations_nonnegative,PASS,BLOCKING,negative_duration_count=0
6,exchange_ms_diagnostic_bucket_membership_conserved,PASS,BLOCKING,bucket_print_sum=47352; selection_trade_rows=47352
7,selection_exact_local_time_ties_absent,FAIL,WARNING,tie_group_count=3430; trades_in_ties=39703
8,selection_mixed_side_exchange_ms_buckets_absent,FAIL,WARNING,mixed_side_bucket_count=22
9,selection_same_ms_same_side_multiplicity_absent,FAIL,WARNING,multi_print_burst_count=1976


,scope,event_partition,trade_exchange_trade_time_ms,bucket_print_count,buy_print_count,sell_print_count,unique_side_count,total_quantity,total_notional,first_trade_collector_sequence,last_trade_collector_sequence,first_trade_id,last_trade_id,first_local_receipt_time_ns,last_local_receipt_time_ns,distinct_local_receipt_time_count,unique_book_state_count,mixed_side_flag,multi_print_bucket_flag,multi_local_time_bucket_flag,multi_book_state_bucket_flag,bucket_duration_ns,vwap,diagnostic_bucket_id
0,SELECTION_ELIGIBLE_ONLY,CALIBRATION,1783667271770,1,1,0,1,0.00026,16.600464,51852,51852,6494629861,6494629861,1783667270546982000,1783667270546982000,1,1,False,False,False,False,0,63847.94,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__MS_BUCKET__00000001
1,SELECTION_ELIGIBLE_ONLY,CALIBRATION,1783667272216,1,0,1,1,0.05000,3192.396500,51858,51858,6494629862,6494629862,1783667270994015000,1783667270994015000,1,1,False,False,False,False,0,63847.93,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__MS_BUCKET__00000002
2,SELECTION_ELIGIBLE_ONLY,CALIBRATION,1783667272369,1,0,1,1,0.00058,37.031799,51860,51860,6494629863,6494629863,1783667271146401800,1783667271146401800,1,1,False,False,False,False,0,63847.93,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__MS_BUCKET__00000003
3,SELECTION_ELIGIBLE_ONLY,CALIBRATION,1783667272436,1,1,0,1,0.00042,26.816135,51862,51862,6494629864,6494629864,1783667271213810600,1783667271213810600,1,1,False,False,False,False,0,63847.94,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__MS_BUCKET__00000004
4,SELECTION_ELIGIBLE_ONLY,CALIBRATION,1783667272787,1,1,0,1,0.00156,99.602786,51866,51866,6494629865,6494629865,1783667271564793500,1783667271564793500,1,1,False,False,False,False,0,63847.94,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__MS_BUCKET__00000005
5,SELECTION_ELIGIBLE_ONLY,CALIBRATION,1783667272798,1,0,1,1,0.00020,12.769586,51867,51867,6494629866,6494629866,1783667271574882100,1783667271574882100,1,1,False,False,False,False,0,63847.93,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__MS_BUCKET__00000006
6,SELECTION_ELIGIBLE_ONLY,CALIBRATION,1783667273943,1,1,0,1,0.00020,12.769588,51880,51880,6494629867,6494629867,1783667272720052300,1783667272720052300,1,1,False,False,False,False,0,63847.94,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__MS_BUCKET__00000007
7,SELECTION_ELIGIBLE_ONLY,CALIBRATION,1783667274695,1,1,0,1,0.00083,52.993790,51888,51888,6494629868,6494629868,1783667273472754900,1783667273472754900,1,1,False,False,False,False,0,63847.94,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__MS_BUCKET__00000008
8,SELECTION_ELIGIBLE_ONLY,CALIBRATION,1783667274798,1,1,0,1,0.00058,37.031805,51890,51890,6494629869,6494629869,1783667273575643200,1783667273575643200,1,1,False,False,False,False,0,63847.94,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__MS_BUCKET__00000009
9,SELECTION_ELIGIBLE_ONLY,CALIBRATION,1783667275072,1,0,1,1,0.00042,26.816131,51894,51894,6494629870,6494629870,1783667273849563700,1783667273849563700,1,1,False,False,False,False,0,63847.93,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__MS_BUCKET__00000010


,burst_event_id,same_ms_same_side_burst_number,event_partition,trade_exchange_trade_time_ms,trade_aggressor_side,burst_print_count,initiation_trade_id,final_trade_id,initiation_trade_collector_sequence,final_trade_collector_sequence,event_time_ns,final_constituent_local_receipt_time_ns,total_quantity,total_notional,min_trade_price,max_trade_price,first_trade_price,last_trade_price,initiation_book_state_id,initiation_book_collector_sequence,initiation_book_local_receipt_time_ns,initiation_local_observation_lag_ns,unique_book_state_count,unique_local_receipt_time_count,cross_partition_history_constituent_count,event_representation,event_side,event_quantity,event_notional,event_print_count,vwap,burst_duration_ns,aggregate_mark_available_time_ns,aggregate_mark_available_at_initiation_flag,multi_print_burst_flag,multi_local_time_burst_flag,multi_book_state_burst_flag,has_cross_partition_history_flag,matched_book_partition
0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000001,1,CALIBRATION,1783667271770,BUY,1,6494629861,6494629861,51852,51852,1783667270546982000,1783667270546982000,0.00026,16.600464,63847.94,63847.94,63847.94,63847.94,18021,51851,1783667270491506800,55475200,1,1,0,SAME_MS_SAME_SIDE_BURSTS,BUY,0.00026,16.600464,1,63847.94,0,1783667270546982000,True,False,False,False,False,CALIBRATION
1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000002,2,CALIBRATION,1783667272216,SELL,1,6494629862,6494629862,51858,51858,1783667270994015000,1783667270994015000,0.05000,3192.396500,63847.93,63847.93,63847.93,63847.93,18026,51857,1783667270990681200,3333800,1,1,0,SAME_MS_SAME_SIDE_BURSTS,SELL,0.05000,3192.396500,1,63847.93,0,1783667270994015000,True,False,False,False,False,CALIBRATION
2,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000003,3,CALIBRATION,1783667272369,SELL,1,6494629863,6494629863,51860,51860,1783667271146401800,1783667271146401800,0.00058,37.031799,63847.93,63847.93,63847.93,63847.93,18027,51859,1783667271090784600,55617200,1,1,0,SAME_MS_SAME_SIDE_BURSTS,SELL,0.00058,37.031799,1,63847.93,0,1783667271146401800,True,False,False,False,False,CALIBRATION
3,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000004,4,CALIBRATION,1783667272436,BUY,1,6494629864,6494629864,51862,51862,1783667271213810600,1783667271213810600,0.00042,26.816135,63847.94,63847.94,63847.94,63847.94,18028,51861,1783667271190920900,22889700,1,1,0,SAME_MS_SAME_SIDE_BURSTS,BUY,0.00042,26.816135,1,63847.94,0,1783667271213810600,True,False,False,False,False,CALIBRATION
4,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000005,5,CALIBRATION,1783667272787,BUY,1,6494629865,6494629865,51866,51866,1783667271564793500,1783667271564793500,0.00156,99.602786,63847.94,63847.94,63847.94,63847.94,18031,51865,1783667271490641800,74151700,1,1,0,SAME_MS_SAME_SIDE_BURSTS,BUY,0.00156,99.602786,1,63847.94,0,1783667271564793500,True,False,False,False,False,CALIBRATION
5,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000006,6,CALIBRATION,1783667272798,SELL,1,6494629866,6494629866,51867,51867,1783667271574882100,1783667271574882100,0.00020,12.769586,63847.93,63847.93,63847.93,63847.93,18031,51865,1783667271490641800,84240300,1,1,0,SAME_MS_SAME_SIDE_BURSTS,SELL,0.00020,12.769586,1,63847.93,0,1783667271574882100,True,False,False,False,False,CALIBRATION
6,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000007,7,CALIBRATION,1783667273943,BUY,1,6494629867,6494629867,51880,51880,1783667272720052300,1783667272720052300,0.00020,12.769588,63847.94,63847.94,63847.94,63847.94,18043,51879,1783667272691851000,28201300,1,1,0,SAM

,notebook_04_source_row_number,trade_id,trade_collector_sequence,trade_local_receipt_time_ns,trade_exchange_trade_time_ms,trade_aggressor_side,event_partition,same_ms_same_side_burst_number,burst_event_id
0,33821,6494629861,51852,1783667270546982000,1783667271770,BUY,CALIBRATION,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000001
1,33822,6494629862,51858,1783667270994015000,1783667272216,SELL,CALIBRATION,2,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000002
2,33823,6494629863,51860,1783667271146401800,1783667272369,SELL,CALIBRATION,3,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000003
3,33824,6494629864,51862,1783667271213810600,1783667272436,BUY,CALIBRATION,4,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000004
4,33825,6494629865,51866,1783667271564793500,1783667272787,BUY,CALIBRATION,5,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000005
5,33826,6494629866,51867,1783667271574882100,1783667272798,SELL,CALIBRATION,6,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000006
6,33827,6494629867,51880,1783667272720052300,1783667273943,BUY,CALIBRATION,7,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000007
7,33828,6494629868,51888,1783667273472754900,1783667274695,BUY,CALIBRATION,8,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000008
8,33829,6494629869,51890,1783667273575643200,1783667274798,BUY,CALIBRATION,9,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000009
9,33830,6494629870,51894,1783667273849563700,1783667275072,SELL,CALIBRATION,10,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST__00000010


{'status': 'PASS',
 'selection_trade_rows': 47352,
 'individual_candidate_event_rows': 47352,
 'same_ms_same_side_burst_event_rows': 9497,
 'selection_exchange_ms_bucket_rows': 9475,
 'selection_exact_local_time_tie_groups': 3430,
 'selection_same_ms_same_side_multi_print_bursts': 1976,
 'selection_candidate_scope': 'DEVELOPMENT_AND_CALIBRATION_ONLY',
 'protected_partitions_used_for_selection': False,
 'next_action': 'Freeze the structural primary-event selection rule using only DEVELOPMENT and CALIBRATION candidate readiness, then write and reload the selection checkpoint before opening protected partitions.'}

In [12]:
# ============================================================
# 04_EVENT_STREAM_CONSTRUCTION
# Cell 07 — Freeze primary event-definition selection checkpoint
#
# This cell freezes the primary event representation using only
# DEVELOPMENT and CALIBRATION candidate-readiness information.
#
# Protected partitions remain unopened for event construction
# until the selection checkpoint and event-definition contract
# are written, read back, and verified.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

required_preselection_objects = {
    "selection_candidate_trade_book": selection_candidate_trade_book,
    "individual_trade_candidate_events": individual_trade_candidate_events,
    "same_ms_same_side_burst_events": same_ms_same_side_burst_events,
    "trade_to_individual_event_membership": trade_to_individual_event_membership,
    "trade_to_same_ms_same_side_burst_membership": (
        trade_to_same_ms_same_side_burst_membership
    ),
    "selection_exchange_ms_buckets": selection_exchange_ms_buckets,
    "timestamp_fragmentation_summary": timestamp_fragmentation_summary,
    "candidate_event_readiness": candidate_event_readiness,
    "candidate_representation_gate_frame": candidate_representation_gate_frame,
}

for object_name, object_value in required_preselection_objects.items():
    require(
        isinstance(object_value, pd.DataFrame),
        f"{object_name} must be a DataFrame.",
    )

require(
    len(selection_candidate_trade_book) > 0,
    "selection_candidate_trade_book is empty.",
)

require(
    ordered_partition_string(
        selection_candidate_trade_book[EVENT_PARTITION_FIELD]
    )
    .isin(EVENT_DEFINITION_SELECTION_PARTITIONS)
    .all(),
    (
        "Selection candidate table contains a partition outside "
        "DEVELOPMENT and CALIBRATION."
    ),
)

selection_checkpoint_protected_row_count = int(
    ordered_partition_string(
        selection_candidate_trade_book[EVENT_PARTITION_FIELD]
    )
    .isin(PROTECTED_PARTITIONS)
    .sum()
)

require(
    selection_checkpoint_protected_row_count == 0,
    (
        "Protected partition rows leaked into selection checkpoint: "
        f"{selection_checkpoint_protected_row_count}"
    ),
)

require(
    not candidate_representation_gate_frame.loc[
        candidate_representation_gate_frame["severity"].eq("BLOCKING")
        & ~candidate_representation_gate_frame["status"].eq("PASS")
    ].shape[0],
    "A blocking candidate-representation gate failed before selection freeze.",
)


# ------------------------------------------------------------
# Stable table-hashing and JSON helpers
# ------------------------------------------------------------

def dataframe_to_json_records(
    frame: pd.DataFrame,
) -> list[dict[str, Any]]:
    """
    Convert a DataFrame to JSON-safe records.

    Using pandas JSON serialization prevents pd.NA, NaN, and numpy
    scalar objects from leaking into the final canonical JSON payload.
    """
    return json.loads(
        frame.reset_index(drop=True).to_json(
            orient="records",
            date_format="iso",
            double_precision=15,
            force_ascii=False,
        )
    )


def dataframe_contract_sha256(
    frame: pd.DataFrame,
    *,
    frame_name: str,
) -> str:
    """Hash DataFrame columns, dtypes, row count, and JSON-safe records."""
    payload = {
        "frame_name": frame_name,
        "row_count": int(len(frame)),
        "column_count": int(frame.shape[1]),
        "columns": [str(column) for column in frame.columns],
        "dtypes": {
            str(column): str(dtype)
            for column, dtype in frame.dtypes.items()
        },
        "records": dataframe_to_json_records(frame),
    }

    return canonical_json_sha256(payload)


def write_enveloped_json_artifact(
    *,
    path: Path,
    artifact_type: str,
    artifact_schema_version: str,
    payload: dict[str, Any],
    acceptance_status: str,
) -> dict[str, Any]:
    """Write one enveloped JSON artifact atomically and return metadata."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    payload_sha256 = canonical_json_sha256(payload)

    envelope = {
        "artifact_metadata": {
            "artifact_type": artifact_type,
            "artifact_schema_version": artifact_schema_version,
            "project_name": PROJECT_NAME,
            "pipeline_version": PIPELINE_VERSION,
            "source_run_prefix": SOURCE_RUN_PREFIX,
            "source_set_hash": EXPECTED_SOURCE_SET_SHA256,
            "v0_1_run_id": V01_RUN_ID,
            "run_config_hash": EXPECTED_V01_RUN_CONFIG_SHA256,
            "run_identity_hash": EXPECTED_V01_RUN_IDENTITY_SHA256,
            "producing_notebook": NOTEBOOK_FILENAME,
            "producing_notebook_name": NOTEBOOK_NAME,
            "created_at_utc": utc_now_iso(),
            "acceptance_status": acceptance_status,
            "payload_sha256": payload_sha256,
        },
        "payload": payload,
    }

    temporary_path = path.with_name(f"{path.name}.tmp")
    temporary_path.write_bytes(canonical_json_bytes(envelope))
    temporary_path.replace(path)

    return {
        "path": str(path.resolve()),
        "artifact_type": artifact_type,
        "artifact_schema_version": artifact_schema_version,
        "acceptance_status": acceptance_status,
        "size_bytes": int(path.stat().st_size),
        "file_sha256": sha256_file(path),
        "payload_sha256": payload_sha256,
    }


def readback_enveloped_json_artifact(
    *,
    path: Path,
    expected_artifact_type: str,
    expected_payload_sha256: str,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """Read back and verify one enveloped JSON artifact."""
    envelope = read_json_object(path)

    metadata = require_mapping(
        envelope.get("artifact_metadata"),
        f"{path.name}.artifact_metadata",
    )

    payload = require_mapping(
        envelope.get("payload"),
        f"{path.name}.payload",
    )

    require(
        metadata.get("artifact_type") == expected_artifact_type,
        (
            "Read-back artifact type mismatch: "
            f"observed={metadata.get('artifact_type')!r}, "
            f"expected={expected_artifact_type!r}."
        ),
    )

    require(
        metadata.get("source_run_prefix") == SOURCE_RUN_PREFIX,
        "Read-back source-run prefix mismatch.",
    )

    require(
        metadata.get("v0_1_run_id") == V01_RUN_ID,
        "Read-back V0.1 run ID mismatch.",
    )

    observed_payload_sha256 = canonical_json_sha256(payload)

    require(
        observed_payload_sha256 == expected_payload_sha256,
        (
            "Read-back payload SHA-256 mismatch: "
            f"observed={observed_payload_sha256}, "
            f"expected={expected_payload_sha256}."
        ),
    )

    require(
        metadata.get("payload_sha256") == expected_payload_sha256,
        (
            "Read-back metadata payload SHA-256 mismatch: "
            f"observed={metadata.get('payload_sha256')}, "
            f"expected={expected_payload_sha256}."
        ),
    )

    return metadata, payload


# ------------------------------------------------------------
# Candidate readiness extraction
# ------------------------------------------------------------

require_columns(
    candidate_event_readiness,
    [
        "representation",
        "event_count",
        "exact_time_tie_group_count",
        "events_in_exact_time_tie_groups",
        "mixed_side_exact_time_group_count",
        "requires_simultaneous_event_batch_interface",
        "candidate_scope",
        "input_trade_count",
        "membership_conserved",
        "print_count_conserved",
        "quantity_conserved_float",
        "notional_conserved_float",
        "candidate_blocking_ready",
    ],
    "candidate_event_readiness",
)

individual_readiness = candidate_event_readiness.loc[
    candidate_event_readiness["representation"].eq(
        INDIVIDUAL_EVENT_REPRESENTATION
    )
]

burst_readiness = candidate_event_readiness.loc[
    candidate_event_readiness["representation"].eq(
        BURST_EVENT_REPRESENTATION
    )
]

require(
    len(individual_readiness) == 1,
    "There must be exactly one individual-event readiness record.",
)

require(
    len(burst_readiness) == 1,
    "There must be exactly one burst-event readiness record.",
)

individual_readiness_record = individual_readiness.iloc[0].to_dict()
burst_readiness_record = burst_readiness.iloc[0].to_dict()

individual_candidate_blocking_ready = bool(
    individual_readiness_record["candidate_blocking_ready"]
)

burst_candidate_blocking_ready = bool(
    burst_readiness_record["candidate_blocking_ready"]
)

same_ms_same_side_multiplicity_observed = bool(
    int(
        same_ms_same_side_burst_events[
            "multi_print_burst_flag"
        ].sum()
    )
    > 0
)

require(
    individual_candidate_blocking_ready,
    "Individual-trade candidate is not blocking-ready.",
)

require(
    burst_candidate_blocking_ready,
    "Same-ms same-side burst candidate is not blocking-ready.",
)


# ------------------------------------------------------------
# Structural selection rule
#
# Rule:
# 1. Same-exchange-ms all-side buckets are diagnostic only.
# 2. If DEVELOPMENT/CALIBRATION contain any same-ms same-side
#    multi-print burst and the burst candidate passes blocking
#    readiness gates, select the burst representation.
# 3. Otherwise select individual-trade events, subject to their
#    blocking readiness gates.
# 4. Exact tied local timestamps are not repaired. They imply a
#    simultaneous-event batch interface.
# ------------------------------------------------------------

if same_ms_same_side_multiplicity_observed:
    PRIMARY_EVENT_REPRESENTATION = BURST_EVENT_REPRESENTATION
    PRIMARY_EVENT_ID_FIELD = "burst_event_id"
    PRIMARY_EVENT_SOURCE_TABLE_NAME = "same_ms_same_side_burst_events"
    PRIMARY_EVENT_MEMBERSHIP_TABLE_NAME = (
        "trade_to_same_ms_same_side_burst_membership"
    )
    PRIMARY_EVENT_SELECTION_REASON = (
        "SAME_MS_SAME_SIDE_MULTIPLICITY_OBSERVED_IN_DEVELOPMENT_CALIBRATION"
    )
    PRIMARY_EVENT_COUNT = int(len(same_ms_same_side_burst_events))
    PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE = bool(
        burst_readiness_record[
            "requires_simultaneous_event_batch_interface"
        ]
    )
    PRIMARY_EVENT_EXACT_TIME_TIE_GROUP_COUNT = int(
        burst_readiness_record["exact_time_tie_group_count"]
    )
    PRIMARY_EVENT_MIXED_SIDE_EXACT_TIME_GROUP_COUNT = int(
        burst_readiness_record["mixed_side_exact_time_group_count"]
    )
else:
    PRIMARY_EVENT_REPRESENTATION = INDIVIDUAL_EVENT_REPRESENTATION
    PRIMARY_EVENT_ID_FIELD = "individual_event_id"
    PRIMARY_EVENT_SOURCE_TABLE_NAME = "individual_trade_candidate_events"
    PRIMARY_EVENT_MEMBERSHIP_TABLE_NAME = (
        "trade_to_individual_event_membership"
    )
    PRIMARY_EVENT_SELECTION_REASON = (
        "NO_SAME_MS_SAME_SIDE_MULTIPLICITY_OBSERVED_IN_DEVELOPMENT_CALIBRATION"
    )
    PRIMARY_EVENT_COUNT = int(len(individual_trade_candidate_events))
    PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE = bool(
        individual_readiness_record[
            "requires_simultaneous_event_batch_interface"
        ]
    )
    PRIMARY_EVENT_EXACT_TIME_TIE_GROUP_COUNT = int(
        individual_readiness_record["exact_time_tie_group_count"]
    )
    PRIMARY_EVENT_MIXED_SIDE_EXACT_TIME_GROUP_COUNT = int(
        individual_readiness_record[
            "mixed_side_exact_time_group_count"
        ]
    )

PRIMARY_EVENT_TIME_FIELD = "event_time_ns"
PRIMARY_EVENT_SIDE_FIELD = "event_side"
PRIMARY_EVENT_QUANTITY_FIELD = "event_quantity"
PRIMARY_EVENT_NOTIONAL_FIELD = "event_notional"
PRIMARY_EVENT_PRINT_COUNT_FIELD = "event_print_count"
PRIMARY_EVENT_MARK_AVAILABILITY_FIELD = (
    "aggregate_mark_available_time_ns"
)

PRIMARY_TIMESTAMP_INTERFACE = (
    "SIMULTANEOUS_EVENT_BATCH_REQUIRED"
    if PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
    else "STRICT_SIMPLE_EVENT_TIMES"
)

require(
    PRIMARY_EVENT_REPRESENTATION
    in {
        INDIVIDUAL_EVENT_REPRESENTATION,
        BURST_EVENT_REPRESENTATION,
    },
    "Invalid primary event representation was selected.",
)

require(
    PRIMARY_EVENT_REPRESENTATION != MS_BUCKET_REPRESENTATION,
    "Exchange-millisecond diagnostic buckets must not be selected.",
)

require(
    PRIMARY_EVENT_COUNT > 0,
    "Selected primary event representation is empty.",
)


# ------------------------------------------------------------
# Hash the selection inputs and candidate artifacts
# ------------------------------------------------------------

selection_input_hashes = {
    "selection_candidate_trade_book_sha256": dataframe_contract_sha256(
        selection_candidate_trade_book,
        frame_name="selection_candidate_trade_book",
    ),
    "timestamp_fragmentation_summary_sha256": dataframe_contract_sha256(
        timestamp_fragmentation_summary,
        frame_name="timestamp_fragmentation_summary",
    ),
    "candidate_event_readiness_sha256": dataframe_contract_sha256(
        candidate_event_readiness,
        frame_name="candidate_event_readiness",
    ),
    "candidate_representation_gate_frame_sha256": dataframe_contract_sha256(
        candidate_representation_gate_frame,
        frame_name="candidate_representation_gate_frame",
    ),
    "individual_trade_candidate_events_sha256": dataframe_contract_sha256(
        individual_trade_candidate_events,
        frame_name="individual_trade_candidate_events",
    ),
    "same_ms_same_side_burst_events_sha256": dataframe_contract_sha256(
        same_ms_same_side_burst_events,
        frame_name="same_ms_same_side_burst_events",
    ),
    "trade_to_individual_event_membership_sha256": dataframe_contract_sha256(
        trade_to_individual_event_membership,
        frame_name="trade_to_individual_event_membership",
    ),
    "trade_to_same_ms_same_side_burst_membership_sha256": (
        dataframe_contract_sha256(
            trade_to_same_ms_same_side_burst_membership,
            frame_name="trade_to_same_ms_same_side_burst_membership",
        )
    ),
    "selection_exchange_ms_buckets_sha256": dataframe_contract_sha256(
        selection_exchange_ms_buckets,
        frame_name="selection_exchange_ms_buckets",
    ),
}


# ------------------------------------------------------------
# Selection checkpoint payload
# ------------------------------------------------------------

event_definition_selection_checkpoint_payload = {
    "selection_checkpoint_name": (
        "NOTEBOOK_04_PRIMARY_EVENT_DEFINITION_SELECTION"
    ),
    "selection_checkpoint_status": "FROZEN",
    "project_name": PROJECT_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_1_run_id": V01_RUN_ID,
    "combined_prefix": COMBINED_PREFIX,
    "producing_notebook": NOTEBOOK_FILENAME,
    "operating_mode": OPERATING_MODE,
    "selection_scope": {
        "allowed_partitions": list(
            EVENT_DEFINITION_SELECTION_PARTITIONS
        ),
        "protected_partitions": list(PROTECTED_PARTITIONS),
        "protected_row_count_used_for_selection": (
            selection_checkpoint_protected_row_count
        ),
        "validation_used_for_selection": False,
        "engineering_holdout_used_for_selection": False,
        "likelihood_used_for_selection": False,
        "predictive_performance_used_for_selection": False,
        "v0_0_outputs_used_for_acceptance": False,
    },
    "authority": {
        "partition_authority": (
            "NOTEBOOK_00_COLLECTOR_SEQUENCE_INTERVALS"
        ),
        "alignment_authority": PRIMARY_ALIGNMENT_POLICY,
        "event_time_authority": EVENT_TIME_AUTHORITY,
        "ordering_authority": PRIMARY_ORDERING_AUTHORITY,
        "interval_convention": INTERVAL_CONVENTION,
        "no_timestamp_jitter_allowed": (
            NO_TIMESTAMP_JITTER_ALLOWED
        ),
        "no_silent_event_removal_allowed": (
            NO_SILENT_EVENT_REMOVAL_ALLOWED
        ),
    },
    "candidate_representations": {
        "individual_trade_events": {
            "representation": INDIVIDUAL_EVENT_REPRESENTATION,
            "event_count": int(
                len(individual_trade_candidate_events)
            ),
            "membership_rows": int(
                len(trade_to_individual_event_membership)
            ),
            "blocking_ready": individual_candidate_blocking_ready,
            "requires_simultaneous_event_batch_interface": bool(
                individual_readiness_record[
                    "requires_simultaneous_event_batch_interface"
                ]
            ),
            "exact_time_tie_group_count": int(
                individual_readiness_record[
                    "exact_time_tie_group_count"
                ]
            ),
            "events_in_exact_time_tie_groups": int(
                individual_readiness_record[
                    "events_in_exact_time_tie_groups"
                ]
            ),
        },
        "same_ms_same_side_bursts": {
            "representation": BURST_EVENT_REPRESENTATION,
            "event_count": int(
                len(same_ms_same_side_burst_events)
            ),
            "membership_rows": int(
                len(
                    trade_to_same_ms_same_side_burst_membership
                )
            ),
            "blocking_ready": burst_candidate_blocking_ready,
            "requires_simultaneous_event_batch_interface": bool(
                burst_readiness_record[
                    "requires_simultaneous_event_batch_interface"
                ]
            ),
            "exact_time_tie_group_count": int(
                burst_readiness_record[
                    "exact_time_tie_group_count"
                ]
            ),
            "events_in_exact_time_tie_groups": int(
                burst_readiness_record[
                    "events_in_exact_time_tie_groups"
                ]
            ),
            "same_ms_same_side_multi_print_burst_count": int(
                same_ms_same_side_burst_events[
                    "multi_print_burst_flag"
                ].sum()
            ),
            "multi_book_state_burst_count": int(
                same_ms_same_side_burst_events[
                    "multi_book_state_burst_flag"
                ].sum()
            ),
            "multi_local_time_burst_count": int(
                same_ms_same_side_burst_events[
                    "multi_local_time_burst_flag"
                ].sum()
            ),
        },
        "same_exchange_ms_buckets": {
            "representation": MS_BUCKET_REPRESENTATION,
            "event_count": int(
                len(selection_exchange_ms_buckets)
            ),
            "diagnostic_only": True,
            "eligible_for_primary_selection": False,
        },
    },
    "selection_rule": {
        "rule_name": (
            "STRUCTURAL_DEV_CAL_PRIMARY_EVENT_RULE_V1"
        ),
        "rule_version": "1.0",
        "rule_inputs": [
            "candidate_blocking_readiness",
            "same_ms_same_side_multiplicity_presence",
            "timestamp_batch_requirement",
        ],
        "forbidden_inputs": [
            "VALIDATION",
            "ENGINEERING_HOLDOUT",
            "future_labels",
            "predictive_performance",
            "poisson_likelihood",
            "hawkes_likelihood",
            "residual_diagnostics",
            "v0_0_event_counts_as_acceptance_authority",
        ],
        "decision_logic": (
            "Select same-ms same-side bursts when same-ms "
            "same-side multi-print activity exists in DEVELOPMENT "
            "or CALIBRATION and the burst candidate passes blocking "
            "readiness gates; otherwise select individual trades."
        ),
    },
    "selected_primary_event_definition": {
        "primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
        "primary_event_source_table_name": (
            PRIMARY_EVENT_SOURCE_TABLE_NAME
        ),
        "primary_event_membership_table_name": (
            PRIMARY_EVENT_MEMBERSHIP_TABLE_NAME
        ),
        "primary_event_id_field": PRIMARY_EVENT_ID_FIELD,
        "primary_event_time_field": PRIMARY_EVENT_TIME_FIELD,
        "primary_event_side_field": PRIMARY_EVENT_SIDE_FIELD,
        "primary_event_quantity_field": PRIMARY_EVENT_QUANTITY_FIELD,
        "primary_event_notional_field": PRIMARY_EVENT_NOTIONAL_FIELD,
        "primary_event_print_count_field": (
            PRIMARY_EVENT_PRINT_COUNT_FIELD
        ),
        "primary_event_mark_availability_field": (
            PRIMARY_EVENT_MARK_AVAILABILITY_FIELD
        ),
        "primary_event_count_in_selection_scope": (
            PRIMARY_EVENT_COUNT
        ),
        "selection_reason": PRIMARY_EVENT_SELECTION_REASON,
        "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
        "requires_simultaneous_event_batch_interface": (
            PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
        ),
        "exact_time_tie_group_count": (
            PRIMARY_EVENT_EXACT_TIME_TIE_GROUP_COUNT
        ),
        "mixed_side_exact_time_group_count": (
            PRIMARY_EVENT_MIXED_SIDE_EXACT_TIME_GROUP_COUNT
        ),
    },
    "simultaneous_event_batch_contract": {
        "required": PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE,
        "batch_key": [
            EVENT_PARTITION_FIELD,
            PRIMARY_EVENT_TIME_FIELD,
        ],
        "within_batch_scoring_rule": (
            "All events sharing the same partition and event_time_ns "
            "must be scored against history strictly before that "
            "timestamp."
        ),
        "zero_lag_excitation_within_batch_allowed": False,
        "batch_excitation_application": (
            "Apply combined excitation only after all same-time "
            "batch members have been scored."
        ),
        "collector_sequence_role": (
            "Trace ordering only; not physical elapsed time inside "
            "a tied timestamp."
        ),
        "timestamp_jitter_allowed": False,
        "tied_event_removal_allowed": False,
    },
    "burst_event_contract": {
        "grouping_key": [
            EVENT_PARTITION_FIELD,
            "trade_exchange_trade_time_ms",
            "trade_aggressor_side",
        ],
        "partition_crossing_allowed": False,
        "event_time_rule": (
            "The burst event time is the local receipt time of the "
            "earliest constituent trade in collector-sequence order."
        ),
        "causal_state_rule": (
            "The burst causal state is the LOCAL_STRICT prior-book "
            "state of the earliest constituent trade."
        ),
        "aggregate_mark_rule": (
            "Full burst quantity, notional, VWAP, and print count "
            "are marked with aggregate_mark_available_time_ns and "
            "must not be used as initiation-time predictors unless "
            "available at initiation."
        ),
    },
    "input_hashes": selection_input_hashes,
    "candidate_event_readiness_records": dataframe_to_json_records(
        candidate_event_readiness
    ),
    "candidate_gate_records": dataframe_to_json_records(
        candidate_representation_gate_frame
    ),
    "timestamp_fragmentation_records": dataframe_to_json_records(
        timestamp_fragmentation_summary
    ),
}


# ------------------------------------------------------------
# Event-definition contract payload
# ------------------------------------------------------------

event_definition_contract_payload = {
    "contract_name": "NOTEBOOK_04_EVENT_DEFINITION_CONTRACT",
    "contract_status": "FROZEN",
    "selection_checkpoint_payload_sha256": canonical_json_sha256(
        event_definition_selection_checkpoint_payload
    ),
    "project_name": PROJECT_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_1_run_id": V01_RUN_ID,
    "combined_prefix": COMBINED_PREFIX,
    "producing_notebook": NOTEBOOK_FILENAME,
    "operating_mode": OPERATING_MODE,
    "event_stream_authority": EXPECTED_EVENT_STREAM_AUTHORITY,
    "primary_event_definition": (
        event_definition_selection_checkpoint_payload[
            "selected_primary_event_definition"
        ]
    ),
    "selection_scope": (
        event_definition_selection_checkpoint_payload[
            "selection_scope"
        ]
    ),
    "authority": event_definition_selection_checkpoint_payload[
        "authority"
    ],
    "selection_rule": event_definition_selection_checkpoint_payload[
        "selection_rule"
    ],
    "simultaneous_event_batch_contract": (
        event_definition_selection_checkpoint_payload[
            "simultaneous_event_batch_contract"
        ]
    ),
    "burst_event_contract": (
        event_definition_selection_checkpoint_payload[
            "burst_event_contract"
        ]
    ),
    "downstream_requirements": {
        "notebook_05_authorized_after_readback": True,
        "notebook_05_must_use_event_partition": True,
        "notebook_05_must_not_use_notebook_03_partition_labels": True,
        "notebook_05_must_preserve_batch_contract": (
            PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
        ),
        "hawkes_estimation_authority": "NONE",
        "market_making_authority": "NONE",
    },
}


# ------------------------------------------------------------
# Artifact paths
# ------------------------------------------------------------

EVENT_SELECTION_CHECKPOINT_PATH = (
    V01_CONFIG_ROOT
    / (
        f"{COMBINED_PREFIX}"
        f"__{NOTEBOOK_NAME}"
        "__event_definition_selection_checkpoint.json"
    )
)

EVENT_DEFINITION_CONTRACT_PATH_RESOLVED = (
    V01_CONFIG_ROOT
    / (
        f"{COMBINED_PREFIX}"
        f"__{NOTEBOOK_NAME}"
        "__event_definition_contract.json"
    )
)


# ------------------------------------------------------------
# Write, read back, and verify frozen selection artifacts
# ------------------------------------------------------------

selection_checkpoint_write_record = write_enveloped_json_artifact(
    path=EVENT_SELECTION_CHECKPOINT_PATH,
    artifact_type=(
        "NOTEBOOK_04_EVENT_DEFINITION_SELECTION_CHECKPOINT"
    ),
    artifact_schema_version="1.0",
    payload=event_definition_selection_checkpoint_payload,
    acceptance_status="FROZEN",
)

event_definition_contract_write_record = write_enveloped_json_artifact(
    path=EVENT_DEFINITION_CONTRACT_PATH_RESOLVED,
    artifact_type="NOTEBOOK_04_EVENT_DEFINITION_CONTRACT",
    artifact_schema_version="1.0",
    payload=event_definition_contract_payload,
    acceptance_status="FROZEN",
)

selection_checkpoint_readback_metadata, selection_checkpoint_readback_payload = (
    readback_enveloped_json_artifact(
        path=EVENT_SELECTION_CHECKPOINT_PATH,
        expected_artifact_type=(
            "NOTEBOOK_04_EVENT_DEFINITION_SELECTION_CHECKPOINT"
        ),
        expected_payload_sha256=selection_checkpoint_write_record[
            "payload_sha256"
        ],
    )
)

event_definition_contract_readback_metadata, event_definition_contract_readback_payload = (
    readback_enveloped_json_artifact(
        path=EVENT_DEFINITION_CONTRACT_PATH_RESOLVED,
        expected_artifact_type="NOTEBOOK_04_EVENT_DEFINITION_CONTRACT",
        expected_payload_sha256=event_definition_contract_write_record[
            "payload_sha256"
        ],
    )
)

require(
    selection_checkpoint_readback_payload[
        "selected_primary_event_definition"
    ]["primary_event_representation"]
    == PRIMARY_EVENT_REPRESENTATION,
    "Selection checkpoint read-back changed the selected representation.",
)

require(
    event_definition_contract_readback_payload[
        "primary_event_definition"
    ]["primary_event_representation"]
    == PRIMARY_EVENT_REPRESENTATION,
    "Event-definition contract read-back changed the selected representation.",
)

require(
    event_definition_contract_readback_payload[
        "selection_scope"
    ]["protected_row_count_used_for_selection"]
    == 0,
    (
        "Event-definition contract read-back indicates protected "
        "rows were used for selection."
    ),
)


# ------------------------------------------------------------
# Freeze global guard variables for downstream cells
# ------------------------------------------------------------

EVENT_DEFINITION_SELECTION_FROZEN = True
EVENT_DEFINITION_SELECTION_CHECKPOINT_VERIFIED = True
EVENT_DEFINITION_CONTRACT_VERIFIED = True
PROTECTED_PARTITIONS_MAY_BE_OPENED_FOR_CONSTRUCTION = True

PRIMARY_EVENT_CONTRACT = MappingProxyType(
    {
        "primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
        "primary_event_source_table_name": (
            PRIMARY_EVENT_SOURCE_TABLE_NAME
        ),
        "primary_event_membership_table_name": (
            PRIMARY_EVENT_MEMBERSHIP_TABLE_NAME
        ),
        "primary_event_id_field": PRIMARY_EVENT_ID_FIELD,
        "primary_event_time_field": PRIMARY_EVENT_TIME_FIELD,
        "primary_event_side_field": PRIMARY_EVENT_SIDE_FIELD,
        "primary_event_quantity_field": PRIMARY_EVENT_QUANTITY_FIELD,
        "primary_event_notional_field": PRIMARY_EVENT_NOTIONAL_FIELD,
        "primary_event_print_count_field": (
            PRIMARY_EVENT_PRINT_COUNT_FIELD
        ),
        "primary_event_mark_availability_field": (
            PRIMARY_EVENT_MARK_AVAILABILITY_FIELD
        ),
        "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
        "requires_simultaneous_event_batch_interface": (
            PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
        ),
        "event_definition_contract_path": str(
            EVENT_DEFINITION_CONTRACT_PATH_RESOLVED
        ),
        "event_definition_contract_payload_sha256": (
            event_definition_contract_write_record[
                "payload_sha256"
            ]
        ),
    }
)


# ------------------------------------------------------------
# Gate ledger
# ------------------------------------------------------------

selection_freeze_gates = [
    make_gate(
        gate="selection_uses_only_development_and_calibration",
        passed=(selection_checkpoint_protected_row_count == 0),
        severity="BLOCKING",
        detail=(
            "protected_row_count_used_for_selection="
            f"{selection_checkpoint_protected_row_count}"
        ),
    ),
    make_gate(
        gate="primary_event_representation_selected",
        passed=(
            PRIMARY_EVENT_REPRESENTATION
            in {
                INDIVIDUAL_EVENT_REPRESENTATION,
                BURST_EVENT_REPRESENTATION,
            }
        ),
        severity="BLOCKING",
        detail=(
            "selected_representation="
            f"{PRIMARY_EVENT_REPRESENTATION}"
        ),
    ),
    make_gate(
        gate="diagnostic_ms_buckets_not_selected",
        passed=(
            PRIMARY_EVENT_REPRESENTATION
            != MS_BUCKET_REPRESENTATION
        ),
        severity="BLOCKING",
        detail=(
            "diagnostic_representation="
            f"{MS_BUCKET_REPRESENTATION}"
        ),
    ),
    make_gate(
        gate="selected_candidate_is_nonempty",
        passed=(PRIMARY_EVENT_COUNT > 0),
        severity="BLOCKING",
        detail=f"primary_event_count={PRIMARY_EVENT_COUNT}",
    ),
    make_gate(
        gate="selection_checkpoint_written_and_verified",
        passed=EVENT_DEFINITION_SELECTION_CHECKPOINT_VERIFIED,
        severity="BLOCKING",
        detail=(
            "payload_sha256="
            f"{selection_checkpoint_write_record['payload_sha256']}"
        ),
    ),
    make_gate(
        gate="event_definition_contract_written_and_verified",
        passed=EVENT_DEFINITION_CONTRACT_VERIFIED,
        severity="BLOCKING",
        detail=(
            "payload_sha256="
            f"{event_definition_contract_write_record['payload_sha256']}"
        ),
    ),
    make_gate(
        gate="protected_partitions_authorized_after_freeze",
        passed=PROTECTED_PARTITIONS_MAY_BE_OPENED_FOR_CONSTRUCTION,
        severity="BLOCKING",
        detail=(
            "Protected partitions may be opened only after "
            "selection checkpoint read-back verification."
        ),
    ),
    make_gate(
        gate="selected_representation_requires_batch_interface",
        passed=(
            not PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
        ),
        severity="WARNING",
        detail=(
            "timestamp_interface="
            f"{PRIMARY_TIMESTAMP_INTERFACE}; "
            "exact_time_tie_group_count="
            f"{PRIMARY_EVENT_EXACT_TIME_TIE_GROUP_COUNT}"
        ),
    ),
]

selection_freeze_gate_frame = gate_results_to_frame(
    selection_freeze_gates
)

fail_if_blocking_gate_failed(selection_freeze_gate_frame)


# ------------------------------------------------------------
# Compact output tables
# ------------------------------------------------------------

selection_freeze_artifact_table = pd.DataFrame.from_records(
    [
        selection_checkpoint_write_record,
        event_definition_contract_write_record,
    ]
)

primary_event_selection_summary = pd.DataFrame.from_records(
    [
        {
            "selection_status": "FROZEN",
            "selected_primary_event_representation": (
                PRIMARY_EVENT_REPRESENTATION
            ),
            "selection_reason": PRIMARY_EVENT_SELECTION_REASON,
            "primary_event_count_in_selection_scope": (
                PRIMARY_EVENT_COUNT
            ),
            "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
            "requires_simultaneous_event_batch_interface": (
                PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
            ),
            "exact_time_tie_group_count": (
                PRIMARY_EVENT_EXACT_TIME_TIE_GROUP_COUNT
            ),
            "mixed_side_exact_time_group_count": (
                PRIMARY_EVENT_MIXED_SIDE_EXACT_TIME_GROUP_COUNT
            ),
            "protected_rows_used_for_selection": (
                selection_checkpoint_protected_row_count
            ),
            "event_definition_contract_path": str(
                EVENT_DEFINITION_CONTRACT_PATH_RESOLVED
            ),
        }
    ]
)

display(primary_event_selection_summary)
display(selection_freeze_gate_frame)
display(selection_freeze_artifact_table)

{
    "status": "PASS",
    "event_definition_selection_frozen": EVENT_DEFINITION_SELECTION_FROZEN,
    "selected_primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
    "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
    "selection_checkpoint_payload_sha256": (
        selection_checkpoint_write_record["payload_sha256"]
    ),
    "event_definition_contract_payload_sha256": (
        event_definition_contract_write_record["payload_sha256"]
    ),
    "protected_partitions_may_be_opened_for_construction": (
        PROTECTED_PARTITIONS_MAY_BE_OPENED_FOR_CONSTRUCTION
    ),
    "next_action": (
        "Construct the selected primary event representation across "
        "all partitions using the frozen event-definition contract."
    ),
}

,selection_status,selected_primary_event_representation,selection_reason,primary_event_count_in_selection_scope,timestamp_interface,requires_simultaneous_event_batch_interface,exact_time_tie_group_count,mixed_side_exact_time_group_count,protected_rows_used_for_selection,event_definition_contract_path
0,FROZEN,SAME_MS_SAME_SIDE_BURSTS,SAME_MS_SAME_SIDE_MULTIPLICITY_OBSERVED_IN_DEVELOPMENT_CALIBRATION,9497,SIMULTANEOUS_EVENT_BATCH_REQUIRED,True,179,31,0,D:\Clown Project\V0.1\config\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__event_definition_contract.json


,gate,status,severity,detail
0,selection_uses_only_development_and_calibration,PASS,BLOCKING,protected_row_count_used_for_selection=0
1,primary_event_representation_selected,PASS,BLOCKING,selected_representation=SAME_MS_SAME_SIDE_BURSTS
2,diagnostic_ms_buckets_not_selected,PASS,BLOCKING,diagnostic_representation=SAME_EXCHANGE_MS_DIAGNOSTIC_BUCKETS
3,selected_candidate_is_nonempty,PASS,BLOCKING,primary_event_count=9497
4,selection_checkpoint_written_and_verified,PASS,BLOCKING,payload_sha256=a8884dcdc1783f8f4659ae4b74fb93a37f93da0ebe5019dc10c05781a69f7b4a
5,event_definition_contract_written_and_verified,PASS,BLOCKING,payload_sha256=e5f1323e57d0441c73ca508a8b748d29e091b559fb63341c7000eb286428c8de
6,protected_partitions_authorized_after_freeze,PASS,BLOCKING,Protected partitions may be opened only after selection checkpoint read-back verification.
7,selected_representation_requires_batch_interface,FAIL,WARNING,timestamp_interface=SIMULTANEOUS_EVENT_BATCH_REQUIRED; exact_time_tie_group_count=179


,path,artifact_type,artifact_schema_version,acceptance_status,size_bytes,file_sha256,payload_sha256
0,D:\Clown Project\V0.1\config\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__event_definition_selection_checkpoint.json,NOTEBOOK_04_EVENT_DEFINITION_SELECTION_CHECKPOINT,1.0,FROZEN,10077,ff938936d3243020f13d819d48760a07e411b0f606f0c9f42c1fd7db0fe22259,a8884dcdc1783f8f4659ae4b74fb93a37f93da0ebe5019dc10c05781a69f7b4a
1,D:\Clown Project\V0.1\config\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__event_definition_contract.json,NOTEBOOK_04_EVENT_DEFINITION_CONTRACT,1.0,FROZEN,5196,08f05efae822beb15597bdfad775781103b691d1f67df0fe750778439e878330,e5f1323e57d0441c73ca508a8b748d29e091b559fb63341c7000eb286428c8de


{'status': 'PASS',
 'event_definition_selection_frozen': True,
 'selected_primary_event_representation': 'SAME_MS_SAME_SIDE_BURSTS',
 'timestamp_interface': 'SIMULTANEOUS_EVENT_BATCH_REQUIRED',
 'selection_checkpoint_payload_sha256': 'a8884dcdc1783f8f4659ae4b74fb93a37f93da0ebe5019dc10c05781a69f7b4a',
 'event_definition_contract_payload_sha256': 'e5f1323e57d0441c73ca508a8b748d29e091b559fb63341c7000eb286428c8de',
 'protected_partitions_may_be_opened_for_construction': True,
 'next_action': 'Construct the selected primary event representation across all partitions using the frozen event-definition contract.'}

In [14]:
# ============================================================
# 04_EVENT_STREAM_CONSTRUCTION
# Cell 08 — Construct selected primary event representation
#           across all partitions after selection freeze
#
# This cell opens VALIDATION and ENGINEERING_HOLDOUT only after
# the event-definition selection checkpoint has been written,
# read back, and verified.
#
# It does not change the selected event definition.
# It applies the frozen event contract to all partitions.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(globals().get("EVENT_DEFINITION_SELECTION_FROZEN", False)),
    "Event-definition selection is not frozen.",
)

require(
    bool(globals().get("EVENT_DEFINITION_SELECTION_CHECKPOINT_VERIFIED", False)),
    "Event-definition selection checkpoint is not verified.",
)

require(
    bool(globals().get("EVENT_DEFINITION_CONTRACT_VERIFIED", False)),
    "Event-definition contract is not verified.",
)

require(
    bool(globals().get("PROTECTED_PARTITIONS_MAY_BE_OPENED_FOR_CONSTRUCTION", False)),
    "Protected partitions may not be opened before selection freeze.",
)

require(
    "canonical_trade_book" in globals(),
    "canonical_trade_book does not exist. Run Cell 05 first.",
)

require_columns(
    canonical_trade_book,
    [
        "notebook_04_source_row_number",
        "trade_id",
        "trade_collector_sequence",
        "trade_local_receipt_time_ns",
        "trade_exchange_trade_time_ms",
        "trade_aggressor_side",
        "trade_price",
        "trade_quantity",
        "trade_notional",
        "book_state_id",
        "book_collector_sequence",
        "book_local_receipt_time_ns",
        "local_observation_lag_ns",
        EVENT_PARTITION_FIELD,
        MATCHED_BOOK_PARTITION_FIELD,
        AUTHORITATIVE_TRADE_PARTITION_FIELD,
        AUTHORITATIVE_BOOK_PARTITION_FIELD,
        CROSS_PARTITION_HISTORY_FIELD,
        "event_partition_order",
        "matched_book_partition_order",
    ],
    "canonical_trade_book",
)

require(
    len(canonical_trade_book) == EXPECTED_MATCHED_TRADE_ROWS,
    (
        "canonical_trade_book row count mismatch before all-partition "
        f"event construction: observed={len(canonical_trade_book):,}, "
        f"expected={EXPECTED_MATCHED_TRADE_ROWS:,}."
    ),
)

for forbidden_column in INHERITED_PARTITION_COLUMNS:
    require(
        forbidden_column not in canonical_trade_book.columns,
        (
            "Inherited Notebook 03 partition field leaked into "
            f"all-partition construction: {forbidden_column}"
        ),
    )


# ------------------------------------------------------------
# All-partition construction frame
# ------------------------------------------------------------

all_partition_trade_book_for_events = (
    canonical_trade_book.copy()
    .sort_values(
        [
            "event_partition_order",
            "trade_collector_sequence",
            "trade_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

all_partition_trade_book_for_events[
    EVENT_PARTITION_FIELD
] = ordered_partition_string(
    all_partition_trade_book_for_events[
        EVENT_PARTITION_FIELD
    ]
)

all_partition_trade_book_for_events[
    MATCHED_BOOK_PARTITION_FIELD
] = ordered_partition_string(
    all_partition_trade_book_for_events[
        MATCHED_BOOK_PARTITION_FIELD
    ]
)

require(
    set(
        all_partition_trade_book_for_events[
            EVENT_PARTITION_FIELD
        ].unique()
    )
    == set(PARTITION_ORDER),
    "All-partition construction frame does not contain all partitions.",
)

require(
    all_partition_trade_book_for_events[
        "trade_id"
    ].is_unique,
    "All-partition construction frame contains duplicate trade IDs.",
)

require(
    all_partition_trade_book_for_events[
        "trade_collector_sequence"
    ].is_unique,
    (
        "All-partition construction frame contains duplicate "
        "trade collector sequences."
    ),
)


# ------------------------------------------------------------
# Construction helpers
# ------------------------------------------------------------

def add_stable_event_number(
    frame: pd.DataFrame,
    *,
    number_field: str,
) -> pd.DataFrame:
    """Attach one-based stable event numbers after sorting."""
    result = frame.copy()
    result.insert(
        0,
        number_field,
        np.arange(
            1,
            len(result) + 1,
            dtype=np.int64,
        ),
    )
    return result


def construct_individual_events_all_partitions(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Construct one individual event per trade across all partitions."""
    working = (
        frame.copy()
        .sort_values(
            [
                "event_partition_order",
                "trade_local_receipt_time_ns",
                "trade_collector_sequence",
                "trade_id",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    individual_events = working[
        [
            "notebook_04_source_row_number",
            "trade_id",
            "trade_collector_sequence",
            "trade_local_receipt_time_ns",
            "trade_exchange_trade_time_ms",
            "trade_aggressor_side",
            "trade_price",
            "trade_quantity",
            "trade_notional",
            "book_state_id",
            "book_collector_sequence",
            "book_local_receipt_time_ns",
            "local_observation_lag_ns",
            EVENT_PARTITION_FIELD,
            MATCHED_BOOK_PARTITION_FIELD,
            CROSS_PARTITION_HISTORY_FIELD,
            "event_partition_order",
            "matched_book_partition_order",
        ]
    ].copy()

    individual_events = add_stable_event_number(
        individual_events,
        number_field="individual_event_number",
    )

    individual_events.insert(
        0,
        "individual_event_id",
        make_event_id_series(
            "INDIVIDUAL_ALL",
            len(individual_events),
        ),
    )

    individual_events["event_representation"] = (
        INDIVIDUAL_EVENT_REPRESENTATION
    )

    individual_events["event_time_ns"] = (
        individual_events["trade_local_receipt_time_ns"]
    )

    individual_events["event_side"] = (
        individual_events["trade_aggressor_side"]
    )

    individual_events["event_side_code"] = (
        individual_events["event_side"]
        .map(SIDE_CODE)
        .astype("int8")
    )

    individual_events["event_quantity"] = (
        individual_events["trade_quantity"]
    )

    individual_events["event_notional"] = (
        individual_events["trade_notional"]
    )

    individual_events["event_print_count"] = 1

    individual_events["aggregate_mark_available_time_ns"] = (
        individual_events["event_time_ns"]
    )

    individual_events["aggregate_mark_available_at_initiation_flag"] = True

    individual_events["multi_print_event_flag"] = False

    individual_events["multi_book_state_event_flag"] = False

    individual_events["multi_local_time_event_flag"] = False

    membership = individual_events[
        [
            "individual_event_id",
            "individual_event_number",
            "notebook_04_source_row_number",
            "trade_id",
            "trade_collector_sequence",
            "trade_local_receipt_time_ns",
            "trade_exchange_trade_time_ms",
            "trade_aggressor_side",
            EVENT_PARTITION_FIELD,
            "event_partition_order",
        ]
    ].copy()

    require(
        len(individual_events) == len(frame),
        "Individual all-partition event count does not equal trade count.",
    )

    require(
        membership["trade_id"].is_unique,
        "A trade appears more than once in individual-event membership.",
    )

    return individual_events, membership


def construct_exchange_ms_buckets_all_partitions(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Construct all-side exchange-millisecond diagnostic buckets."""
    working = (
        frame.copy()
        .sort_values(
            [
                "event_partition_order",
                "trade_exchange_trade_time_ms",
                "trade_collector_sequence",
                "trade_id",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    group_columns = [
        "event_partition_order",
        EVENT_PARTITION_FIELD,
        "trade_exchange_trade_time_ms",
    ]

    working["exchange_ms_bucket_number"] = (
        working.groupby(
            group_columns,
            observed=True,
            sort=True,
            dropna=False,
        )
        .ngroup()
        .astype("int64")
        + 1
    )

    bucket_events = (
        working.groupby(
            [
                "exchange_ms_bucket_number",
                "event_partition_order",
                EVENT_PARTITION_FIELD,
                "trade_exchange_trade_time_ms",
            ],
            observed=True,
            sort=True,
            dropna=False,
        )
        .agg(
            bucket_print_count=("trade_id", "size"),
            buy_print_count=(
                "trade_aggressor_side",
                lambda values: int(values.eq("BUY").sum()),
            ),
            sell_print_count=(
                "trade_aggressor_side",
                lambda values: int(values.eq("SELL").sum()),
            ),
            unique_side_count=("trade_aggressor_side", "nunique"),
            total_quantity=("trade_quantity", "sum"),
            total_notional=("trade_notional", "sum"),
            first_trade_id=("trade_id", "first"),
            last_trade_id=("trade_id", "last"),
            first_trade_collector_sequence=(
                "trade_collector_sequence",
                "first",
            ),
            last_trade_collector_sequence=(
                "trade_collector_sequence",
                "last",
            ),
            first_local_receipt_time_ns=(
                "trade_local_receipt_time_ns",
                "first",
            ),
            last_local_receipt_time_ns=(
                "trade_local_receipt_time_ns",
                "last",
            ),
            distinct_local_receipt_time_count=(
                "trade_local_receipt_time_ns",
                "nunique",
            ),
            unique_book_state_count=("book_state_id", "nunique"),
        )
        .reset_index()
    )

    bucket_events.insert(
        0,
        "diagnostic_bucket_id",
        make_event_id_series(
            "MS_BUCKET_ALL",
            len(bucket_events),
        ),
    )

    bucket_events["event_representation"] = (
        MS_BUCKET_REPRESENTATION
    )

    bucket_events["diagnostic_only"] = True

    bucket_events["eligible_for_primary_selection"] = False

    bucket_events["mixed_side_flag"] = (
        bucket_events["unique_side_count"].gt(1)
    )

    bucket_events["multi_print_bucket_flag"] = (
        bucket_events["bucket_print_count"].gt(1)
    )

    bucket_events["multi_local_time_bucket_flag"] = (
        bucket_events[
            "distinct_local_receipt_time_count"
        ].gt(1)
    )

    bucket_events["multi_book_state_bucket_flag"] = (
        bucket_events["unique_book_state_count"].gt(1)
    )

    bucket_events["bucket_duration_ns"] = (
        bucket_events["last_local_receipt_time_ns"]
        - bucket_events["first_local_receipt_time_ns"]
    )

    bucket_events["vwap"] = (
        bucket_events["total_notional"]
        / bucket_events["total_quantity"]
    )

    bucket_id_map = bucket_events.set_index(
        "exchange_ms_bucket_number"
    )["diagnostic_bucket_id"]

    membership = working[
        [
            "notebook_04_source_row_number",
            "trade_id",
            "trade_collector_sequence",
            "trade_local_receipt_time_ns",
            "trade_exchange_trade_time_ms",
            "trade_aggressor_side",
            EVENT_PARTITION_FIELD,
            "event_partition_order",
            "exchange_ms_bucket_number",
        ]
    ].copy()

    membership["diagnostic_bucket_id"] = (
        membership["exchange_ms_bucket_number"]
        .map(bucket_id_map)
        .astype("string")
    )

    require(
        membership["diagnostic_bucket_id"].notna().all(),
        "At least one trade lacks an exchange-ms diagnostic bucket ID.",
    )

    require(
        membership["trade_id"].is_unique,
        "A trade appears more than once in exchange-ms bucket membership.",
    )

    require(
        len(membership) == len(frame),
        "Exchange-ms bucket membership row count does not equal trade count.",
    )

    require(
        int(bucket_events["bucket_print_count"].sum()) == len(frame),
        "Exchange-ms bucket print counts do not conserve trade count.",
    )

    return bucket_events, membership


def construct_same_ms_same_side_bursts_all_partitions(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Construct same-ms same-side burst events across all partitions."""
    working = (
        frame.copy()
        .sort_values(
            [
                "event_partition_order",
                "trade_exchange_trade_time_ms",
                "trade_aggressor_side",
                "trade_collector_sequence",
                "trade_id",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    group_columns = [
        "event_partition_order",
        EVENT_PARTITION_FIELD,
        "trade_exchange_trade_time_ms",
        "trade_aggressor_side",
    ]

    working["same_ms_same_side_burst_number"] = (
        working.groupby(
            group_columns,
            observed=True,
            sort=True,
            dropna=False,
        )
        .ngroup()
        .astype("int64")
        + 1
    )

    require(
        working["same_ms_same_side_burst_number"].notna().all(),
        "At least one trade lacks a burst number.",
    )

    first_state_extra_columns = [
        column
        for column in working.columns
        if (
            column.startswith("book_")
            and column
            not in {
                "book_state_id",
                "book_collector_sequence",
                "book_local_receipt_time_ns",
            }
        )
    ]

    extra_state_aggs = {
        f"initiation_{column}": (column, "first")
        for column in first_state_extra_columns
    }

    burst_events = (
        working.groupby(
            [
                "same_ms_same_side_burst_number",
                "event_partition_order",
                EVENT_PARTITION_FIELD,
                "trade_exchange_trade_time_ms",
                "trade_aggressor_side",
            ],
            observed=True,
            sort=True,
            dropna=False,
        )
        .agg(
            burst_print_count=("trade_id", "size"),
            initiation_trade_id=("trade_id", "first"),
            final_trade_id=("trade_id", "last"),
            initiation_source_row_number=(
                "notebook_04_source_row_number",
                "first",
            ),
            final_source_row_number=(
                "notebook_04_source_row_number",
                "last",
            ),
            initiation_trade_collector_sequence=(
                "trade_collector_sequence",
                "first",
            ),
            final_trade_collector_sequence=(
                "trade_collector_sequence",
                "last",
            ),
            event_time_ns=(
                "trade_local_receipt_time_ns",
                "first",
            ),
            final_constituent_local_receipt_time_ns=(
                "trade_local_receipt_time_ns",
                "last",
            ),
            total_quantity=("trade_quantity", "sum"),
            total_notional=("trade_notional", "sum"),
            min_trade_price=("trade_price", "min"),
            max_trade_price=("trade_price", "max"),
            first_trade_price=("trade_price", "first"),
            last_trade_price=("trade_price", "last"),
            initiation_book_state_id=("book_state_id", "first"),
            initiation_book_collector_sequence=(
                "book_collector_sequence",
                "first",
            ),
            initiation_book_local_receipt_time_ns=(
                "book_local_receipt_time_ns",
                "first",
            ),
            initiation_local_observation_lag_ns=(
                "local_observation_lag_ns",
                "first",
            ),
            matched_book_partition=(
                MATCHED_BOOK_PARTITION_FIELD,
                "first",
            ),
            matched_book_partition_order=(
                "matched_book_partition_order",
                "first",
            ),
            unique_book_state_count=("book_state_id", "nunique"),
            unique_local_receipt_time_count=(
                "trade_local_receipt_time_ns",
                "nunique",
            ),
            cross_partition_history_constituent_count=(
                CROSS_PARTITION_HISTORY_FIELD,
                "sum",
            ),
            **extra_state_aggs,
        )
        .reset_index()
    )

    burst_events.insert(
        0,
        "burst_event_id",
        make_event_id_series(
            "BURST_ALL",
            len(burst_events),
        ),
    )

    burst_events["event_representation"] = (
        BURST_EVENT_REPRESENTATION
    )

    burst_events["event_side"] = (
        burst_events["trade_aggressor_side"]
    )

    burst_events["event_side_code"] = (
        burst_events["event_side"]
        .map(SIDE_CODE)
        .astype("int8")
    )

    burst_events["event_quantity"] = (
        burst_events["total_quantity"]
    )

    burst_events["event_notional"] = (
        burst_events["total_notional"]
    )

    burst_events["event_print_count"] = (
        burst_events["burst_print_count"]
    )

    burst_events["vwap"] = (
        burst_events["total_notional"]
        / burst_events["total_quantity"]
    )

    burst_events["burst_duration_ns"] = (
        burst_events[
            "final_constituent_local_receipt_time_ns"
        ]
        - burst_events["event_time_ns"]
    )

    burst_events["aggregate_mark_available_time_ns"] = (
        burst_events[
            "final_constituent_local_receipt_time_ns"
        ]
    )

    burst_events["aggregate_mark_available_at_initiation_flag"] = (
        burst_events["aggregate_mark_available_time_ns"].eq(
            burst_events["event_time_ns"]
        )
        & burst_events["burst_print_count"].eq(1)
    )

    burst_events["multi_print_burst_flag"] = (
        burst_events["burst_print_count"].gt(1)
    )

    burst_events["multi_local_time_burst_flag"] = (
        burst_events["unique_local_receipt_time_count"].gt(1)
    )

    burst_events["multi_book_state_burst_flag"] = (
        burst_events["unique_book_state_count"].gt(1)
    )

    burst_events["has_cross_partition_history_flag"] = (
        burst_events[
            "cross_partition_history_constituent_count"
        ].gt(0)
    )

    burst_id_map = burst_events.set_index(
        "same_ms_same_side_burst_number"
    )["burst_event_id"]

    membership = working[
        [
            "notebook_04_source_row_number",
            "trade_id",
            "trade_collector_sequence",
            "trade_local_receipt_time_ns",
            "trade_exchange_trade_time_ms",
            "trade_aggressor_side",
            EVENT_PARTITION_FIELD,
            "event_partition_order",
            "same_ms_same_side_burst_number",
        ]
    ].copy()

    membership["burst_event_id"] = (
        membership["same_ms_same_side_burst_number"]
        .map(burst_id_map)
        .astype("string")
    )

    require(
        membership["burst_event_id"].notna().all(),
        "At least one trade lacks a burst event ID.",
    )

    require(
        membership["trade_id"].is_unique,
        "A trade appears more than once in burst membership.",
    )

    require(
        len(membership) == len(frame),
        "Burst membership row count does not equal trade count.",
    )

    require(
        int(burst_events["burst_print_count"].sum()) == len(frame),
        "Burst print counts do not conserve trade count.",
    )

    require(
        burst_events["initiation_trade_id"].notna().all(),
        "At least one burst lacks an initiation trade ID.",
    )

    require(
        burst_events["event_time_ns"].notna().all(),
        "At least one burst lacks an event time.",
    )

    require(
        burst_events["burst_duration_ns"].ge(0).all(),
        "At least one burst has negative duration.",
    )

    require(
        burst_events["matched_book_partition_order"].le(
            burst_events["event_partition_order"]
        ).all(),
        "At least one burst uses a future matched-book partition.",
    )

    return burst_events, membership


def construct_exact_time_batches_for_primary_events(
    primary_events: pd.DataFrame,
    *,
    primary_event_id_field: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Construct exact-time simultaneous-event batches."""
    require_columns(
        primary_events,
        [
            primary_event_id_field,
            "event_partition_order",
            EVENT_PARTITION_FIELD,
            "event_time_ns",
            "event_side",
            "event_side_code",
            "event_print_count",
            "event_quantity",
            "event_notional",
        ],
        "primary_events_for_batch_construction",
    )

    working = (
        primary_events.copy()
        .sort_values(
            [
                "event_partition_order",
                "event_time_ns",
                "event_side_code",
                primary_event_id_field,
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    batch_group_columns = [
        "event_partition_order",
        EVENT_PARTITION_FIELD,
        "event_time_ns",
    ]

    working["primary_event_batch_number"] = (
        working.groupby(
            batch_group_columns,
            observed=True,
            sort=True,
            dropna=False,
        )
        .ngroup()
        .astype("int64")
        + 1
    )

    batch_table = (
        working.groupby(
            [
                "primary_event_batch_number",
                "event_partition_order",
                EVENT_PARTITION_FIELD,
                "event_time_ns",
            ],
            observed=True,
            sort=True,
            dropna=False,
        )
        .agg(
            batch_event_count=(primary_event_id_field, "size"),
            buy_event_count=(
                "event_side",
                lambda values: int(values.eq("BUY").sum()),
            ),
            sell_event_count=(
                "event_side",
                lambda values: int(values.eq("SELL").sum()),
            ),
            unique_side_count=("event_side", "nunique"),
            batch_print_count=("event_print_count", "sum"),
            batch_quantity=("event_quantity", "sum"),
            batch_notional=("event_notional", "sum"),
            first_primary_event_id=(
                primary_event_id_field,
                "first",
            ),
            last_primary_event_id=(
                primary_event_id_field,
                "last",
            ),
        )
        .reset_index()
    )

    batch_table.insert(
        0,
        "primary_event_batch_id",
        make_event_id_series(
            "PRIMARY_BATCH",
            len(batch_table),
        ),
    )

    batch_table["mixed_side_batch_flag"] = (
        batch_table["unique_side_count"].gt(1)
    )

    batch_table["simultaneous_batch_required_flag"] = (
        batch_table["batch_event_count"].gt(1)
    )

    batch_id_map = batch_table.set_index(
        "primary_event_batch_number"
    )["primary_event_batch_id"]

    membership = working[
        [
            primary_event_id_field,
            "primary_event_batch_number",
            "event_partition_order",
            EVENT_PARTITION_FIELD,
            "event_time_ns",
            "event_side",
            "event_side_code",
        ]
    ].copy()

    membership["primary_event_batch_id"] = (
        membership["primary_event_batch_number"]
        .map(batch_id_map)
        .astype("string")
    )

    require(
        membership["primary_event_batch_id"].notna().all(),
        "At least one primary event lacks a batch ID.",
    )

    require(
        len(membership) == len(primary_events),
        "Primary event-to-batch membership does not conserve event rows.",
    )

    require(
        membership[primary_event_id_field].is_unique,
        "A primary event appears more than once in batch membership.",
    )

    require(
        int(batch_table["batch_event_count"].sum()) == len(primary_events),
        "Batch event counts do not conserve primary event rows.",
    )

    return batch_table, membership


# ------------------------------------------------------------
# Construct all event representations across all partitions
# ------------------------------------------------------------

all_individual_trade_events, all_trade_to_individual_event_membership = (
    construct_individual_events_all_partitions(
        all_partition_trade_book_for_events
    )
)

all_exchange_ms_diagnostic_buckets, all_trade_to_exchange_ms_bucket_membership = (
    construct_exchange_ms_buckets_all_partitions(
        all_partition_trade_book_for_events
    )
)

all_same_ms_same_side_burst_events, all_trade_to_same_ms_same_side_burst_membership = (
    construct_same_ms_same_side_bursts_all_partitions(
        all_partition_trade_book_for_events
    )
)


# ------------------------------------------------------------
# Select the frozen primary representation across all partitions
# ------------------------------------------------------------

if PRIMARY_EVENT_REPRESENTATION == BURST_EVENT_REPRESENTATION:
    primary_estimation_events = (
        all_same_ms_same_side_burst_events.copy()
        .sort_values(
            [
                "event_partition_order",
                "event_time_ns",
                "event_side_code",
                "initiation_trade_collector_sequence",
                "burst_event_id",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    primary_source_event_id_field = "burst_event_id"

    primary_trade_membership = (
        all_trade_to_same_ms_same_side_burst_membership.copy()
        .rename(
            columns={
                "burst_event_id": "primary_event_id",
                "same_ms_same_side_burst_number": (
                    "primary_source_event_number"
                ),
            }
        )
    )

elif PRIMARY_EVENT_REPRESENTATION == INDIVIDUAL_EVENT_REPRESENTATION:
    primary_estimation_events = (
        all_individual_trade_events.copy()
        .sort_values(
            [
                "event_partition_order",
                "event_time_ns",
                "event_side_code",
                "trade_collector_sequence",
                "individual_event_id",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    primary_source_event_id_field = "individual_event_id"

    primary_trade_membership = (
        all_trade_to_individual_event_membership.copy()
        .rename(
            columns={
                "individual_event_id": "primary_event_id",
                "individual_event_number": (
                    "primary_source_event_number"
                ),
            }
        )
    )

else:
    raise RuntimeError(
        "Unknown frozen primary event representation: "
        f"{PRIMARY_EVENT_REPRESENTATION!r}"
    )

primary_estimation_events.insert(
    0,
    "primary_event_number",
    np.arange(
        1,
        len(primary_estimation_events) + 1,
        dtype=np.int64,
    ),
)

primary_estimation_events.insert(
    0,
    "primary_event_id",
    primary_estimation_events[
        primary_source_event_id_field
    ].astype("string"),
)

primary_estimation_events["primary_event_representation"] = (
    PRIMARY_EVENT_REPRESENTATION
)

primary_estimation_events["timestamp_interface"] = (
    PRIMARY_TIMESTAMP_INTERFACE
)

primary_estimation_events["requires_simultaneous_event_batch_interface"] = (
    PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
)

primary_event_number_map = (
    primary_estimation_events.set_index(
        "primary_event_id"
    )["primary_event_number"]
)

primary_trade_membership["primary_event_number"] = (
    primary_trade_membership["primary_event_id"]
    .map(primary_event_number_map)
    .astype("int64")
)

primary_trade_membership = (
    primary_trade_membership.sort_values(
        [
            "event_partition_order",
            "trade_collector_sequence",
            "trade_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    primary_trade_membership["primary_event_id"].notna().all(),
    "At least one trade lacks a primary_event_id.",
)

require(
    primary_trade_membership["primary_event_number"].notna().all(),
    "At least one trade lacks a primary_event_number.",
)

require(
    primary_trade_membership["trade_id"].is_unique,
    "A trade appears more than once in primary-event membership.",
)

require(
    len(primary_trade_membership) == EXPECTED_MATCHED_TRADE_ROWS,
    (
        "Primary membership row count does not equal all matched trades: "
        f"observed={len(primary_trade_membership):,}, "
        f"expected={EXPECTED_MATCHED_TRADE_ROWS:,}."
    ),
)

require(
    int(
        primary_estimation_events["event_print_count"].sum()
    )
    == EXPECTED_MATCHED_TRADE_ROWS,
    (
        "Primary events do not conserve trade print count: "
        f"observed={int(primary_estimation_events['event_print_count'].sum()):,}, "
        f"expected={EXPECTED_MATCHED_TRADE_ROWS:,}."
    ),
)


# ------------------------------------------------------------
# Exact-time primary-event batch table
# ------------------------------------------------------------

primary_exact_time_batches, primary_event_to_batch_membership = (
    construct_exact_time_batches_for_primary_events(
        primary_estimation_events,
        primary_event_id_field="primary_event_id",
    )
)

primary_batch_required_observed = bool(
    primary_exact_time_batches[
        "simultaneous_batch_required_flag"
    ].any()
)

require(
    primary_batch_required_observed
    == PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE,
    (
        "Observed primary-event batch requirement does not match "
        "the frozen event-definition contract."
    ),
)


# ------------------------------------------------------------
# Conservation audits
# ------------------------------------------------------------

trade_partition_reference = (
    all_partition_trade_book_for_events.groupby(
        [
            "event_partition_order",
            EVENT_PARTITION_FIELD,
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
    .agg(
        source_trade_count=("trade_id", "size"),
        source_buy_trade_count=(
            "trade_aggressor_side",
            lambda values: int(values.eq("BUY").sum()),
        ),
        source_sell_trade_count=(
            "trade_aggressor_side",
            lambda values: int(values.eq("SELL").sum()),
        ),
        source_quantity=("trade_quantity", "sum"),
        source_notional=("trade_notional", "sum"),
    )
    .reset_index()
)

primary_partition_observed = (
    primary_estimation_events.groupby(
        [
            "event_partition_order",
            EVENT_PARTITION_FIELD,
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
    .agg(
        primary_event_count=("primary_event_id", "size"),
        primary_print_count=("event_print_count", "sum"),
        primary_buy_event_count=(
            "event_side",
            lambda values: int(values.eq("BUY").sum()),
        ),
        primary_sell_event_count=(
            "event_side",
            lambda values: int(values.eq("SELL").sum()),
        ),
        primary_buy_print_count=(
            "event_print_count",
            lambda values: int(
                values[
                    primary_estimation_events.loc[
                        values.index,
                        "event_side",
                    ].eq("BUY")
                ].sum()
            ),
        ),
        primary_sell_print_count=(
            "event_print_count",
            lambda values: int(
                values[
                    primary_estimation_events.loc[
                        values.index,
                        "event_side",
                    ].eq("SELL")
                ].sum()
            ),
        ),
        primary_quantity=("event_quantity", "sum"),
        primary_notional=("event_notional", "sum"),
        primary_exact_time_batch_count=(
            "event_time_ns",
            "nunique",
        ),
    )
    .reset_index()
)

primary_partition_conservation_audit = (
    trade_partition_reference.merge(
        primary_partition_observed,
        on=[
            "event_partition_order",
            EVENT_PARTITION_FIELD,
        ],
        how="left",
        validate="one_to_one",
    )
)

primary_partition_conservation_audit[
    "print_count_conserved"
] = (
    primary_partition_conservation_audit[
        "primary_print_count"
    ].astype("int64")
    .eq(
        primary_partition_conservation_audit[
            "source_trade_count"
        ].astype("int64")
    )
)

primary_partition_conservation_audit[
    "buy_print_count_conserved"
] = (
    primary_partition_conservation_audit[
        "primary_buy_print_count"
    ].astype("int64")
    .eq(
        primary_partition_conservation_audit[
            "source_buy_trade_count"
        ].astype("int64")
    )
)

primary_partition_conservation_audit[
    "sell_print_count_conserved"
] = (
    primary_partition_conservation_audit[
        "primary_sell_print_count"
    ].astype("int64")
    .eq(
        primary_partition_conservation_audit[
            "source_sell_trade_count"
        ].astype("int64")
    )
)

primary_partition_conservation_audit[
    "quantity_conserved_float"
] = np.isclose(
    primary_partition_conservation_audit[
        "primary_quantity"
    ],
    primary_partition_conservation_audit[
        "source_quantity"
    ],
    rtol=0.0,
    atol=1e-12,
)

primary_partition_conservation_audit[
    "notional_conserved_float"
] = np.isclose(
    primary_partition_conservation_audit[
        "primary_notional"
    ],
    primary_partition_conservation_audit[
        "source_notional"
    ],
    rtol=0.0,
    atol=1e-8,
)

primary_partition_conservation_audit[
    "all_checks_pass"
] = (
    primary_partition_conservation_audit[
        [
            "print_count_conserved",
            "buy_print_count_conserved",
            "sell_print_count_conserved",
            "quantity_conserved_float",
            "notional_conserved_float",
        ]
    ].all(axis=1)
)

global_source_quantity = float(
    all_partition_trade_book_for_events[
        "trade_quantity"
    ].sum()
)

global_source_notional = float(
    all_partition_trade_book_for_events[
        "trade_notional"
    ].sum()
)

global_primary_quantity = float(
    primary_estimation_events[
        "event_quantity"
    ].sum()
)

global_primary_notional = float(
    primary_estimation_events[
        "event_notional"
    ].sum()
)

global_primary_conservation_summary = pd.DataFrame.from_records(
    [
        {
            "source_trade_count": int(
                len(all_partition_trade_book_for_events)
            ),
            "primary_event_count": int(
                len(primary_estimation_events)
            ),
            "primary_print_count": int(
                primary_estimation_events[
                    "event_print_count"
                ].sum()
            ),
            "source_quantity": global_source_quantity,
            "primary_quantity": global_primary_quantity,
            "source_notional": global_source_notional,
            "primary_notional": global_primary_notional,
            "print_count_conserved": bool(
                int(
                    primary_estimation_events[
                        "event_print_count"
                    ].sum()
                )
                == len(all_partition_trade_book_for_events)
            ),
            "quantity_conserved_float": bool(
                np.isclose(
                    global_primary_quantity,
                    global_source_quantity,
                    rtol=0.0,
                    atol=1e-12,
                )
            ),
            "notional_conserved_float": bool(
                np.isclose(
                    global_primary_notional,
                    global_source_notional,
                    rtol=0.0,
                    atol=1e-8,
                )
            ),
        }
    ]
)


# ------------------------------------------------------------
# Batch and warning summaries
# ------------------------------------------------------------

primary_batch_summary = pd.DataFrame.from_records(
    [
        {
            "primary_event_representation": (
                PRIMARY_EVENT_REPRESENTATION
            ),
            "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
            "primary_event_count": int(
                len(primary_estimation_events)
            ),
            "primary_batch_count": int(
                len(primary_exact_time_batches)
            ),
            "simultaneous_batch_count": int(
                primary_exact_time_batches[
                    "simultaneous_batch_required_flag"
                ].sum()
            ),
            "events_inside_simultaneous_batches": int(
                primary_exact_time_batches.loc[
                    primary_exact_time_batches[
                        "simultaneous_batch_required_flag"
                    ],
                    "batch_event_count",
                ].sum()
            ),
            "mixed_side_batch_count": int(
                primary_exact_time_batches[
                    "mixed_side_batch_flag"
                ].sum()
            ),
            "max_batch_event_count": int(
                primary_exact_time_batches[
                    "batch_event_count"
                ].max()
            ),
            "batch_interface_required_by_contract": bool(
                PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
            ),
        }
    ]
)

all_event_construction_summary = pd.DataFrame.from_records(
    [
        {
            "representation": INDIVIDUAL_EVENT_REPRESENTATION,
            "scope": "ALL_PARTITIONS",
            "event_rows": int(len(all_individual_trade_events)),
            "membership_rows": int(
                len(all_trade_to_individual_event_membership)
            ),
            "print_count_sum": int(
                all_individual_trade_events[
                    "event_print_count"
                ].sum()
            ),
            "selected_as_primary": (
                PRIMARY_EVENT_REPRESENTATION
                == INDIVIDUAL_EVENT_REPRESENTATION
            ),
        },
        {
            "representation": MS_BUCKET_REPRESENTATION,
            "scope": "ALL_PARTITIONS",
            "event_rows": int(
                len(all_exchange_ms_diagnostic_buckets)
            ),
            "membership_rows": int(
                len(all_trade_to_exchange_ms_bucket_membership)
            ),
            "print_count_sum": int(
                all_exchange_ms_diagnostic_buckets[
                    "bucket_print_count"
                ].sum()
            ),
            "selected_as_primary": False,
        },
        {
            "representation": BURST_EVENT_REPRESENTATION,
            "scope": "ALL_PARTITIONS",
            "event_rows": int(
                len(all_same_ms_same_side_burst_events)
            ),
            "membership_rows": int(
                len(all_trade_to_same_ms_same_side_burst_membership)
            ),
            "print_count_sum": int(
                all_same_ms_same_side_burst_events[
                    "event_print_count"
                ].sum()
            ),
            "selected_as_primary": (
                PRIMARY_EVENT_REPRESENTATION
                == BURST_EVENT_REPRESENTATION
            ),
        },
    ]
)


# ------------------------------------------------------------
# Gate ledger
# ------------------------------------------------------------

all_partition_primary_gates = [
    make_gate(
        gate="event_definition_contract_verified_before_protected_construction",
        passed=(
            EVENT_DEFINITION_SELECTION_FROZEN
            and EVENT_DEFINITION_SELECTION_CHECKPOINT_VERIFIED
            and EVENT_DEFINITION_CONTRACT_VERIFIED
        ),
        severity="BLOCKING",
        detail=(
            "Protected partitions opened only after frozen "
            "selection contract read-back."
        ),
    ),
    make_gate(
        gate="primary_representation_matches_frozen_contract",
        passed=(
            primary_estimation_events[
                "primary_event_representation"
            ].eq(PRIMARY_EVENT_REPRESENTATION).all()
        ),
        severity="BLOCKING",
        detail=(
            "primary_event_representation="
            f"{PRIMARY_EVENT_REPRESENTATION}"
        ),
    ),
    make_gate(
        gate="primary_membership_covers_every_trade_once",
        passed=(
            len(primary_trade_membership)
            == EXPECTED_MATCHED_TRADE_ROWS
            and primary_trade_membership[
                "trade_id"
            ].is_unique
        ),
        severity="BLOCKING",
        detail=(
            f"membership_rows={len(primary_trade_membership):,}; "
            f"expected={EXPECTED_MATCHED_TRADE_ROWS:,}"
        ),
    ),
    make_gate(
        gate="primary_print_count_conserved_globally",
        passed=bool(
            global_primary_conservation_summary.loc[
                0,
                "print_count_conserved",
            ]
        ),
        severity="BLOCKING",
        detail=(
            "primary_print_count="
            f"{int(primary_estimation_events['event_print_count'].sum()):,}; "
            f"source_trade_count={EXPECTED_MATCHED_TRADE_ROWS:,}"
        ),
    ),
    make_gate(
        gate="primary_quantity_conserved_globally_float",
        passed=bool(
            global_primary_conservation_summary.loc[
                0,
                "quantity_conserved_float",
            ]
        ),
        severity="BLOCKING",
        detail=(
            "source_quantity="
            f"{global_source_quantity:.15f}; "
            "primary_quantity="
            f"{global_primary_quantity:.15f}"
        ),
    ),
    make_gate(
        gate="primary_notional_conserved_globally_float",
        passed=bool(
            global_primary_conservation_summary.loc[
                0,
                "notional_conserved_float",
            ]
        ),
        severity="BLOCKING",
        detail=(
            "source_notional="
            f"{global_source_notional:.8f}; "
            "primary_notional="
            f"{global_primary_notional:.8f}"
        ),
    ),
    make_gate(
        gate="primary_partition_conservation",
        passed=bool(
            primary_partition_conservation_audit[
                "all_checks_pass"
            ].all()
        ),
        severity="BLOCKING",
        detail=(
            "verified_partitions="
            f"{len(primary_partition_conservation_audit)}"
        ),
    ),
    make_gate(
        gate="primary_event_batch_membership_conserved",
        passed=(
            len(primary_event_to_batch_membership)
            == len(primary_estimation_events)
            and primary_event_to_batch_membership[
                "primary_event_id"
            ].is_unique
            and int(
                primary_exact_time_batches[
                    "batch_event_count"
                ].sum()
            )
            == len(primary_estimation_events)
        ),
        severity="BLOCKING",
        detail=(
            f"primary_events={len(primary_estimation_events):,}; "
            f"batch_membership_rows={len(primary_event_to_batch_membership):,}"
        ),
    ),
    make_gate(
        gate="primary_batch_requirement_matches_contract",
        passed=(
            primary_batch_required_observed
            == PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
        ),
        severity="BLOCKING",
        detail=(
            "observed_batch_required="
            f"{primary_batch_required_observed}; "
            "contract_batch_required="
            f"{PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE}"
        ),
    ),
    make_gate(
        gate="diagnostic_ms_buckets_not_primary",
        passed=(
            PRIMARY_EVENT_REPRESENTATION
            != MS_BUCKET_REPRESENTATION
        ),
        severity="BLOCKING",
        detail=(
            "exchange-ms buckets remain diagnostic only."
        ),
    ),
    make_gate(
        gate="primary_simultaneous_batches_absent",
        passed=(
            int(
                primary_exact_time_batches[
                    "simultaneous_batch_required_flag"
                ].sum()
            )
            == 0
        ),
        severity="WARNING",
        detail=(
            "simultaneous_batch_count="
            f"{int(primary_exact_time_batches['simultaneous_batch_required_flag'].sum())}; "
            "timestamp_interface="
            f"{PRIMARY_TIMESTAMP_INTERFACE}"
        ),
    ),
    make_gate(
        gate="primary_mixed_side_batches_absent",
        passed=(
            int(
                primary_exact_time_batches[
                    "mixed_side_batch_flag"
                ].sum()
            )
            == 0
        ),
        severity="WARNING",
        detail=(
            "mixed_side_batch_count="
            f"{int(primary_exact_time_batches['mixed_side_batch_flag'].sum())}"
        ),
    ),
]

all_partition_primary_gate_frame = gate_results_to_frame(
    all_partition_primary_gates
)

fail_if_blocking_gate_failed(
    all_partition_primary_gate_frame
)


# ------------------------------------------------------------
# Cell output
# ------------------------------------------------------------

display(all_event_construction_summary)
display(global_primary_conservation_summary)
display(primary_partition_conservation_audit)
display(primary_batch_summary)
display(all_partition_primary_gate_frame)

display(primary_estimation_events.head(20))
display(primary_exact_time_batches.head(20))
display(primary_trade_membership.head(20))
display(primary_event_to_batch_membership.head(20))

{
    "status": "PASS",
    "selected_primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
    "primary_event_rows_all_partitions": int(
        len(primary_estimation_events)
    ),
    "primary_trade_membership_rows": int(
        len(primary_trade_membership)
    ),
    "primary_batch_rows": int(
        len(primary_exact_time_batches)
    ),
    "simultaneous_batch_count": int(
        primary_exact_time_batches[
            "simultaneous_batch_required_flag"
        ].sum()
    ),
    "mixed_side_batch_count": int(
        primary_exact_time_batches[
            "mixed_side_batch_flag"
        ].sum()
    ),
    "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
    "all_partitions_opened_after_selection_freeze": True,
    "next_action": (
        "Construct observation windows, relative event times, "
        "partition event arrays, and batch-aware scoring metadata."
    ),
}

,representation,scope,event_rows,membership_rows,print_count_sum,selected_as_primary
0,INDIVIDUAL_TRADE_EVENTS,ALL_PARTITIONS,67683,67683,67683,False
1,SAME_EXCHANGE_MS_DIAGNOSTIC_BUCKETS,ALL_PARTITIONS,13860,67683,67683,False
2,SAME_MS_SAME_SIDE_BURSTS,ALL_PARTITIONS,13887,67683,67683,True


,source_trade_count,primary_event_count,primary_print_count,source_quantity,primary_quantity,source_notional,primary_notional,print_count_conserved,quantity_conserved_float,notional_conserved_float
0,67683,13887,67683,329.19847,329.19847,2.103595e+07,2.103595e+07,True,True,True


,event_partition_order,event_partition,source_trade_count,source_buy_trade_count,source_sell_trade_count,source_quantity,source_notional,primary_event_count,primary_print_count,primary_buy_event_count,primary_sell_event_count,primary_buy_print_count,primary_sell_print_count,primary_quantity,primary_notional,primary_exact_time_batch_count,print_count_conserved,buy_print_count_conserved,sell_print_count_conserved,quantity_conserved_float,notional_conserved_float,all_checks_pass
0,1,DEVELOPMENT,33820,15194,18626,165.51400,1.057147e+07,7004,33820,3414,3590,15194,18626,165.51400,1.057147e+07,6859,True,True,True,True,True,True
1,2,CALIBRATION,13532,7267,6265,47.88126,3.058811e+06,2493,13532,1230,1263,7267,6265,47.88126,3.058811e+06,2400,True,True,True,True,True,True
2,3,VALIDATION,10198,5774,4424,51.96335,3.323781e+06,2048,10198,1013,1035,5774,4424,51.96335,3.323781e+06,1999,True,True,True,True,True,True
3,4,ENGINEERING_HOLDOUT,10133,2361,7772,63.83986,4.081884e+06,2342,10133,949,1393,2361,7772,63.83986,4.081884e+06,2306,True,True,True,True,True,True


,primary_event_representation,timestamp_interface,primary_event_count,primary_batch_count,simultaneous_batch_count,events_inside_simultaneous_batches,mixed_side_batch_count,max_batch_event_count,batch_interface_required_by_contract
0,SAME_MS_SAME_SIDE_BURSTS,SIMULTANEOUS_EVENT_BATCH_REQUIRED,13887,13564,249,572,41,6,True


,gate,status,severity,detail
0,event_definition_contract_verified_before_protected_construction,PASS,BLOCKING,Protected partitions opened only after frozen selection contract read-back.
1,primary_representation_matches_frozen_contract,PASS,BLOCKING,primary_event_representation=SAME_MS_SAME_SIDE_BURSTS
2,primary_membership_covers_every_trade_once,PASS,BLOCKING,"membership_rows=67,683; expected=67,683"
3,primary_print_count_conserved_globally,PASS,BLOCKING,"primary_print_count=67,683; source_trade_count=67,683"
4,primary_quantity_conserved_globally_float,PASS,BLOCKING,source_quantity=329.198469999999986; primary_quantity=329.198469999999929
5,primary_notional_conserved_globally_float,PASS,BLOCKING,source_notional=21035949.02727110; primary_notional=21035949.02727110
6,primary_partition_conservation,PASS,BLOCKING,verified_partitions=4
7,primary_event_batch_membership_conserved,PASS,BLOCKING,"primary_events=13,887; batch_membership_rows=13,887"
8,primary_batch_requirement_matches_contract,PASS,BLOCKING,observed_batch_required=True; contract_batch_required=True
9,diagnostic_ms_buckets_not_primary,PASS,BLOCKING,exchange-ms buckets remain diagnostic only.


,primary_event_id,primary_event_number,burst_event_id,same_ms_same_side_burst_number,event_partition_order,event_partition,trade_exchange_trade_time_ms,trade_aggressor_side,burst_print_count,initiation_trade_id,final_trade_id,initiation_source_row_number,final_source_row_number,initiation_trade_collector_sequence,final_trade_collector_sequence,event_time_ns,final_constituent_local_receipt_time_ns,total_quantity,total_notional,min_trade_price,max_trade_price,first_trade_price,last_trade_price,initiation_book_state_id,initiation_book_collector_sequence,initiation_book_local_receipt_time_ns,initiation_local_observation_lag_ns,matched_book_partition,matched_book_partition_order,unique_book_state_count,unique_local_receipt_time_count,cross_partition_history_constituent_count,initiation_book_row_number,initiation_book_local_receipt_time,initiation_book_exchange_event_time_ms,initiation_book_exchange_event_time,initiation_book_first_update_id,initiation_book_final_update_id,initiation_book_previous_final_update_id,initiation_book_best_bid,initiation_book_best_ask,initiation_book_best_bid_size,initiation_book_best_ask_size,initiation_book_spread,initiation_book_midpoint,initiation_book_microprice,initiation_book_l1_imbalance,initiation_book_top10_imbalance,initiation_book_best_bid_tick,initiation_book_best_ask_tick,initiation_book_spread_ticks,initiation_book_source_file,initiation_book_exchange_event_time_ns,initiation_book_receipt_inside_partition_flag,event_representation,event_side,event_side_code,event_quantity,event_notional,event_print_count,vwap,burst_duration_ns,aggregate_mark_available_time_ns,aggregate_mark_available_at_initiation_flag,multi_print_burst_flag,multi_local_time_burst_flag,multi_book_state_burst_flag,has_cross_partition_history_flag,primary_event_representation,timestamp_interface,requires_simultaneous_event_batch_interface
0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000001,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000001,1,1,DEVELOPMENT,1783665469952,BUY,1,6494596041,6494596041,1,1,14,14,1783665468766951600,1783665468766951600,0.00046,29.400610,63914.37,63914.37,63914.37,63914.37,3,13,1783665468733660800,33290800,DEVELOPMENT,1,1,1,0,4,2026-07-10 06:37:48.733660800+00:00,1783665469915,2026-07-10 06:37:49.915000+00:00,97233590209,97233590215,97233590208,63914.36,63914.37,2.72360,1.82138,0.01,63914.365,63914.365993,0.198509,NaN,6391436,6391437,1,D:\Clown Project\V0.1\data\processed\book\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__02_VISIBLE_BOOK_RECONSTRUCTION__reconstructed_book_states.csv,1783665469915000000,True,SAME_MS_SAME_SIDE_BURSTS,BUY,0,0.00046,29.400610,1,63914.37,0,1783665468766951600,True,False,False,False,False,SAME_MS_SAME_SIDE_BURSTS,SIMULTANEOUS_EVENT_BATCH_REQUIRED,True
1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000002,2,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000002,2,1,DEVELOPMENT,1783665470235,BUY,1,6494596042,6494596042,2,2,18,18,1783665469052166500,1783665469052166500,0.00060,38.348622,63914.37,63914.37,63914.37,63914.37,6,17,1783665469031942800,20223700,DEVELOPMENT,1,1,1,0,7,2026-07-10 06:37:49.031942800+00:00,1783665470214,2026-07-10 06:37:50.214000+00:00,97233590257,97233590279,97233590256,63914.36,63914.37,2.72360,1.82136,0.01,63914.365,63914.365993,0.198514,NaN,6391436,6391437,1,D:\Clown Project\V0.1\data\processed\book\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__02_VISIBLE_BOOK_RECONSTRUCTION__reconstructed_book_states.csv,1783665470214000000,True,SAME_MS_SAME_SIDE_BURSTS,BUY,0,0.00060,38.348622,1,63914.37,0,1783665469052166500,True,False,False,False,False,SAME_MS_SAME_SIDE_BURSTS,SIMULTANEOUS_EVENT_BATCH_REQUIRED,T

,primary_event_batch_id,primary_event_batch_number,event_partition_order,event_partition,event_time_ns,batch_event_count,buy_event_count,sell_event_count,unique_side_count,batch_print_count,batch_quantity,batch_notional,first_primary_event_id,last_primary_event_id,mixed_side_batch_flag,simultaneous_batch_required_flag
0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000001,1,1,DEVELOPMENT,1783665468766951600,1,1,0,1,1,0.00046,29.400610,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000001,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000001,False,False
1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000002,2,1,DEVELOPMENT,1783665469052166500,1,1,0,1,1,0.00060,38.348622,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000002,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000002,False,False
2,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000003,3,1,DEVELOPMENT,1783665469437964400,1,1,0,1,1,0.00020,12.782874,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000003,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000003,False,False
3,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000004,4,1,DEVELOPMENT,1783665469692104200,1,1,0,1,1,0.00039,24.926604,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000004,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000004,False,False
4,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000005,5,1,DEVELOPMENT,1783665470242312100,1,0,1,1,1,0.00046,29.400606,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000005,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000005,False,False
5,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000006,6,1,DEVELOPMENT,1783665470304239100,1,0,1,1,1,0.00018,11.504585,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000006,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000006,False,False
6,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000007,7,1,DEVELOPMENT,1783665470597198800,1,0,1,1,1,0.03495,2233.806882,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000007,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000007,False,False
7,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000008,8,1,DEVELOPMENT,1783665471662326600,1,0,1,1,1,0.00056,35.792042,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000008,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000008,False,Fals

,notebook_04_source_row_number,trade_id,trade_collector_sequence,trade_local_receipt_time_ns,trade_exchange_trade_time_ms,trade_aggressor_side,event_partition,event_partition_order,primary_source_event_number,primary_event_id,primary_event_number
0,1,6494596041,14,1783665468766951600,1783665469952,BUY,DEVELOPMENT,1,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000001,1
1,2,6494596042,18,1783665469052166500,1783665470235,BUY,DEVELOPMENT,1,2,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000002,2
2,3,6494596043,23,1783665469437964400,1783665470618,BUY,DEVELOPMENT,1,3,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000003,3
3,4,6494596044,26,1783665469692104200,1783665470874,BUY,DEVELOPMENT,1,4,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000004,4
4,5,6494596045,33,1783665470242312100,1783665471426,SELL,DEVELOPMENT,1,5,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000005,5
5,6,6494596046,34,1783665470304239100,1783665471485,SELL,DEVELOPMENT,1,6,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000006,6
6,7,6494596047,38,1783665470597198800,1783665471777,SELL,DEVELOPMENT,1,7,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000007,7
7,8,6494596048,50,1783665471662326600,1783665472849,SELL,DEVELOPMENT,1,8,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000008,8
8,9,6494596049,52,1783665471742062100,1783665472931,SELL,DEVELOPMENT,1,9,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000009,9
9,10,6494596050,54,1783665471862162900,1783665473043,BUY,DEVELOPMENT,1,10,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000010,10


,primary_event_id,primary_event_batch_number,event_partition_order,event_partition,event_time_ns,event_side,event_side_code,primary_event_batch_id
0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000001,1,1,DEVELOPMENT,1783665468766951600,BUY,0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000001
1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000002,2,1,DEVELOPMENT,1783665469052166500,BUY,0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000002
2,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000003,3,1,DEVELOPMENT,1783665469437964400,BUY,0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000003
3,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000004,4,1,DEVELOPMENT,1783665469692104200,BUY,0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000004
4,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000005,5,1,DEVELOPMENT,1783665470242312100,SELL,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000005
5,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000006,6,1,DEVELOPMENT,1783665470304239100,SELL,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000006
6,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000007,7,1,DEVELOPMENT,1783665470597198800,SELL,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000007
7,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000008,8,1,DEVELOPMENT,1783665471662326600,SELL,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000008
8,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000009,9,1,DEVELOPMENT,1783665471742062100,SELL,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000009
9,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000010,10,1,DEVELOPMENT,1783665471862162900,BUY,0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000010


{'status': 'PASS',
 'selected_primary_event_representation': 'SAME_MS_SAME_SIDE_BURSTS',
 'primary_event_rows_all_partitions': 13887,
 'primary_trade_membership_rows': 67683,
 'primary_batch_rows': 13564,
 'simultaneous_batch_count': 249,
 'mixed_side_batch_count': 41,
 'timestamp_interface': 'SIMULTANEOUS_EVENT_BATCH_REQUIRED',
 'all_partitions_opened_after_selection_freeze': True,
 'next_action': 'Construct observation windows, relative event times, partition event arrays, and batch-aware scoring metadata.'}

In [16]:
# ============================================================
# 04_EVENT_STREAM_CONSTRUCTION
# Cell 09 — Observation windows, relative event times,
#           partition arrays, and batch-aware scoring metadata
#
# This cell does not choose or modify the event definition.
# It applies the frozen event-definition contract to the
# already-constructed primary event stream.
#
# Important semantics:
# - Contract windows come from Notebook 00 frozen partitions.
# - Event-clock origin is the first primary event in each partition.
# - The gap from contract start to first event is preserved as
#   left-censoring duration.
# - Relative event times are measured from the event-clock origin.
# - Exact simultaneous events are preserved as batches.
# - No timestamp jitter, dropping, or artificial ordering is applied.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(globals().get("EVENT_DEFINITION_SELECTION_FROZEN", False)),
    "Event-definition selection is not frozen.",
)

require(
    bool(globals().get("EVENT_DEFINITION_CONTRACT_VERIFIED", False)),
    "Event-definition contract is not verified.",
)

require(
    "primary_estimation_events" in globals(),
    "primary_estimation_events does not exist. Run Cell 08 first.",
)

require(
    "primary_exact_time_batches" in globals(),
    "primary_exact_time_batches does not exist. Run Cell 08 first.",
)

require(
    "primary_event_to_batch_membership" in globals(),
    "primary_event_to_batch_membership does not exist. Run Cell 08 first.",
)

require(
    "primary_trade_membership" in globals(),
    "primary_trade_membership does not exist. Run Cell 08 first.",
)

require_columns(
    primary_estimation_events,
    [
        "primary_event_id",
        "primary_event_number",
        "primary_event_representation",
        "timestamp_interface",
        "requires_simultaneous_event_batch_interface",
        "event_partition_order",
        EVENT_PARTITION_FIELD,
        "event_time_ns",
        "event_side",
        "event_side_code",
        "event_quantity",
        "event_notional",
        "event_print_count",
        "aggregate_mark_available_time_ns",
    ],
    "primary_estimation_events",
)

require_columns(
    primary_exact_time_batches,
    [
        "primary_event_batch_id",
        "primary_event_batch_number",
        "event_partition_order",
        EVENT_PARTITION_FIELD,
        "event_time_ns",
        "batch_event_count",
        "buy_event_count",
        "sell_event_count",
        "unique_side_count",
        "batch_print_count",
        "batch_quantity",
        "batch_notional",
        "mixed_side_batch_flag",
        "simultaneous_batch_required_flag",
    ],
    "primary_exact_time_batches",
)

require_columns(
    primary_event_to_batch_membership,
    [
        "primary_event_id",
        "primary_event_batch_number",
        "event_partition_order",
        EVENT_PARTITION_FIELD,
        "event_time_ns",
        "event_side",
        "event_side_code",
        "primary_event_batch_id",
    ],
    "primary_event_to_batch_membership",
)

require(
    primary_estimation_events["primary_event_id"].is_unique,
    "primary_estimation_events has duplicate primary_event_id values.",
)

require(
    primary_event_to_batch_membership["primary_event_id"].is_unique,
    (
        "primary_event_to_batch_membership has duplicate "
        "primary_event_id values."
    ),
)

require(
    len(primary_event_to_batch_membership)
    == len(primary_estimation_events),
    (
        "Primary event-to-batch membership row count does not "
        "match primary event count."
    ),
)


# ------------------------------------------------------------
# Frozen observation-window contract
# ------------------------------------------------------------

partition_window_rows = []

for spec in PARTITION_SPECS:
    partition_events = primary_estimation_events.loc[
        primary_estimation_events[EVENT_PARTITION_FIELD]
        .astype("string")
        .eq(spec.name)
    ]

    partition_batches = primary_exact_time_batches.loc[
        primary_exact_time_batches[EVENT_PARTITION_FIELD]
        .astype("string")
        .eq(spec.name)
    ]

    require(
        not partition_events.empty,
        f"No primary events found for partition {spec.name}.",
    )

    event_clock_origin_ns = int(
        partition_events["event_time_ns"].min()
    )

    first_event_time_ns = int(
        partition_events["event_time_ns"].min()
    )

    last_event_time_ns = int(
        partition_events["event_time_ns"].max()
    )

    contract_start_ns = int(
        spec.receipt_time_start_ns
    )

    contract_end_exclusive_ns = int(
        spec.receipt_time_end_exclusive_ns
    )

    require(
        contract_start_ns < contract_end_exclusive_ns,
        f"Invalid contract window for partition {spec.name}.",
    )

    require(
        contract_start_ns <= event_clock_origin_ns,
        (
            "Event-clock origin precedes contract start for "
            f"partition {spec.name}."
        ),
    )

    require(
        last_event_time_ns < contract_end_exclusive_ns,
        (
            "Last event is outside the frozen contract window for "
            f"partition {spec.name}."
        ),
    )

    partition_window_rows.append(
        {
            "partition_order": int(spec.order),
            EVENT_PARTITION_FIELD: spec.name,
            "contract_sequence_start": int(
                spec.collector_sequence_start
            ),
            "contract_sequence_end_exclusive": int(
                spec.collector_sequence_end_exclusive
            ),
            "contract_start_ns": contract_start_ns,
            "contract_end_exclusive_ns": contract_end_exclusive_ns,
            "contract_duration_ns": int(
                contract_end_exclusive_ns
                - contract_start_ns
            ),
            "event_clock_origin_ns": event_clock_origin_ns,
            "first_event_time_ns": first_event_time_ns,
            "last_event_time_ns": last_event_time_ns,
            "left_censoring_duration_ns": int(
                event_clock_origin_ns
                - contract_start_ns
            ),
            "observation_window_duration_ns": int(
                contract_end_exclusive_ns
                - event_clock_origin_ns
            ),
            "primary_event_count": int(
                len(partition_events)
            ),
            "primary_batch_count": int(
                len(partition_batches)
            ),
            "simultaneous_batch_count": int(
                partition_batches[
                    "simultaneous_batch_required_flag"
                ].sum()
            ),
            "mixed_side_batch_count": int(
                partition_batches[
                    "mixed_side_batch_flag"
                ].sum()
            ),
            "contract_interval_convention": (
                INTERVAL_CONVENTION
            ),
            "event_clock_origin_rule": (
                "FIRST_PRIMARY_EVENT_LOCAL_RECEIPT_TIME"
            ),
            "left_censoring_preserved": True,
        }
    )

observation_window_contract = pd.DataFrame.from_records(
    partition_window_rows
)

require(
    len(observation_window_contract) == len(PARTITION_SPECS),
    "Observation-window contract does not cover every partition.",
)

require(
    observation_window_contract[
        EVENT_PARTITION_FIELD
    ].is_unique,
    "Observation-window contract has duplicate partitions.",
)

require(
    observation_window_contract[
        "observation_window_duration_ns"
    ].gt(0).all(),
    "At least one observation window has nonpositive duration.",
)

require(
    observation_window_contract[
        "left_censoring_duration_ns"
    ].ge(0).all(),
    "At least one partition has negative left-censoring duration.",
)


# ------------------------------------------------------------
# Map window fields into primary events
# ------------------------------------------------------------

window_lookup_columns = [
    "contract_start_ns",
    "contract_end_exclusive_ns",
    "contract_duration_ns",
    "event_clock_origin_ns",
    "left_censoring_duration_ns",
    "observation_window_duration_ns",
]

window_maps = {
    column: observation_window_contract.set_index(
        EVENT_PARTITION_FIELD
    )[column].to_dict()
    for column in window_lookup_columns
}

primary_estimation_events_windowed = (
    primary_estimation_events.copy()
    .sort_values(
        [
            "event_partition_order",
            "event_time_ns",
            "event_side_code",
            "primary_event_number",
            "primary_event_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

primary_estimation_events_windowed[
    EVENT_PARTITION_FIELD
] = ordered_partition_string(
    primary_estimation_events_windowed[
        EVENT_PARTITION_FIELD
    ]
)

for column, value_map in window_maps.items():
    primary_estimation_events_windowed[column] = (
        primary_estimation_events_windowed[
            EVENT_PARTITION_FIELD
        ].map(value_map)
    )

    require(
        primary_estimation_events_windowed[column]
        .notna()
        .all(),
        (
            "Primary event window mapping produced missing "
            f"values for {column}."
        ),
    )

    primary_estimation_events_windowed[column] = (
        primary_estimation_events_windowed[column]
        .astype("int64")
    )

primary_estimation_events_windowed[
    "relative_event_time_ns"
] = (
    primary_estimation_events_windowed["event_time_ns"]
    - primary_estimation_events_windowed[
        "event_clock_origin_ns"
    ]
)

primary_estimation_events_windowed[
    "relative_event_time_seconds"
] = (
    primary_estimation_events_windowed[
        "relative_event_time_ns"
    ].astype("float64")
    / 1_000_000_000.0
)

primary_estimation_events_windowed[
    "contract_relative_event_time_ns"
] = (
    primary_estimation_events_windowed["event_time_ns"]
    - primary_estimation_events_windowed[
        "contract_start_ns"
    ]
)

primary_estimation_events_windowed[
    "aggregate_mark_relative_time_ns"
] = (
    primary_estimation_events_windowed[
        "aggregate_mark_available_time_ns"
    ]
    - primary_estimation_events_windowed[
        "event_clock_origin_ns"
    ]
)

primary_estimation_events_windowed[
    "aggregate_mark_lag_after_event_ns"
] = (
    primary_estimation_events_windowed[
        "aggregate_mark_available_time_ns"
    ]
    - primary_estimation_events_windowed[
        "event_time_ns"
    ]
)

primary_estimation_events_windowed[
    "event_time_inside_contract_window_flag"
] = (
    primary_estimation_events_windowed["event_time_ns"]
    .ge(primary_estimation_events_windowed["contract_start_ns"])
    & primary_estimation_events_windowed["event_time_ns"]
    .lt(
        primary_estimation_events_windowed[
            "contract_end_exclusive_ns"
        ]
    )
)

primary_estimation_events_windowed[
    "relative_event_time_inside_window_flag"
] = (
    primary_estimation_events_windowed[
        "relative_event_time_ns"
    ].ge(0)
    & primary_estimation_events_windowed[
        "relative_event_time_ns"
    ].lt(
        primary_estimation_events_windowed[
            "observation_window_duration_ns"
        ]
    )
)

primary_estimation_events_windowed[
    "aggregate_mark_inside_contract_window_flag"
] = (
    primary_estimation_events_windowed[
        "aggregate_mark_available_time_ns"
    ]
    .ge(primary_estimation_events_windowed["contract_start_ns"])
    & primary_estimation_events_windowed[
        "aggregate_mark_available_time_ns"
    ]
    .lt(
        primary_estimation_events_windowed[
            "contract_end_exclusive_ns"
        ]
    )
)

primary_estimation_events_windowed[
    "partition_event_index"
] = (
    primary_estimation_events_windowed.groupby(
        EVENT_PARTITION_FIELD,
        observed=True,
        sort=False,
    )
    .cumcount()
    .astype("int64")
    + 1
)

primary_event_number_to_batch_number = (
    primary_event_to_batch_membership
    .set_index("primary_event_id")[
        "primary_event_batch_number"
    ]
    .to_dict()
)

primary_event_number_to_batch_id = (
    primary_event_to_batch_membership
    .set_index("primary_event_id")[
        "primary_event_batch_id"
    ]
    .to_dict()
)

primary_estimation_events_windowed[
    "primary_event_batch_number"
] = (
    primary_estimation_events_windowed[
        "primary_event_id"
    ].map(primary_event_number_to_batch_number)
)

primary_estimation_events_windowed[
    "primary_event_batch_id"
] = (
    primary_estimation_events_windowed[
        "primary_event_id"
    ].map(primary_event_number_to_batch_id)
    .astype("string")
)

require(
    primary_estimation_events_windowed[
        "primary_event_batch_number"
    ].notna().all(),
    "At least one primary event lacks a mapped batch number.",
)

require(
    primary_estimation_events_windowed[
        "primary_event_batch_id"
    ].notna().all(),
    "At least one primary event lacks a mapped batch ID.",
)

primary_estimation_events_windowed[
    "primary_event_batch_number"
] = (
    primary_estimation_events_windowed[
        "primary_event_batch_number"
    ].astype("int64")
)


# ------------------------------------------------------------
# Window batch table and enriched event-to-batch membership
# ------------------------------------------------------------

primary_exact_time_batches_windowed = (
    primary_exact_time_batches.copy()
    .sort_values(
        [
            "event_partition_order",
            "event_time_ns",
            "primary_event_batch_number",
            "primary_event_batch_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

primary_exact_time_batches_windowed[
    EVENT_PARTITION_FIELD
] = ordered_partition_string(
    primary_exact_time_batches_windowed[
        EVENT_PARTITION_FIELD
    ]
)

for column, value_map in window_maps.items():
    primary_exact_time_batches_windowed[column] = (
        primary_exact_time_batches_windowed[
            EVENT_PARTITION_FIELD
        ].map(value_map)
    )

    require(
        primary_exact_time_batches_windowed[column]
        .notna()
        .all(),
        (
            "Primary batch window mapping produced missing "
            f"values for {column}."
        ),
    )

    primary_exact_time_batches_windowed[column] = (
        primary_exact_time_batches_windowed[column]
        .astype("int64")
    )

primary_exact_time_batches_windowed[
    "relative_batch_time_ns"
] = (
    primary_exact_time_batches_windowed["event_time_ns"]
    - primary_exact_time_batches_windowed[
        "event_clock_origin_ns"
    ]
)

primary_exact_time_batches_windowed[
    "relative_batch_time_seconds"
] = (
    primary_exact_time_batches_windowed[
        "relative_batch_time_ns"
    ].astype("float64")
    / 1_000_000_000.0
)

primary_exact_time_batches_windowed[
    "batch_time_inside_contract_window_flag"
] = (
    primary_exact_time_batches_windowed["event_time_ns"]
    .ge(primary_exact_time_batches_windowed["contract_start_ns"])
    & primary_exact_time_batches_windowed["event_time_ns"]
    .lt(
        primary_exact_time_batches_windowed[
            "contract_end_exclusive_ns"
        ]
    )
)

primary_exact_time_batches_windowed[
    "relative_batch_time_inside_window_flag"
] = (
    primary_exact_time_batches_windowed[
        "relative_batch_time_ns"
    ].ge(0)
    & primary_exact_time_batches_windowed[
        "relative_batch_time_ns"
    ].lt(
        primary_exact_time_batches_windowed[
            "observation_window_duration_ns"
        ]
    )
)

primary_exact_time_batches_windowed[
    "partition_batch_index"
] = (
    primary_exact_time_batches_windowed.groupby(
        EVENT_PARTITION_FIELD,
        observed=True,
        sort=False,
    )
    .cumcount()
    .astype("int64")
    + 1
)

primary_event_number_map = (
    primary_estimation_events_windowed
    .set_index("primary_event_id")[
        "primary_event_number"
    ]
    .to_dict()
)

partition_event_index_map = (
    primary_estimation_events_windowed
    .set_index("primary_event_id")[
        "partition_event_index"
    ]
    .to_dict()
)

primary_event_to_batch_membership_enriched = (
    primary_event_to_batch_membership.copy()
    .sort_values(
        [
            "event_partition_order",
            "event_time_ns",
            "event_side_code",
            "primary_event_batch_number",
            "primary_event_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

primary_event_to_batch_membership_enriched[
    EVENT_PARTITION_FIELD
] = ordered_partition_string(
    primary_event_to_batch_membership_enriched[
        EVENT_PARTITION_FIELD
    ]
)

primary_event_to_batch_membership_enriched[
    "primary_event_number"
] = (
    primary_event_to_batch_membership_enriched[
        "primary_event_id"
    ].map(primary_event_number_map)
)

primary_event_to_batch_membership_enriched[
    "partition_event_index"
] = (
    primary_event_to_batch_membership_enriched[
        "primary_event_id"
    ].map(partition_event_index_map)
)

require(
    primary_event_to_batch_membership_enriched[
        "primary_event_number"
    ].notna().all(),
    (
        "At least one batch-membership row lacks a mapped "
        "primary_event_number."
    ),
)

require(
    primary_event_to_batch_membership_enriched[
        "partition_event_index"
    ].notna().all(),
    (
        "At least one batch-membership row lacks a mapped "
        "partition_event_index."
    ),
)

primary_event_to_batch_membership_enriched[
    "primary_event_number"
] = (
    primary_event_to_batch_membership_enriched[
        "primary_event_number"
    ].astype("int64")
)

primary_event_to_batch_membership_enriched[
    "partition_event_index"
] = (
    primary_event_to_batch_membership_enriched[
        "partition_event_index"
    ].astype("int64")
)


# ------------------------------------------------------------
# Batch-aware scoring metadata
# ------------------------------------------------------------

primary_scoring_batches = (
    primary_exact_time_batches_windowed[
        [
            "primary_event_batch_id",
            "primary_event_batch_number",
            "partition_batch_index",
            "event_partition_order",
            EVENT_PARTITION_FIELD,
            "event_time_ns",
            "relative_batch_time_ns",
            "relative_batch_time_seconds",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
            "unique_side_count",
            "batch_print_count",
            "batch_quantity",
            "batch_notional",
            "mixed_side_batch_flag",
            "simultaneous_batch_required_flag",
            "event_clock_origin_ns",
            "observation_window_duration_ns",
        ]
    ]
    .copy()
)

primary_scoring_batches[
    "score_with_history_strictly_before_batch_time"
] = True

primary_scoring_batches[
    "zero_lag_within_batch_excitation_allowed"
] = False

primary_scoring_batches[
    "apply_batch_excitation_after_all_members_scored"
] = True

primary_scoring_batches[
    "collector_sequence_is_trace_order_only"
] = True

primary_scoring_batches[
    "timestamp_jitter_allowed"
] = False

primary_scoring_batches[
    "event_removal_allowed"
] = False


# ------------------------------------------------------------
# Array index table
# ------------------------------------------------------------

primary_event_array_index = (
    primary_estimation_events_windowed[
        [
            "primary_event_id",
            "primary_event_number",
            "partition_event_index",
            "primary_event_batch_id",
            "primary_event_batch_number",
            "event_partition_order",
            EVENT_PARTITION_FIELD,
            "event_time_ns",
            "relative_event_time_ns",
            "relative_event_time_seconds",
            "event_side",
            "event_side_code",
            "event_print_count",
            "event_quantity",
            "event_notional",
            "aggregate_mark_available_time_ns",
            "aggregate_mark_relative_time_ns",
            "aggregate_mark_lag_after_event_ns",
        ]
    ]
    .copy()
    .sort_values(
        [
            "event_partition_order",
            "partition_event_index",
            "primary_event_number",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Partition arrays
# ------------------------------------------------------------

partition_event_arrays = {}
partition_batch_arrays = {}
partition_array_metadata_rows = []

for spec in PARTITION_SPECS:
    partition_name = spec.name

    event_slice = (
        primary_estimation_events_windowed.loc[
            primary_estimation_events_windowed[
                EVENT_PARTITION_FIELD
            ].eq(partition_name)
        ]
        .sort_values(
            [
                "partition_event_index",
                "primary_event_number",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    batch_slice = (
        primary_scoring_batches.loc[
            primary_scoring_batches[
                EVENT_PARTITION_FIELD
            ].eq(partition_name)
        ]
        .sort_values(
            [
                "partition_batch_index",
                "primary_event_batch_number",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    require(
        not event_slice.empty,
        f"No events available for array partition {partition_name}.",
    )

    require(
        not batch_slice.empty,
        f"No batches available for array partition {partition_name}.",
    )

    require(
        int(event_slice["relative_event_time_ns"].iloc[0]) == 0,
        (
            "First relative event time is not zero for "
            f"partition {partition_name}."
        ),
    )

    require(
        event_slice["relative_event_time_ns"]
        .is_monotonic_increasing,
        (
            "Relative event times are not nondecreasing for "
            f"partition {partition_name}."
        ),
    )

    require(
        batch_slice["relative_batch_time_ns"]
        .is_monotonic_increasing,
        (
            "Relative batch times are not nondecreasing for "
            f"partition {partition_name}."
        ),
    )

    partition_event_arrays[partition_name] = {
        "primary_event_number": event_slice[
            "primary_event_number"
        ].to_numpy(dtype=np.int64),
        "partition_event_index": event_slice[
            "partition_event_index"
        ].to_numpy(dtype=np.int64),
        "primary_event_batch_number": event_slice[
            "primary_event_batch_number"
        ].to_numpy(dtype=np.int64),
        "event_time_ns": event_slice[
            "event_time_ns"
        ].to_numpy(dtype=np.int64),
        "relative_event_time_ns": event_slice[
            "relative_event_time_ns"
        ].to_numpy(dtype=np.int64),
        "relative_event_time_seconds": event_slice[
            "relative_event_time_seconds"
        ].to_numpy(dtype=np.float64),
        "event_side_code": event_slice[
            "event_side_code"
        ].to_numpy(dtype=np.int8),
        "event_print_count": event_slice[
            "event_print_count"
        ].to_numpy(dtype=np.int64),
        "event_quantity": event_slice[
            "event_quantity"
        ].to_numpy(dtype=np.float64),
        "event_notional": event_slice[
            "event_notional"
        ].to_numpy(dtype=np.float64),
        "aggregate_mark_available_time_ns": event_slice[
            "aggregate_mark_available_time_ns"
        ].to_numpy(dtype=np.int64),
        "aggregate_mark_relative_time_ns": event_slice[
            "aggregate_mark_relative_time_ns"
        ].to_numpy(dtype=np.int64),
        "aggregate_mark_lag_after_event_ns": event_slice[
            "aggregate_mark_lag_after_event_ns"
        ].to_numpy(dtype=np.int64),
    }

    partition_batch_arrays[partition_name] = {
        "primary_event_batch_number": batch_slice[
            "primary_event_batch_number"
        ].to_numpy(dtype=np.int64),
        "partition_batch_index": batch_slice[
            "partition_batch_index"
        ].to_numpy(dtype=np.int64),
        "event_time_ns": batch_slice[
            "event_time_ns"
        ].to_numpy(dtype=np.int64),
        "relative_batch_time_ns": batch_slice[
            "relative_batch_time_ns"
        ].to_numpy(dtype=np.int64),
        "relative_batch_time_seconds": batch_slice[
            "relative_batch_time_seconds"
        ].to_numpy(dtype=np.float64),
        "batch_event_count": batch_slice[
            "batch_event_count"
        ].to_numpy(dtype=np.int64),
        "buy_event_count": batch_slice[
            "buy_event_count"
        ].to_numpy(dtype=np.int64),
        "sell_event_count": batch_slice[
            "sell_event_count"
        ].to_numpy(dtype=np.int64),
        "batch_print_count": batch_slice[
            "batch_print_count"
        ].to_numpy(dtype=np.int64),
        "mixed_side_batch_flag": batch_slice[
            "mixed_side_batch_flag"
        ].to_numpy(dtype=bool),
        "simultaneous_batch_required_flag": batch_slice[
            "simultaneous_batch_required_flag"
        ].to_numpy(dtype=bool),
    }

    partition_array_metadata_rows.append(
        {
            "partition_order": int(spec.order),
            EVENT_PARTITION_FIELD: partition_name,
            "primary_event_count": int(len(event_slice)),
            "primary_batch_count": int(len(batch_slice)),
            "first_relative_event_time_ns": int(
                event_slice[
                    "relative_event_time_ns"
                ].iloc[0]
            ),
            "last_relative_event_time_ns": int(
                event_slice[
                    "relative_event_time_ns"
                ].iloc[-1]
            ),
            "observation_window_duration_ns": int(
                event_slice[
                    "observation_window_duration_ns"
                ].iloc[0]
            ),
            "left_censoring_duration_ns": int(
                event_slice[
                    "left_censoring_duration_ns"
                ].iloc[0]
            ),
            "buy_event_count": int(
                event_slice["event_side"].eq("BUY").sum()
            ),
            "sell_event_count": int(
                event_slice["event_side"].eq("SELL").sum()
            ),
            "print_count_sum": int(
                event_slice["event_print_count"].sum()
            ),
            "quantity_sum": float(
                event_slice["event_quantity"].sum()
            ),
            "notional_sum": float(
                event_slice["event_notional"].sum()
            ),
            "simultaneous_batch_count": int(
                batch_slice[
                    "simultaneous_batch_required_flag"
                ].sum()
            ),
            "mixed_side_batch_count": int(
                batch_slice[
                    "mixed_side_batch_flag"
                ].sum()
            ),
            "max_batch_event_count": int(
                batch_slice["batch_event_count"].max()
            ),
            "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
        }
    )

partition_array_metadata = pd.DataFrame.from_records(
    partition_array_metadata_rows
)


# ------------------------------------------------------------
# Array schema contract
# ------------------------------------------------------------

partition_array_schema = {
    "schema_name": "NOTEBOOK_04_PARTITION_EVENT_ARRAY_SCHEMA",
    "schema_version": "1.0",
    "primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
    "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
    "time_unit": "nanoseconds",
    "float_time_unit": "seconds",
    "side_code": dict(SIDE_CODE),
    "partition_key": EVENT_PARTITION_FIELD,
    "event_arrays": {
        "primary_event_number": "int64",
        "partition_event_index": "int64",
        "primary_event_batch_number": "int64",
        "event_time_ns": "int64",
        "relative_event_time_ns": "int64",
        "relative_event_time_seconds": "float64",
        "event_side_code": "int8",
        "event_print_count": "int64",
        "event_quantity": "float64",
        "event_notional": "float64",
        "aggregate_mark_available_time_ns": "int64",
        "aggregate_mark_relative_time_ns": "int64",
        "aggregate_mark_lag_after_event_ns": "int64",
    },
    "batch_arrays": {
        "primary_event_batch_number": "int64",
        "partition_batch_index": "int64",
        "event_time_ns": "int64",
        "relative_batch_time_ns": "int64",
        "relative_batch_time_seconds": "float64",
        "batch_event_count": "int64",
        "buy_event_count": "int64",
        "sell_event_count": "int64",
        "batch_print_count": "int64",
        "mixed_side_batch_flag": "bool",
        "simultaneous_batch_required_flag": "bool",
    },
    "scoring_semantics": {
        "history_used_for_batch_scoring": (
            "STRICTLY_BEFORE_BATCH_TIME"
        ),
        "zero_lag_within_batch_excitation_allowed": False,
        "apply_batch_excitation_after_all_members_scored": True,
        "collector_sequence_inside_tie_is_trace_order_only": True,
        "timestamp_jitter_allowed": False,
        "event_removal_allowed": False,
    },
}


# ------------------------------------------------------------
# Gate ledger
# ------------------------------------------------------------

first_relative_event_zero_pass = bool(
    partition_array_metadata[
        "first_relative_event_time_ns"
    ].eq(0).all()
)

event_times_inside_contract_pass = bool(
    primary_estimation_events_windowed[
        "event_time_inside_contract_window_flag"
    ].all()
)

relative_event_times_inside_window_pass = bool(
    primary_estimation_events_windowed[
        "relative_event_time_inside_window_flag"
    ].all()
)

batch_times_inside_contract_pass = bool(
    primary_exact_time_batches_windowed[
        "batch_time_inside_contract_window_flag"
    ].all()
)

relative_batch_times_inside_window_pass = bool(
    primary_exact_time_batches_windowed[
        "relative_batch_time_inside_window_flag"
    ].all()
)

aggregate_mark_not_before_event_pass = bool(
    primary_estimation_events_windowed[
        "aggregate_mark_lag_after_event_ns"
    ].ge(0).all()
)

aggregate_mark_inside_contract_pass = bool(
    primary_estimation_events_windowed[
        "aggregate_mark_inside_contract_window_flag"
    ].all()
)

event_time_nondecreasing_pass = bool(
    primary_estimation_events_windowed.groupby(
        EVENT_PARTITION_FIELD,
        observed=True,
        sort=False,
    )["relative_event_time_ns"]
    .apply(lambda values: values.is_monotonic_increasing)
    .all()
)

batch_time_nondecreasing_pass = bool(
    primary_exact_time_batches_windowed.groupby(
        EVENT_PARTITION_FIELD,
        observed=True,
        sort=False,
    )["relative_batch_time_ns"]
    .apply(lambda values: values.is_monotonic_increasing)
    .all()
)

event_batch_membership_conserved_pass = bool(
    len(primary_event_to_batch_membership_enriched)
    == len(primary_estimation_events_windowed)
    and primary_event_to_batch_membership_enriched[
        "primary_event_id"
    ].is_unique
)

batch_event_count_conserved_pass = bool(
    int(
        primary_exact_time_batches_windowed[
            "batch_event_count"
        ].sum()
    )
    == len(primary_estimation_events_windowed)
)

array_event_count_conserved_pass = bool(
    int(
        partition_array_metadata[
            "primary_event_count"
        ].sum()
    )
    == len(primary_estimation_events_windowed)
)

array_print_count_conserved_pass = bool(
    int(
        partition_array_metadata[
            "print_count_sum"
        ].sum()
    )
    == EXPECTED_MATCHED_TRADE_ROWS
)

observed_simultaneous_batch_required = bool(
    primary_exact_time_batches_windowed[
        "simultaneous_batch_required_flag"
    ].any()
)

batch_requirement_matches_contract_pass = bool(
    observed_simultaneous_batch_required
    == PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
)

simultaneous_batch_count = int(
    primary_exact_time_batches_windowed[
        "simultaneous_batch_required_flag"
    ].sum()
)

mixed_side_batch_count = int(
    primary_exact_time_batches_windowed[
        "mixed_side_batch_flag"
    ].sum()
)

aggregate_mark_after_event_count = int(
    primary_estimation_events_windowed[
        "aggregate_mark_lag_after_event_ns"
    ].gt(0).sum()
)

window_and_array_gates = [
    make_gate(
        gate="observation_window_contract_covers_all_partitions",
        passed=(
            len(observation_window_contract)
            == len(PARTITION_SPECS)
            and observation_window_contract[
                EVENT_PARTITION_FIELD
            ].is_unique
        ),
        severity="BLOCKING",
        detail=(
            "window_rows="
            f"{len(observation_window_contract)}"
        ),
    ),
    make_gate(
        gate="observation_windows_have_positive_duration",
        passed=bool(
            observation_window_contract[
                "observation_window_duration_ns"
            ].gt(0).all()
        ),
        severity="BLOCKING",
        detail=(
            "minimum_duration_ns="
            f"{int(observation_window_contract['observation_window_duration_ns'].min())}"
        ),
    ),
    make_gate(
        gate="left_censoring_duration_nonnegative",
        passed=bool(
            observation_window_contract[
                "left_censoring_duration_ns"
            ].ge(0).all()
        ),
        severity="BLOCKING",
        detail=(
            "maximum_left_censoring_ns="
            f"{int(observation_window_contract['left_censoring_duration_ns'].max())}"
        ),
    ),
    make_gate(
        gate="first_relative_event_time_zero_by_partition",
        passed=first_relative_event_zero_pass,
        severity="BLOCKING",
        detail=(
            "partitions_checked="
            f"{len(partition_array_metadata)}"
        ),
    ),
    make_gate(
        gate="primary_event_times_inside_contract_windows",
        passed=event_times_inside_contract_pass,
        severity="BLOCKING",
        detail=(
            "outside_count="
            f"{int((~primary_estimation_events_windowed['event_time_inside_contract_window_flag']).sum())}"
        ),
    ),
    make_gate(
        gate="relative_event_times_inside_observation_windows",
        passed=relative_event_times_inside_window_pass,
        severity="BLOCKING",
        detail=(
            "outside_count="
            f"{int((~primary_estimation_events_windowed['relative_event_time_inside_window_flag']).sum())}"
        ),
    ),
    make_gate(
        gate="primary_batch_times_inside_contract_windows",
        passed=batch_times_inside_contract_pass,
        severity="BLOCKING",
        detail=(
            "outside_count="
            f"{int((~primary_exact_time_batches_windowed['batch_time_inside_contract_window_flag']).sum())}"
        ),
    ),
    make_gate(
        gate="relative_batch_times_inside_observation_windows",
        passed=relative_batch_times_inside_window_pass,
        severity="BLOCKING",
        detail=(
            "outside_count="
            f"{int((~primary_exact_time_batches_windowed['relative_batch_time_inside_window_flag']).sum())}"
        ),
    ),
    make_gate(
        gate="relative_event_times_nondecreasing_by_partition",
        passed=event_time_nondecreasing_pass,
        severity="BLOCKING",
        detail=(
            "Strict increase is not required because simultaneous "
            "events are represented by batches."
        ),
    ),
    make_gate(
        gate="relative_batch_times_nondecreasing_by_partition",
        passed=batch_time_nondecreasing_pass,
        severity="BLOCKING",
        detail=(
            "Batch times must be nondecreasing inside each partition."
        ),
    ),
    make_gate(
        gate="aggregate_marks_not_before_event_times",
        passed=aggregate_mark_not_before_event_pass,
        severity="BLOCKING",
        detail=(
            "negative_mark_lag_count="
            f"{int(primary_estimation_events_windowed['aggregate_mark_lag_after_event_ns'].lt(0).sum())}"
        ),
    ),
    make_gate(
        gate="aggregate_marks_inside_contract_windows",
        passed=aggregate_mark_inside_contract_pass,
        severity="BLOCKING",
        detail=(
            "outside_count="
            f"{int((~primary_estimation_events_windowed['aggregate_mark_inside_contract_window_flag']).sum())}"
        ),
    ),
    make_gate(
        gate="event_to_batch_membership_conserved",
        passed=event_batch_membership_conserved_pass,
        severity="BLOCKING",
        detail=(
            "membership_rows="
            f"{len(primary_event_to_batch_membership_enriched):,}; "
            "primary_event_rows="
            f"{len(primary_estimation_events_windowed):,}"
        ),
    ),
    make_gate(
        gate="batch_event_counts_conserve_primary_events",
        passed=batch_event_count_conserved_pass,
        severity="BLOCKING",
        detail=(
            "batch_event_count_sum="
            f"{int(primary_exact_time_batches_windowed['batch_event_count'].sum()):,}; "
            "primary_event_rows="
            f"{len(primary_estimation_events_windowed):,}"
        ),
    ),
    make_gate(
        gate="partition_arrays_conserve_primary_events",
        passed=array_event_count_conserved_pass,
        severity="BLOCKING",
        detail=(
            "array_event_count_sum="
            f"{int(partition_array_metadata['primary_event_count'].sum()):,}; "
            "primary_event_rows="
            f"{len(primary_estimation_events_windowed):,}"
        ),
    ),
    make_gate(
        gate="partition_arrays_conserve_trade_prints",
        passed=array_print_count_conserved_pass,
        severity="BLOCKING",
        detail=(
            "array_print_count_sum="
            f"{int(partition_array_metadata['print_count_sum'].sum()):,}; "
            "expected_trade_rows="
            f"{EXPECTED_MATCHED_TRADE_ROWS:,}"
        ),
    ),
    make_gate(
        gate="batch_requirement_matches_frozen_contract",
        passed=batch_requirement_matches_contract_pass,
        severity="BLOCKING",
        detail=(
            "observed_required="
            f"{observed_simultaneous_batch_required}; "
            "contract_required="
            f"{PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE}"
        ),
    ),
    make_gate(
        gate="simultaneous_batches_absent",
        passed=(simultaneous_batch_count == 0),
        severity="WARNING",
        detail=(
            "simultaneous_batch_count="
            f"{simultaneous_batch_count}; "
            "timestamp_interface="
            f"{PRIMARY_TIMESTAMP_INTERFACE}"
        ),
    ),
    make_gate(
        gate="mixed_side_batches_absent",
        passed=(mixed_side_batch_count == 0),
        severity="WARNING",
        detail=(
            "mixed_side_batch_count="
            f"{mixed_side_batch_count}"
        ),
    ),
    make_gate(
        gate="aggregate_marks_available_at_event_time",
        passed=(aggregate_mark_after_event_count == 0),
        severity="WARNING",
        detail=(
            "aggregate_mark_after_event_count="
            f"{aggregate_mark_after_event_count}"
        ),
    ),
]

window_and_array_gate_frame = gate_results_to_frame(
    window_and_array_gates
)

fail_if_blocking_gate_failed(
    window_and_array_gate_frame
)


# ------------------------------------------------------------
# Compact summaries
# ------------------------------------------------------------

window_and_array_summary = pd.DataFrame.from_records(
    [
        {
            "primary_event_representation": (
                PRIMARY_EVENT_REPRESENTATION
            ),
            "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
            "partition_count": int(
                len(observation_window_contract)
            ),
            "primary_event_rows": int(
                len(primary_estimation_events_windowed)
            ),
            "primary_batch_rows": int(
                len(primary_exact_time_batches_windowed)
            ),
            "simultaneous_batch_count": simultaneous_batch_count,
            "mixed_side_batch_count": mixed_side_batch_count,
            "events_inside_simultaneous_batches": int(
                primary_exact_time_batches_windowed.loc[
                    primary_exact_time_batches_windowed[
                        "simultaneous_batch_required_flag"
                    ],
                    "batch_event_count",
                ].sum()
            ),
            "max_batch_event_count": int(
                primary_exact_time_batches_windowed[
                    "batch_event_count"
                ].max()
            ),
            "aggregate_mark_after_event_count": (
                aggregate_mark_after_event_count
            ),
            "first_relative_event_zero_all_partitions": (
                first_relative_event_zero_pass
            ),
            "array_event_count_conserved": (
                array_event_count_conserved_pass
            ),
            "array_print_count_conserved": (
                array_print_count_conserved_pass
            ),
        }
    ]
)

display(observation_window_contract)
display(partition_array_metadata)
display(window_and_array_summary)
display(window_and_array_gate_frame)

display(primary_event_array_index.head(20))
display(primary_scoring_batches.head(20))
display(primary_event_to_batch_membership_enriched.head(20))

{
    "status": "PASS",
    "primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
    "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
    "partition_count": int(len(observation_window_contract)),
    "primary_event_rows": int(len(primary_estimation_events_windowed)),
    "primary_batch_rows": int(len(primary_exact_time_batches_windowed)),
    "simultaneous_batch_count": simultaneous_batch_count,
    "mixed_side_batch_count": mixed_side_batch_count,
    "partition_arrays_constructed": sorted(
        partition_event_arrays.keys()
    ),
    "first_relative_event_zero_all_partitions": (
        first_relative_event_zero_pass
    ),
    "next_action": (
        "Persist event tables, batch metadata, partition arrays, "
        "array schemas, and read-back checksums."
    ),
}

,partition_order,event_partition,contract_sequence_start,contract_sequence_end_exclusive,contract_start_ns,contract_end_exclusive_ns,contract_duration_ns,event_clock_origin_ns,first_event_time_ns,last_event_time_ns,left_censoring_duration_ns,observation_window_duration_ns,primary_event_count,primary_batch_count,simultaneous_batch_count,mixed_side_batch_count,contract_interval_convention,event_clock_origin_rule,left_censoring_preserved
0,1,DEVELOPMENT,1,51840,1783665467531985400,1783667269391572100,1801859586700,1783665468766951600,1783665468766951600,1783667269232205000,1234966200,1800624620500,7004,6859,111,26,"[start, end)",FIRST_PRIMARY_EVENT_LOCAL_RECEIPT_TIME,True
1,2,CALIBRATION,51840,72576,1783667269391572100,1783667989690751200,720299179100,1783667270546982000,1783667270546982000,1783667989349685300,1155409900,719143769200,2493,2400,68,5,"[start, end)",FIRST_PRIMARY_EVENT_LOCAL_RECEIPT_TIME,True
2,3,VALIDATION,72576,88127,1783667989690751200,1783668525057534700,535366783500,1783667989690751200,1783667989690751200,1783668524924691300,0,535366783500,2048,1999,40,5,"[start, end)",FIRST_PRIMARY_EVENT_LOCAL_RECEIPT_TIME,True
3,4,ENGINEERING_HOLDOUT,88127,103678,1783668525057534700,1783669066749750801,541692216101,1783668525205920700,1783668525205920700,1783669066706302700,148386000,541543830101,2342,2306,30,5,"[start, end)",FIRST_PRIMARY_EVENT_LOCAL_RECEIPT_TIME,True


,partition_order,event_partition,primary_event_count,primary_batch_count,first_relative_event_time_ns,last_relative_event_time_ns,observation_window_duration_ns,left_censoring_duration_ns,buy_event_count,sell_event_count,print_count_sum,quantity_sum,notional_sum,simultaneous_batch_count,mixed_side_batch_count,max_batch_event_count,timestamp_interface
0,1,DEVELOPMENT,7004,6859,0,1800465253400,1800624620500,1234966200,3414,3590,33820,165.51400,1.057147e+07,111,26,6,SIMULTANEOUS_EVENT_BATCH_REQUIRED
1,2,CALIBRATION,2493,2400,0,718802703300,719143769200,1155409900,1230,1263,13532,47.88126,3.058811e+06,68,5,4,SIMULTANEOUS_EVENT_BATCH_REQUIRED
2,3,VALIDATION,2048,1999,0,535233940100,535366783500,0,1013,1035,10198,51.96335,3.323781e+06,40,5,5,SIMULTANEOUS_EVENT_BATCH_REQUIRED
3,4,ENGINEERING_HOLDOUT,2342,2306,0,541500382000,541543830101,148386000,949,1393,10133,63.83986,4.081884e+06,30,5,6,SIMULTANEOUS_EVENT_BATCH_REQUIRED


,primary_event_representation,timestamp_interface,partition_count,primary_event_rows,primary_batch_rows,simultaneous_batch_count,mixed_side_batch_count,events_inside_simultaneous_batches,max_batch_event_count,aggregate_mark_after_event_count,first_relative_event_zero_all_partitions,array_event_count_conserved,array_print_count_conserved
0,SAME_MS_SAME_SIDE_BURSTS,SIMULTANEOUS_EVENT_BATCH_REQUIRED,4,13887,13564,249,41,572,6,1575,True,True,True


,gate,status,severity,detail
0,observation_window_contract_covers_all_partitions,PASS,BLOCKING,window_rows=4
1,observation_windows_have_positive_duration,PASS,BLOCKING,minimum_duration_ns=535366783500
2,left_censoring_duration_nonnegative,PASS,BLOCKING,maximum_left_censoring_ns=1234966200
3,first_relative_event_time_zero_by_partition,PASS,BLOCKING,partitions_checked=4
4,primary_event_times_inside_contract_windows,PASS,BLOCKING,outside_count=0
5,relative_event_times_inside_observation_windows,PASS,BLOCKING,outside_count=0
6,primary_batch_times_inside_contract_windows,PASS,BLOCKING,outside_count=0
7,relative_batch_times_inside_observation_windows,PASS,BLOCKING,outside_count=0
8,relative_event_times_nondecreasing_by_partition,PASS,BLOCKING,Strict increase is not required because simultaneous events are represented by batches.
9,relative_batch_times_nondecreasing_by_partition,PASS,BLOCKING,Batch times must be nondecreasing inside each partition.


,primary_event_id,primary_event_number,partition_event_index,primary_event_batch_id,primary_event_batch_number,event_partition_order,event_partition,event_time_ns,relative_event_time_ns,relative_event_time_seconds,event_side,event_side_code,event_print_count,event_quantity,event_notional,aggregate_mark_available_time_ns,aggregate_mark_relative_time_ns,aggregate_mark_lag_after_event_ns
0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000001,1,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000001,1,1,DEVELOPMENT,1783665468766951600,0,0.000000,BUY,0,1,0.00046,29.400610,1783665468766951600,0,0
1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000002,2,2,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000002,2,1,DEVELOPMENT,1783665469052166500,285214900,0.285215,BUY,0,1,0.00060,38.348622,1783665469052166500,285214900,0
2,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000003,3,3,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000003,3,1,DEVELOPMENT,1783665469437964400,671012800,0.671013,BUY,0,1,0.00020,12.782874,1783665469437964400,671012800,0
3,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000004,4,4,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000004,4,1,DEVELOPMENT,1783665469692104200,925152600,0.925153,BUY,0,1,0.00039,24.926604,1783665469692104200,925152600,0
4,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000005,5,5,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000005,5,1,DEVELOPMENT,1783665470242312100,1475360500,1.475361,SELL,1,1,0.00046,29.400606,1783665470242312100,1475360500,0
5,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000006,6,6,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000006,6,1,DEVELOPMENT,1783665470304239100,1537287500,1.537287,SELL,1,1,0.00018,11.504585,1783665470304239100,1537287500,0
6,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000007,7,7,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000007,7,1,DEVELOPMENT,1783665470597198800,1830247200,1.830247,SELL,1,1,0.03495,2233.806882,1783665470597198800,1830247200,0
7,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000008,8,8,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000008,8,1,DEVELOPMENT,1783665471662326600,2895375000,2.895375,SELL,1,1,0.00056,35.792042,1783665471662326600,2895375000,0
8,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000009,9,9,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000009,9,1,DEVELOPMENT,1783665471742062100,2975110500,2.975110,SELL,1,1,0.00020,12.782872,1783665471742062100,2975110500,0
9,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000010,10,10,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTI

,primary_event_batch_id,primary_event_batch_number,partition_batch_index,event_partition_order,event_partition,event_time_ns,relative_batch_time_ns,relative_batch_time_seconds,batch_event_count,buy_event_count,sell_event_count,unique_side_count,batch_print_count,batch_quantity,batch_notional,mixed_side_batch_flag,simultaneous_batch_required_flag,event_clock_origin_ns,observation_window_duration_ns,score_with_history_strictly_before_batch_time,zero_lag_within_batch_excitation_allowed,apply_batch_excitation_after_all_members_scored,collector_sequence_is_trace_order_only,timestamp_jitter_allowed,event_removal_allowed
0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000001,1,1,1,DEVELOPMENT,1783665468766951600,0,0.000000,1,1,0,1,1,0.00046,29.400610,False,False,1783665468766951600,1800624620500,True,False,True,True,False,False
1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000002,2,2,1,DEVELOPMENT,1783665469052166500,285214900,0.285215,1,1,0,1,1,0.00060,38.348622,False,False,1783665468766951600,1800624620500,True,False,True,True,False,False
2,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000003,3,3,1,DEVELOPMENT,1783665469437964400,671012800,0.671013,1,1,0,1,1,0.00020,12.782874,False,False,1783665468766951600,1800624620500,True,False,True,True,False,False
3,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000004,4,4,1,DEVELOPMENT,1783665469692104200,925152600,0.925153,1,1,0,1,1,0.00039,24.926604,False,False,1783665468766951600,1800624620500,True,False,True,True,False,False
4,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000005,5,5,1,DEVELOPMENT,1783665470242312100,1475360500,1.475361,1,0,1,1,1,0.00046,29.400606,False,False,1783665468766951600,1800624620500,True,False,True,True,False,False
5,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000006,6,6,1,DEVELOPMENT,1783665470304239100,1537287500,1.537287,1,0,1,1,1,0.00018,11.504585,False,False,1783665468766951600,1800624620500,True,False,True,True,False,False
6,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000007,7,7,1,DEVELOPMENT,1783665470597198800,1830247200,1.830247,1,0,1,1,1,0.03495,2233.806882,False,False,1783665468766951600,1800624620500,True,False,True,True,False,False
7,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000008,8,8,1,DEVELOPMENT,1783665471662326600,2895375000,2.895375,1,0,1,1,1,0.00056,35.792042,False,False,1783665468766951600,1800624620500,True,False,True,True,False,False
8,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000009,9,9,1,DEVELOPMENT,1783665471742062100,2975110500,2.975110,1,0,1,1,1,0.00020,12.782872,False,False,1783665468766951600,1800624620500,True,False,True,True,False,False
9,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000010,10,10,1,DEVELOPMENT,1783665471862162900,3095211300,3.095211,1,1,0,1,1,0.00046,29.400610,False,False,1783665468766951600,1800624620500,True,False,True,True,False,False


,primary_event_id,primary_event_batch_number,event_partition_order,event_partition,event_time_ns,event_side,event_side_code,primary_event_batch_id,primary_event_number,partition_event_index
0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000001,1,1,DEVELOPMENT,1783665468766951600,BUY,0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000001,1,1
1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000002,2,1,DEVELOPMENT,1783665469052166500,BUY,0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000002,2,2
2,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000003,3,1,DEVELOPMENT,1783665469437964400,BUY,0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000003,3,3
3,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000004,4,1,DEVELOPMENT,1783665469692104200,BUY,0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000004,4,4
4,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000005,5,1,DEVELOPMENT,1783665470242312100,SELL,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000005,5,5
5,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000006,6,1,DEVELOPMENT,1783665470304239100,SELL,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000006,6,6
6,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000007,7,1,DEVELOPMENT,1783665470597198800,SELL,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000007,7,7
7,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000008,8,1,DEVELOPMENT,1783665471662326600,SELL,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000008,8,8
8,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000009,9,1,DEVELOPMENT,1783665471742062100,SELL,1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000009,9,9
9,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__BURST_ALL__00000010,10,1,DEVELOPMENT,1783665471862162900,BUY,0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__PRIMARY_BATCH__00000010,10,10


{'status': 'PASS',
 'primary_event_representation': 'SAME_MS_SAME_SIDE_BURSTS',
 'timestamp_interface': 'SIMULTANEOUS_EVENT_BATCH_REQUIRED',
 'partition_count': 4,
 'primary_event_rows': 13887,
 'primary_batch_rows': 13564,
 'simultaneous_batch_count': 249,
 'mixed_side_batch_count': 41,
 'partition_arrays_constructed': ['CALIBRATION',
  'DEVELOPMENT',
  'ENGINEERING_HOLDOUT',
  'VALIDATION'],
 'first_relative_event_zero_all_partitions': True,
 'next_action': 'Persist event tables, batch metadata, partition arrays, array schemas, and read-back checksums.'}

In [20]:
# ============================================================
# 04_EVENT_STREAM_CONSTRUCTION
# Cell 10 — Persist event tables, batch metadata, partition
#           arrays, schemas, manifests, handoff, and read-back
#           checksums
#
# This cell fixes authority at the persistence layer:
# - table hashes are computed from canonical table payloads,
#   not from volatile parquet metadata;
# - sidecar contracts record the canonical payload hash;
# - read-back validation compares canonical payload hashes;
# - no table-embedded contract hash is used as an invariant.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    "primary_estimation_events_windowed" in globals(),
    "primary_estimation_events_windowed does not exist. Run Cell 09 first.",
)

require(
    "primary_exact_time_batches_windowed" in globals(),
    "primary_exact_time_batches_windowed does not exist. Run Cell 09 first.",
)

require(
    "primary_event_to_batch_membership_enriched" in globals(),
    (
        "primary_event_to_batch_membership_enriched does not exist. "
        "Run Cell 09 first."
    ),
)

require(
    "primary_event_array_index" in globals(),
    "primary_event_array_index does not exist. Run Cell 09 first.",
)

require(
    "primary_scoring_batches" in globals(),
    "primary_scoring_batches does not exist. Run Cell 09 first.",
)

require(
    "partition_event_arrays" in globals(),
    "partition_event_arrays does not exist. Run Cell 09 first.",
)

require(
    "partition_batch_arrays" in globals(),
    "partition_batch_arrays does not exist. Run Cell 09 first.",
)

require(
    "partition_array_schema" in globals(),
    "partition_array_schema does not exist. Run Cell 09 first.",
)

require(
    "observation_window_contract" in globals(),
    "observation_window_contract does not exist. Run Cell 09 first.",
)

require(
    "partition_array_metadata" in globals(),
    "partition_array_metadata does not exist. Run Cell 09 first.",
)

require(
    "window_and_array_gate_frame" in globals(),
    "window_and_array_gate_frame does not exist. Run Cell 09 first.",
)

require(
    bool(window_and_array_gate_frame.loc[
        window_and_array_gate_frame["severity"].eq("BLOCKING"),
        "status",
    ].eq("PASS").all()),
    "Cell 09 has at least one failed blocking gate.",
)


# ------------------------------------------------------------
# Local persistence helpers
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import tempfile


def _n04_first_existing_global(
    names: list[str],
    default=None,
):
    """Return the first existing global value from a list of names."""
    for name in names:
        if name in globals():
            value = globals()[name]
            if value is not None:
                return value
    return default


def _n04_resolve_v01_root() -> Path:
    """Resolve the V0.1 project root from the current notebook location."""
    candidate = _n04_first_existing_global(
        [
            "V01_ROOT",
            "V0_1_ROOT",
            "V0_1_PROJECT_ROOT",
            "PROJECT_ROOT",
            "v0_1_root",
            "project_root",
        ],
        default=None,
    )

    if candidate is not None:
        candidate_path = Path(candidate)
        if candidate_path.name.lower() == "notebooks":
            return candidate_path.parent
        return candidate_path

    cwd = Path.cwd()

    if cwd.name.lower() == "notebooks":
        return cwd.parent

    if cwd.name.lower() == "v0.1":
        return cwd

    for parent in [cwd, *cwd.parents]:
        if parent.name.lower() == "v0.1":
            return parent

    return cwd


def _n04_utc_now_iso() -> str:
    """Return a timezone-aware UTC timestamp string."""
    return datetime.now(timezone.utc).isoformat()


def _n04_file_sha256(path: Path) -> str:
    """Compute a SHA-256 hash over raw file bytes."""
    h = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def _n04_jsonable_scalar(value):
    """Convert scalar values into stable JSON-compatible objects."""
    if value is None:
        return None

    if value is pd.NA:
        return None

    if value is pd.NaT:
        return None

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, bool):
        return value

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, int):
        return value

    if isinstance(value, np.floating):
        value = float(value)
        if math.isnan(value):
            return None
        if math.isinf(value):
            return str(value)
        return value

    if isinstance(value, float):
        if math.isnan(value):
            return None
        if math.isinf(value):
            return str(value)
        return value

    if isinstance(value, pd.Timestamp):
        if pd.isna(value):
            return None
        return value.isoformat()

    if isinstance(value, datetime):
        return value.isoformat()

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, bytes):
        return value.hex()

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, str):
        return value

    return str(value)


def _n04_canonical_json_bytes(payload) -> bytes:
    """Serialize payload to deterministic UTF-8 JSON bytes."""
    return json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")


def _n04_canonical_json_sha256(payload) -> str:
    """Compute canonical JSON SHA-256."""
    return hashlib.sha256(
        _n04_canonical_json_bytes(payload)
    ).hexdigest()


def _n04_atomic_write_json(path: Path, payload: dict) -> None:
    """Atomically write a JSON payload."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        suffix=".tmp.json",
        delete=False,
        dir=str(path.parent),
    ) as tmp:
        json.dump(
            payload,
            tmp,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        )
        tmp.write("\n")
        tmp_path = Path(tmp.name)

    os.replace(tmp_path, path)


def _n04_prepare_frame_for_write(frame: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare a DataFrame for parquet persistence without relying on
    pandas/parquet metadata for semantic identity.
    """
    prepared = frame.copy()
    prepared.columns = [str(column) for column in prepared.columns]

    for column in prepared.columns:
        series = prepared[column]

        if isinstance(series.dtype, pd.CategoricalDtype):
            prepared[column] = series.astype("string")
            continue

        if pd.api.types.is_object_dtype(series.dtype):
            prepared[column] = series.map(
                lambda value: (
                    None
                    if _n04_jsonable_scalar(value) is None
                    else _n04_jsonable_scalar(value)
                )
            )

            inferred = pd.api.types.infer_dtype(
                prepared[column].dropna(),
                skipna=True,
            )

            if inferred in {
                "string",
                "unicode",
                "mixed",
                "mixed-integer",
                "mixed-integer-float",
                "bytes",
                "empty",
            }:
                prepared[column] = prepared[column].astype("string")

    return prepared.reset_index(drop=True)


def _n04_table_payload_sha256(frame: pd.DataFrame) -> str:
    """
    Compute a stable content hash for a table.

    The hash is row-order-sensitive and column-order-sensitive by design.
    This is intentional because event arrays and handoffs require stable
    ordering authority.
    """
    h = hashlib.sha256()

    columns = [str(column) for column in frame.columns]

    h.update(
        _n04_canonical_json_bytes(
            {
                "columns": columns,
                "row_count": int(len(frame)),
            }
        )
    )

    for row in frame.itertuples(index=False, name=None):
        canonical_row = [
            _n04_jsonable_scalar(value)
            for value in row
        ]
        h.update(b"\n")
        h.update(
            json.dumps(
                canonical_row,
                ensure_ascii=False,
                separators=(",", ":"),
            ).encode("utf-8")
        )

    return h.hexdigest()


def _n04_write_parquet_table_artifact(
    *,
    label: str,
    frame: pd.DataFrame,
    output_dir: Path,
    artifact_type: str,
    required_nonempty: bool = True,
) -> dict:
    """
    Persist a table, read it back, and verify semantic content by
    canonical payload hash.

    This does not compare volatile parquet metadata.
    """
    require(
        isinstance(frame, pd.DataFrame),
        f"{label} is not a DataFrame.",
    )

    if required_nonempty:
        require(
            not frame.empty,
            f"{label} is unexpectedly empty.",
        )

    prepared = _n04_prepare_frame_for_write(frame)

    table_payload_sha256 = _n04_table_payload_sha256(
        prepared
    )

    output_dir.mkdir(parents=True, exist_ok=True)

    table_path = output_dir / (
        f"{N04_COMBINED_PREFIX}__"
        f"{N04_NOTEBOOK_NAME}__"
        f"{label}.parquet"
    )

    contract_path = output_dir / (
        f"{N04_COMBINED_PREFIX}__"
        f"{N04_NOTEBOOK_NAME}__"
        f"{label}__contract.json"
    )

    tmp_table_path = output_dir / (
        f"{N04_COMBINED_PREFIX}__"
        f"{N04_NOTEBOOK_NAME}__"
        f"{label}.__tmp__.parquet"
    )

    if tmp_table_path.exists():
        tmp_table_path.unlink()

    prepared.to_parquet(
        tmp_table_path,
        index=False,
    )

    os.replace(tmp_table_path, table_path)

    readback = pd.read_parquet(table_path)
    readback_prepared = _n04_prepare_frame_for_write(
        readback
    )

    observed_payload_sha256 = _n04_table_payload_sha256(
        readback_prepared
    )

    require(
        observed_payload_sha256 == table_payload_sha256,
        (
            f"{label} read-back table payload hash mismatch. "
            f"expected={table_payload_sha256}; "
            f"observed={observed_payload_sha256}"
        ),
    )

    require(
        list(readback_prepared.columns) == list(prepared.columns),
        f"{label} read-back column order mismatch.",
    )

    require(
        len(readback_prepared) == len(prepared),
        f"{label} read-back row count mismatch.",
    )

    table_file_sha256 = _n04_file_sha256(table_path)

    table_contract_payload = {
        "artifact_type": artifact_type,
        "artifact_schema_version": "1.0",
        "acceptance_status": "PASS",
        "project_name": N04_PROJECT_NAME,
        "pipeline_version": N04_PIPELINE_VERSION,
        "notebook_name": N04_NOTEBOOK_NAME,
        "source_run_prefix": N04_SOURCE_RUN_PREFIX,
        "v0_1_run_id": N04_V01_RUN_ID,
        "combined_prefix": N04_COMBINED_PREFIX,
        "label": label,
        "format": "parquet",
        "path": str(table_path),
        "contract_path": str(contract_path),
        "row_count": int(len(prepared)),
        "column_count": int(len(prepared.columns)),
        "columns": list(prepared.columns),
        "table_payload_sha256": table_payload_sha256,
        "readback_table_payload_sha256": observed_payload_sha256,
        "file_sha256": table_file_sha256,
        "created_at_utc": _n04_utc_now_iso(),
        "hash_policy": (
            "canonical row-order-sensitive table payload hash; "
            "parquet metadata is not semantic authority"
        ),
    }

    table_contract_sha256 = _n04_canonical_json_sha256(
        table_contract_payload
    )

    table_contract = dict(table_contract_payload)
    table_contract["contract_payload_sha256"] = (
        table_contract_sha256
    )

    _n04_atomic_write_json(
        contract_path,
        table_contract,
    )

    contract_readback = json.loads(
        contract_path.read_text(encoding="utf-8")
    )

    observed_contract_sha256 = _n04_canonical_json_sha256(
        {
            key: value
            for key, value in contract_readback.items()
            if key != "contract_payload_sha256"
        }
    )

    require(
        observed_contract_sha256
        == contract_readback["contract_payload_sha256"],
        f"{label} sidecar contract read-back hash mismatch.",
    )

    return {
        "label": label,
        "artifact_type": artifact_type,
        "format": "parquet",
        "path": str(table_path),
        "contract_path": str(contract_path),
        "row_count": int(len(prepared)),
        "column_count": int(len(prepared.columns)),
        "table_payload_sha256": table_payload_sha256,
        "file_sha256": table_file_sha256,
        "contract_payload_sha256": table_contract_sha256,
        "readback_verified": True,
    }


def _n04_array_payload_sha256(
    array_dict: dict[str, np.ndarray],
) -> str:
    """Compute stable hash for a dictionary of numeric arrays."""
    h = hashlib.sha256()

    normalized_schema = {}

    for key in sorted(array_dict.keys()):
        arr = np.ascontiguousarray(array_dict[key])
        normalized_schema[key] = {
            "dtype": str(arr.dtype),
            "shape": [int(x) for x in arr.shape],
        }

    h.update(
        _n04_canonical_json_bytes(
            {
                "array_schema": normalized_schema,
                "array_count": len(array_dict),
            }
        )
    )

    for key in sorted(array_dict.keys()):
        arr = np.ascontiguousarray(array_dict[key])
        h.update(b"\n")
        h.update(key.encode("utf-8"))
        h.update(b"\n")
        h.update(arr.tobytes(order="C"))

    return h.hexdigest()


def _n04_flatten_partition_arrays(
    nested: dict[str, dict[str, np.ndarray]],
    *,
    family: str,
) -> dict[str, np.ndarray]:
    """Flatten partition-keyed arrays into NPZ-compatible keys."""
    flat = {}

    for partition_name in sorted(nested.keys()):
        for array_name in sorted(nested[partition_name].keys()):
            key = (
                f"{partition_name}__"
                f"{family}__"
                f"{array_name}"
            )
            arr = np.asarray(nested[partition_name][array_name])

            require(
                arr.dtype != object,
                (
                    "Object dtype is not allowed in persisted NPZ "
                    f"array {key}."
                ),
            )

            flat[key] = np.ascontiguousarray(arr)

    return flat


def _n04_write_npz_array_artifact(
    *,
    label: str,
    array_dict: dict[str, np.ndarray],
    output_dir: Path,
    artifact_type: str,
    schema_payload: dict,
) -> dict:
    """Persist numeric arrays as NPZ and verify by read-back hash."""
    require(
        len(array_dict) > 0,
        f"{label} array dictionary is empty.",
    )

    output_dir.mkdir(parents=True, exist_ok=True)

    expected_payload_sha256 = _n04_array_payload_sha256(
        array_dict
    )

    npz_path = output_dir / (
        f"{N04_COMBINED_PREFIX}__"
        f"{N04_NOTEBOOK_NAME}__"
        f"{label}.npz"
    )

    schema_path = output_dir / (
        f"{N04_COMBINED_PREFIX}__"
        f"{N04_NOTEBOOK_NAME}__"
        f"{label}__schema.json"
    )

    contract_path = output_dir / (
        f"{N04_COMBINED_PREFIX}__"
        f"{N04_NOTEBOOK_NAME}__"
        f"{label}__contract.json"
    )

    tmp_npz_path = output_dir / (
        f"{N04_COMBINED_PREFIX}__"
        f"{N04_NOTEBOOK_NAME}__"
        f"{label}.__tmp__.npz"
    )

    if tmp_npz_path.exists():
        tmp_npz_path.unlink()

    np.savez_compressed(
        tmp_npz_path,
        **array_dict,
    )

    os.replace(tmp_npz_path, npz_path)

    with np.load(npz_path, allow_pickle=False) as loaded:
        readback_array_dict = {
            key: np.ascontiguousarray(loaded[key])
            for key in loaded.files
        }

    observed_payload_sha256 = _n04_array_payload_sha256(
        readback_array_dict
    )

    require(
        observed_payload_sha256 == expected_payload_sha256,
        (
            f"{label} NPZ read-back payload hash mismatch. "
            f"expected={expected_payload_sha256}; "
            f"observed={observed_payload_sha256}"
        ),
    )

    schema_contract = {
        "artifact_type": f"{artifact_type}_SCHEMA",
        "artifact_schema_version": "1.0",
        "label": label,
        "path": str(schema_path),
        "source_run_prefix": N04_SOURCE_RUN_PREFIX,
        "v0_1_run_id": N04_V01_RUN_ID,
        "combined_prefix": N04_COMBINED_PREFIX,
        "schema_payload": schema_payload,
        "created_at_utc": _n04_utc_now_iso(),
    }

    schema_payload_sha256 = _n04_canonical_json_sha256(
        schema_contract
    )

    schema_contract["schema_payload_sha256"] = (
        schema_payload_sha256
    )

    _n04_atomic_write_json(
        schema_path,
        schema_contract,
    )

    npz_file_sha256 = _n04_file_sha256(npz_path)

    artifact_contract_payload = {
        "artifact_type": artifact_type,
        "artifact_schema_version": "1.0",
        "acceptance_status": "PASS",
        "project_name": N04_PROJECT_NAME,
        "pipeline_version": N04_PIPELINE_VERSION,
        "notebook_name": N04_NOTEBOOK_NAME,
        "source_run_prefix": N04_SOURCE_RUN_PREFIX,
        "v0_1_run_id": N04_V01_RUN_ID,
        "combined_prefix": N04_COMBINED_PREFIX,
        "label": label,
        "format": "npz",
        "path": str(npz_path),
        "schema_path": str(schema_path),
        "contract_path": str(contract_path),
        "array_count": int(len(array_dict)),
        "array_payload_sha256": expected_payload_sha256,
        "readback_array_payload_sha256": observed_payload_sha256,
        "file_sha256": npz_file_sha256,
        "schema_payload_sha256": schema_payload_sha256,
        "created_at_utc": _n04_utc_now_iso(),
        "hash_policy": (
            "canonical key-sorted array hash over dtype, shape, "
            "and raw contiguous bytes"
        ),
    }

    artifact_contract_sha256 = _n04_canonical_json_sha256(
        artifact_contract_payload
    )

    artifact_contract = dict(artifact_contract_payload)
    artifact_contract["contract_payload_sha256"] = (
        artifact_contract_sha256
    )

    _n04_atomic_write_json(
        contract_path,
        artifact_contract,
    )

    return {
        "label": label,
        "artifact_type": artifact_type,
        "format": "npz",
        "path": str(npz_path),
        "schema_path": str(schema_path),
        "contract_path": str(contract_path),
        "array_count": int(len(array_dict)),
        "array_payload_sha256": expected_payload_sha256,
        "file_sha256": npz_file_sha256,
        "schema_payload_sha256": schema_payload_sha256,
        "contract_payload_sha256": artifact_contract_sha256,
        "readback_verified": True,
    }


# ------------------------------------------------------------
# Identity and output directories
# ------------------------------------------------------------

N04_PROJECT_NAME = _n04_first_existing_global(
    ["PROJECT_NAME", "project_name"],
    default="The Clown Project",
)

N04_PIPELINE_VERSION = _n04_first_existing_global(
    ["PIPELINE_VERSION", "pipeline_version"],
    default="V0.1",
)

N04_NOTEBOOK_NAME = "04_EVENT_STREAM_CONSTRUCTION"

N04_SOURCE_RUN_PREFIX = _n04_first_existing_global(
    ["SOURCE_RUN_PREFIX", "source_run_prefix"],
    default="BTCUSDT_spot_20260710T063746Z_c8b5bf12",
)

N04_V01_RUN_ID = _n04_first_existing_global(
    ["V0_1_RUN_ID", "V01_RUN_ID", "v0_1_run_id"],
    default="v0_1_20260714T090616Z_e82325081a81",
)

N04_COMBINED_PREFIX = _n04_first_existing_global(
    ["COMBINED_PREFIX", "combined_prefix"],
    default=f"{N04_SOURCE_RUN_PREFIX}__{N04_V01_RUN_ID}",
)

N04_V01_ROOT = _n04_resolve_v01_root()

N04_CONFIG_DIR = N04_V01_ROOT / "config"
N04_PROCESSED_EVENTS_DIR = (
    N04_V01_ROOT
    / "data"
    / "processed"
    / "events"
)
N04_ARRAY_DIR = (
    N04_V01_ROOT
    / "data"
    / "processed"
    / "point_process"
)
N04_AUDIT_DIR = (
    N04_V01_ROOT
    / "artifacts"
    / "audit_tables"
)
N04_MANIFEST_DIR = (
    N04_V01_ROOT
    / "artifacts"
    / "manifests"
)
N04_HANDOFF_DIR = (
    N04_V01_ROOT
    / "artifacts"
    / "handoff"
)

for directory in [
    N04_CONFIG_DIR,
    N04_PROCESSED_EVENTS_DIR,
    N04_ARRAY_DIR,
    N04_AUDIT_DIR,
    N04_MANIFEST_DIR,
    N04_HANDOFF_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Table artifact specifications
# ------------------------------------------------------------

table_artifact_specs = [
    {
        "label": "individual_trade_events",
        "frame": all_individual_trade_events,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_INDIVIDUAL_TRADE_EVENTS",
    },
    {
        "label": "same_exchange_ms_diagnostic_buckets",
        "frame": all_exchange_ms_diagnostic_buckets,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_SAME_EXCHANGE_MS_DIAGNOSTIC_BUCKETS",
    },
    {
        "label": "same_ms_same_side_burst_events",
        "frame": all_same_ms_same_side_burst_events,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_SAME_MS_SAME_SIDE_BURST_EVENTS",
    },
    {
        "label": "primary_estimation_events",
        "frame": primary_estimation_events_windowed,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_PRIMARY_ESTIMATION_EVENTS",
    },
    {
        "label": "primary_exact_time_batches",
        "frame": primary_exact_time_batches_windowed,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_PRIMARY_EXACT_TIME_BATCHES",
    },
    {
        "label": "trade_to_individual_event_membership",
        "frame": all_trade_to_individual_event_membership,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_TRADE_TO_INDIVIDUAL_EVENT_MEMBERSHIP",
    },
    {
        "label": "trade_to_exchange_ms_bucket_membership",
        "frame": all_trade_to_exchange_ms_bucket_membership,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_TRADE_TO_EXCHANGE_MS_BUCKET_MEMBERSHIP",
    },
    {
        "label": "trade_to_same_ms_same_side_burst_membership",
        "frame": all_trade_to_same_ms_same_side_burst_membership,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_TRADE_TO_BURST_MEMBERSHIP",
    },
    {
        "label": "primary_trade_membership",
        "frame": primary_trade_membership,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_PRIMARY_TRADE_MEMBERSHIP",
    },
    {
        "label": "primary_event_to_batch_membership",
        "frame": primary_event_to_batch_membership_enriched,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_PRIMARY_EVENT_TO_BATCH_MEMBERSHIP",
    },
    {
        "label": "primary_event_array_index",
        "frame": primary_event_array_index,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_PRIMARY_EVENT_ARRAY_INDEX",
    },
    {
        "label": "primary_scoring_batches",
        "frame": primary_scoring_batches,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_PRIMARY_SCORING_BATCHES",
    },
    {
        "label": "observation_window_contract",
        "frame": observation_window_contract,
        "output_dir": N04_CONFIG_DIR,
        "artifact_type": "NOTEBOOK_04_OBSERVATION_WINDOW_CONTRACT",
    },
    {
        "label": "partition_array_metadata",
        "frame": partition_array_metadata,
        "output_dir": N04_PROCESSED_EVENTS_DIR,
        "artifact_type": "NOTEBOOK_04_PARTITION_ARRAY_METADATA",
    },
    {
        "label": "window_and_array_gate_frame",
        "frame": window_and_array_gate_frame,
        "output_dir": N04_AUDIT_DIR,
        "artifact_type": "NOTEBOOK_04_WINDOW_AND_ARRAY_GATE_FRAME",
    },
]

optional_audit_tables = [
    "all_event_construction_summary",
    "global_primary_conservation_summary",
    "primary_partition_conservation_audit",
    "primary_batch_summary",
    "all_partition_primary_gate_frame",
    "window_and_array_summary",
    "event_definition_selection_gate_frame",
    "event_definition_selection_summary",
    "candidate_event_audit_summary",
    "selection_candidate_gate_frame",
    "canonicalization_summary",
    "canonicalization_gate_frame",
]

for optional_name in optional_audit_tables:
    if optional_name in globals():
        optional_frame = globals()[optional_name]

        if isinstance(optional_frame, pd.DataFrame):
            table_artifact_specs.append(
                {
                    "label": optional_name,
                    "frame": optional_frame,
                    "output_dir": N04_AUDIT_DIR,
                    "artifact_type": (
                        "NOTEBOOK_04_AUDIT_TABLE"
                    ),
                    "required_nonempty": False,
                }
            )


# ------------------------------------------------------------
# Write and verify table artifacts
# ------------------------------------------------------------

table_write_records = []

for spec in table_artifact_specs:
    table_write_records.append(
        _n04_write_parquet_table_artifact(
            label=spec["label"],
            frame=spec["frame"],
            output_dir=spec["output_dir"],
            artifact_type=spec["artifact_type"],
            required_nonempty=spec.get(
                "required_nonempty",
                True,
            ),
        )
    )

notebook_04_table_artifact_audit = pd.DataFrame.from_records(
    table_write_records
).sort_values(
    ["artifact_type", "label"],
    kind="stable",
).reset_index(drop=True)

require(
    notebook_04_table_artifact_audit[
        "readback_verified"
    ].all(),
    "At least one table artifact failed read-back verification.",
)


# ------------------------------------------------------------
# Write and verify partition array artifacts
# ------------------------------------------------------------

flat_partition_event_arrays = _n04_flatten_partition_arrays(
    partition_event_arrays,
    family="event",
)

flat_partition_batch_arrays = _n04_flatten_partition_arrays(
    partition_batch_arrays,
    family="batch",
)

array_write_records = [
    _n04_write_npz_array_artifact(
        label="partition_event_arrays",
        array_dict=flat_partition_event_arrays,
        output_dir=N04_ARRAY_DIR,
        artifact_type="NOTEBOOK_04_PARTITION_EVENT_ARRAYS",
        schema_payload=partition_array_schema,
    ),
    _n04_write_npz_array_artifact(
        label="partition_batch_arrays",
        array_dict=flat_partition_batch_arrays,
        output_dir=N04_ARRAY_DIR,
        artifact_type="NOTEBOOK_04_PARTITION_BATCH_ARRAYS",
        schema_payload=partition_array_schema,
    ),
]

notebook_04_array_artifact_audit = pd.DataFrame.from_records(
    array_write_records
).sort_values(
    ["artifact_type", "label"],
    kind="stable",
).reset_index(drop=True)

require(
    notebook_04_array_artifact_audit[
        "readback_verified"
    ].all(),
    "At least one array artifact failed read-back verification.",
)


# ------------------------------------------------------------
# Persist event-stream contract update
# ------------------------------------------------------------

event_stream_persistence_contract_payload = {
    "artifact_type": "NOTEBOOK_04_EVENT_STREAM_PERSISTENCE_CONTRACT",
    "artifact_schema_version": "1.0",
    "acceptance_status": "PASS",
    "project_name": N04_PROJECT_NAME,
    "pipeline_version": N04_PIPELINE_VERSION,
    "notebook_name": N04_NOTEBOOK_NAME,
    "source_run_prefix": N04_SOURCE_RUN_PREFIX,
    "v0_1_run_id": N04_V01_RUN_ID,
    "combined_prefix": N04_COMBINED_PREFIX,
    "primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
    "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
    "requires_simultaneous_event_batch_interface": bool(
        PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
    ),
    "side_code": dict(SIDE_CODE),
    "event_partition_field": EVENT_PARTITION_FIELD,
    "matched_book_partition_field": MATCHED_BOOK_PARTITION_FIELD,
    "cross_partition_history_field": CROSS_PARTITION_HISTORY_FIELD,
    "primary_event_rows": int(
        len(primary_estimation_events_windowed)
    ),
    "primary_batch_rows": int(
        len(primary_exact_time_batches_windowed)
    ),
    "trade_rows_conserved": int(
        primary_estimation_events_windowed[
            "event_print_count"
        ].sum()
    ),
    "expected_trade_rows": int(EXPECTED_MATCHED_TRADE_ROWS),
    "simultaneous_batch_count": int(
        primary_exact_time_batches_windowed[
            "simultaneous_batch_required_flag"
        ].sum()
    ),
    "mixed_side_batch_count": int(
        primary_exact_time_batches_windowed[
            "mixed_side_batch_flag"
        ].sum()
    ),
    "aggregate_mark_after_event_count": int(
        primary_estimation_events_windowed[
            "aggregate_mark_lag_after_event_ns"
        ].gt(0).sum()
    ),
    "table_artifacts": table_write_records,
    "array_artifacts": array_write_records,
    "created_at_utc": _n04_utc_now_iso(),
    "persistence_hash_policy": {
        "tables": (
            "canonical row-order-sensitive table payload hash; "
            "parquet metadata is not semantic authority"
        ),
        "arrays": (
            "canonical key-sorted array hash over dtype, shape, "
            "and raw contiguous bytes"
        ),
    },
}

event_stream_persistence_contract_sha256 = (
    _n04_canonical_json_sha256(
        event_stream_persistence_contract_payload
    )
)

event_stream_persistence_contract = dict(
    event_stream_persistence_contract_payload
)

event_stream_persistence_contract[
    "contract_payload_sha256"
] = event_stream_persistence_contract_sha256

event_stream_persistence_contract_path = (
    N04_CONFIG_DIR
    / (
        f"{N04_COMBINED_PREFIX}__"
        f"{N04_NOTEBOOK_NAME}__"
        "event_stream_persistence_contract.json"
    )
)

_n04_atomic_write_json(
    event_stream_persistence_contract_path,
    event_stream_persistence_contract,
)

event_stream_persistence_contract_readback = json.loads(
    event_stream_persistence_contract_path.read_text(
        encoding="utf-8"
    )
)

observed_persistence_contract_sha256 = (
    _n04_canonical_json_sha256(
        {
            key: value
            for key, value
            in event_stream_persistence_contract_readback.items()
            if key != "contract_payload_sha256"
        }
    )
)

require(
    observed_persistence_contract_sha256
    == event_stream_persistence_contract_sha256,
    "Event-stream persistence contract read-back hash mismatch.",
)


# ------------------------------------------------------------
# Notebook 04 output manifest
# ------------------------------------------------------------

notebook_04_output_manifest_payload = {
    "artifact_type": "NOTEBOOK_04_OUTPUT_MANIFEST",
    "artifact_schema_version": "1.0",
    "acceptance_status": "PASS_WITH_EVENT_STREAM_WARNINGS",
    "project_name": N04_PROJECT_NAME,
    "pipeline_version": N04_PIPELINE_VERSION,
    "notebook_name": N04_NOTEBOOK_NAME,
    "source_run_prefix": N04_SOURCE_RUN_PREFIX,
    "v0_1_run_id": N04_V01_RUN_ID,
    "combined_prefix": N04_COMBINED_PREFIX,
    "operating_mode": _n04_first_existing_global(
        ["OPERATING_MODE", "operating_mode"],
        default="ENGINEERING_REPRODUCTION_MODE",
    ),
    "previous_notebook": "03_CAUSAL_TRADE_BOOK_ALIGNMENT",
    "next_notebook": "05_MARKET_STATE_FEATURES",
    "primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
    "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
    "primary_alignment_policy": _n04_first_existing_global(
        ["PRIMARY_ALIGNMENT_POLICY", "primary_alignment_policy"],
        default="LOCAL_STRICT",
    ),
    "primary_ordering_authority": _n04_first_existing_global(
        ["PRIMARY_ORDERING_AUTHORITY", "primary_ordering_authority"],
        default="collector_sequence",
    ),
    "protected_partitions_opened_only_after_selection_freeze": True,
    "table_artifact_count": int(
        len(notebook_04_table_artifact_audit)
    ),
    "array_artifact_count": int(
        len(notebook_04_array_artifact_audit)
    ),
    "table_artifacts": table_write_records,
    "array_artifacts": array_write_records,
    "event_stream_persistence_contract_path": str(
        event_stream_persistence_contract_path
    ),
    "event_stream_persistence_contract_payload_sha256": (
        event_stream_persistence_contract_sha256
    ),
    "blocking_gate_failures": 0,
    "warning_findings": {
        "simultaneous_batch_count": int(
            primary_exact_time_batches_windowed[
                "simultaneous_batch_required_flag"
            ].sum()
        ),
        "mixed_side_batch_count": int(
            primary_exact_time_batches_windowed[
                "mixed_side_batch_flag"
            ].sum()
        ),
        "aggregate_mark_after_event_count": int(
            primary_estimation_events_windowed[
                "aggregate_mark_lag_after_event_ns"
            ].gt(0).sum()
        ),
    },
    "created_at_utc": _n04_utc_now_iso(),
}

notebook_04_output_manifest_payload_sha256 = (
    _n04_canonical_json_sha256(
        notebook_04_output_manifest_payload
    )
)

notebook_04_output_manifest = dict(
    notebook_04_output_manifest_payload
)

notebook_04_output_manifest[
    "manifest_payload_sha256"
] = notebook_04_output_manifest_payload_sha256

notebook_04_output_manifest_path = (
    N04_MANIFEST_DIR
    / (
        f"{N04_COMBINED_PREFIX}__"
        f"{N04_NOTEBOOK_NAME}__"
        "notebook_04_output_manifest.json"
    )
)

_n04_atomic_write_json(
    notebook_04_output_manifest_path,
    notebook_04_output_manifest,
)

notebook_04_output_manifest_readback = json.loads(
    notebook_04_output_manifest_path.read_text(
        encoding="utf-8"
    )
)

observed_manifest_payload_sha256 = _n04_canonical_json_sha256(
    {
        key: value
        for key, value
        in notebook_04_output_manifest_readback.items()
        if key != "manifest_payload_sha256"
    }
)

require(
    observed_manifest_payload_sha256
    == notebook_04_output_manifest_payload_sha256,
    "Notebook 04 output manifest read-back hash mismatch.",
)


# ------------------------------------------------------------
# Notebook 04 to Notebook 05 handoff
# ------------------------------------------------------------

def _n04_find_artifact_record(label: str) -> dict:
    matches = [
        record
        for record in table_write_records + array_write_records
        if record["label"] == label
    ]

    require(
        len(matches) == 1,
        f"Expected exactly one artifact record for {label}.",
    )

    return matches[0]


notebook_04_to_notebook_05_handoff_payload = {
    "artifact_type": "NOTEBOOK_04_TO_NOTEBOOK_05_HANDOFF",
    "artifact_schema_version": "1.0",
    "acceptance_status": "PASS_WITH_EVENT_STREAM_WARNINGS",
    "project_name": N04_PROJECT_NAME,
    "pipeline_version": N04_PIPELINE_VERSION,
    "source_run_prefix": N04_SOURCE_RUN_PREFIX,
    "v0_1_run_id": N04_V01_RUN_ID,
    "combined_prefix": N04_COMBINED_PREFIX,
    "producing_notebook": N04_NOTEBOOK_NAME,
    "next_notebook": "05_MARKET_STATE_FEATURES",
    "event_stream_authorized": True,
    "feature_construction_authorized": True,
    "hawkes_estimation_authorized": False,
    "market_making_authorized": False,
    "primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
    "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
    "requires_simultaneous_event_batch_interface": bool(
        PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
    ),
    "side_code": dict(SIDE_CODE),
    "primary_event_rows": int(
        len(primary_estimation_events_windowed)
    ),
    "primary_batch_rows": int(
        len(primary_exact_time_batches_windowed)
    ),
    "trade_print_rows_conserved": int(
        primary_estimation_events_windowed[
            "event_print_count"
        ].sum()
    ),
    "expected_trade_rows": int(EXPECTED_MATCHED_TRADE_ROWS),
    "event_partition_field": EVENT_PARTITION_FIELD,
    "matched_book_partition_field": MATCHED_BOOK_PARTITION_FIELD,
    "cross_partition_history_field": CROSS_PARTITION_HISTORY_FIELD,
    "strict_no_jitter_contract": True,
    "strict_no_event_removal_contract": True,
    "scoring_batch_semantics": {
        "history_used_for_batch_scoring": (
            "STRICTLY_BEFORE_BATCH_TIME"
        ),
        "zero_lag_within_batch_excitation_allowed": False,
        "apply_batch_excitation_after_all_members_scored": True,
        "collector_sequence_inside_tie_is_trace_order_only": True,
    },
    "observation_window_contract": {
        "path": _n04_find_artifact_record(
            "observation_window_contract"
        )["path"],
        "table_payload_sha256": _n04_find_artifact_record(
            "observation_window_contract"
        )["table_payload_sha256"],
    },
    "primary_estimation_events": {
        "path": _n04_find_artifact_record(
            "primary_estimation_events"
        )["path"],
        "table_payload_sha256": _n04_find_artifact_record(
            "primary_estimation_events"
        )["table_payload_sha256"],
    },
    "primary_exact_time_batches": {
        "path": _n04_find_artifact_record(
            "primary_exact_time_batches"
        )["path"],
        "table_payload_sha256": _n04_find_artifact_record(
            "primary_exact_time_batches"
        )["table_payload_sha256"],
    },
    "primary_trade_membership": {
        "path": _n04_find_artifact_record(
            "primary_trade_membership"
        )["path"],
        "table_payload_sha256": _n04_find_artifact_record(
            "primary_trade_membership"
        )["table_payload_sha256"],
    },
    "primary_event_to_batch_membership": {
        "path": _n04_find_artifact_record(
            "primary_event_to_batch_membership"
        )["path"],
        "table_payload_sha256": _n04_find_artifact_record(
            "primary_event_to_batch_membership"
        )["table_payload_sha256"],
    },
    "primary_event_array_index": {
        "path": _n04_find_artifact_record(
            "primary_event_array_index"
        )["path"],
        "table_payload_sha256": _n04_find_artifact_record(
            "primary_event_array_index"
        )["table_payload_sha256"],
    },
    "primary_scoring_batches": {
        "path": _n04_find_artifact_record(
            "primary_scoring_batches"
        )["path"],
        "table_payload_sha256": _n04_find_artifact_record(
            "primary_scoring_batches"
        )["table_payload_sha256"],
    },
    "partition_event_arrays": {
        "path": _n04_find_artifact_record(
            "partition_event_arrays"
        )["path"],
        "array_payload_sha256": _n04_find_artifact_record(
            "partition_event_arrays"
        )["array_payload_sha256"],
        "schema_path": _n04_find_artifact_record(
            "partition_event_arrays"
        )["schema_path"],
    },
    "partition_batch_arrays": {
        "path": _n04_find_artifact_record(
            "partition_batch_arrays"
        )["path"],
        "array_payload_sha256": _n04_find_artifact_record(
            "partition_batch_arrays"
        )["array_payload_sha256"],
        "schema_path": _n04_find_artifact_record(
            "partition_batch_arrays"
        )["schema_path"],
    },
    "notebook_04_output_manifest": {
        "path": str(notebook_04_output_manifest_path),
        "manifest_payload_sha256": (
            notebook_04_output_manifest_payload_sha256
        ),
    },
    "created_at_utc": _n04_utc_now_iso(),
}

notebook_04_to_notebook_05_handoff_payload_sha256 = (
    _n04_canonical_json_sha256(
        notebook_04_to_notebook_05_handoff_payload
    )
)

notebook_04_to_notebook_05_handoff = dict(
    notebook_04_to_notebook_05_handoff_payload
)

notebook_04_to_notebook_05_handoff[
    "handoff_payload_sha256"
] = notebook_04_to_notebook_05_handoff_payload_sha256

notebook_04_to_notebook_05_handoff_path = (
    N04_HANDOFF_DIR
    / (
        f"{N04_COMBINED_PREFIX}__"
        f"{N04_NOTEBOOK_NAME}__"
        "notebook_04_to_notebook_05_handoff.json"
    )
)

_n04_atomic_write_json(
    notebook_04_to_notebook_05_handoff_path,
    notebook_04_to_notebook_05_handoff,
)

notebook_04_to_notebook_05_handoff_readback = json.loads(
    notebook_04_to_notebook_05_handoff_path.read_text(
        encoding="utf-8"
    )
)

observed_handoff_payload_sha256 = _n04_canonical_json_sha256(
    {
        key: value
        for key, value
        in notebook_04_to_notebook_05_handoff_readback.items()
        if key != "handoff_payload_sha256"
    }
)

require(
    observed_handoff_payload_sha256
    == notebook_04_to_notebook_05_handoff_payload_sha256,
    "Notebook 04 to Notebook 05 handoff read-back hash mismatch.",
)


# ------------------------------------------------------------
# Final persistence gate ledger
# ------------------------------------------------------------

persistence_gates = [
    make_gate(
        gate="all_table_artifacts_written_and_readback_verified",
        passed=bool(
            notebook_04_table_artifact_audit[
                "readback_verified"
            ].all()
        ),
        severity="BLOCKING",
        detail=(
            "table_artifact_count="
            f"{len(notebook_04_table_artifact_audit)}"
        ),
    ),
    make_gate(
        gate="all_array_artifacts_written_and_readback_verified",
        passed=bool(
            notebook_04_array_artifact_audit[
                "readback_verified"
            ].all()
        ),
        severity="BLOCKING",
        detail=(
            "array_artifact_count="
            f"{len(notebook_04_array_artifact_audit)}"
        ),
    ),
    make_gate(
        gate="event_stream_persistence_contract_written_and_verified",
        passed=(
            observed_persistence_contract_sha256
            == event_stream_persistence_contract_sha256
        ),
        severity="BLOCKING",
        detail=(
            "payload_sha256="
            f"{event_stream_persistence_contract_sha256}"
        ),
    ),
    make_gate(
        gate="notebook_04_output_manifest_written_and_verified",
        passed=(
            observed_manifest_payload_sha256
            == notebook_04_output_manifest_payload_sha256
        ),
        severity="BLOCKING",
        detail=(
            "payload_sha256="
            f"{notebook_04_output_manifest_payload_sha256}"
        ),
    ),
    make_gate(
        gate="notebook_04_to_05_handoff_written_and_verified",
        passed=(
            observed_handoff_payload_sha256
            == notebook_04_to_notebook_05_handoff_payload_sha256
        ),
        severity="BLOCKING",
        detail=(
            "payload_sha256="
            f"{notebook_04_to_notebook_05_handoff_payload_sha256}"
        ),
    ),
    make_gate(
        gate="primary_event_prints_conserve_all_trades",
        passed=(
            int(
                primary_estimation_events_windowed[
                    "event_print_count"
                ].sum()
            )
            == int(EXPECTED_MATCHED_TRADE_ROWS)
        ),
        severity="BLOCKING",
        detail=(
            "primary_print_sum="
            f"{int(primary_estimation_events_windowed['event_print_count'].sum()):,}; "
            "expected_trade_rows="
            f"{int(EXPECTED_MATCHED_TRADE_ROWS):,}"
        ),
    ),
    make_gate(
        gate="feature_construction_authorized",
        passed=True,
        severity="BLOCKING",
        detail=(
            "Notebook 05 may use Notebook 04 handoff artifacts. "
            "Hawkes estimation and market making remain unauthorized."
        ),
    ),
    make_gate(
        gate="simultaneous_batches_absent",
        passed=(
            int(
                primary_exact_time_batches_windowed[
                    "simultaneous_batch_required_flag"
                ].sum()
            )
            == 0
        ),
        severity="WARNING",
        detail=(
            "simultaneous_batch_count="
            f"{int(primary_exact_time_batches_windowed['simultaneous_batch_required_flag'].sum())}; "
            "batch-aware scoring is required downstream."
        ),
    ),
    make_gate(
        gate="mixed_side_batches_absent",
        passed=(
            int(
                primary_exact_time_batches_windowed[
                    "mixed_side_batch_flag"
                ].sum()
            )
            == 0
        ),
        severity="WARNING",
        detail=(
            "mixed_side_batch_count="
            f"{int(primary_exact_time_batches_windowed['mixed_side_batch_flag'].sum())}"
        ),
    ),
]

notebook_04_persistence_gate_frame = gate_results_to_frame(
    persistence_gates
)

fail_if_blocking_gate_failed(
    notebook_04_persistence_gate_frame
)


# ------------------------------------------------------------
# Cell output
# ------------------------------------------------------------

display(notebook_04_table_artifact_audit)
display(notebook_04_array_artifact_audit)
display(notebook_04_persistence_gate_frame)

{
    "status": "PASS_WITH_EVENT_STREAM_WARNINGS",
    "table_artifact_count": int(
        len(notebook_04_table_artifact_audit)
    ),
    "array_artifact_count": int(
        len(notebook_04_array_artifact_audit)
    ),
    "primary_event_rows": int(
        len(primary_estimation_events_windowed)
    ),
    "primary_batch_rows": int(
        len(primary_exact_time_batches_windowed)
    ),
    "trade_print_rows_conserved": int(
        primary_estimation_events_windowed[
            "event_print_count"
        ].sum()
    ),
    "notebook_04_output_manifest_path": str(
        notebook_04_output_manifest_path
    ),
    "notebook_04_output_manifest_payload_sha256": (
        notebook_04_output_manifest_payload_sha256
    ),
    "notebook_04_to_notebook_05_handoff_path": str(
        notebook_04_to_notebook_05_handoff_path
    ),
    "notebook_04_to_notebook_05_handoff_payload_sha256": (
        notebook_04_to_notebook_05_handoff_payload_sha256
    ),
    "feature_construction_authorized": True,
    "hawkes_estimation_authorized": False,
    "market_making_authorized": False,
    "next_action": (
        "Run final Notebook 04 acceptance audit, then start "
        "05_MARKET_STATE_FEATURES from the Notebook 04 handoff."
    ),
}

,label,artifact_type,format,path,contract_path,row_count,column_count,table_payload_sha256,file_sha256,contract_payload_sha256,readback_verified
0,all_event_construction_summary,NOTEBOOK_04_AUDIT_TABLE,parquet,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__all_event_construction_su...,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__all_event_construction_su...,3,6,ca519c208ee81f865309c65d5fde18e98a7894519b4abc1e8dfdbe7927048d9c,95d05793ffc2919880c686575bb3a0a8974a6b8aa61f6bd9dc3d451b6a7cc153,13135258bfdab280223ef51f775062c02159f1678972a9173917a3d9bba2ca1a,True
1,all_partition_primary_gate_frame,NOTEBOOK_04_AUDIT_TABLE,parquet,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__all_partition_primary_gat...,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__all_partition_primary_gat...,12,4,06cc13679a645fbc7343b676c6f7c09d5a7495c44196e2eab48ec7e495648c13,2f360403c4f1dd539a5f0ea2781031e28554716865c473bccbeb619122d28b15,b6734723532bf11662195d620685706d753fb60245fff0a8a765601aa922b57f,True
2,global_primary_conservation_summary,NOTEBOOK_04_AUDIT_TABLE,parquet,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__global_primary_conservati...,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__global_primary_conservati...,1,10,e3f947eb423c22175472af93d43e9362e3c6a7bf112b7dc133a4539948a09349,bc5064f2855ea03a16f8652092c125ad67744d008cc425f1a4ad22e0122c3d45,5f125582f8216702598088b0a6a06405c7c8cd2dfe3b58cece2ea550f1906bbd,True
3,primary_batch_summary,NOTEBOOK_04_AUDIT_TABLE,parquet,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__primary_batch_summary.par...,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__primary_batch_summary__co...,1,9,9987de284d32328f890826ace6fe515510bcd22a97fe08fb79b577403cd245ce,8fcbfe6e276535eb199b69f5a555d9501bb5dae7787134b61581c0b5ce26ffb2,1421776d9f06378236c4cb4d62f0a0c7acf6dd712052a4bc798fc3bef72144ba,True
4,primary_partition_conservation_audit,NOTEBOOK_04_AUDIT_TABLE,parquet,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__primary_partition_conserv...,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__primary_partition_conserv...,4,22,6af3cdcbe128d6aee42af162947cddb37c187eba43ff8ed489f57fa916dccb82,5c6da4255d859a2faed2638c4494266b8164b95e38cad1ae828579ab443704f8,447eef7d7ed8c50a065ad0b191641a1af3f0b743f02b53e67b5f6f23bcc41aea,True
5,window_and_array_summary,NOTEBOOK_04_AUDIT_TABLE,parquet,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__window_and_array_summary....,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__window_and_array_summary_...,1,13,89e83265edcf4fc4aac96fcedd1d85f1170c56609a82a9417dc8f6160d177d75,c3fb00b083cb93ede4b3dd7477b03791e4a876b910657b4478d87cab8da939ed,e24437bf1cc70495a090b4ec7cb62b36dd20eb8a7e9556f5b24415542aa32f58,True
6,individual_trade_events,NOTEBOOK_04_INDIVIDUAL_TRADE_EVENTS,parquet,D:\Clown 

,label,artifact_type,format,path,schema_path,contract_path,array_count,array_payload_sha256,file_sha256,schema_payload_sha256,contract_payload_sha256,readback_verified
0,partition_batch_arrays,NOTEBOOK_04_PARTITION_BATCH_ARRAYS,npz,D:\Clown Project\V0.1\data\processed\point_process\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__partition_batch_arr...,D:\Clown Project\V0.1\data\processed\point_process\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__partition_batch_arr...,D:\Clown Project\V0.1\data\processed\point_process\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__partition_batch_arr...,44,56da4e6d4a2bf4f0be8e20a372d935c15b6c9ae45c51e285b8bcab2024ab7844,c639e065a4d0e63146770df90bc8d12ffa7b89bf793857e0ebaa93b03c512c1d,1906437b616ab628d8251392dc63a7d88f3957ae47841a386ef8452243b8daec,4020a4de8bc5f1c7044b9d682e37ac5baa81405ba84eca2bf0d0ac5e23b362f1,True
1,partition_event_arrays,NOTEBOOK_04_PARTITION_EVENT_ARRAYS,npz,D:\Clown Project\V0.1\data\processed\point_process\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__partition_event_arr...,D:\Clown Project\V0.1\data\processed\point_process\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__partition_event_arr...,D:\Clown Project\V0.1\data\processed\point_process\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__partition_event_arr...,52,ee6049585ea1975c83aba3aa8c3bd1037fd3845c88486815dd0034ae2ca2469e,fbc20bb575c691f51523938de9ef69dd9729d6bcc46082375712e58737c41de3,ae29c15303ba84394985939f050a3b595559d266765dc696c980ed2883a7ea53,bd6d329d3083f336b79887b15c3f215cfe0d5fc06fbf0dbe887bbe369faecf12,True


,gate,status,severity,detail
0,all_table_artifacts_written_and_readback_verified,PASS,BLOCKING,table_artifact_count=21
1,all_array_artifacts_written_and_readback_verified,PASS,BLOCKING,array_artifact_count=2
2,event_stream_persistence_contract_written_and_verified,PASS,BLOCKING,payload_sha256=4c486bab5ea30cc9dbe94bc995033fd530dca8ef8f1880680f06db63e1b88230
3,notebook_04_output_manifest_written_and_verified,PASS,BLOCKING,payload_sha256=c8454d2059f1b5e50675aa156e1eabd7762642cbf624065b496bea1edb56ff53
4,notebook_04_to_05_handoff_written_and_verified,PASS,BLOCKING,payload_sha256=f0544516d59fbabc24d1c8ff42063c96e7c6c4b72fcd5cedbd8ef0a2dfc38a28
5,primary_event_prints_conserve_all_trades,PASS,BLOCKING,"primary_print_sum=67,683; expected_trade_rows=67,683"
6,feature_construction_authorized,PASS,BLOCKING,Notebook 05 may use Notebook 04 handoff artifacts. Hawkes estimation and market making remain unauthorized.
7,simultaneous_batches_absent,FAIL,WARNING,simultaneous_batch_count=249; batch-aware scoring is required downstream.
8,mixed_side_batches_absent,FAIL,WARNING,mixed_side_batch_count=41


{'status': 'PASS_WITH_EVENT_STREAM_WARNINGS',
 'table_artifact_count': 21,
 'array_artifact_count': 2,
 'primary_event_rows': 13887,
 'primary_batch_rows': 13564,
 'trade_print_rows_conserved': 67683,
 'notebook_04_output_manifest_path': 'D:\\Clown Project\\V0.1\\artifacts\\manifests\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__notebook_04_output_manifest.json',
 'notebook_04_output_manifest_payload_sha256': 'c8454d2059f1b5e50675aa156e1eabd7762642cbf624065b496bea1edb56ff53',
 'notebook_04_to_notebook_05_handoff_path': 'D:\\Clown Project\\V0.1\\artifacts\\handoff\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__notebook_04_to_notebook_05_handoff.json',
 'notebook_04_to_notebook_05_handoff_payload_sha256': 'f0544516d59fbabc24d1c8ff42063c96e7c6c4b72fcd5cedbd8ef0a2dfc38a28',
 'feature_construction_authorized': True,
 'hawkes_estimation_authorized': False,
 'market_making_

In [21]:
# ============================================================
# 04_EVENT_STREAM_CONSTRUCTION
# Cell 11 — Final Notebook 04 acceptance audit
#
# This cell performs the terminal audit for Notebook 04:
# - reloads the output manifest and Notebook 05 handoff;
# - verifies payload hashes;
# - re-reads persisted table and array artifacts;
# - confirms all blocking gates are PASS;
# - records remaining warnings as explicit downstream contracts;
# - freezes final Notebook 04 terminal status.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

required_final_audit_objects = [
    "notebook_04_table_artifact_audit",
    "notebook_04_array_artifact_audit",
    "notebook_04_persistence_gate_frame",
    "notebook_04_output_manifest_path",
    "notebook_04_output_manifest_payload_sha256",
    "notebook_04_to_notebook_05_handoff_path",
    "notebook_04_to_notebook_05_handoff_payload_sha256",
    "primary_estimation_events_windowed",
    "primary_exact_time_batches_windowed",
    "primary_trade_membership",
    "primary_event_to_batch_membership_enriched",
    "observation_window_contract",
    "partition_array_metadata",
]

for object_name in required_final_audit_objects:
    require(
        object_name in globals(),
        f"{object_name} is missing. Run the persistence cell first.",
    )

require(
    callable(globals().get("_n04_file_sha256")),
    "_n04_file_sha256 helper is missing. Run the persistence cell first.",
)

require(
    callable(globals().get("_n04_table_payload_sha256")),
    "_n04_table_payload_sha256 helper is missing. Run the persistence cell first.",
)

require(
    callable(globals().get("_n04_array_payload_sha256")),
    "_n04_array_payload_sha256 helper is missing. Run the persistence cell first.",
)

require(
    callable(globals().get("_n04_prepare_frame_for_write")),
    "_n04_prepare_frame_for_write helper is missing. Run the persistence cell first.",
)

require(
    callable(globals().get("_n04_atomic_write_json")),
    "_n04_atomic_write_json helper is missing. Run the persistence cell first.",
)

require(
    callable(globals().get("_n04_canonical_json_sha256")),
    "_n04_canonical_json_sha256 helper is missing. Run the persistence cell first.",
)


# ------------------------------------------------------------
# Final-audit helpers
# ------------------------------------------------------------

def _n04_read_json_contract(path: Path) -> dict:
    """Read a JSON contract from disk."""
    require(
        path.exists(),
        f"JSON contract does not exist: {path}",
    )

    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def _n04_verify_embedded_payload_hash(
    *,
    payload: dict,
    hash_field: str,
    label: str,
) -> str:
    """Verify a JSON payload hash field against canonical payload bytes."""
    require(
        hash_field in payload,
        f"{label} is missing embedded hash field {hash_field}.",
    )

    observed = _n04_canonical_json_sha256(
        {
            key: value
            for key, value in payload.items()
            if key != hash_field
        }
    )

    expected = payload[hash_field]

    require(
        observed == expected,
        (
            f"{label} embedded payload hash mismatch. "
            f"expected={expected}; observed={observed}"
        ),
    )

    return observed


def _n04_verify_table_record_readback(record: dict) -> dict:
    """Verify a persisted parquet table and its sidecar contract."""
    label = str(record["label"])
    table_path = Path(record["path"])
    contract_path = Path(record["contract_path"])

    require(
        table_path.exists(),
        f"{label} parquet file is missing: {table_path}",
    )

    require(
        contract_path.exists(),
        f"{label} sidecar contract is missing: {contract_path}",
    )

    contract = _n04_read_json_contract(contract_path)

    _n04_verify_embedded_payload_hash(
        payload=contract,
        hash_field="contract_payload_sha256",
        label=f"{label} sidecar contract",
    )

    observed_file_sha256 = _n04_file_sha256(table_path)
    expected_file_sha256 = str(record["file_sha256"])

    require(
        observed_file_sha256 == expected_file_sha256,
        (
            f"{label} parquet file SHA-256 mismatch. "
            f"expected={expected_file_sha256}; "
            f"observed={observed_file_sha256}"
        ),
    )

    readback = pd.read_parquet(table_path)
    readback_prepared = _n04_prepare_frame_for_write(readback)

    observed_table_payload_sha256 = _n04_table_payload_sha256(
        readback_prepared
    )

    expected_table_payload_sha256 = str(
        record["table_payload_sha256"]
    )

    require(
        observed_table_payload_sha256
        == expected_table_payload_sha256,
        (
            f"{label} canonical table payload SHA-256 mismatch. "
            f"expected={expected_table_payload_sha256}; "
            f"observed={observed_table_payload_sha256}"
        ),
    )

    expected_row_count = int(record["row_count"])
    expected_column_count = int(record["column_count"])

    require(
        len(readback_prepared) == expected_row_count,
        (
            f"{label} row-count mismatch. "
            f"expected={expected_row_count}; "
            f"observed={len(readback_prepared)}"
        ),
    )

    require(
        len(readback_prepared.columns) == expected_column_count,
        (
            f"{label} column-count mismatch. "
            f"expected={expected_column_count}; "
            f"observed={len(readback_prepared.columns)}"
        ),
    )

    return {
        "label": label,
        "artifact_type": str(record["artifact_type"]),
        "format": "parquet",
        "path": str(table_path),
        "row_count": expected_row_count,
        "column_count": expected_column_count,
        "file_sha256_verified": True,
        "payload_sha256_verified": True,
        "contract_sha256_verified": True,
        "readback_verified": True,
    }


def _n04_verify_array_record_readback(record: dict) -> dict:
    """Verify a persisted NPZ array artifact and its contracts."""
    label = str(record["label"])
    npz_path = Path(record["path"])
    schema_path = Path(record["schema_path"])
    contract_path = Path(record["contract_path"])

    require(
        npz_path.exists(),
        f"{label} NPZ file is missing: {npz_path}",
    )

    require(
        schema_path.exists(),
        f"{label} schema file is missing: {schema_path}",
    )

    require(
        contract_path.exists(),
        f"{label} sidecar contract is missing: {contract_path}",
    )

    contract = _n04_read_json_contract(contract_path)

    _n04_verify_embedded_payload_hash(
        payload=contract,
        hash_field="contract_payload_sha256",
        label=f"{label} array contract",
    )

    schema_contract = _n04_read_json_contract(schema_path)

    _n04_verify_embedded_payload_hash(
        payload=schema_contract,
        hash_field="schema_payload_sha256",
        label=f"{label} array schema",
    )

    observed_file_sha256 = _n04_file_sha256(npz_path)
    expected_file_sha256 = str(record["file_sha256"])

    require(
        observed_file_sha256 == expected_file_sha256,
        (
            f"{label} NPZ file SHA-256 mismatch. "
            f"expected={expected_file_sha256}; "
            f"observed={observed_file_sha256}"
        ),
    )

    with np.load(npz_path, allow_pickle=False) as loaded:
        loaded_arrays = {
            key: np.ascontiguousarray(loaded[key])
            for key in loaded.files
        }

    observed_array_payload_sha256 = _n04_array_payload_sha256(
        loaded_arrays
    )

    expected_array_payload_sha256 = str(
        record["array_payload_sha256"]
    )

    require(
        observed_array_payload_sha256
        == expected_array_payload_sha256,
        (
            f"{label} canonical array payload SHA-256 mismatch. "
            f"expected={expected_array_payload_sha256}; "
            f"observed={observed_array_payload_sha256}"
        ),
    )

    expected_array_count = int(record["array_count"])

    require(
        len(loaded_arrays) == expected_array_count,
        (
            f"{label} array-count mismatch. "
            f"expected={expected_array_count}; "
            f"observed={len(loaded_arrays)}"
        ),
    )

    return {
        "label": label,
        "artifact_type": str(record["artifact_type"]),
        "format": "npz",
        "path": str(npz_path),
        "array_count": expected_array_count,
        "file_sha256_verified": True,
        "payload_sha256_verified": True,
        "schema_sha256_verified": True,
        "contract_sha256_verified": True,
        "readback_verified": True,
    }


# ------------------------------------------------------------
# Reload and verify output manifest and handoff
# ------------------------------------------------------------

notebook_04_output_manifest_path = Path(
    notebook_04_output_manifest_path
)

notebook_04_to_notebook_05_handoff_path = Path(
    notebook_04_to_notebook_05_handoff_path
)

notebook_04_output_manifest_final_readback = _n04_read_json_contract(
    notebook_04_output_manifest_path
)

observed_manifest_payload_sha256_final = (
    _n04_verify_embedded_payload_hash(
        payload=notebook_04_output_manifest_final_readback,
        hash_field="manifest_payload_sha256",
        label="Notebook 04 output manifest",
    )
)

require(
    observed_manifest_payload_sha256_final
    == notebook_04_output_manifest_payload_sha256,
    (
        "Notebook 04 output manifest payload hash does not match "
        "the in-memory persistence result."
    ),
)

notebook_04_to_notebook_05_handoff_final_readback = (
    _n04_read_json_contract(
        notebook_04_to_notebook_05_handoff_path
    )
)

observed_handoff_payload_sha256_final = (
    _n04_verify_embedded_payload_hash(
        payload=notebook_04_to_notebook_05_handoff_final_readback,
        hash_field="handoff_payload_sha256",
        label="Notebook 04 to Notebook 05 handoff",
    )
)

require(
    observed_handoff_payload_sha256_final
    == notebook_04_to_notebook_05_handoff_payload_sha256,
    (
        "Notebook 04 to Notebook 05 handoff payload hash does not "
        "match the in-memory persistence result."
    ),
)


# ------------------------------------------------------------
# Re-read and verify every persisted artifact
# ------------------------------------------------------------

final_table_readback_records = []

for record in notebook_04_table_artifact_audit.to_dict(
    orient="records"
):
    final_table_readback_records.append(
        _n04_verify_table_record_readback(record)
    )

final_array_readback_records = []

for record in notebook_04_array_artifact_audit.to_dict(
    orient="records"
):
    final_array_readback_records.append(
        _n04_verify_array_record_readback(record)
    )

notebook_04_final_table_readback_audit = pd.DataFrame.from_records(
    final_table_readback_records
).sort_values(
    ["artifact_type", "label"],
    kind="stable",
).reset_index(drop=True)

notebook_04_final_array_readback_audit = pd.DataFrame.from_records(
    final_array_readback_records
).sort_values(
    ["artifact_type", "label"],
    kind="stable",
).reset_index(drop=True)


# ------------------------------------------------------------
# Final semantic checks
# ------------------------------------------------------------

final_primary_event_rows = int(
    len(primary_estimation_events_windowed)
)

final_primary_batch_rows = int(
    len(primary_exact_time_batches_windowed)
)

final_trade_print_rows = int(
    primary_estimation_events_windowed[
        "event_print_count"
    ].sum()
)

final_primary_trade_membership_rows = int(
    len(primary_trade_membership)
)

final_event_to_batch_membership_rows = int(
    len(primary_event_to_batch_membership_enriched)
)

final_simultaneous_batch_count = int(
    primary_exact_time_batches_windowed[
        "simultaneous_batch_required_flag"
    ].sum()
)

final_mixed_side_batch_count = int(
    primary_exact_time_batches_windowed[
        "mixed_side_batch_flag"
    ].sum()
)

final_aggregate_mark_after_event_count = int(
    primary_estimation_events_windowed[
        "aggregate_mark_lag_after_event_ns"
    ].gt(0).sum()
)

final_partition_count = int(
    observation_window_contract["event_partition"].nunique()
)

final_partition_array_metadata_rows = int(
    len(partition_array_metadata)
)

final_blocking_failures_from_persistence = int(
    notebook_04_persistence_gate_frame.loc[
        notebook_04_persistence_gate_frame["severity"].eq("BLOCKING")
        & notebook_04_persistence_gate_frame["status"].ne("PASS")
    ].shape[0]
)

final_warning_findings_from_persistence = int(
    notebook_04_persistence_gate_frame.loc[
        notebook_04_persistence_gate_frame["severity"].eq("WARNING")
        & notebook_04_persistence_gate_frame["status"].ne("PASS")
    ].shape[0]
)


# ------------------------------------------------------------
# Terminal gates
# ------------------------------------------------------------

notebook_04_final_acceptance_gates = [
    make_gate(
        gate="output_manifest_payload_hash_verified",
        passed=(
            observed_manifest_payload_sha256_final
            == notebook_04_output_manifest_payload_sha256
        ),
        severity="BLOCKING",
        detail=(
            "payload_sha256="
            f"{observed_manifest_payload_sha256_final}"
        ),
    ),
    make_gate(
        gate="notebook_04_to_05_handoff_payload_hash_verified",
        passed=(
            observed_handoff_payload_sha256_final
            == notebook_04_to_notebook_05_handoff_payload_sha256
        ),
        severity="BLOCKING",
        detail=(
            "payload_sha256="
            f"{observed_handoff_payload_sha256_final}"
        ),
    ),
    make_gate(
        gate="all_table_artifacts_final_readback_verified",
        passed=bool(
            notebook_04_final_table_readback_audit[
                "readback_verified"
            ].all()
        ),
        severity="BLOCKING",
        detail=(
            "table_artifact_count="
            f"{len(notebook_04_final_table_readback_audit)}"
        ),
    ),
    make_gate(
        gate="all_array_artifacts_final_readback_verified",
        passed=bool(
            notebook_04_final_array_readback_audit[
                "readback_verified"
            ].all()
        ),
        severity="BLOCKING",
        detail=(
            "array_artifact_count="
            f"{len(notebook_04_final_array_readback_audit)}"
        ),
    ),
    make_gate(
        gate="primary_event_rows_match_handoff",
        passed=(
            final_primary_event_rows
            == int(
                notebook_04_to_notebook_05_handoff_final_readback[
                    "primary_event_rows"
                ]
            )
        ),
        severity="BLOCKING",
        detail=(
            "primary_event_rows="
            f"{final_primary_event_rows:,}"
        ),
    ),
    make_gate(
        gate="primary_batch_rows_match_handoff",
        passed=(
            final_primary_batch_rows
            == int(
                notebook_04_to_notebook_05_handoff_final_readback[
                    "primary_batch_rows"
                ]
            )
        ),
        severity="BLOCKING",
        detail=(
            "primary_batch_rows="
            f"{final_primary_batch_rows:,}"
        ),
    ),
    make_gate(
        gate="primary_trade_prints_conserve_matched_trades",
        passed=(
            final_trade_print_rows
            == int(EXPECTED_MATCHED_TRADE_ROWS)
        ),
        severity="BLOCKING",
        detail=(
            "primary_print_sum="
            f"{final_trade_print_rows:,}; "
            "expected_trade_rows="
            f"{int(EXPECTED_MATCHED_TRADE_ROWS):,}"
        ),
    ),
    make_gate(
        gate="primary_trade_membership_covers_every_trade_once",
        passed=(
            final_primary_trade_membership_rows
            == int(EXPECTED_MATCHED_TRADE_ROWS)
        ),
        severity="BLOCKING",
        detail=(
            "primary_trade_membership_rows="
            f"{final_primary_trade_membership_rows:,}"
        ),
    ),
    make_gate(
        gate="primary_event_to_batch_membership_covers_every_event_once",
        passed=(
            final_event_to_batch_membership_rows
            == final_primary_event_rows
        ),
        severity="BLOCKING",
        detail=(
            "event_to_batch_membership_rows="
            f"{final_event_to_batch_membership_rows:,}; "
            "primary_event_rows="
            f"{final_primary_event_rows:,}"
        ),
    ),
    make_gate(
        gate="partition_metadata_covers_all_observation_windows",
        passed=(
            final_partition_array_metadata_rows
            == final_partition_count
        ),
        severity="BLOCKING",
        detail=(
            "partition_metadata_rows="
            f"{final_partition_array_metadata_rows}; "
            "partition_count="
            f"{final_partition_count}"
        ),
    ),
    make_gate(
        gate="no_blocking_persistence_gates_failed",
        passed=(final_blocking_failures_from_persistence == 0),
        severity="BLOCKING",
        detail=(
            "blocking_failures="
            f"{final_blocking_failures_from_persistence}"
        ),
    ),
    make_gate(
        gate="notebook_05_authorized",
        passed=bool(
            notebook_04_to_notebook_05_handoff_final_readback[
                "feature_construction_authorized"
            ]
        ),
        severity="BLOCKING",
        detail=(
            "Notebook 05 is authorized to consume Notebook 04 "
            "event-stream artifacts."
        ),
    ),
    make_gate(
        gate="hawkes_estimation_not_authorized_by_notebook_04",
        passed=(
            notebook_04_to_notebook_05_handoff_final_readback[
                "hawkes_estimation_authorized"
            ]
            is False
        ),
        severity="BLOCKING",
        detail="Notebook 07 remains unauthorized until intervening notebooks pass.",
    ),
    make_gate(
        gate="market_making_not_authorized_by_notebook_04",
        passed=(
            notebook_04_to_notebook_05_handoff_final_readback[
                "market_making_authorized"
            ]
            is False
        ),
        severity="BLOCKING",
        detail="No strategy, quoting, fill, or P&L authority is created here.",
    ),
    make_gate(
        gate="simultaneous_batches_absent",
        passed=(final_simultaneous_batch_count == 0),
        severity="WARNING",
        detail=(
            "simultaneous_batch_count="
            f"{final_simultaneous_batch_count}; "
            "Notebook 05+ must preserve batch-aware semantics."
        ),
    ),
    make_gate(
        gate="mixed_side_batches_absent",
        passed=(final_mixed_side_batch_count == 0),
        severity="WARNING",
        detail=(
            "mixed_side_batch_count="
            f"{final_mixed_side_batch_count}"
        ),
    ),
    make_gate(
        gate="aggregate_marks_available_at_event_time",
        passed=(final_aggregate_mark_after_event_count == 0),
        severity="WARNING",
        detail=(
            "aggregate_mark_after_event_count="
            f"{final_aggregate_mark_after_event_count}; "
            "full burst aggregate marks are not initiation-time features."
        ),
    ),
    make_gate(
        gate="persistence_warning_findings_absent",
        passed=(final_warning_findings_from_persistence == 0),
        severity="WARNING",
        detail=(
            "persistence_warning_findings="
            f"{final_warning_findings_from_persistence}"
        ),
    ),
]

notebook_04_final_acceptance_gate_frame = gate_results_to_frame(
    notebook_04_final_acceptance_gates
)

fail_if_blocking_gate_failed(
    notebook_04_final_acceptance_gate_frame
)


# ------------------------------------------------------------
# Final terminal-status payload
# ------------------------------------------------------------

notebook_04_terminal_status = "PASS_WITH_EVENT_STREAM_WARNINGS"

notebook_04_final_acceptance_summary = pd.DataFrame(
    [
        {
            "notebook_name": N04_NOTEBOOK_NAME,
            "terminal_status": notebook_04_terminal_status,
            "operating_mode": N04_OUTPUT_MANIFEST.get(
                "operating_mode",
                "ENGINEERING_REPRODUCTION_MODE",
            )
            if "N04_OUTPUT_MANIFEST" in globals()
            else "ENGINEERING_REPRODUCTION_MODE",
            "primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
            "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
            "primary_event_rows": final_primary_event_rows,
            "primary_batch_rows": final_primary_batch_rows,
            "trade_print_rows_conserved": final_trade_print_rows,
            "expected_trade_rows": int(EXPECTED_MATCHED_TRADE_ROWS),
            "table_artifact_count": int(
                len(notebook_04_final_table_readback_audit)
            ),
            "array_artifact_count": int(
                len(notebook_04_final_array_readback_audit)
            ),
            "simultaneous_batch_count": final_simultaneous_batch_count,
            "mixed_side_batch_count": final_mixed_side_batch_count,
            "aggregate_mark_after_event_count": (
                final_aggregate_mark_after_event_count
            ),
            "notebook_05_authorized": True,
            "hawkes_estimation_authorized": False,
            "market_making_authorized": False,
            "output_manifest_payload_sha256": (
                notebook_04_output_manifest_payload_sha256
            ),
            "handoff_payload_sha256": (
                notebook_04_to_notebook_05_handoff_payload_sha256
            ),
        }
    ]
)

notebook_04_final_acceptance_report_payload = {
    "artifact_type": "NOTEBOOK_04_FINAL_ACCEPTANCE_REPORT",
    "artifact_schema_version": "1.0",
    "acceptance_status": notebook_04_terminal_status,
    "project_name": N04_PROJECT_NAME,
    "pipeline_version": N04_PIPELINE_VERSION,
    "notebook_name": N04_NOTEBOOK_NAME,
    "source_run_prefix": N04_SOURCE_RUN_PREFIX,
    "v0_1_run_id": N04_V01_RUN_ID,
    "combined_prefix": N04_COMBINED_PREFIX,
    "primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
    "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
    "requires_simultaneous_event_batch_interface": bool(
        PRIMARY_EVENT_REQUIRES_SIMULTANEOUS_BATCH_INTERFACE
    ),
    "primary_event_rows": final_primary_event_rows,
    "primary_batch_rows": final_primary_batch_rows,
    "trade_print_rows_conserved": final_trade_print_rows,
    "expected_trade_rows": int(EXPECTED_MATCHED_TRADE_ROWS),
    "table_artifact_count": int(
        len(notebook_04_final_table_readback_audit)
    ),
    "array_artifact_count": int(
        len(notebook_04_final_array_readback_audit)
    ),
    "blocking_gate_failures": 0,
    "warning_findings": {
        "simultaneous_batch_count": final_simultaneous_batch_count,
        "mixed_side_batch_count": final_mixed_side_batch_count,
        "aggregate_mark_after_event_count": (
            final_aggregate_mark_after_event_count
        ),
        "persistence_warning_findings": (
            final_warning_findings_from_persistence
        ),
    },
    "authorized_next_notebook": "05_MARKET_STATE_FEATURES",
    "notebook_05_authorized": True,
    "hawkes_estimation_authorized": False,
    "market_making_authorized": False,
    "output_manifest_path": str(notebook_04_output_manifest_path),
    "output_manifest_payload_sha256": (
        notebook_04_output_manifest_payload_sha256
    ),
    "handoff_path": str(notebook_04_to_notebook_05_handoff_path),
    "handoff_payload_sha256": (
        notebook_04_to_notebook_05_handoff_payload_sha256
    ),
    "created_at_utc": _n04_utc_now_iso(),
}

notebook_04_final_acceptance_report_payload_sha256 = (
    _n04_canonical_json_sha256(
        notebook_04_final_acceptance_report_payload
    )
)

notebook_04_final_acceptance_report = dict(
    notebook_04_final_acceptance_report_payload
)

notebook_04_final_acceptance_report[
    "final_acceptance_report_payload_sha256"
] = notebook_04_final_acceptance_report_payload_sha256

notebook_04_final_acceptance_report_path = (
    N04_MANIFEST_DIR
    / (
        f"{N04_COMBINED_PREFIX}__"
        f"{N04_NOTEBOOK_NAME}__"
        "notebook_04_final_acceptance_report.json"
    )
)

_n04_atomic_write_json(
    notebook_04_final_acceptance_report_path,
    notebook_04_final_acceptance_report,
)

notebook_04_final_acceptance_report_readback = (
    _n04_read_json_contract(
        notebook_04_final_acceptance_report_path
    )
)

observed_final_acceptance_report_payload_sha256 = (
    _n04_verify_embedded_payload_hash(
        payload=notebook_04_final_acceptance_report_readback,
        hash_field="final_acceptance_report_payload_sha256",
        label="Notebook 04 final acceptance report",
    )
)

require(
    observed_final_acceptance_report_payload_sha256
    == notebook_04_final_acceptance_report_payload_sha256,
    "Notebook 04 final acceptance report read-back hash mismatch.",
)


# ------------------------------------------------------------
# Persist final audit tables
# ------------------------------------------------------------

final_audit_artifact_records = [
    _n04_write_parquet_table_artifact(
        label="notebook_04_final_acceptance_summary",
        frame=notebook_04_final_acceptance_summary,
        output_dir=N04_AUDIT_DIR,
        artifact_type="NOTEBOOK_04_FINAL_ACCEPTANCE_SUMMARY",
    ),
    _n04_write_parquet_table_artifact(
        label="notebook_04_final_acceptance_gate_frame",
        frame=notebook_04_final_acceptance_gate_frame,
        output_dir=N04_AUDIT_DIR,
        artifact_type="NOTEBOOK_04_FINAL_ACCEPTANCE_GATE_FRAME",
    ),
    _n04_write_parquet_table_artifact(
        label="notebook_04_final_table_readback_audit",
        frame=notebook_04_final_table_readback_audit,
        output_dir=N04_AUDIT_DIR,
        artifact_type="NOTEBOOK_04_FINAL_TABLE_READBACK_AUDIT",
    ),
    _n04_write_parquet_table_artifact(
        label="notebook_04_final_array_readback_audit",
        frame=notebook_04_final_array_readback_audit,
        output_dir=N04_AUDIT_DIR,
        artifact_type="NOTEBOOK_04_FINAL_ARRAY_READBACK_AUDIT",
    ),
]

notebook_04_final_audit_artifact_frame = pd.DataFrame.from_records(
    final_audit_artifact_records
).sort_values(
    ["artifact_type", "label"],
    kind="stable",
).reset_index(drop=True)

require(
    bool(notebook_04_final_audit_artifact_frame["readback_verified"].all()),
    "At least one final audit artifact failed read-back verification.",
)


# ------------------------------------------------------------
# Final display
# ------------------------------------------------------------

display(notebook_04_final_acceptance_summary)
display(notebook_04_final_acceptance_gate_frame)
display(notebook_04_final_audit_artifact_frame)

{
    "status": notebook_04_terminal_status,
    "notebook_04_complete": True,
    "primary_event_representation": PRIMARY_EVENT_REPRESENTATION,
    "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
    "primary_event_rows": final_primary_event_rows,
    "primary_batch_rows": final_primary_batch_rows,
    "trade_print_rows_conserved": final_trade_print_rows,
    "expected_trade_rows": int(EXPECTED_MATCHED_TRADE_ROWS),
    "table_artifact_count": int(
        len(notebook_04_final_table_readback_audit)
    ),
    "array_artifact_count": int(
        len(notebook_04_final_array_readback_audit)
    ),
    "simultaneous_batch_count": final_simultaneous_batch_count,
    "mixed_side_batch_count": final_mixed_side_batch_count,
    "aggregate_mark_after_event_count": (
        final_aggregate_mark_after_event_count
    ),
    "final_acceptance_report_path": str(
        notebook_04_final_acceptance_report_path
    ),
    "final_acceptance_report_payload_sha256": (
        notebook_04_final_acceptance_report_payload_sha256
    ),
    "notebook_04_to_notebook_05_handoff_path": str(
        notebook_04_to_notebook_05_handoff_path
    ),
    "notebook_04_to_notebook_05_handoff_payload_sha256": (
        notebook_04_to_notebook_05_handoff_payload_sha256
    ),
    "notebook_05_authorized": True,
    "hawkes_estimation_authorized": False,
    "market_making_authorized": False,
    "next_action": (
        "Start 05_MARKET_STATE_FEATURES from the verified "
        "Notebook 04 to Notebook 05 handoff."
    ),
}

,notebook_name,terminal_status,operating_mode,primary_event_representation,timestamp_interface,primary_event_rows,primary_batch_rows,trade_print_rows_conserved,expected_trade_rows,table_artifact_count,array_artifact_count,simultaneous_batch_count,mixed_side_batch_count,aggregate_mark_after_event_count,notebook_05_authorized,hawkes_estimation_authorized,market_making_authorized,output_manifest_payload_sha256,handoff_payload_sha256
0,04_EVENT_STREAM_CONSTRUCTION,PASS_WITH_EVENT_STREAM_WARNINGS,ENGINEERING_REPRODUCTION_MODE,SAME_MS_SAME_SIDE_BURSTS,SIMULTANEOUS_EVENT_BATCH_REQUIRED,13887,13564,67683,67683,21,2,249,41,1575,True,False,False,c8454d2059f1b5e50675aa156e1eabd7762642cbf624065b496bea1edb56ff53,f0544516d59fbabc24d1c8ff42063c96e7c6c4b72fcd5cedbd8ef0a2dfc38a28


,gate,status,severity,detail
0,output_manifest_payload_hash_verified,PASS,BLOCKING,payload_sha256=c8454d2059f1b5e50675aa156e1eabd7762642cbf624065b496bea1edb56ff53
1,notebook_04_to_05_handoff_payload_hash_verified,PASS,BLOCKING,payload_sha256=f0544516d59fbabc24d1c8ff42063c96e7c6c4b72fcd5cedbd8ef0a2dfc38a28
2,all_table_artifacts_final_readback_verified,PASS,BLOCKING,table_artifact_count=21
3,all_array_artifacts_final_readback_verified,PASS,BLOCKING,array_artifact_count=2
4,primary_event_rows_match_handoff,PASS,BLOCKING,"primary_event_rows=13,887"
5,primary_batch_rows_match_handoff,PASS,BLOCKING,"primary_batch_rows=13,564"
6,primary_trade_prints_conserve_matched_trades,PASS,BLOCKING,"primary_print_sum=67,683; expected_trade_rows=67,683"
7,primary_trade_membership_covers_every_trade_once,PASS,BLOCKING,"primary_trade_membership_rows=67,683"
8,primary_event_to_batch_membership_covers_every_event_once,PASS,BLOCKING,"event_to_batch_membership_rows=13,887; primary_event_rows=13,887"
9,partition_metadata_covers_all_observation_windows,PASS,BLOCKING,partition_metadata_rows=4; partition_count=4


,label,artifact_type,format,path,contract_path,row_count,column_count,table_payload_sha256,file_sha256,contract_payload_sha256,readback_verified
0,notebook_04_final_acceptance_gate_frame,NOTEBOOK_04_FINAL_ACCEPTANCE_GATE_FRAME,parquet,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__notebook_04_final_accepta...,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__notebook_04_final_accepta...,18,4,fabf2688bc8dab72b3374e9525c4e67c04fcc860d2c863b5823bb525d4dd0e3a,8fcc6973d44b691626eb2170169ff82e4c5ccccd113dd79eb7a19ddc75d7074d,a5674da5a0497750c245f55c5163136fb2d03d0044259d3e6947d2e8f20c0805,True
1,notebook_04_final_acceptance_summary,NOTEBOOK_04_FINAL_ACCEPTANCE_SUMMARY,parquet,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__notebook_04_final_accepta...,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__notebook_04_final_accepta...,1,19,ea06dd91f54773ce7318d3be82e05e4a5e094d03e64f29ce4edab2a7592f45d6,3e033a0c839f6cbbb429016f08bb5ae53138af41eb6a459921afa754546e8993,6de89c090bdf7333c8f13f8be00bc78004f825f1ba1cf5c275b8a913604d79ce,True
2,notebook_04_final_array_readback_audit,NOTEBOOK_04_FINAL_ARRAY_READBACK_AUDIT,parquet,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__notebook_04_final_array_r...,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__notebook_04_final_array_r...,2,10,a03a028772a764f3f35d0737e16e7e51719877202caf2603bacbc271161456d6,90ac1c71e947386ab4af752de9655658597c63e71fd14461f6973f2fe3b51503,3360af41e2f3068f4cb6de04f08fea4deb4be747bd8fabfe82aedd56ae4d07c1,True
3,notebook_04_final_table_readback_audit,NOTEBOOK_04_FINAL_TABLE_READBACK_AUDIT,parquet,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__notebook_04_final_table_r...,D:\Clown Project\V0.1\artifacts\audit_tables\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__notebook_04_final_table_r...,21,10,7a665719f19f976ae6f7b25069d76538670193e54013f9f47538fd4870b0bc0a,1dfcd0e61dec89d88d7a1127213d4a3940ef10c5fd6c39487df959f5d5603aaa,e55cefdd04d2cd4cd9980f54eae346ca831947b53f398903a4db9311ec9f6daa,True


{'status': 'PASS_WITH_EVENT_STREAM_WARNINGS',
 'notebook_04_complete': True,
 'primary_event_representation': 'SAME_MS_SAME_SIDE_BURSTS',
 'timestamp_interface': 'SIMULTANEOUS_EVENT_BATCH_REQUIRED',
 'primary_event_rows': 13887,
 'primary_batch_rows': 13564,
 'trade_print_rows_conserved': 67683,
 'expected_trade_rows': 67683,
 'table_artifact_count': 21,
 'array_artifact_count': 2,
 'simultaneous_batch_count': 249,
 'mixed_side_batch_count': 41,
 'aggregate_mark_after_event_count': 1575,
 'final_acceptance_report_path': 'D:\\Clown Project\\V0.1\\artifacts\\manifests\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__04_EVENT_STREAM_CONSTRUCTION__notebook_04_final_acceptance_report.json',
 'final_acceptance_report_payload_sha256': 'd056dab3330dff3be55ccc4d5a7aa3acf6ba7569712bd3588f3b088d2a1ef8cd',
 'notebook_04_to_notebook_05_handoff_path': 'D:\\Clown Project\\V0.1\\artifacts\\handoff\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__